# ARC-AGI-3 — rulebook agent submission (SungsuHyun)

Our solver: an explicit rulebook of hypotheses (environment / rules / win conditions) written by Qwen3.6-27B-FP8 (vLLM in the notebook) at level start; a deterministic predictor (entities and regions, click statistics by colour and region, button slot permutations, marker coupling) says what every candidate action will do; the model only chooses; every action is verified and mismatches trigger a review. Win predicates (gone / inside / equal / pose / pressed) are learned from completed levels and program-made plans (match / press BFS / marker) are offered as candidates. Sources are embedded from `rulebook/` and `arcnav/` by `scripts/build_rulebook_notebook.py`; do not edit cells here.

Commit = 2-game offline smoke (12 min); the competition rerun plays every gateway game.

In [ ]:
import json, os, subprocess, sys, time
T0 = time.time()
RERUN = bool(os.getenv('KAGGLE_IS_COMPETITION_RERUN'))
WORK = '/kaggle/working'
print('competition rerun:', RERUN)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-index', '--find-links', '/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels', 'arc-agi'], check=True)
SOURCES = json.loads("{\"arcnav\": {\"__init__.py\": \"\\\"\\\"\\\"arcnav \\u2014 our ARC-AGI-3 solver: a tool-using LLM harness built around\\nnavigation helpers, solver synthesis and programmatic safety policies.\\n\\nModules: frame (observation encoding), nav (navigation helper), sandbox\\n(python tool), llm (OpenAI-compatible tool-calling client), prompts, solver\\n(persistent solver policies), agent (per-game loop), runner (local/Kaggle play).\\n\\\"\\\"\\\"\\n__version__ = \\\"0.1.0\\\"\\n\", \"advisor.py\": \"\\\"\\\"\\\"Layer 2: the model as a macro planner, not a per-step actor.\\n\\nThe explorer (layer 0) supplies structured evidence: which macro actions exist, what each kind of touch did on earlier\\nlevels (effect table), how the earlier levels were won, and the current board. The model answers with a short ordered\\nlist of macro labels (JSON). The harness executes them, reports what changed, and asks again \\u2014 a handful of rounds per\\nlevel, tens of actions per round, no free-form per-step reasoning.\\n\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport json\\nimport re\\nfrom collections import Counter\\nfrom typing import Optional\\n\\nfrom .frame import summarize_diff\\n\\nSYSTEM = \\\"\\\"\\\"You plan moves for a grid puzzle game played by a program. You never act step by step: you return an ordered list of\\nmacro actions (by their exact labels) that the program executes for you, then you see what changed and plan again.\\nAnswer with JSON only: {\\\"hypothesis\\\": \\\"<one sentence: what the level requires>\\\", \\\"plan\\\": [\\\"<label>\\\", \\\"<label>\\\", ...]}.\\nUse only labels from the AVAILABLE MACROS list, at most 12 per answer. Prefer macros whose kind/colour did something on earlier\\nlevels. If a plan made no progress, try a genuinely different hypothesis instead of repeating it.\\\"\\\"\\\"\\n\\n\\ndef _board_ascii(frame) -> str:\\n    return frame.ascii\\n\\n\\ndef _objects_by_colour(frame, limit: int = 12) -> str:\\n    rows = []\\n    by: dict[int, list] = {}\\n    for n in frame.segmentation[\\\"nodes\\\"]:\\n        if n[\\\"hud\\\"]:\\n            continue\\n        by.setdefault(n[\\\"color\\\"], []).append(n)\\n    for c, ns in sorted(by.items(), key=lambda kv: -len(kv[1])):\\n        ns.sort(key=lambda n: n[\\\"pixels\\\"])\\n        pos = \\\", \\\".join(f\\\"({n['center'][0]},{n['center'][1]}){'' if n['pixels'] == ns[0]['pixels'] else f'[{n['pixels']}px]'}\\\" for n in ns[:limit])\\n        rows.append(f\\\"  colour {c}: {len(ns)} object(s), sizes {sorted(set(n['pixels'] for n in ns))[:6]}, at {pos}{' ...' if len(ns) > limit else ''}\\\")\\n    return \\\"\\\\n\\\".join(rows)\\n\\n\\ndef build_brief(explorer, macros: list) -> str:\\n    s = explorer.s\\n    parts = [f\\\"GAME: level {s.level} of {s.levels_total}; valid actions {s.valid_actions}; actions used so far {s.actions_used}.\\\"]\\n    if explorer.solutions:\\n        for lvl, path in sorted(explorer.solutions.items()):\\n            parts.append(f\\\"LEVEL {lvl} WAS WON by this macro sequence: {path[-16:]}\\\")\\n    if s.level_recaps:\\n        parts.append(\\\"HARNESS RECAP OF THE LAST WIN: \\\" + s.level_recaps[-1][:700])\\n    if s.goal_hypotheses:\\n        parts.append(\\\"GOAL HYPOTHESES FROM EARLIER LEVELS: \\\" + \\\" | \\\".join(h[\\\"text\\\"][:160] for h in s.goal_hypotheses[:3]))\\n    if explorer.effect_stats:\\n        eff = sorted(((k, v) for k, v in explorer.effect_stats.items() if v[0]), key=lambda kv: -kv[1][1] / kv[1][0])\\n        parts.append(\\\"EFFECT TABLE (what touching each colour did so far; 1.0 = always changed the board, 0 = never): \\\" +\\n                     \\\", \\\".join(f\\\"{k[0]} colour {k[1]}: {v[1] / v[0]:.2f} over {v[0] // 2} tries\\\" for k, v in eff[:14]))\\n    if getattr(s, \\\"rules_text\\\", \\\"\\\"):\\n        parts.append(\\\"INDUCED RULES: \\\" + s.rules_text[:600])\\n    parts.append(\\\"CURRENT BOARD OBJECTS (row,col centres):\\\\n\\\" + _objects_by_colour(s.frame))\\n    parts.append(\\\"CURRENT BOARD (64x64, one hex digit per cell, HUD included):\\\\n\\\" + _board_ascii(s.frame))\\n    parts.append(\\\"AVAILABLE MACROS (use these exact labels):\\\\n  \\\" + \\\"\\\\n  \\\".join(m.label for m in macros[:60]))\\n    return \\\"\\\\n\\\\n\\\".join(parts)\\n\\n\\ndef parse_plan(text: str) -> tuple[str, list[str]]:\\n    text = text.strip()\\n    m = re.search(r\\\"\\\\{.*\\\\}\\\", text, re.S)\\n    if not m:\\n        return \\\"\\\", []\\n    try:\\n        obj = json.loads(m.group(0))\\n    except Exception:\\n        # tolerate trailing commas / single quotes\\n        try:\\n            obj = json.loads(re.sub(r\\\",\\\\s*([\\\\]}])\\\", r\\\"\\\\1\\\", m.group(0)).replace(\\\"'\\\", '\\\"'))\\n        except Exception:\\n            return \\\"\\\", []\\n    plan = obj.get(\\\"plan\\\") or []\\n    return str(obj.get(\\\"hypothesis\\\", \\\"\\\"))[:300], [str(p) for p in plan][:12]\\n\\n\\ndef advise_level(explorer, client, *, rounds: int = 6, max_actions: int = 150, log=print) -> dict:\\n    \\\"\\\"\\\"Run up to `rounds` plan\\u2192execute\\u2192report cycles on the current level. Returns a summary dict.\\\"\\\"\\\"\\n    s = explorer.s\\n    level0 = s.level; a0 = s.actions_used\\n    messages = [{\\\"role\\\": \\\"system\\\", \\\"content\\\": SYSTEM}]\\n    history: list[str] = []\\n    for rnd in range(rounds):\\n        if s.level != level0 or s.state == \\\"WIN\\\" or s.actions_used - a0 >= max_actions:\\n            break\\n        node = explorer.node()\\n        macros = explorer.macros(node)\\n        by_label = {m.label: m for m in macros}\\n        brief = build_brief(explorer, macros)\\n        if history:\\n            brief = \\\"WHAT YOUR EARLIER PLANS DID ON THIS LEVEL:\\\\n\\\" + \\\"\\\\n\\\".join(history[-6:]) + \\\"\\\\n\\\\n\\\" + brief\\n        messages = [messages[0], {\\\"role\\\": \\\"user\\\", \\\"content\\\": brief}]   # one fresh brief per round: the state, not the chat, is the memory\\n        try:\\n            r = client.chat(messages, tools=None, tool_choice=\\\"none\\\")\\n        except Exception as e:\\n            log(f\\\"[advisor] model call failed: {e!r}\\\"); break\\n        hyp, plan = parse_plan(r.message.get(\\\"content\\\", \\\"\\\"))\\n        log(f\\\"[advisor] round {rnd + 1}: hypothesis={hyp!r} plan={plan}\\\")\\n        explorer.trace.append(f\\\"advisor r{rnd + 1}: {hyp[:80]} -> {plan}\\\")\\n        if not plan:\\n            history.append(f\\\"round {rnd + 1}: no valid plan returned\\\"); continue\\n        outcomes = []\\n        for label in plan:\\n            m = by_label.get(label)\\n            if m is None:\\n                # labels go stale as the board changes: re-resolve by kind+colour when possible\\n                macros = explorer.macros(explorer.node()); by_label = {x.label: x for x in macros}\\n                m = by_label.get(label)\\n                if m is None:\\n                    col = explorer._macro_color_label(label) if hasattr(explorer, \\\"_macro_color_label\\\") else None\\n                    cands = [x for x in macros if x.kind == label.split(\\\"(\\\")[0] and (col is None or explorer._macro_color(x) == col)]\\n                    m = cands[0] if cands else None\\n            if m is None:\\n                outcomes.append(f\\\"{label}: not available now\\\"); continue\\n            before = s.frame\\n            res = explorer._run_macro(explorer.node(), m)\\n            d = summarize_diff(before, s.frame) if s.frame is not None else {}\\n            outcomes.append(f\\\"{label}: {d.get('changed_cells', 0)} cells changed\\\" +\\n                            (f\\\", vanished {[(x['color'], x['pixels']) for x in d.get('disappeared', [])][:3]}\\\" if d.get(\\\"disappeared\\\") else \\\"\\\") +\\n                            (f\\\", appeared {[(x['color'], x['pixels']) for x in d.get('appeared', [])][:3]}\\\" if d.get(\\\"appeared\\\") else \\\"\\\") +\\n                            (\\\" -> LEVEL COMPLETED\\\" if res[\\\"level_completed\\\"] else \\\" -> GAME OVER (level reset)\\\" if res[\\\"game_over\\\"] else \\\"\\\"))\\n            if res[\\\"level_completed\\\"] or res[\\\"game_over\\\"] or s.actions_used - a0 >= max_actions:\\n                break\\n        history.append(f\\\"round {rnd + 1} (hypothesis: {hyp[:120]}): \\\" + \\\"; \\\".join(outcomes))\\n        if s.level > level0:\\n            explorer.solutions[level0] = [f\\\"advisor:{hyp[:60]}\\\"] + plan\\n            return {\\\"completed\\\": True, \\\"rounds\\\": rnd + 1, \\\"actions\\\": s.actions_used - a0, \\\"hypothesis\\\": hyp}\\n    return {\\\"completed\\\": s.level > level0, \\\"rounds\\\": rounds, \\\"actions\\\": s.actions_used - a0}\\n\", \"agent.py\": \"\\\"\\\"\\\"Per-game loop: environment stepping, model turns, python tool, solver auto-run.\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport json\\nimport time\\nfrom pathlib import Path\\nfrom typing import Any, Optional\\n\\nfrom arcengine import GameAction, GameState\\n\\nfrom . import solver as solver_policy\\nfrom .frame import Frame, masked_ascii, summarize_diff, grid_to_png_b64\\nfrom .llm import ChatClient, ContextLengthError\\nfrom .nav import NavHelper\\nfrom . import rules as rule_induction\\nfrom .autopilot import run_two_body, run_reach\\nfrom . import goals as goal_inference\\nfrom .prompts import PROPOSE_TOOL, PYTHON_TOOL, system_prompt, turn_header\\nfrom .sandbox import Sandbox\\n\\nMODEL_TO_ENGINE = {\\\"UP\\\": \\\"ACTION1\\\", \\\"DOWN\\\": \\\"ACTION2\\\", \\\"LEFT\\\": \\\"ACTION3\\\", \\\"RIGHT\\\": \\\"ACTION4\\\", \\\"SPACE\\\": \\\"ACTION5\\\", \\\"MOUSE\\\": \\\"ACTION6\\\", \\\"ACTION7\\\": \\\"ACTION7\\\"}\\nENGINE_TO_MODEL = {v: k for k, v in MODEL_TO_ENGINE.items()}\\nMAX_ACTIONS_PER_CALL = 24   # blind 50-action batches walked straight into game over on s5i5\\nPROBE_SWEEP_AFTER_TURNS = 8   # iter4: after this many model turns on a level without completing it, the harness probes every untried action once\\nMAX_ACTIONS_PER_TOOL_RUN = 30   # also caps loops of single-action calls inside one python tool run\\nPROPOSAL_ATTEMPTS = 0   # >0: after a level completion, action() is blocked and propose_solver is forced via tool_choice for this many turns.\\n                        # Set E (forced, 3 turns): 0.75, level>=2 0/4 \\u2014 forced solvers burned 50\\u201380 actions per level; disabled by default.\\n\\n\\nclass _T:  # transition view for the host-side NavHelper\\n    def __init__(self, action, before_frame, after_frame):\\n        self.action, self.before_frame, self.after_frame = action, before_frame, after_frame\\n\\n\\nclass GameSession:\\n    def __init__(self, env, game_id: str, client: Optional[ChatClient], *, log_dir: Path, max_minutes: float = 20.0,\\n                 max_actions: int = 3000, max_model_turns: int = 400, keep_full_turns: int = 3, tool_timeout: int = 30,\\n                 context_tokens: int = 32768, verbose: bool = True, deadline: Optional[float] = None, think_first_turns: int = 0,\\n                 tool_choice_required: bool = False, oracle_rules: str = \\\"\\\", image_context: bool = False):\\n        self.env, self.game_id, self.client = env, game_id, client\\n        self.log_dir = Path(log_dir); self.log_dir.mkdir(parents=True, exist_ok=True)\\n        self.max_minutes, self.max_actions, self.max_model_turns = max_minutes, max_actions, max_model_turns\\n        self.keep_full_turns, self.tool_timeout, self.context_tokens, self.verbose = keep_full_turns, tool_timeout, context_tokens, verbose\\n        self.deadline = deadline   # absolute epoch seconds (global run cap), optional\\n        self.image_context = image_context   # attach the current board as an image to the latest user message (VLM)\\n        self.oracle_rules = oracle_rules   # D1 diagnostic: ground-truth rules injected into every turn (never in submissions)\\n        self.tool_choice_required = tool_choice_required   # force a tool call every turn (models with flaky tool formatting)\\n        self.think_first_turns = think_first_turns   # iter3: chain-of-thought ON for the first N model turns of every level\\n        self.level_turn_start = 0\\n        self.transitions: list[dict] = []          # payloads for the sandbox\\n        self.host_transitions: list[_T] = []       # Frame views for the host NavHelper\\n        self.attempt_start_index = 0               # index into host_transitions where the current level attempt began\\n        self.cycle_hits_this_level = 0\\n        self.frame: Optional[Frame] = None\\n        self.level, self.levels_total, self.state = 1, 0, \\\"NOT_PLAYED\\\"\\n        self.valid_actions: list[str] = []\\n        self.actions_used = self.level_actions = self.resets = self.step_no = 0\\n        self.level_action_log: list[int] = []\\n        self.solver: Optional[dict] = None\\n        self.model_turns = self.solver_turns = 0\\n        self.level_just_completed = False\\n        self.last_outcome: list[str] = []\\n        self.notes = \\\"\\\"\\n        self.checklist: dict = {\\\"goal\\\": \\\"\\\", \\\"roles\\\": {}, \\\"plan\\\": \\\"\\\", \\\"tried\\\": []}   # model-owned fields; harness adds facts\\n        self.checklist_level = 1\\n        self.rules_text = \\\"\\\"\\n        self.goal_hypotheses: list[dict] = []\\n        self.autopilot_tries: dict[int, int] = {}   # level -> attempts of the rule-based autopilot\\n        self._run_actions = 0\\n        self.game_overs_this_level = 0\\n        self.rejections_in_row = 0\\n        self.zero_action_turns = 0   # consecutive model turns that executed nothing\\n        self.probe_sweeps: dict[int, str] = {}   # iter4: level -> transition table from the harness probe sweep\\n        self.level_recaps: list[str] = []   # harness-written summaries of how each completed level was won\\n        self.recent_codes: list[str] = []   # repetition guard: identical tool code is not executed twice in a row\\n        self.proposal_required = 0   # >0: action() blocked until propose_solver is called (set after a level completion)\\n        self.messages: list[dict] = []\\n        self.t0 = time.time()\\n        self.transcript = open(self.log_dir / f\\\"{game_id}.log\\\", \\\"a\\\")\\n        self.events = open(self.log_dir / f\\\"{game_id}.jsonl\\\", \\\"a\\\")\\n        self.sandbox = Sandbox(action_handler=self._on_action, solver_handler=self._on_solver, state_provider=self._state, timeout=tool_timeout)\\n\\n    # ------------------------------------------------------------ environment\\n    def _apply(self, raw) -> None:\\n        self.state = str(raw.state).split(\\\".\\\")[-1]\\n        self.levels_total = int(raw.win_levels or 0)\\n        self.level = int(raw.levels_completed) + 1\\n        self.valid_actions = [ENGINE_TO_MODEL.get(f\\\"ACTION{a}\\\", f\\\"ACTION{a}\\\") for a in (raw.available_actions or [])]\\n        grids = [g.tolist() if hasattr(g, \\\"tolist\\\") else g for g in raw.frame]\\n        if grids:\\n            self.frame = Frame(grids[-1], step=self.step_no, level=self.level)\\n\\n    def reset(self) -> None:\\n        raw = self.env.step(GameAction.RESET)\\n        self.resets += 1; self._apply(raw)\\n        self.level_actions = 0\\n\\n    def _step_engine(self, act: dict):\\n        name = act[\\\"action\\\"]\\n        ga = GameAction.from_name(MODEL_TO_ENGINE[name]) if hasattr(GameAction, \\\"from_name\\\") else GameAction[MODEL_TO_ENGINE[name]]\\n        data = {\\\"x\\\": int(act[\\\"col\\\"]), \\\"y\\\": int(act[\\\"row\\\"])} if name == \\\"MOUSE\\\" else {}\\n        ga.set_data(data) if data else None\\n        return self.env.step(ga, data=ga.action_data.model_dump())\\n\\n    def execute(self, actions: list[dict]) -> dict:\\n        \\\"\\\"\\\"Run actions until one completes a level or ends the game. Records transitions.\\\"\\\"\\\"\\n        results, executed, changed_any, level_completed, game_over, stopped = [], 0, False, False, False, None\\n        if len(actions) > MAX_ACTIONS_PER_CALL:\\n            actions = actions[:MAX_ACTIONS_PER_CALL]; stopped = f\\\"only the first {MAX_ACTIONS_PER_CALL} actions of a call are executed; look at the board, then continue\\\"\\n        for act in actions:\\n            if act[\\\"action\\\"] not in self.valid_actions and self.valid_actions:\\n                stopped = f\\\"{act['action']} is not a valid action now ({self.valid_actions})\\\"; break\\n            if act[\\\"action\\\"] == \\\"MOUSE\\\" and (\\\"row\\\" not in act or \\\"col\\\" not in act):\\n                stopped = \\\"MOUSE needs row and col\\\"; break\\n            if act[\\\"action\\\"] == \\\"MOUSE\\\" and not (0 <= int(act[\\\"row\\\"]) <= 63 and 0 <= int(act[\\\"col\\\"]) <= 63):\\n                stopped = f\\\"MOUSE coordinates out of range (row={act['row']}, col={act['col']}; valid 0..63)\\\"; break\\n            if self.actions_used >= self.max_actions:\\n                stopped = \\\"action budget exhausted\\\"; break\\n            before = self.frame; prev_level = self.level\\n            raw = self._step_engine(act)\\n            self.step_no += 1; self.actions_used += 1; self.level_actions += 1; executed += 1\\n            self._apply(raw)\\n            after = self.frame\\n            label = act[\\\"action\\\"] if act[\\\"action\\\"] != \\\"MOUSE\\\" else {\\\"action\\\": \\\"MOUSE\\\", \\\"row\\\": act[\\\"row\\\"], \\\"col\\\": act[\\\"col\\\"]}\\n            self.transitions.append({\\\"action\\\": label, \\\"before\\\": before.to_payload(), \\\"after\\\": after.to_payload()})\\n            self.host_transitions.append(_T(label, before, after))   # dict label for MOUSE keeps row/col (rule induction needs them)\\n            ch = before.ascii != after.ascii; changed_any |= ch\\n            results.append({\\\"action\\\": label, \\\"changed\\\": ch})\\n            if self.level > prev_level or self.state == \\\"WIN\\\":\\n                level_completed = True; self.level_action_log.append(self.level_actions); self.level_actions = 0; self.game_overs_this_level = 0; self.cycle_hits_this_level = 0\\n                self.attempt_start_index = len(self.host_transitions)   # cycle guard compares states within the current attempt only\\n                self.level_turn_start = self.model_turns\\n                try:\\n                    self.level_recaps.append(self._level_recap(prev_level))\\n                except Exception as e:  # never break play on a recap error\\n                    self._log(f\\\"recap failed: {e!r}\\\")\\n                try:\\n                    cur = [t for t in self.host_transitions if t.before_frame.level == prev_level]\\n                    first_ascii = cur[0].before_frame.ascii if cur else None; start = 0\\n                    for i, t in enumerate(cur):   # last attempt = after the last reset on that level\\n                        if i > 0 and t.before_frame.ascii == first_ascii:\\n                            start = i\\n                    hyps = goal_inference.infer(cur[start:], prev_level, movement_keys=any(a in self.valid_actions for a in (\\\"UP\\\", \\\"DOWN\\\", \\\"LEFT\\\", \\\"RIGHT\\\")))\\n                    if hyps:\\n                        self.goal_hypotheses = hyps + [h for h in self.goal_hypotheses if h.get(\\\"level\\\") != prev_level]\\n                        self._log(\\\"goal hypotheses: \\\" + \\\" | \\\".join(h[\\\"text\\\"] for h in hyps[:3]))\\n                        self._event(kind=\\\"goals\\\", level=prev_level, hypotheses=hyps)\\n                except Exception as e:\\n                    self._log(f\\\"goal inference failed: {e!r}\\\")\\n                self._log(f\\\"*** level {prev_level} completed after {self.level_action_log[-1]} actions (total {self.actions_used})\\\")\\n                stopped = \\\"level completed\\\" if self.state != \\\"WIN\\\" else \\\"game won\\\"; break\\n            if self.state == \\\"GAME_OVER\\\":\\n                game_over = True; self.game_overs_this_level += 1; self._log(\\\"*** game over -> reset\\\"); self.reset(); stopped = \\\"game over (reset done, level restarted)\\\"\\n                self.attempt_start_index = len(self.host_transitions); self.cycle_hits_this_level = 0\\n                self.level_turn_start = self.model_turns   # re-open the thinking window: the level restarts, re-plan with reasoning\\n                break\\n        return {\\\"executed_count\\\": executed, \\\"board_changed\\\": changed_any, \\\"level_completed\\\": level_completed, \\\"game_over\\\": game_over,\\n                \\\"stopped_reason\\\": stopped, \\\"results\\\": results[-12:]}\\n\\n    # ------------------------------------------------------------ sandbox hooks\\n    def _state(self) -> dict:\\n        return {\\\"frame\\\": self.frame.to_payload() if self.frame else None, \\\"valid_actions\\\": self.valid_actions, \\\"level\\\": self.level,\\n                \\\"levels_total\\\": self.levels_total, \\\"transitions\\\": self.transitions[-400:], \\\"notes\\\": self.notes, \\\"level_recaps\\\": self.level_recaps,\\n                \\\"checklist\\\": self.checklist, \\\"rules_text\\\": self.rules_text, \\\"goal_hypotheses\\\": self.goal_hypotheses}\\n\\n    def _on_action(self, actions: list[dict]) -> tuple[dict, dict]:\\n        if self.proposal_required > 0:\\n            return {\\\"executed_count\\\": 0, \\\"board_changed\\\": False, \\\"level_completed\\\": False, \\\"game_over\\\": False,\\n                    \\\"stopped_reason\\\": \\\"blocked: a level was just completed, so call propose_solver(code) first (inspection is fine, actions are not)\\\",\\n                    \\\"results\\\": []}, self._state()\\n        left = MAX_ACTIONS_PER_TOOL_RUN - self._run_actions\\n        if left <= 0:\\n            return {\\\"executed_count\\\": 0, \\\"board_changed\\\": False, \\\"level_completed\\\": False, \\\"game_over\\\": False,\\n                    \\\"stopped_reason\\\": f\\\"this tool run already executed {MAX_ACTIONS_PER_TOOL_RUN} actions: finish the call, read the board and the outcome, then continue in the next turn\\\",\\n                    \\\"results\\\": []}, self._state()\\n        res = self.execute(actions[:left])\\n        self._run_actions += res[\\\"executed_count\\\"]\\n        if res[\\\"level_completed\\\"]:\\n            self.level_just_completed = True\\n            if self.solver and self.solver[\\\"status\\\"] == \\\"active\\\":\\n                self.solver[\\\"no_progress_actions\\\"] = 0\\n            elif self.state != \\\"WIN\\\":\\n                self.proposal_required = PROPOSAL_ATTEMPTS\\n        return res, self._state()\\n\\n    def _on_solver(self, code: str, report: dict) -> dict:\\n        if report.get(\\\"ok\\\"):\\n            self.solver = solver_policy.new_solver(code, report)\\n            self._log(f\\\"solver stored ({self.solver['status']}):\\\\n\\\" + code)\\n            note = (\\\"Solver stored and VERIFIED; the harness will run solve() automatically from the next turn.\\\" if self.solver[\\\"status\\\"] == \\\"active\\\" else\\n                    \\\"Solver stored as a DRAFT (no predict() or accuracy < 0.8): the harness will show its suggestion each turn but will not run it by itself.\\\")\\n            return {**report, \\\"stored\\\": True, \\\"note\\\": note}\\n        self.solver = {\\\"status\\\": \\\"rejected\\\", \\\"reason\\\": report.get(\\\"reason\\\"), \\\"report\\\": report, \\\"code\\\": code}\\n        self._log(f\\\"solver rejected: {report.get('reason')}\\\")\\n        return {**report, \\\"stored\\\": False}\\n\\n    # ------------------------------------------------------------ model turns\\n    def _log(self, text: str) -> None:\\n        line = f\\\"[{time.time() - self.t0:7.1f}s a={self.actions_used} L{self.level}] {text}\\\"\\n        self.transcript.write(line + \\\"\\\\n\\\"); self.transcript.flush()\\n        if self.verbose:\\n            print(f\\\"{self.game_id} {line[:220]}\\\", flush=True)\\n\\n    def _event(self, **kw) -> None:\\n        self.events.write(json.dumps({\\\"t\\\": round(time.time() - self.t0, 1), **kw}, default=str) + \\\"\\\\n\\\"); self.events.flush()\\n\\n    def _nav_lines(self) -> list[str]:\\n        try:\\n            cur = [t for t in self.host_transitions if t.before_frame.level == t.after_frame.level == self.level]\\n            nav = NavHelper(cur, self.frame)\\n            if nav.moves:\\n                lines = [\\\"Navigation helper summary:\\\", nav.summary()]\\n                ready = []\\n                g = nav.gauge(); left = g.get(\\\"actions_left\\\") if g else None\\n                for t in [t for t in nav.targets() if t.get(\\\"path_len\\\")][:4]:\\n                    path = nav.path_to(t[\\\"row\\\"], t[\\\"col\\\"])\\n                    if path:\\n                        tag = f\\\"colour {t.get('color')}, {'visited' if t.get('visited') else 'unvisited'}\\\"\\n                        if left is not None and len(path) >= left:\\n                            tag += f\\\", TOO LONG for the gauge ({len(path)} >= {left} left)\\\"\\n                        ready.append(f\\\"  nav.path_to({t['row']}, {t['col']}) -> {path}  # {tag}\\\")\\n                fr = nav.frontier()\\n                if fr and (path := nav.path_to(fr[0], fr[1])):\\n                    ready.append(f\\\"  nav.path_to({fr[0]}, {fr[1]}) -> {path}  # nearest unexplored block\\\")\\n                if ready:\\n                    lines.append(\\\"Ready-made routes (pass one straight to action(...)):\\\\n\\\" + \\\"\\\\n\\\".join(ready))\\n                lines += self._gauge_lines(nav, cur)\\n                lines += self._budget_lines(nav, cur)\\n                return lines\\n            g = nav.gauge()\\n            if g:\\n                return [f\\\"Action gauge detected: colour {g.get('color')}, {g.get('size')} cells, {g.get('per_action')} per action, ~{g.get('actions_left')} actions left. \\\"\\n                        \\\"This strip is a counter of your remaining actions (a HUD), NOT a goal to fill or empty: ignore it when choosing targets and finish the level before it runs out.\\\"]\\n        except Exception as e:  # never let the helper break a turn\\n            return [f\\\"Navigation helper unavailable ({type(e).__name__}).\\\"]\\n        if any(a in self.valid_actions for a in (\\\"UP\\\", \\\"DOWN\\\", \\\"LEFT\\\", \\\"RIGHT\\\")):\\n            return [\\\"Navigation helper: no movement learned yet. Press each of UP/DOWN/LEFT/RIGHT once, then read nav.summary().\\\"]\\n        return [\\\"Navigation helper: this level has no movement keys (click game). Use current_frame.segmentation to enumerate objects; nodes with hud=True are edge strips (counters), not targets.\\\"]\\n\\n    def _seconds_left(self) -> float:\\n        left = self.max_minutes * 60 - (time.time() - self.t0)\\n        if self.deadline is not None:\\n            left = min(left, self.deadline - time.time())\\n        return left\\n\\n    def _level_recap(self, level: int) -> str:\\n        \\\"\\\"\\\"What won the level: the last attempt's action sequence, objects that vanished (collected), gauge refills.\\\"\\\"\\\"\\n        cur = [t for t in self.host_transitions if t.before_frame.level == level]\\n        # last attempt = transitions after the last reset on this level (a reset restores the initial board)\\n        first = cur[0].before_frame.ascii if cur else None\\n        start = 0\\n        for i, t in enumerate(cur):\\n            if i > 0 and t.before_frame.ascii == first:\\n                start = i\\n        att = cur[start:]\\n        acts = [t.action if isinstance(t.action, str) else f\\\"MOUSE({t.action['row']},{t.action['col']})\\\" for t in att]\\n        seq, run = [], None\\n        for a in acts:  # compress runs: UP x3\\n            if run and run[0] == a:\\n                run[1] += 1\\n            else:\\n                run = [a, 1]; seq.append(run)\\n        seq_txt = \\\", \\\".join(f\\\"{a} x{n}\\\" if n > 1 else a for a, n in seq)[:400]\\n        before_objs = {n[\\\"hash\\\"]: n for n in att[0].before_frame.segmentation[\\\"nodes\\\"]} if att else {}\\n        after_objs = {n[\\\"hash\\\"]: n for n in att[-1].before_frame.segmentation[\\\"nodes\\\"]} if att else {}\\n        gone = [f\\\"colour {n['color']} ({n['pixels']}px at {n['center']})\\\" for h, n in before_objs.items()\\n                if h not in after_objs and not n[\\\"hud\\\"] and n[\\\"pixels\\\"] <= 300][:6]\\n        refills = []\\n        nav = NavHelper(att, att[-1].after_frame) if att else None\\n        g = nav.gauge() if nav else None\\n        if g:\\n            color = g[\\\"color\\\"]\\n            for t in att[:-1]:   # the last transition enters the next level (its full gauge is not a refill)\\n                b = sum(v == color for row in t.before_frame.grid for v in row); a = sum(v == color for row in t.after_frame.grid for v in row)\\n                if a > b:\\n                    refills.append(f\\\"after {t.action if isinstance(t.action, str) else 'MOUSE'} (+{a - b})\\\")\\n        parts = [f\\\"Level {level} was completed in {len(att)} actions (attempt {sum(1 for i, t in enumerate(cur) if i > 0 and t.before_frame.ascii == first) + 1}).\\\",\\n                 f\\\"Winning sequence: {seq_txt}.\\\"]\\n        if gone:\\n            parts.append(\\\"Objects that disappeared during the win (probably collected/used): \\\" + \\\"; \\\".join(gone) + \\\".\\\")\\n        if refills:\\n            parts.append(\\\"Gauge refilled \\\" + \\\", \\\".join(refills[:4]) + \\\".\\\")\\n        parts.append(\\\"The next level normally keeps the same rules with a new layout: find the analogous objects and repeat the strategy with nav.path_to.\\\")\\n        return \\\" \\\".join(parts)\\n\\n    def _probe_sweep(self) -> str:\\n        \\\"\\\"\\\"Try every untried action type once (each movement key, SPACE, clicks on up to 6 untried objects) and\\n        tabulate what each did. Runs at most once per level and only with enough gauge left.\\\"\\\"\\\"\\n        cur = [t for t in self.host_transitions if t.after_frame.level == self.level]\\n        used = {t.action if isinstance(t.action, str) else \\\"MOUSE\\\" for t in cur}\\n        clicked = {(t.action[\\\"row\\\"] // 4, t.action[\\\"col\\\"] // 4) for t in cur if isinstance(t.action, dict)}\\n        plan = [{\\\"action\\\": a} for a in (\\\"UP\\\", \\\"DOWN\\\", \\\"LEFT\\\", \\\"RIGHT\\\", \\\"SPACE\\\", \\\"ACTION7\\\") if a in self.valid_actions and a not in used]\\n        if \\\"MOUSE\\\" in self.valid_actions and self.frame is not None:\\n            nodes = [n for n in self.frame.segmentation[\\\"nodes\\\"] if not n[\\\"hud\\\"] and (n[\\\"center\\\"][0] // 4, n[\\\"center\\\"][1] // 4) not in clicked]\\n            nodes.sort(key=lambda n: n[\\\"pixels\\\"])\\n            seen_colors = set()\\n            for n in nodes:  # one click per colour, smallest objects first\\n                if n[\\\"color\\\"] in seen_colors:\\n                    continue\\n                seen_colors.add(n[\\\"color\\\"]); plan.append({\\\"action\\\": \\\"MOUSE\\\", \\\"row\\\": n[\\\"center\\\"][0], \\\"col\\\": n[\\\"center\\\"][1], \\\"colour\\\": n[\\\"color\\\"]})\\n                if len(plan) >= 10:\\n                    break\\n        try:\\n            g = NavHelper([t for t in cur if t.before_frame.level == self.level], self.frame).gauge() if self.frame else None\\n            if g and g.get(\\\"actions_left\\\") is not None:\\n                plan = plan[:max(0, int(g[\\\"actions_left\\\"]) - 4)]\\n        except Exception:\\n            pass\\n        rows = []\\n        for act in plan:   # may be empty when the model already pressed every key: the interaction probes below still run\\n            before = self.frame\\n            res = self.execute([{k: v for k, v in act.items() if k != \\\"colour\\\"}])\\n            if not res[\\\"executed_count\\\"]:\\n                break\\n            after = self.frame\\n            diff = summarize_diff(before, after)\\n            label = act[\\\"action\\\"] if act[\\\"action\\\"] != \\\"MOUSE\\\" else f\\\"MOUSE({act['row']},{act['col']}) on colour {act.get('colour')}\\\"\\n            what = \\\"no change\\\" if diff[\\\"changed_cells\\\"] == 0 else (f\\\"{diff['changed_cells']} cells changed\\\" +\\n                    (f\\\"; moved: {[(m['color'], m['from'], m['to']) for m in diff.get('moved', [])][:3]}\\\" if diff.get(\\\"moved\\\") else \\\"\\\") +\\n                    (f\\\"; appeared: {[(m['color'], m['center']) for m in diff.get('appeared', [])][:3]}\\\" if diff.get(\\\"appeared\\\") else \\\"\\\") +\\n                    (f\\\"; disappeared: {[(m['color'], m['center']) for m in diff.get('disappeared', [])][:3]}\\\" if diff.get(\\\"disappeared\\\") else \\\"\\\"))\\n            rows.append(f\\\"  {label}: {what}\\\")\\n            if res[\\\"level_completed\\\"] or res[\\\"game_over\\\"]:\\n                rows.append(f\\\"  -> {'LEVEL COMPLETED' if res['level_completed'] else 'GAME OVER (level restarted)'}\\\")\\n                break\\n        # interaction probes: go to the nearest targets and press the interaction key there (many games end when you\\n        # interact ON an object, which single-key probes never reveal)\\n        inter = [a for a in (\\\"SPACE\\\", \\\"ACTION7\\\") if a in self.valid_actions]\\n        try:\\n            nav = NavHelper([t for t in cur if t.before_frame.level == self.level], self.frame)\\n            if nav.moves and nav.avatar():\\n                done = 0\\n                for t in [t for t in nav.targets(max_n=12) if t.get(\\\"path_len\\\") and not t.get(\\\"visited\\\")][:3 if inter else 2]:\\n                    path = nav.path_to(t[\\\"row\\\"], t[\\\"col\\\"])\\n                    if not path or len(path) > 14:\\n                        continue\\n                    g = nav.gauge()\\n                    if g and g.get(\\\"actions_left\\\") is not None and len(path) + 1 > int(g[\\\"actions_left\\\"]) - 2:\\n                        continue   # would exhaust the gauge: not worth a game over\\n                    before = self.frame\\n                    res = self.execute([{\\\"action\\\": a} for a in path] + ([{\\\"action\\\": inter[0]}] if inter else []))\\n                    diff = summarize_diff(before, self.frame) if self.frame else {}\\n                    rows.append(f\\\"  go to colour {t['color']} at ({t['row']},{t['col']}){' then ' + inter[0] if inter else ''}: {diff.get('changed_cells', 0)} cells changed\\\"\\n                                + (f\\\"; disappeared: {[(m['color'], m['center']) for m in diff.get('disappeared', [])][:2]}\\\" if diff.get(\\\"disappeared\\\") else \\\"\\\"))\\n                    done += 1\\n                    if res[\\\"level_completed\\\"] or res[\\\"game_over\\\"]:\\n                        rows.append(f\\\"  -> {'LEVEL COMPLETED' if res['level_completed'] else 'GAME OVER (level restarted)'}\\\")\\n                        break\\n                    nav = NavHelper([t2 for t2 in self.host_transitions if t2.before_frame.level == t2.after_frame.level == self.level], self.frame)\\n                    if not nav.avatar():\\n                        break\\n        except Exception as e:\\n            rows.append(f\\\"  (interaction probes skipped: {type(e).__name__})\\\")\\n        self._log(f\\\"probe sweep on level {self.level}: {len(rows)} probes\\\")\\n        if not rows:\\n            return \\\"\\\"\\n        return \\\"Harness probe sweep (each untried action once, then 'go to a target and interact'):\\\\n\\\" + \\\"\\\\n\\\".join(rows)\\n\\n    def _host_probe(self) -> str:\\n        \\\"\\\"\\\"After repeated identical calls: execute ONE untried action so the model gets new information.\\\"\\\"\\\"\\n        cur = [t for t in self.host_transitions if t.after_frame.level == self.level]\\n        used = {t.action if isinstance(t.action, str) else \\\"MOUSE\\\" for t in cur}\\n        clicked = {(t.action[\\\"row\\\"] // 4, t.action[\\\"col\\\"] // 4) for t in cur if isinstance(t.action, dict)}\\n        probe = None\\n        for a in (\\\"SPACE\\\", \\\"ACTION7\\\", \\\"UP\\\", \\\"DOWN\\\", \\\"LEFT\\\", \\\"RIGHT\\\"):\\n            if a in self.valid_actions and a not in used:\\n                probe = {\\\"action\\\": a}; break\\n        if probe is None and \\\"MOUSE\\\" in self.valid_actions and self.frame is not None:\\n            nodes = [n for n in self.frame.segmentation[\\\"nodes\\\"] if not n[\\\"hud\\\"] and (n[\\\"center\\\"][0] // 4, n[\\\"center\\\"][1] // 4) not in clicked]\\n            nodes.sort(key=lambda n: n[\\\"pixels\\\"])\\n            if nodes:\\n                r, c = nodes[0][\\\"center\\\"]; probe = {\\\"action\\\": \\\"MOUSE\\\", \\\"row\\\": r, \\\"col\\\": c}\\n        if probe is None:\\n            return \\\"Harness probe: nothing untried is left on this level; change the ORDER or the target of your actions.\\\"\\n        before = self.frame\\n        res = self.execute([probe])\\n        diff = summarize_diff(before, self.frame) if self.frame else {}\\n        label = probe[\\\"action\\\"] if probe[\\\"action\\\"] != \\\"MOUSE\\\" else f\\\"MOUSE(row={probe['row']}, col={probe['col']})\\\"\\n        self._log(f\\\"host probe: {label} -> changed={res['board_changed']}\\\")\\n        return (f\\\"Harness probe (because you repeated yourself): executed {label} -> board_changed={res['board_changed']}, \\\"\\n                f\\\"level_completed={res['level_completed']}, game_over={res['game_over']}; change: {json.dumps(diff, default=str)[:400]}. Build on this.\\\")\\n\\n    def _checklist_lines(self) -> list[str]:\\n        \\\"\\\"\\\"The turn starts from this state instead of from zero: harness facts + the model's own fields, with the next gap named.\\\"\\\"\\\"\\n        if self.checklist_level != self.level:   # new level: plan and tried restart; goal/roles carry over (same rules)\\n            self.checklist[\\\"plan\\\"] = \\\"\\\"; self.checklist[\\\"tried\\\"] = []; self.checklist_level = self.level\\n        cur = [t for t in self.host_transitions if t.before_frame.level == t.after_frame.level == self.level]\\n        used = {t.action if isinstance(t.action, str) else \\\"MOUSE\\\" for t in cur}\\n        nav = None\\n        try:\\n            nav = NavHelper(cur, self.frame) if self.frame else None\\n        except Exception:\\n            pass\\n        items = []\\n        keys = [a for a in self.valid_actions if a != \\\"MOUSE\\\"]\\n        untried = [a for a in keys if a not in used]\\n        moves = nav.moves if nav else {}\\n        ctrl = \\\", \\\".join(f\\\"{k}={moves[k]}\\\" for k in moves) if moves else \\\"no movement learned\\\"\\n        items.append((not untried and (moves or not keys), \\\"controls\\\", ctrl + (f\\\"; untried keys: {untried}\\\" if untried else \\\"\\\") +\\n                      (\\\"; MOUSE available\\\" if \\\"MOUSE\\\" in self.valid_actions else \\\"\\\")))\\n        av = nav.avatar() if nav else None\\n        if keys:\\n            items.append((bool(av), \\\"avatar\\\", f\\\"colours {av['colors']} at ({av['row']},{av['col']})\\\" if av else \\\"not identified (press each movement key once)\\\"))\\n        g = nav.gauge() if nav else None\\n        lim = (f\\\"gauge colour {g['color']}, ~{g.get('actions_left')} actions left\\\" if g else \\\"no gauge detected yet\\\") + f\\\"; game overs on this level: {self.game_overs_this_level}\\\"\\n        items.append((True, \\\"limits\\\", lim))\\n        if self.frame is not None:\\n            seg = [n for n in self.frame.segmentation[\\\"nodes\\\"] if not n[\\\"hud\\\"]]\\n            counts: dict[int, int] = {}\\n            for n in seg:\\n                counts[n[\\\"color\\\"]] = counts.get(n[\\\"color\\\"], 0) + 1\\n            roles = self.checklist.get(\\\"roles\\\") or {}\\n            desc = \\\", \\\".join(f\\\"colour {c} x{k}\\\" + (f\\\" = {roles.get(str(c)) or roles.get(c)}\\\" if (roles.get(str(c)) or roles.get(c)) else \\\" (role ?)\\\") for c, k in sorted(counts.items(), key=lambda x: -x[1])[:8])\\n            known = sum(1 for c in counts if roles.get(str(c)) or roles.get(c))\\n            items.append((known >= min(2, len(counts)), \\\"objects/roles\\\", desc + \\\"   <- fill checklist['roles'][colour] = 'role'\\\"))\\n        goal = self.checklist.get(\\\"goal\\\") or \\\"\\\"\\n        items.append((bool(goal), \\\"goal\\\", goal or \\\"(none)   <- fill checklist['goal'] = 'hypothesis' after a probe\\\"))\\n        tried = self.checklist.get(\\\"tried\\\") or []\\n        items.append((True, \\\"tried\\\", \\\"; \\\".join(tried[-5:]) if tried else \\\"(nothing recorded)   <- checklist['tried'].append('what + outcome')\\\"))\\n        plan = self.checklist.get(\\\"plan\\\") or \\\"\\\"\\n        items.append((bool(plan), \\\"plan\\\", plan or \\\"(none)   <- fill checklist['plan'] = 'next concrete step'\\\"))\\n        done = sum(1 for ok, _, _ in items if ok); first_gap = next((name for ok, name, _ in items if not ok), None)\\n        head = f\\\"CHECKLIST (level {self.level}) \\u2014 {done}/{len(items)} settled. \\\" + (f\\\"Resolve first: {first_gap.upper()}.\\\" if first_gap else \\\"All settled: execute the plan.\\\")\\n        return [head] + [f\\\"[{'x' if ok else ' '}] {name}: {text}\\\" for ok, name, text in items]\\n\\n    def _rule_lines(self) -> list[str]:\\n        \\\"\\\"\\\"Symbolic rules induced from this level's transitions (exact fit; confirmed vs tentative).\\\"\\\"\\\"\\n        cur = [t for t in self.host_transitions if t.before_frame.level == t.after_frame.level == self.level]\\n        if len(cur) < 2 or self.frame is None:\\n            return []\\n        try:\\n            rules, unex = rule_induction.induce(cur, self.frame)\\n            self.rules_text = rule_induction.summary(rules, unex)\\n            return [self.rules_text + \\\"\\\\n(These are induced by the harness from what actually happened; extend them, do not contradict them.)\\\"]\\n        except Exception as e:\\n            return [f\\\"(rule induction unavailable: {type(e).__name__})\\\"]\\n\\n    def _new_kinds_lines(self) -> list[str]:\\n        \\\"\\\"\\\"Object kinds (colour, size class) present on this level that never appeared on earlier levels: probe these first.\\\"\\\"\\\"\\n        if self.level <= 1 or self.frame is None:\\n            return []\\n        def kinds(frame):\\n            out = set()\\n            for n in frame.segmentation[\\\"nodes\\\"]:\\n                if n[\\\"hud\\\"]:\\n                    continue\\n                size = \\\"tiny\\\" if n[\\\"pixels\\\"] <= 4 else \\\"small\\\" if n[\\\"pixels\\\"] <= 30 else \\\"medium\\\" if n[\\\"pixels\\\"] <= 200 else \\\"large\\\"\\n                out.add((n[\\\"color\\\"], size))\\n            return out\\n        seen = set()\\n        for t in self.host_transitions:\\n            if t.before_frame.level < self.level:\\n                seen |= kinds(t.before_frame)\\n        now = kinds(self.frame)\\n        new = sorted(now - seen)\\n        if not new:\\n            return []\\n        ex = {}\\n        for n in self.frame.segmentation[\\\"nodes\\\"]:\\n            size = \\\"tiny\\\" if n[\\\"pixels\\\"] <= 4 else \\\"small\\\" if n[\\\"pixels\\\"] <= 30 else \\\"medium\\\" if n[\\\"pixels\\\"] <= 200 else \\\"large\\\"\\n            if (n[\\\"color\\\"], size) in new and (n[\\\"color\\\"], size) not in ex:\\n                ex[(n[\\\"color\\\"], size)] = n[\\\"center\\\"]\\n        items = \\\", \\\".join(f\\\"colour {c} ({sz}, e.g. at {ex.get((c, sz))})\\\" for c, sz in new[:6])\\n        return [f\\\"NEW ON THIS LEVEL (not seen on earlier levels): {items}. A new kind usually carries the new rule of this level: \\\"\\n                \\\"touch/click each one once early and record what it did.\\\"]\\n\\n    def _budget_lines(self, nav, cur) -> list[str]:\\n        \\\"\\\"\\\"Arithmetic the model tends to skip: moves left on the gauge vs. route lengths, and whether refills are known.\\\"\\\"\\\"\\n        g = nav.gauge() if nav else None\\n        if not g or g.get(\\\"actions_left\\\") is None:\\n            return []\\n        per = abs(float(g.get(\\\"per_action\\\") or 1)) or 1\\n        full = int(round(g[\\\"size\\\"] / per)) if g.get(\\\"size\\\") else None\\n        lines = [f\\\"BUDGET CHECK: ~{g['actions_left']} moves left on this gauge\\\" + (f\\\" (a full gauge is ~{full} moves at {per:g} per move)\\\" if full else \\\"\\\") + \\\".\\\"]\\n        try:\\n            reach = [t for t in nav.targets() if t.get(\\\"path_len\\\")]\\n            if reach:\\n                far = max(t[\\\"path_len\\\"] for t in reach); near = min(t[\\\"path_len\\\"] for t in reach)\\n                lines.append(f\\\"Known targets are {near}-{far} moves away. Anything beyond {g['actions_left']} moves is unreachable before the gauge runs out \\\"\\n                             \\\"unless you pass a refill item on the way (objects that increased the gauge when touched, see gauge lines).\\\")\\n        except Exception:\\n            pass\\n        return lines\\n\\n    def _micro_diff_lines(self, before_n: int) -> list[str]:\\n        \\\"\\\"\\\"When the last actions changed only a few cells (a sprite rotated/recoloured), zoom into that region.\\\"\\\"\\\"\\n        new = self.host_transitions[before_n:]\\n        if not new:\\n            return []\\n        b, a = new[0].before_frame, new[-1].after_frame\\n        mb, ma = masked_ascii(b).splitlines(), masked_ascii(a).splitlines()\\n        cells = [(r, c) for r in range(len(mb)) for c in range(len(mb[r])) if mb[r][c] != ma[r][c]]\\n        if not cells or len(cells) > 40:\\n            return []\\n        rs = [r for r, _ in cells]; cs = [c for _, c in cells]\\n        r0, r1 = max(0, min(rs) - 2), min(63, max(rs) + 2); c0, c1 = max(0, min(cs) - 2), min(63, max(cs) + 2)\\n        if (r1 - r0) > 14 or (c1 - c0) > 14:\\n            return []\\n        crop = lambda rows: \\\"\\\\n\\\".join(\\\"    \\\" + rows[r][c0:c1 + 1] for r in range(r0, r1 + 1))\\n        return [f\\\"SMALL CHANGE ZOOM (rows {r0}-{r1}, cols {c0}-{c1}; {len(cells)} cells changed, HUD excluded) \\u2014 before / after:\\\",\\n                crop(b.ascii.splitlines()), \\\"    ->\\\", crop(a.ascii.splitlines())]\\n\\n    def _gauge_lines(self, nav, cur) -> list[str]:\\n        g = nav.gauge()\\n        if not g:\\n            return []\\n        out = [f\\\"Gauge budget: ~{g.get('actions_left')} actions left before the level restarts (colour {g.get('color')}, {g.get('per_action')} per action). \\\"\\n               \\\"Any route longer than that fails: pick a target reachable within the budget or find what refills the gauge first.\\\"]\\n        # refill clues: transitions of this level where the gauge object grew\\n        color = g.get(\\\"color\\\"); grew = []\\n        for t in cur[-60:]:\\n            try:\\n                b = sum(1 for row in t.before_frame.grid for v in row if v == color)\\n                a = sum(1 for row in t.after_frame.grid for v in row if v == color)\\n            except Exception:\\n                continue\\n            if a > b:\\n                grew.append(f\\\"{t.action} (+{a - b})\\\")\\n        if grew:\\n            out.append(\\\"Gauge REFILLED after: \\\" + \\\", \\\".join(grew[-4:]) + \\\" \\u2014 whatever the avatar touched then refills it; plan to collect such objects on the way.\\\")\\n        return out\\n\\n    def _budget_line(self) -> str:\\n        left = self._seconds_left()\\n        return f\\\"Time left: {max(0, left) / 60:.1f} min. Action budget left: {self.max_actions - self.actions_used}.\\\"\\n\\n    def _user_message(self) -> str:\\n        parts = [turn_header(level=self.level, levels_total=self.levels_total, actions_used=self.actions_used, level_actions=self.level_actions,\\n                             valid_actions=self.valid_actions, budget_line=self._budget_line())]\\n        if self.oracle_rules:\\n            parts.append(\\\"KNOWN RULES OF THIS GAME (given, trust them fully):\\\\n\\\" + self.oracle_rules)\\n        try:\\n            block = self._checklist_lines() + self._rule_lines() + self._new_kinds_lines()\\n            key = \\\"\\\\n\\\".join(block)\\n            if key != getattr(self, \\\"_last_block\\\", None) or self.model_turns % 5 == 0:\\n                parts += block; self._last_block = key\\n            else:\\n                parts.append(\\\"(checklist and rules unchanged since last turn)\\\")\\n        except Exception as e:\\n            parts.append(f\\\"(checklist unavailable: {type(e).__name__})\\\")\\n        if self.last_outcome:\\n            parts += self.last_outcome\\n        parts += self._nav_lines()\\n        turns_here = self.model_turns - self.level_turn_start\\n        early = False   # a shrinking gauge means the budget will be gone before turn 8: sweep from turn 4 while routes are still affordable\\n        try:\\n            g = NavHelper([t for t in self.host_transitions if t.before_frame.level == t.after_frame.level == self.level], self.frame).gauge() if self.frame else None\\n            early = bool(g and g.get(\\\"actions_left\\\") is not None and int(g[\\\"actions_left\\\"]) <= 25 and turns_here >= 4)\\n        except Exception:\\n            pass\\n        if (PROBE_SWEEP_AFTER_TURNS and self.level not in self.probe_sweeps and (turns_here >= PROBE_SWEEP_AFTER_TURNS or early)\\n                and not (self.solver and self.solver.get(\\\"status\\\") == \\\"active\\\")):\\n            lvl = self.level\\n            text = self._probe_sweep()\\n            self.probe_sweeps[lvl] = text if self.level == lvl else \\\"\\\"   # the sweep itself completed the level: nothing to show on the new level\\n            if self.level != lvl:\\n                self.probe_sweeps.setdefault(self.level, \\\"\\\")\\n                self.level_turn_start = self.model_turns\\n        if self.goal_hypotheses:\\n            parts.append(goal_inference.summary(self.goal_hypotheses))\\n            if not self.checklist.get(\\\"goal\\\"):\\n                self.checklist[\\\"goal\\\"] = \\\"(harness hypothesis) \\\" + self.goal_hypotheses[0][\\\"text\\\"][:200]\\n        if self.probe_sweeps.get(self.level):\\n            parts.append(self.probe_sweeps[self.level])\\n        if self.level_recaps:\\n            parts.append(\\\"Recap of previous levels (written by the harness):\\\\n\\\" + \\\"\\\\n\\\".join(f\\\"- {r}\\\" for r in self.level_recaps[-2:]))\\n        parts.append(\\\"Your notes (the sandbox variable `notes`; keep it current instead of re-deriving the rules each turn):\\\\n\\\" +\\n                     (self.notes or \\\"(empty \\u2014 write what each key does, the goal hypothesis and the next plan)\\\"))\\n        if self.proposal_required > 0:\\n            parts.append(f\\\"REQUIRED: you just completed a level, so you know the rules. Call propose_solver(code) with a solve() that reproduces \\\"\\n                         f\\\"the winning strategy from `transitions` of the previous level (use nav.path_to / segmentation, not fixed coordinates). \\\"\\n                         f\\\"action() stays blocked until you do ({self.proposal_required} turn(s) left before the requirement is waived); \\\"\\n                         \\\"inspecting the board is allowed.\\\")\\n        if self.solver and self.solver.get(\\\"status\\\") == \\\"draft\\\":\\n            try:\\n                res = self.sandbox.run(f\\\"__solver_code = {json.dumps(self.solver['code'])}\\\\nexec(compile(__solver_code, 'solver.py', 'exec'), globals())\\\\nprint(list(solve() or [])[:12])\\\", timeout=15)\\n                self.solver[\\\"suggestion\\\"] = (res[\\\"stdout\\\"].strip() or res.get(\\\"error\\\") or \\\"?\\\")[:300]\\n            except Exception as e:\\n                self.solver[\\\"suggestion\\\"] = f\\\"(dry run failed: {e!r})\\\"\\n        parts += solver_policy.status_lines(self.solver, model_turns=self.model_turns, level_just_completed=self.level_just_completed)\\n        self.level_just_completed = False\\n        if self.solver and self.solver.get(\\\"status\\\") in (\\\"failed\\\", \\\"rejected\\\"):\\n            self.solver[\\\"status\\\"] = \\\"shown\\\"\\n        parts.append(\\\"Current board:\\\\n\\\" + self.frame.ascii)\\n        return \\\"\\\\n\\\".join(parts)\\n\\n    def _trim_context(self) -> None:\\n        \\\"\\\"\\\"Older turns lose their board text and long tool output; on overflow drop the oldest turns.\\\"\\\"\\\"\\n        user_idx = [i for i, m in enumerate(self.messages) if m[\\\"role\\\"] == \\\"user\\\"]\\n        for i in user_idx[:-self.keep_full_turns]:\\n            c = self.messages[i][\\\"content\\\"]\\n            if isinstance(c, list):\\n                continue\\n            if \\\"Current board:\\\" in c:\\n                self.messages[i][\\\"content\\\"] = c.split(\\\"Current board:\\\")[0] + \\\"[board omitted]\\\"\\n        tool_idx = [i for i, m in enumerate(self.messages) if m[\\\"role\\\"] == \\\"tool\\\"]\\n        for i in tool_idx[:-self.keep_full_turns]:\\n            c = self.messages[i][\\\"content\\\"]\\n            if len(c) > 800:\\n                self.messages[i][\\\"content\\\"] = c[:600] + \\\"\\\\n...[older tool output trimmed]...\\\"\\n        # token estimate calibrated on the last real prompt_tokens (hex boards tokenize at ~1 token per character)\\n        budget = (self.context_tokens - self.client.max_tokens) * 0.85 if self.client else self.context_tokens * 0.8\\n        est = self._estimate_tokens()\\n        while est > budget and len(self.messages) > 4:\\n            self._drop_oldest_turn(); est = self._estimate_tokens()\\n\\n    def _estimate_tokens(self) -> float:\\n        def _len(c):\\n            return len(c) if isinstance(c, str) else sum(len(p.get(\\\"text\\\", \\\"\\\")) for p in c if p.get(\\\"type\\\") == \\\"text\\\") + 1200 * sum(1 for p in c if p.get(\\\"type\\\") == \\\"image_url\\\")\\n        chars = sum(_len(m.get(\\\"content\\\") or \\\"\\\") + len(json.dumps(m.get(\\\"tool_calls\\\") or \\\"\\\")) for m in self.messages)\\n        return chars * getattr(self, \\\"_tok_per_char\\\", 0.45) + 300\\n\\n    def _drop_oldest_turn(self) -> None:\\n        # messages[0] is the system prompt; a turn is user, assistant(, tool)\\n        k = 1\\n        while k < len(self.messages) and self.messages[k][\\\"role\\\"] != \\\"user\\\":\\n            k += 1\\n        end = k + 1\\n        while end < len(self.messages) and self.messages[end][\\\"role\\\"] != \\\"user\\\":\\n            end += 1\\n        if end - k >= len(self.messages) - 1:  # never drop the only turn\\n            return\\n        del self.messages[k:end]\\n\\n    def _run_tool(self, code: str, *, who: str) -> str:\\n        self._run_actions = 0\\n        res = self.sandbox.run(code, timeout=self.tool_timeout)\\n        if res.get(\\\"notes\\\"):\\n            self.notes = res[\\\"notes\\\"]\\n        if isinstance(res.get(\\\"checklist\\\"), dict) and res[\\\"checklist\\\"]:\\n            ck = res[\\\"checklist\\\"]\\n            self.checklist = {\\\"goal\\\": str(ck.get(\\\"goal\\\") or \\\"\\\")[:300], \\\"roles\\\": ck.get(\\\"roles\\\") if isinstance(ck.get(\\\"roles\\\"), dict) else {},\\n                              \\\"plan\\\": str(ck.get(\\\"plan\\\") or \\\"\\\")[:300], \\\"tried\\\": [str(t)[:120] for t in (ck.get(\\\"tried\\\") or [])][-8:]}\\n        out = res[\\\"stdout\\\"]\\n        if res[\\\"error\\\"]:\\n            out += (\\\"\\\\n\\\" if out else \\\"\\\") + \\\"Error: \\\" + res[\\\"error\\\"]\\n        self._event(kind=\\\"tool\\\", who=who, code=code, stdout=res[\\\"stdout\\\"][:4000], error=res[\\\"error\\\"], actions=res[\\\"actions_executed\\\"])\\n        return out or \\\"(no output)\\\"\\n\\n    def _untried_here(self) -> str:\\n        \\\"\\\"\\\"What has never been tried on this level: keys, clicked objects (by 4x4 cell), reachable targets.\\\"\\\"\\\"\\n        cur = [t for t in self.host_transitions if t.before_frame.level == t.after_frame.level == self.level]\\n        used = {t.action for t in cur if isinstance(t.action, str)}\\n        parts = []\\n        keys = [a for a in self.valid_actions if a != \\\"MOUSE\\\" and a not in used]\\n        if keys:\\n            parts.append(f\\\"keys never pressed: {keys}\\\")\\n        if \\\"MOUSE\\\" in self.valid_actions and self.frame is not None:\\n            clicked = {(t.action[\\\"row\\\"] // 4, t.action[\\\"col\\\"] // 4) for t in cur if isinstance(t.action, dict)}\\n            nodes = [n for n in self.frame.segmentation[\\\"nodes\\\"] if not n[\\\"hud\\\"] and (n[\\\"center\\\"][0] // 4, n[\\\"center\\\"][1] // 4) not in clicked]\\n            nodes.sort(key=lambda n: n[\\\"pixels\\\"])\\n            if nodes:\\n                parts.append(\\\"objects never clicked: \\\" + \\\", \\\".join(f\\\"colour {n['color']} at {tuple(n['center'])}\\\" for n in nodes[:6]) + (f\\\" (+{len(nodes) - 6} more)\\\" if len(nodes) > 6 else \\\"\\\"))\\n        try:\\n            nav = NavHelper(cur, self.frame) if self.frame else None\\n            if nav and nav.avatar():\\n                targets = [tg for tg in nav.targets() if tg.get(\\\"path_len\\\") is not None and not tg.get(\\\"visited\\\")]\\n                if targets:\\n                    parts.append(\\\"reachable targets: \\\" + \\\", \\\".join(f\\\"colour {tg['color']} at ({tg['row']},{tg['col']}) in {tg['path_len']} moves (never visited)\\\" for tg in targets[:5]))\\n        except Exception:\\n            pass\\n        return \\\"; \\\".join(parts) if parts else \\\"nothing obvious is untried: change the ORDER (e.g. interact right after arriving) or combine keys.\\\"\\n\\n    def _cycle_lines(self, before_n: int) -> list[str]:\\n        \\\"\\\"\\\"Detect (a) a board state already visited in this level attempt and (b) a repeated action sequence; both mean the model is looping.\\n        Only evaluated on turns that executed at least one action (a warning is not repeated while the model merely thinks).\\\"\\\"\\\"\\n        if len(self.host_transitions) == before_n:\\n            return []\\n        cur = [t for t in self.host_transitions[self.attempt_start_index:] if t.before_frame.level == t.after_frame.level == self.level]\\n        if len(cur) < 4 or self.frame is None:\\n            return []\\n        out = []\\n        labels = []\\n        for t in cur:\\n            a = t.action\\n            labels.append(a if isinstance(a, str) else f\\\"MOUSE({a['row'] // 4},{a['col'] // 4})\\\")\\n        # (b) period detection on the last actions: the same sequence of length k (2..8) executed 3x in a row for short k, 2x for k >= 4\\n        #     (k = 1 is excluded: walking UP UP UP or clicking a toggle twice is normal play; a real loop shows up as a state revisit)\\n        period = None\\n        for k in range(2, 9):\\n            reps = 3 if k <= 3 else 2\\n            if len(labels) >= reps * k and all(labels[-k:] == labels[-(i + 1) * k:-i * k or None] for i in range(1, reps)) and len(set(labels[-k:])) >= 2:\\n                period = k\\n        # (a) state revisit: the current masked board equals the board after an earlier action on this level\\n        now = masked_ascii(self.frame)\\n        first = None\\n        for i, t in enumerate(cur[:-1]):\\n            if masked_ascii(t.after_frame) == now:\\n                first = i; break\\n        if first is not None and len(cur) - 1 - first >= 2:\\n            since = labels[first + 1:]\\n            compact = []\\n            for a in since:\\n                if compact and compact[-1][0] == a:\\n                    compact[-1][1] += 1\\n                else:\\n                    compact.append([a, 1])\\n            seq = \\\" \\\".join(f\\\"{a}x{n}\\\" if n > 1 else a for a, n in compact)[:200]\\n            out.append(f\\\"LOOP DETECTED: the board is identical to what it was after action #{first + 1} of this level, {len(since)} actions ago. \\\"\\n                       f\\\"The actions since then ({seq}) form a cycle and cannot progress the level. Do NOT repeat them. Untried here: {self._untried_here()}\\\")\\n        elif period:\\n            out.append(f\\\"REPEATED SEQUENCE: your last {period} actions ({' '.join(labels[-period:])}) repeat the {period} before them. \\\"\\n                       f\\\"If the board did not move closer to the goal, stop repeating. Untried here: {self._untried_here()}\\\")\\n        if out:\\n            self.cycle_hits_this_level = getattr(self, \\\"cycle_hits_this_level\\\", 0) + 1\\n            self._log(f\\\"cycle guard: {out[0][:90]!r} (hit {self.cycle_hits_this_level})\\\")\\n            if self.cycle_hits_this_level >= 3 and self.cycle_hits_this_level % 3 == 0 and self.valid_actions:\\n                try:\\n                    out.append(self._host_probe())\\n                except Exception:\\n                    pass\\n        return out\\n\\n    def _outcome_lines(self, before_n: int) -> list[str]:\\n        new = self.host_transitions[before_n:]\\n        extra = []\\n        if self.game_overs_this_level:\\n            extra.append(f\\\"GAME OVER count on this level: {self.game_overs_this_level} (each one reset the level). The action limit or the gauge ran out \\\"\\n                         \\\"before the goal: stop repeating the same sweep, re-read what changed the board, and try a different hypothesis.\\\")\\n        if not new:\\n            return [\\\"Last turn executed no actions.\\\"] + extra\\n        acts = [t.action if isinstance(t.action, str) else \\\"MOUSE\\\" for t in new]\\n        diff = summarize_diff(new[0].before_frame, new[-1].after_frame)\\n        return [f\\\"Last turn executed {len(new)} action(s): {', '.join(acts[:12])}{'...' if len(acts) > 12 else ''}. \\\"\\n                f\\\"Board change over the turn: {json.dumps(diff, default=str)[:700]}\\\"] + extra\\n\\n    def _flatten_old_images(self) -> None:\\n        \\\"\\\"\\\"Keep only the newest user message multimodal; older ones become plain text (saves tokens).\\\"\\\"\\\"\\n        users = [m for m in self.messages if m[\\\"role\\\"] == \\\"user\\\" and isinstance(m.get(\\\"content\\\"), list)]\\n        for m in users[:-1]:\\n            m[\\\"content\\\"] = \\\"\\\\n\\\".join(part.get(\\\"text\\\", \\\"\\\") for part in m[\\\"content\\\"] if part.get(\\\"type\\\") == \\\"text\\\") + \\\"\\\\n[board image omitted]\\\"\\n\\n    def model_turn(self) -> bool:\\n        \\\"\\\"\\\"One model call + tool execution. Returns False when the model produced nothing usable.\\\"\\\"\\\"\\n        text = self._user_message()\\n        self._event(kind=\\\"user\\\", turn=self.model_turns + 1, content=text[:6000])   # what the model saw (post-mortems need the harness lines too)\\n        if self.image_context and self.frame is not None:\\n            self.messages.append({\\\"role\\\": \\\"user\\\", \\\"content\\\": [{\\\"type\\\": \\\"text\\\", \\\"text\\\": text + \\\"\\\\nThe same board is attached as an image (8-cell grid lines).\\\"},\\n                                                              {\\\"type\\\": \\\"image_url\\\", \\\"image_url\\\": {\\\"url\\\": \\\"data:image/png;base64,\\\" + grid_to_png_b64(self.frame.grid)}}]})\\n            self._flatten_old_images()\\n        else:\\n            self.messages.append({\\\"role\\\": \\\"user\\\", \\\"content\\\": text})\\n        self._trim_context()\\n        tools = [PYTHON_TOOL, PROPOSE_TOOL]\\n        # after a level completion: one free inspection turn, then the proposal call is forced via tool_choice\\n        choice = {\\\"type\\\": \\\"function\\\", \\\"function\\\": {\\\"name\\\": \\\"propose_solver\\\"}} if 0 < self.proposal_required < PROPOSAL_ATTEMPTS else (\\\"required\\\" if self.tool_choice_required else \\\"auto\\\")\\n        override = None\\n        if self.think_first_turns and (self.model_turns - self.level_turn_start) < self.think_first_turns:\\n            override = {\\\"chat_template_kwargs\\\": {\\\"enable_thinking\\\": True}, \\\"max_tokens\\\": max(self.client.max_tokens, 8192), \\\"temperature\\\": 0.6, \\\"top_p\\\": 0.95}\\n        r = None\\n        for attempt in range(6):\\n            try:\\n                r = self.client.chat(self.messages, tools=tools, tool_choice=choice, override=override); break\\n            except ContextLengthError:\\n                self._tok_per_char = min(1.2, getattr(self, \\\"_tok_per_char\\\", 0.45) * 1.3)   # we under-estimated: be more aggressive\\n                n_before = len(self.messages)\\n                for _ in range(2 + attempt):\\n                    self._drop_oldest_turn()\\n                if len(self.messages) == n_before:   # nothing left to drop: shrink the board text of the current turn\\n                    if isinstance(self.messages[-1][\\\"content\\\"], str):\\n                        self.messages[-1][\\\"content\\\"] = self.messages[-1][\\\"content\\\"].split(\\\"Current board:\\\")[0] + \\\"[board omitted: context full]\\\"\\n                self._log(f\\\"context overflow: dropped turns ({n_before} -> {len(self.messages)} messages), retry {attempt + 1}\\\")\\n        if r is None:\\n            raise RuntimeError(\\\"context overflow could not be resolved\\\")\\n        # calibrate the estimate with the real prompt size\\n        pt = int(r.usage.get(\\\"prompt_tokens\\\") or 0); chars = sum((len(m.get(\\\"content\\\")) if isinstance(m.get(\\\"content\\\"), str) else 1200) for m in self.messages if m.get(\\\"content\\\"))\\n        if pt and chars:\\n            self._tok_per_char = max(0.25, min(1.2, pt / chars))\\n        self.model_turns += 1\\n        msg = r.message\\n        self._event(kind=\\\"model\\\", turn=self.model_turns, content=msg.get(\\\"content\\\", \\\"\\\")[:2000], reasoning=r.reasoning[:1500],\\n                    tool_calls=msg.get(\\\"tool_calls\\\"), latency=round(r.latency, 1), usage=r.usage)\\n        self.messages.append({\\\"role\\\": \\\"assistant\\\", \\\"content\\\": msg.get(\\\"content\\\", \\\"\\\"), **({\\\"tool_calls\\\": msg[\\\"tool_calls\\\"]} if msg.get(\\\"tool_calls\\\") else {})})\\n        for line in (msg.get(\\\"content\\\") or \\\"\\\").splitlines():   # the five-line memory feeds the checklist\\n            low = line.strip().lower()\\n            if low.startswith(\\\"goal model:\\\"):\\n                self.checklist[\\\"goal\\\"] = line.split(\\\":\\\", 1)[1].strip()[:300]\\n            elif low.startswith(\\\"plan:\\\"):\\n                self.checklist[\\\"plan\\\"] = line.split(\\\":\\\", 1)[1].strip()[:300]\\n        self._log(f\\\"model turn {self.model_turns} ({r.latency:.0f}s, {r.usage.get('completion_tokens', '?')} tok): {(msg.get('content') or r.reasoning)[:200]!r}\\\")\\n        calls = msg.get(\\\"tool_calls\\\") or []\\n        if not calls:\\n            # content-as-code fallback: some models write the python call in the message body instead of a tool call\\n            body = (msg.get(\\\"content\\\") or \\\"\\\").strip()\\n            if body.startswith(\\\"```\\\"):\\n                body = body.strip(\\\"`\\\"); body = body.split(\\\"\\\\n\\\", 1)[1] if \\\"\\\\n\\\" in body else body\\n                body = body.rsplit(\\\"```\\\", 1)[0] if \\\"```\\\" in body else body\\n            # JSON fallback: the tool arguments emitted as a bare JSON object in the body (seen with gpt-oss + tool_choice=required)\\n            if body.startswith(\\\"{\\\") and body.endswith(\\\"}\\\"):\\n                try:\\n                    obj = json.loads(body)\\n                    if isinstance(obj, dict) and isinstance(obj.get(\\\"code\\\"), str) and obj[\\\"code\\\"].strip():\\n                        calls = [{\\\"id\\\": \\\"fallback_json\\\", \\\"type\\\": \\\"function\\\", \\\"function\\\": {\\\"name\\\": \\\"python\\\", \\\"arguments\\\": json.dumps(obj)}}]\\n                        self.messages[-1][\\\"tool_calls\\\"] = calls; self.messages[-1][\\\"content\\\"] = \\\"\\\"\\n                        self._log(\\\"json-content fallback: executing the body's `code` field as the python tool\\\")\\n                except Exception:\\n                    pass\\n            looks_like_code = not calls and any(k in body for k in (\\\"action(\\\", \\\"propose_solver(\\\", \\\"print(\\\", \\\"nav.\\\", \\\"checklist[\\\")) and not body.startswith(\\\"{\\\")\\n            if looks_like_code and len(body) < 6000:\\n                try:\\n                    import ast as _ast; _ast.parse(body)\\n                    calls = [{\\\"id\\\": \\\"fallback_0\\\", \\\"type\\\": \\\"function\\\", \\\"function\\\": {\\\"name\\\": \\\"python\\\", \\\"arguments\\\": json.dumps({\\\"code\\\": body})}}]\\n                    self.messages[-1][\\\"tool_calls\\\"] = calls; self.messages[-1][\\\"content\\\"] = \\\"\\\"\\n                    self._log(\\\"content-as-code fallback: executing the message body as python\\\")\\n                except SyntaxError:\\n                    pass\\n        if not calls:\\n            if r.finish_reason == \\\"length\\\":\\n                self.last_outcome = [\\\"Your last reply was cut off by the output limit while you were still reasoning, so nothing happened. \\\"\\n                                     \\\"Reason briefly (a few sentences) and act with one python tool call; let the code do the arithmetic.\\\"]\\n            else:\\n                self.last_outcome = [\\\"Your last reply contained no python tool call; nothing happened. Reply with exactly one python tool call.\\\"]\\n            return False\\n        before_n = len(self.host_transitions)\\n        for call in calls[:1]:\\n            try:\\n                args = json.loads(call[\\\"function\\\"].get(\\\"arguments\\\") or \\\"{}\\\")\\n                code = args.get(\\\"code\\\") or \\\"\\\"\\n            except Exception:\\n                args, code = {}, \\\"\\\"\\n            for key in (\\\"goal\\\", \\\"plan\\\"):   # checklist fields carried by the tool call itself\\n                if isinstance(args.get(key), str) and args[key].strip():\\n                    self.checklist[key] = args[key].strip()[:300]\\n            if isinstance(args.get(\\\"roles\\\"), str) and args[\\\"roles\\\"].strip():\\n                for part in args[\\\"roles\\\"].replace(\\\";\\\", \\\",\\\").split(\\\",\\\"):\\n                    if \\\"=\\\" in part:\\n                        k, v = part.split(\\\"=\\\", 1)\\n                        if k.strip().isdigit() and v.strip():\\n                            self.checklist.setdefault(\\\"roles\\\", {})[k.strip()] = v.strip()[:40]\\n            if call[\\\"function\\\"].get(\\\"name\\\") == \\\"propose_solver\\\" and code:\\n                code = f\\\"print(propose_solver({json.dumps(code)}))\\\"   # routed through the sandbox's verifier\\n            norm = \\\" \\\".join(code.split())\\n            if code and norm in self.recent_codes[-2:]:\\n                used = {t.action if isinstance(t.action, str) else \\\"MOUSE\\\" for t in self.host_transitions if t.after_frame.level == self.level}\\n                unused = [a for a in self.valid_actions if a not in used]\\n                out = (\\\"Rejected: this code is identical to one of your last two calls, so it was NOT executed (repeating it cannot give new information). \\\"\\n                       \\\"Do something different: \\\" + (f\\\"actions not yet tried on this level: {unused}. \\\" if unused else \\\"\\\") +\\n                       \\\"Try a different key, a different object, SPACE/MOUSE on things you have not touched, or update `notes` with a new hypothesis first.\\\")\\n                self._log(\\\"repetition guard: identical tool code rejected\\\")\\n                self.rejections_in_row += 1\\n                if self.rejections_in_row >= 2:\\n                    out += \\\"\\\\n\\\" + self._host_probe()\\n                    self.rejections_in_row = 0\\n            else:\\n                self.rejections_in_row = 0\\n                out = self._run_tool(code, who=\\\"model\\\") if code else \\\"Error: tool call had no `code` argument\\\"\\n            if code:\\n                self.recent_codes = (self.recent_codes + [norm])[-4:]\\n            if self.proposal_required > 0:\\n                if \\\"propose_solver(\\\" in code:\\n                    self.proposal_required = 0\\n                else:\\n                    self.proposal_required -= 1\\n                    if self.proposal_required == 0:\\n                        out += \\\"\\\\n(The solver requirement is now waived; you may act directly.)\\\"\\n            self._log(f\\\"tool ({len(self.host_transitions) - before_n} actions): {out[:300]!r}\\\")\\n            self.messages.append({\\\"role\\\": \\\"tool\\\", \\\"tool_call_id\\\": call.get(\\\"id\\\", \\\"call_0\\\"), \\\"content\\\": out[:6000]})\\n        for call in calls[1:]:  # extra calls are acknowledged, not executed\\n            self.messages.append({\\\"role\\\": \\\"tool\\\", \\\"tool_call_id\\\": call.get(\\\"id\\\", \\\"call_x\\\"), \\\"content\\\": \\\"Skipped: only one python call per reply is executed.\\\"})\\n        self.last_outcome = self._outcome_lines(before_n)\\n        try:\\n            self.last_outcome += self._micro_diff_lines(before_n)\\n        except Exception:\\n            pass\\n        try:\\n            self.last_outcome += self._cycle_lines(before_n)\\n        except Exception as e:\\n            self._log(f\\\"cycle guard error: {type(e).__name__}: {e}\\\")\\n        if len(self.host_transitions) == before_n:\\n            self.zero_action_turns += 1\\n            if self.zero_action_turns >= 2 and self.valid_actions:\\n                try:   # analysis paralysis: keep the game moving with one untried action and report what it did\\n                    self.last_outcome.append(\\\"You have spent 2 turns without acting. \\\" + self._host_probe())\\n                except Exception:\\n                    pass\\n                self.zero_action_turns = 0\\n        else:\\n            self.zero_action_turns = 0\\n        # harness-written 'tried' entry: what this turn did and what happened\\n        new = self.host_transitions[before_n:]\\n        acts = [t.action if isinstance(t.action, str) else \\\"MOUSE\\\" for t in new]\\n        if acts:\\n            compact = []\\n            for a in acts:\\n                if compact and compact[-1][0] == a:\\n                    compact[-1][1] += 1\\n                else:\\n                    compact.append([a, 1])\\n            what = \\\" \\\".join(f\\\"{a}x{n}\\\" if n > 1 else a for a, n in compact)[:80]\\n            changed = sum(1 for t in new if masked_ascii(t.before_frame) != masked_ascii(t.after_frame))\\n            outcome = (\\\"LEVEL DONE\\\" if any(t.before_frame.level != t.after_frame.level for t in new) else\\n                       f\\\"{changed}/{len(new)} moves changed the board\\\")\\n            self.checklist.setdefault(\\\"tried\\\", []).append(f\\\"T{self.model_turns}: {what} -> {outcome}\\\")\\n            self.checklist[\\\"tried\\\"] = self.checklist[\\\"tried\\\"][-8:]\\n        return True\\n\\n    def solver_turn(self) -> None:\\n        s = self.solver\\n        before_n = len(self.host_transitions); prev_level = self.level; prev_go = self.game_overs_this_level + self.resets\\n        budget = solver_policy.MAX_ACTIONS_PER_TURN\\n        try:  # never let a solver turn spend the last actions of a gauge blindly\\n            cur = [t for t in self.host_transitions if t.before_frame.level == t.after_frame.level == self.level]\\n            g = NavHelper(cur, self.frame).gauge() if self.frame else None\\n            if g and g.get(\\\"actions_left\\\") is not None:\\n                budget = min(budget, int(g[\\\"actions_left\\\"]) - 2)\\n        except Exception:\\n            pass\\n        if budget <= 0:\\n            s[\\\"status\\\"], s[\\\"reason\\\"] = \\\"failed\\\", \\\"the action gauge is nearly exhausted; the solver is paused so you can decide what to do with the last actions\\\"\\n            self._log(\\\"solver paused: gauge nearly exhausted\\\"); self.last_outcome = [f\\\"The stored solver was paused: {s['reason']}.\\\"]\\n            return\\n        out = self._run_tool(solver_policy.run_snippet(s[\\\"code\\\"], budget), who=\\\"solver\\\")\\n        s[\\\"turns\\\"] += 1; self.solver_turns += 1\\n        n = len(self.host_transitions) - before_n; s[\\\"actions_run\\\"] += n\\n        # progress = change outside HUD strips (a ticking gauge alone is not progress)\\n        changed = any(masked_ascii(t.before_frame) != masked_ascii(t.after_frame) for t in self.host_transitions[before_n:])\\n        self._log(f\\\"solver turn {s['turns']}: {n} actions, changed={changed}: {out[:160]!r}\\\")\\n        if \\\"Error:\\\" in out:\\n            s[\\\"status\\\"], s[\\\"reason\\\"] = \\\"failed\\\", \\\"solve() raised: \\\" + out.split(\\\"Error:\\\", 1)[1].strip()[:300]\\n        elif \\\"SOLVER_EMPTY\\\" in out:\\n            s[\\\"status\\\"], s[\\\"reason\\\"] = \\\"failed\\\", \\\"solve() returned [] (it could not decide)\\\"\\n        elif n == 0:\\n            s[\\\"status\\\"], s[\\\"reason\\\"] = \\\"failed\\\", \\\"solve() produced no executable action\\\"\\n        elif self.game_overs_this_level + self.resets > prev_go and self.level <= prev_level:\\n            s[\\\"status\\\"], s[\\\"reason\\\"] = \\\"failed\\\", \\\"its actions ran into a GAME OVER (gauge/action limit exhausted before the goal)\\\"\\n        if self.level > prev_level:\\n            s[\\\"noop_turns\\\"] = 0; s[\\\"no_progress_actions\\\"] = 0; s[\\\"board_seen\\\"] = {}\\n            return\\n        s[\\\"noop_turns\\\"] = 0 if changed else s[\\\"noop_turns\\\"] + 1\\n        s[\\\"no_progress_actions\\\"] += n\\n        key = masked_ascii(self.frame) if self.frame else \\\"\\\"\\n        s[\\\"board_seen\\\"][key] = s[\\\"board_seen\\\"].get(key, 0) + 1\\n        if s[\\\"status\\\"] == \\\"active\\\":\\n            if s[\\\"noop_turns\\\"] >= solver_policy.NOOP_TURN_LIMIT:\\n                s[\\\"status\\\"], s[\\\"reason\\\"] = \\\"failed\\\", f\\\"{s['noop_turns']} consecutive solver turns without any board change\\\"\\n            elif s[\\\"no_progress_actions\\\"] >= solver_policy.NO_PROGRESS_ACTIONS:\\n                s[\\\"status\\\"], s[\\\"reason\\\"] = \\\"failed\\\", f\\\"{s['no_progress_actions']} solver actions without completing the level\\\"\\n            elif s[\\\"board_seen\\\"][key] >= solver_policy.CYCLE_WINDOW:\\n                s[\\\"status\\\"], s[\\\"reason\\\"] = \\\"failed\\\", \\\"the solver keeps returning to the same board (cycle)\\\"\\n        if s[\\\"status\\\"] == \\\"failed\\\":\\n            self._log(f\\\"solver failed: {s['reason']}\\\")\\n            self.last_outcome = [f\\\"The stored solver ran {s['actions_run']} actions in total and was stopped: {s['reason']}.\\\"]\\n\\n    # ------------------------------------------------------------ main loop\\n    def _out_of_budget(self) -> Optional[str]:\\n        if self.state == \\\"WIN\\\":\\n            return \\\"won\\\"\\n        if self._seconds_left() <= 0:\\n            return \\\"time\\\"\\n        if self.actions_used >= self.max_actions:\\n            return \\\"actions\\\"\\n        if self.model_turns >= self.max_model_turns:\\n            return \\\"turns\\\"\\n        return None\\n\\n    def _maybe_autopilot(self) -> bool:\\n        \\\"\\\"\\\"Programmatic-first: when the induced rules show a mirrored second body and movement keys, let the\\n        cell-space planner try the level before spending model turns. Returns True if it acted.\\\"\\\"\\\"\\n        if not any(a in self.valid_actions for a in (\\\"UP\\\", \\\"DOWN\\\", \\\"LEFT\\\", \\\"RIGHT\\\")) or self.frame is None:\\n            return False\\n        if self.autopilot_tries.get(self.level, 0) >= 2:\\n            return False\\n        cur = [t for t in self.host_transitions if t.before_frame.level == t.after_frame.level == self.level]\\n        if len(cur) < 4:\\n            return False\\n        try:\\n            rules, _ = rule_induction.induce(cur, self.frame)\\n        except Exception:\\n            return False\\n        mirror = next((r for r in rules if r.kind == \\\"mirror\\\" and r.support >= 3), None)\\n        if not mirror:\\n            # single-avatar: act on a goal hypothesis inferred from an earlier level\\n            hyp = next((h for h in self.goal_hypotheses if h.get(\\\"type\\\") in (\\\"reach\\\", \\\"collect_reach\\\", \\\"collect_all\\\") and h.get(\\\"level\\\", 0) < self.level), None)\\n            if not hyp or not NavHelper(cur, self.frame).moves:\\n                return False\\n            self.autopilot_tries[self.level] = self.autopilot_tries.get(self.level, 0) + 1\\n            lvl = self.level\\n            res = run_reach(self, hyp, max_actions=60, log=lambda m: self._log(m[:200]))\\n            self._event(kind=\\\"autopilot\\\", level=lvl, result=res, hypothesis=hyp)\\n            if res[\\\"completed\\\"]:\\n                self.last_outcome = [f\\\"HARNESS AUTOPILOT completed level {lvl} in {res['actions']} actions by following the inferred goal ({hyp['text']}). \\\"\\n                                     \\\"Press each movement key once on the new level; it will try again.\\\"]\\n            else:\\n                self.last_outcome = [f\\\"HARNESS AUTOPILOT followed the inferred goal ({hyp['text']}) on level {lvl} and stopped: {res['reason']} after {res['actions']} actions. \\\"\\n                                     \\\"Something else is required first (a key, a switch, an item, a refill, an order): find it.\\\"]\\n            return True\\n        nav = NavHelper(cur, self.frame); floor = set(nav.floor_colors) | {nav.background}\\n        self.autopilot_tries[self.level] = self.autopilot_tries.get(self.level, 0) + 1\\n        lvl = self.level\\n        res = run_two_body(self, body_color=mirror.params[\\\"color\\\"], transform=mirror.params[\\\"how\\\"], floor_colors=floor,\\n                           max_replans=10, max_actions=80, log=lambda m: self._log(m[:200]))\\n        self._event(kind=\\\"autopilot\\\", level=lvl, result=res)\\n        if res[\\\"completed\\\"]:\\n            self.last_outcome = [f\\\"HARNESS AUTOPILOT completed level {lvl} in {res['actions']} actions using the confirmed two-body rules \\\"\\n                                 \\\"(mirrored movement, walls, hazards). Press each movement key once on the new level so the rules re-confirm; the autopilot will try again.\\\"]\\n        else:\\n            self.last_outcome = [f\\\"HARNESS AUTOPILOT tried the two-body plan on level {lvl} and stopped: {res['reason']} after {res['actions']} actions. \\\"\\n                                 \\\"A rule the planner does not know is in play (e.g. a gate or a selectable block): find it.\\\"]\\n        return True\\n\\n    def play(self) -> dict:\\n        self.reset()\\n        self.messages = [{\\\"role\\\": \\\"system\\\", \\\"content\\\": system_prompt()}]\\n        self._log(f\\\"start: {self.levels_total} levels, valid={self.valid_actions}\\\")\\n        idle = 0\\n        while not (why := self._out_of_budget()):\\n            if self.solver and self.solver.get(\\\"status\\\") == \\\"active\\\":\\n                self.solver_turn(); continue\\n            try:\\n                if self._maybe_autopilot():\\n                    continue\\n            except Exception as e:\\n                self._log(f\\\"autopilot error: {e!r}\\\")\\n            if self.client is None:\\n                why = \\\"no model\\\"; break\\n            ok = self.model_turn()\\n            idle = 0 if ok else idle + 1\\n            if idle >= 6:\\n                why = \\\"model produced no tool calls\\\"; break\\n        summary = {\\\"game_id\\\": self.game_id, \\\"state\\\": self.state, \\\"levels_completed\\\": self.level - 1 if self.state != \\\"WIN\\\" else self.levels_total,\\n                   \\\"levels_total\\\": self.levels_total, \\\"actions\\\": self.actions_used, \\\"resets\\\": self.resets, \\\"model_turns\\\": self.model_turns,\\n                   \\\"solver_turns\\\": self.solver_turns, \\\"solver_stored\\\": bool(self.solver and self.solver.get(\\\"code\\\")),\\n                   \\\"level_actions\\\": self.level_action_log, \\\"elapsed_s\\\": round(time.time() - self.t0, 1), \\\"stop_reason\\\": why,\\n                   \\\"prompt_tokens\\\": getattr(self.client, \\\"prompt_tokens\\\", 0), \\\"completion_tokens\\\": getattr(self.client, \\\"completion_tokens\\\", 0)}\\n        self._log(f\\\"end: {json.dumps(summary)}\\\")\\n        self.sandbox.close(); self.transcript.close(); self.events.close()\\n        return summary\\n\", \"autopilot.py\": \"\\\"\\\"\\\"Autopilot for two-body (mirrored) merge games in cell space: lattice from the body, plan on cells,\\nexecute step by step, verify against the prediction, refine passable colours, replan. No model calls.\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nfrom typing import Optional\\n\\nfrom .frame import texture_colors\\nfrom .nav import extract_objects\\nfrom .planner import lattice_from_body, cell_of, cell_colors, two_body_merge_cells\\n\\nTF = {\\\"same\\\": (1, 1), \\\"mirror_x\\\": (1, -1), \\\"mirror_y\\\": (-1, 1), \\\"mirror_xy\\\": (-1, -1)}\\nMOVES = {\\\"UP\\\": (-1, 0), \\\"DOWN\\\": (1, 0), \\\"LEFT\\\": (0, -1), \\\"RIGHT\\\": (0, 1)}\\n\\n\\ndef _bodies(grid, color, min_size=4):\\n    objs, _ = extract_objects(grid)\\n    return sorted([o for o in objs if o[\\\"color\\\"] == color and o[\\\"size\\\"] >= min_size\\n                   and o[\\\"bbox\\\"][2] - o[\\\"bbox\\\"][0] >= 1 and o[\\\"bbox\\\"][3] - o[\\\"bbox\\\"][1] >= 1],   # not a 1-px-thin HUD strip\\n                  key=lambda o: (o[\\\"center\\\"][1], o[\\\"center\\\"][0]))\\n\\n\\ndef run_two_body(session, *, body_color: int, transform: str, floor_colors: set, max_replans: int = 8, max_actions: int = 120, log=print) -> dict:\\n    level0 = session.level; start_actions = session.actions_used\\n    grid = session.frame.grid; tex = texture_colors(grid)\\n    passable = set(floor_colors) | {body_color}; hazards = set(tex); forbidden = set(); blocked_cells = set()\\n    replans = 0\\n    while replans <= max_replans and session.actions_used - start_actions < max_actions:\\n        grid = session.frame.grid; b = _bodies(grid, body_color)\\n        if len(b) != 2:\\n            return {\\\"completed\\\": False, \\\"actions\\\": session.actions_used - start_actions, \\\"replans\\\": replans, \\\"reason\\\": f\\\"{len(b)} bodies visible\\\"}\\n        rows, cols = lattice_from_body(b[0][\\\"bbox\\\"])\\n        cells = cell_colors(grid, rows, cols)\\n        A, B = cell_of(b[0][\\\"bbox\\\"], rows, cols), cell_of(b[1][\\\"bbox\\\"], rows, cols)\\n        plan = two_body_merge_cells(cells, A, B, transform, passable, hazards, forbidden=forbidden, blocked=blocked_cells)\\n        log(f\\\"autopilot: replan {replans}: lattice {len(rows)-1}x{len(cols)-1}, A{A} B{B}, passable={sorted(passable)}, hazards={sorted(hazards)}, forbidden={sorted(forbidden)}, plan={plan}\\\")\\n        if not plan:\\n            return {\\\"completed\\\": False, \\\"actions\\\": session.actions_used - start_actions, \\\"replans\\\": replans, \\\"reason\\\": \\\"no plan\\\"}\\n        replans += 1\\n        tf = TF[transform]\\n        for k, act in enumerate(plan):\\n            dr, dc = MOVES[act]\\n            def stp(pos, r_, c_):\\n                r, c = pos[0] + r_, pos[1] + c_\\n                if not (0 <= r < len(cells) and 0 <= c < len(cells[0])):\\n                    return pos\\n                if cells[r][c][1] & hazards:\\n                    return (r, c)\\n                if not cells[r][c][1] <= passable or (r, c) in blocked_cells:\\n                    return pos\\n                return (r, c)\\n            pa, pb = stp(A, dr, dc), stp(B, tf[0] * dr, tf[1] * dc)\\n            res = session.execute([{\\\"action\\\": act}])\\n            if res[\\\"level_completed\\\"] or session.level > level0:\\n                return {\\\"completed\\\": True, \\\"actions\\\": session.actions_used - start_actions, \\\"replans\\\": replans, \\\"reason\\\": \\\"level completed\\\"}\\n            if res[\\\"game_over\\\"]:\\n                return {\\\"completed\\\": False, \\\"actions\\\": session.actions_used - start_actions, \\\"replans\\\": replans, \\\"reason\\\": \\\"game over\\\"}\\n            grid = session.frame.grid; b2 = _bodies(grid, body_color)\\n            if len(b2) != 2:   # bodies touched and segment as one: finish the next planned steps blindly\\n                rest = plan[k + 1:][:3]\\n                if rest:\\n                    res = session.execute([{\\\"action\\\": x} for x in rest])\\n                    if res[\\\"level_completed\\\"] or session.level > level0:\\n                        return {\\\"completed\\\": True, \\\"actions\\\": session.actions_used - start_actions, \\\"replans\\\": replans, \\\"reason\\\": \\\"level completed\\\"}\\n                break\\n            cells = cell_colors(grid, rows, cols)\\n            c1, c2 = cell_of(b2[0][\\\"bbox\\\"], rows, cols), cell_of(b2[1][\\\"bbox\\\"], rows, cols)\\n            # identity: the bodies may cross; assign observed cells to predictions by total distance\\n            d = lambda x, y: abs(x[0] - y[0]) + abs(x[1] - y[1])\\n            A2, B2 = ((c1, c2) if d(c1, pa) + d(c2, pb) <= d(c2, pa) + d(c1, pb) else (c2, c1))\\n            if A2 == pa and B2 == pb:\\n                A, B = A2, B2\\n                continue\\n            log(f\\\"autopilot: divergence on {act}: predicted A{pa} B{pb}, actual A{A2} B{B2}\\\")\\n            jumped = [(pred, act_pos, body, d) for pred, act_pos, body, d in ((pa, A2, A, (dr, dc)), (pb, B2, B, (tf[0] * dr, tf[1] * dc)))\\n                      if act_pos != pred and act_pos != body and act_pos != (body[0] + d[0], body[1] + d[1])]\\n            if jumped:   # a reset: blame the body whose target cell shows hazard texture; if none does, blame both\\n                blamed = [(body, d) for _, _, body, d in jumped\\n                          if 0 <= body[0] + d[0] < len(cells) and 0 <= body[1] + d[1] < len(cells[0]) and (cells[body[0] + d[0]][body[1] + d[1]][1] & hazards)]\\n                if not blamed:\\n                    blamed = [(body, d) for _, _, body, d in jumped]\\n                for body, d in blamed:\\n                    forbidden.add((body[0] + d[0], body[1] + d[1]))\\n                log(f\\\"autopilot: reset; forbidding {sorted(forbidden)}\\\")\\n                break\\n            for pred, act_pos, body, (r_, c_) in ((pa, A2, A, (dr, dc)), (pb, B2, B, (tf[0] * dr, tf[1] * dc))):\\n                if act_pos == pred:\\n                    continue\\n                target = (body[0] + r_, body[1] + c_)\\n                in_grid = 0 <= target[0] < len(cells) and 0 <= target[1] < len(cells[0])\\n                if act_pos == body and in_grid:        # blocked where we predicted a move\\n                    extra = cells[target[0]][target[1]][1] - passable\\n                    if extra:\\n                        log(f\\\"autopilot: colours {sorted(extra)} block\\\")   # already implied by the whitelist; nothing to change\\n                    else:\\n                        blocked_cells.add(target); log(f\\\"autopilot: cell {target} blocks although it looks like floor\\\")\\n                elif act_pos == target and in_grid:    # moved where we predicted a block\\n                    passable |= cells[target[0]][target[1]][1]; blocked_cells.discard(target); log(f\\\"autopilot: colours {sorted(cells[target[0]][target[1]][1])} are passable\\\")\\n                elif in_grid:\\n                    forbidden.add(target); log(f\\\"autopilot: unexpected position after entering {target}; forbidding it\\\")\\n            break\\n    return {\\\"completed\\\": False, \\\"actions\\\": session.actions_used - start_actions, \\\"replans\\\": replans, \\\"reason\\\": \\\"budget exhausted\\\"}\\n\\n\\n\\ndef run_reach(session, hypothesis: dict, *, max_actions: int = 60, log=print) -> dict:\\n    \\\"\\\"\\\"Single-avatar autopilot for 'reach' / 'collect_reach' / 'collect_all' hypotheses: pick the next target from the\\n    hypothesis (collectibles nearest-first, then the goal colour), route with nav.path_to over known-walkable cells,\\n    execute at most 12 steps at a time, re-plan from the live board. Stops on level completion, game over, no path,\\n    or no movement. Never spends more than max_actions.\\\"\\\"\\\"\\n    from .nav import NavHelper\\n    level0 = session.level; start = session.actions_used; stalls = 0; visited_targets = set()\\n    kind = hypothesis.get(\\\"type\\\"); goal_c = hypothesis.get(\\\"reach\\\"); coll_c = hypothesis.get(\\\"collect\\\")\\n    while session.actions_used - start < max_actions and session.level == level0:\\n        cur = [t for t in session.host_transitions if t.before_frame.level == t.after_frame.level == session.level]\\n        nav = NavHelper(cur, session.frame)\\n        if not nav.moves or not nav.avatar():\\n            return {\\\"completed\\\": False, \\\"actions\\\": session.actions_used - start, \\\"reason\\\": \\\"movement not learned\\\"}\\n        targets = [t for t in nav.targets(max_n=20) if t.get(\\\"path_len\\\")]\\n        want = None\\n        if kind in (\\\"collect_reach\\\", \\\"collect_all\\\") and coll_c is not None:\\n            left = [t for t in targets if t[\\\"color\\\"] == coll_c and (t[\\\"row\\\"], t[\\\"col\\\"]) not in visited_targets]\\n            want = min(left, key=lambda t: t[\\\"path_len\\\"]) if left else None\\n        if want is None and kind in (\\\"reach\\\", \\\"collect_reach\\\") and goal_c is not None:\\n            goals = [t for t in targets if t[\\\"color\\\"] == goal_c]\\n            want = min(goals, key=lambda t: t[\\\"path_len\\\"]) if goals else None\\n        if want is None:\\n            return {\\\"completed\\\": False, \\\"actions\\\": session.actions_used - start, \\\"reason\\\": \\\"no reachable target for the hypothesis\\\"}\\n        path = nav.path_to(want[\\\"row\\\"], want[\\\"col\\\"])\\n        if not path:\\n            return {\\\"completed\\\": False, \\\"actions\\\": session.actions_used - start, \\\"reason\\\": \\\"no path\\\"}\\n        before = nav.avatar(); step = path[:12]\\n        log(f\\\"reach-autopilot: target colour {want['color']} at ({want['row']},{want['col']}), {len(path)} moves, executing {len(step)}\\\")\\n        res = session.execute([{\\\"action\\\": a} for a in step])\\n        if res[\\\"level_completed\\\"] or session.level > level0:\\n            return {\\\"completed\\\": True, \\\"actions\\\": session.actions_used - start, \\\"reason\\\": \\\"level completed\\\"}\\n        if res[\\\"game_over\\\"]:\\n            return {\\\"completed\\\": False, \\\"actions\\\": session.actions_used - start, \\\"reason\\\": \\\"game over\\\"}\\n        after = NavHelper([t for t in session.host_transitions if t.before_frame.level == t.after_frame.level == session.level], session.frame).avatar()\\n        if after and before and (after[\\\"row\\\"], after[\\\"col\\\"]) == (before[\\\"row\\\"], before[\\\"col\\\"]):\\n            stalls += 1\\n            if stalls >= 2:\\n                return {\\\"completed\\\": False, \\\"actions\\\": session.actions_used - start, \\\"reason\\\": \\\"avatar did not move (blocked target?)\\\"}\\n        else:\\n            stalls = 0\\n        if len(step) == len(path):\\n            visited_targets.add((want[\\\"row\\\"], want[\\\"col\\\"]))\\n    return {\\\"completed\\\": False, \\\"actions\\\": session.actions_used - start, \\\"reason\\\": \\\"budget exhausted\\\" if session.level == level0 else \\\"level completed\\\"}\\n\\n\\ndef run_click_sequence(session, hypothesis: dict, *, max_actions: int = 40, log=print) -> dict:\\n    \\\"\\\"\\\"Click-game autopilot for a 'click_sequence' hypothesis learned on an earlier level: click objects of the recorded\\n    colours in the same order on the new level (positions differ; the nearest object of that colour to the previous click\\n    is chosen). Stops on level completion, game over, a missing colour, or the action budget.\\\"\\\"\\\"\\n    level0 = session.level; start = session.actions_used\\n    seq = list(hypothesis.get(\\\"sequence\\\") or [])\\n    if not seq or \\\"MOUSE\\\" not in session.valid_actions:\\n        return {\\\"completed\\\": False, \\\"actions\\\": 0, \\\"reason\\\": \\\"not a click game / empty sequence\\\"}\\n    # compress the recorded sequence into runs: a run of >= 2 clicks on colour c generalises to \\\"click every colour-c object\\\"\\n    runs: list[list] = []\\n    for c in seq:\\n        if runs and runs[-1][0] == c:\\n            runs[-1][1] += 1\\n        else:\\n            runs.append([c, 1])\\n    last = None\\n    for color, count in runs:\\n        clicked_cells: set = set()\\n        todo = 64 if count >= 2 else 1\\n        for _ in range(todo):\\n            if session.actions_used - start >= max_actions or session.level != level0:\\n                break\\n            nodes = [n for n in session.frame.segmentation[\\\"nodes\\\"] if not n[\\\"hud\\\"] and n[\\\"color\\\"] == color\\n                     and (n[\\\"center\\\"][0] // 4, n[\\\"center\\\"][1] // 4) not in clicked_cells]\\n            if not nodes:\\n                if todo == 1:\\n                    return {\\\"completed\\\": False, \\\"actions\\\": session.actions_used - start, \\\"reason\\\": f\\\"no colour-{color} object on this level\\\"}\\n                break   # every colour-c object clicked: next run\\n            if last is not None:\\n                nodes.sort(key=lambda n: abs(n[\\\"center\\\"][0] - last[0]) + abs(n[\\\"center\\\"][1] - last[1]))\\n            else:\\n                nodes.sort(key=lambda n: n[\\\"pixels\\\"])\\n            n = nodes[0]; last = n[\\\"center\\\"]; clicked_cells.add((n[\\\"center\\\"][0] // 4, n[\\\"center\\\"][1] // 4))\\n            res = session.execute([{\\\"action\\\": \\\"MOUSE\\\", \\\"row\\\": n[\\\"center\\\"][0], \\\"col\\\": n[\\\"center\\\"][1]}])\\n            if res[\\\"level_completed\\\"] or session.level > level0:\\n                return {\\\"completed\\\": True, \\\"actions\\\": session.actions_used - start, \\\"reason\\\": \\\"level completed\\\"}\\n            if res[\\\"game_over\\\"]:\\n                return {\\\"completed\\\": False, \\\"actions\\\": session.actions_used - start, \\\"reason\\\": \\\"game over\\\"}\\n    return {\\\"completed\\\": False, \\\"actions\\\": session.actions_used - start, \\\"reason\\\": \\\"sequence replayed without completion\\\"}\\n\\n\\ndef run_goal_search(session, *, goal_colors: list, collect_colors: list, refill_colors: list, hazard_colors: set,\\n                    max_actions: int = 120, log=print, refill_amounts: Optional[dict] = None) -> dict:\\n    \\\"\\\"\\\"Generic movement-game level solver on induced knowledge (no game-specific code):\\n    1. go to the nearest goal-colour object (a door bumped earlier is still allowed as the final step);\\n    2. if arriving did not finish the level or no goal is reachable, collect the known collectible colours nearest-first\\n       (keys open doors), refuel at refill colours when the gauge is short, then retry the goal;\\n    3. stop on completion, game over, budget, or two rounds without any progress.\\\"\\\"\\\"\\n    from .nav import NavHelper\\n    level0 = session.level; start = session.actions_used\\n    visited: set = set(); stale_rounds = 0\\n    refill_amounts = dict(refill_amounts or {})\\n\\n    def nav_now():\\n        cur = [t for t in session.host_transitions if t.before_frame.level == t.after_frame.level == session.level]\\n        return NavHelper(cur, session.frame)\\n\\n    def go(nav, t, final_any=False) -> Optional[dict]:\\n        path = nav.path_to(t[\\\"row\\\"], t[\\\"col\\\"], ignore_wall_target=final_any)\\n        if not path:\\n            return None\\n        g = nav.gauge()\\n        if g and g.get(\\\"actions_left\\\") is not None and len(path) > int(g[\\\"actions_left\\\"]) - 1:\\n            return {\\\"short\\\": True}\\n        res = session.execute([{\\\"action\\\": a} for a in path[:24]])\\n        return res\\n\\n    def targets(nav, colors):\\n        return sorted([t for t in nav.targets(max_n=30) if t[\\\"color\\\"] in colors and (t[\\\"row\\\"], t[\\\"col\\\"]) not in visited],\\n                      key=lambda t: (t.get(\\\"path_len\\\") is None, t.get(\\\"path_len\\\") or 999))\\n\\n    def done(d):\\n        d[\\\"refill_colors\\\"] = list(refill_colors); d[\\\"refill_amounts\\\"] = dict(refill_amounts); return d\\n    while session.actions_used - start < max_actions and session.level == level0:\\n        nav = nav_now()\\n        if not nav.moves or not nav.avatar():\\n            return done({\\\"completed\\\": False, \\\"actions\\\": session.actions_used - start, \\\"reason\\\": \\\"movement not learned\\\"})\\n        progressed = False\\n        # refuel first when the gauge is nearly empty\\n        g = nav.gauge()\\n        need = min([t[\\\"path_len\\\"] for t in nav.targets(max_n=30) if t[\\\"color\\\"] in set(goal_colors) | set(collect_colors) and t.get(\\\"path_len\\\")] or [6])\\n        if g and g.get(\\\"actions_left\\\") is not None and int(g[\\\"actions_left\\\"]) <= max(6, need + 2) and refill_colors:\\n            for t in targets(nav, set(refill_colors))[:2]:\\n                r = go(nav, t)\\n                if r and not r.get(\\\"short\\\") and r[\\\"executed_count\\\"]:\\n                    visited.add((t[\\\"row\\\"], t[\\\"col\\\"])); progressed = True\\n                    log(f\\\"goal-search: refuel at colour {t['color']}\\\"); break\\n            if session.level != level0:\\n                break\\n            nav = nav_now()\\n        # 0. a tight gauge with unknown objects around: learn the nearest unknown first (it is often the refill this level adds)\\n        if g and g.get(\\\"actions_left\\\") is not None and int(g[\\\"actions_left\\\"]) < 2 * need and not refill_colors:\\n            known = set(goal_colors) | set(collect_colors) | set(hazard_colors)\\n            unknown = sorted([t for t in nav.targets(max_n=30) if t[\\\"color\\\"] not in known and t.get(\\\"path_len\\\") and (t[\\\"row\\\"], t[\\\"col\\\"]) not in visited],\\n                             key=lambda t: t[\\\"path_len\\\"])\\n            for t in unknown[:1]:\\n                r = go(nav, t)\\n                if r and not r.get(\\\"short\\\") and r[\\\"executed_count\\\"]:\\n                    visited.add((t[\\\"row\\\"], t[\\\"col\\\"])); progressed = True\\n                    g2 = nav_now().gauge()\\n                    log(f\\\"goal-search: tight gauge ({g['actions_left']} left, need {need}); probed unknown colour {t['color']} -> gauge {g.get('size')}->{g2 and g2.get('size')}\\\")\\n                    if session.level > level0:\\n                        return done({\\\"completed\\\": True, \\\"actions\\\": session.actions_used - start, \\\"reason\\\": \\\"probe completed the level\\\"})\\n                    if r[\\\"game_over\\\"]:\\n                        return done({\\\"completed\\\": False, \\\"actions\\\": session.actions_used - start, \\\"reason\\\": \\\"game over\\\"})\\n                    # measured refill: what the gauge has beyond the plain decrement over the walk, in actions\\n                    try:\\n                        per = abs(float(g.get(\\\"per_action\\\") or 1)); steps = r[\\\"executed_count\\\"]\\n                        gained_px = (g2.get(\\\"size\\\", 0) if g2 else 0) - (g.get(\\\"size\\\", 0) - steps * per)\\n                        gained = int(round(gained_px / per)) if per else 0\\n                        if gained >= 1 and t[\\\"color\\\"] not in refill_colors:\\n                            refill_colors.append(t[\\\"color\\\"]); refill_amounts[t[\\\"color\\\"]] = gained\\n                            log(f\\\"goal-search: colour {t['color']} refills the gauge (+{gained} actions, measured)\\\")\\n                    except Exception:\\n                        pass\\n                    continue\\n            if progressed:\\n                continue\\n        # 0b. resource-aware plan when a gauge is in play: refuel/collect/goal in one route (planner.plan_resources)\\n        if g and g.get(\\\"actions_left\\\") is not None and goal_colors:\\n            from .planner import plan_resources\\n            cells = lambda cols: {nav.block_of(t[\\\"row\\\"], t[\\\"col\\\"]) for t in nav.targets(max_n=40) if t[\\\"color\\\"] in cols}\\n            refill_amt = {c: a for c, a in (refill_amounts or {}).items()}\\n            rc = {nav.block_of(t[\\\"row\\\"], t[\\\"col\\\"]): int(refill_amt.get(t[\\\"color\\\"], 4)) for t in nav.targets(max_n=40) if t[\\\"color\\\"] in set(refill_colors)}\\n            plan = None\\n            for req in (True, False):\\n                plan = plan_resources(nav, goal_cells=cells(set(goal_colors)), collect_cells=cells(set(collect_colors)), refill_cells=rc,\\n                                      hazard_colors=set(hazard_colors), gauge_left=int(g[\\\"actions_left\\\"]), require_collect=req)\\n                if plan:\\n                    break\\n            if plan:\\n                log(f\\\"goal-search: resource plan of {len(plan)} moves (collect first: {req})\\\")\\n                res = session.execute([{\\\"action\\\": a} for a in plan[:24]])\\n                if session.level > level0:\\n                    return done({\\\"completed\\\": True, \\\"actions\\\": session.actions_used - start, \\\"reason\\\": \\\"resource plan reached the goal\\\"})\\n                if res[\\\"game_over\\\"]:\\n                    return done({\\\"completed\\\": False, \\\"actions\\\": session.actions_used - start, \\\"reason\\\": \\\"game over\\\"})\\n                if res[\\\"executed_count\\\"]:\\n                    progressed = True; continue\\n        # 1. goal\\n        for t in targets(nav, set(goal_colors))[:3]:\\n            r = go(nav, t, final_any=True)\\n            if r is None or r.get(\\\"short\\\"):\\n                continue\\n            log(f\\\"goal-search: went to goal colour {t['color']} at ({t['row']},{t['col']}) -> level {'done' if session.level > level0 else 'not done'}\\\")\\n            if session.level > level0:\\n                return done({\\\"completed\\\": True, \\\"actions\\\": session.actions_used - start, \\\"reason\\\": \\\"reached goal\\\"})\\n            if r[\\\"game_over\\\"]:\\n                return done({\\\"completed\\\": False, \\\"actions\\\": session.actions_used - start, \\\"reason\\\": \\\"game over\\\"})\\n            visited.add((t[\\\"row\\\"], t[\\\"col\\\"])); progressed = True\\n            break\\n        if session.level != level0:\\n            break\\n        # 2. prerequisites: collect\\n        nav = nav_now()\\n        for t in targets(nav, set(collect_colors))[:4]:\\n            r = go(nav, t)\\n            if r is None or r.get(\\\"short\\\"):\\n                continue\\n            log(f\\\"goal-search: collected colour {t['color']} at ({t['row']},{t['col']})\\\")\\n            visited.add((t[\\\"row\\\"], t[\\\"col\\\"])); progressed = True\\n            if session.level > level0:\\n                return done({\\\"completed\\\": True, \\\"actions\\\": session.actions_used - start, \\\"reason\\\": \\\"collected -> level done\\\"})\\n            if r[\\\"game_over\\\"]:\\n                return done({\\\"completed\\\": False, \\\"actions\\\": session.actions_used - start, \\\"reason\\\": \\\"game over\\\"})\\n            break\\n        if not progressed:\\n            # every known route was too long for the gauge or nothing known is reachable: learn what the nearest unknown object is\\n            # (new colours on this level first \\u2014 they are usually the new mechanic, e.g. refills); rules.py will classify the effect\\n            nav = nav_now(); known = set(goal_colors) | set(collect_colors) | set(refill_colors) | set(hazard_colors)\\n            unknown = [t for t in nav.targets(max_n=30) if t[\\\"color\\\"] not in known and t.get(\\\"path_len\\\") and (t[\\\"row\\\"], t[\\\"col\\\"]) not in visited]\\n            unknown.sort(key=lambda t: t[\\\"path_len\\\"])\\n            for t in unknown[:2]:\\n                r = go(nav, t)\\n                if r and not r.get(\\\"short\\\") and r[\\\"executed_count\\\"]:\\n                    visited.add((t[\\\"row\\\"], t[\\\"col\\\"])); progressed = True\\n                    log(f\\\"goal-search: probed unknown colour {t['color']} at ({t['row']},{t['col']})\\\")\\n                    if session.level > level0:\\n                        return done({\\\"completed\\\": True, \\\"actions\\\": session.actions_used - start, \\\"reason\\\": \\\"probe completed the level\\\"})\\n                    if r[\\\"game_over\\\"]:\\n                        return done({\\\"completed\\\": False, \\\"actions\\\": session.actions_used - start, \\\"reason\\\": \\\"game over\\\"})\\n                    g2 = nav_now().gauge()\\n                    if g and g2 and g2.get(\\\"size\\\", 0) > g.get(\\\"size\\\", 0) and t[\\\"color\\\"] not in refill_colors:\\n                        refill_colors.append(t[\\\"color\\\"]); log(f\\\"goal-search: colour {t['color']} refills the gauge\\\")\\n                    break\\n        if not progressed:\\n            stale_rounds += 1\\n            if stale_rounds >= 2:\\n                return done({\\\"completed\\\": False, \\\"actions\\\": session.actions_used - start, \\\"reason\\\": \\\"no reachable goal or collectible\\\"})\\n            visited.clear()   # allow a second pass over the same targets once (doors may have opened)\\n        else:\\n            stale_rounds = 0\\n    return done({\\\"completed\\\": session.level > level0, \\\"actions\\\": session.actions_used - start, \\\"reason\\\": \\\"budget\\\" if session.level == level0 else \\\"level completed\\\"})\\n\", \"effects.py\": \"\\\"\\\"\\\"Layer 1 for click games: a position-relative click-effect model, a goal predicate learned from the winning board, and a\\nsearch over click sequences on the learned model (executed with per-step verification).\\n\\nModel: for clicks on cells of colour c, each offset (dr, dc) within RADIUS carries a partial function before->after colour,\\nkept when it is consistent across the observed clicks. Far-away changes (HUD counters) are ignored by the simulator.\\nGoal predicate candidates come from the board just before the level-completing click: colours whose count dropped to zero,\\nplus the colour of the completing click (a 'submit' button).\\n\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nfrom collections import Counter, defaultdict\\nfrom typing import Optional\\n\\nRADIUS = 8\\n\\n\\nclass ClickModel:\\n    def __init__(self):\\n        self.obs: dict[tuple[int, int, int], Counter] = defaultdict(Counter)   # (click colour, dr, dc) -> Counter((before, after))\\n        self.clicks: Counter = Counter()                                      # click colour -> number of observed clicks\\n        self.far: Counter = Counter()                                         # click colour -> clicks with far (HUD) changes only\\n        self.rules: dict[int, dict[tuple[int, int], dict[int, int]]] = {}      # colour -> offset -> {before: after}\\n\\n    def observe(self, before, after, r0: int, c0: int) -> None:\\n        c = before[r0][c0]; self.clicks[c] += 1\\n        local = 0\\n        for r in range(max(0, r0 - RADIUS), min(64, r0 + RADIUS + 1)):\\n            for cc in range(max(0, c0 - RADIUS), min(64, c0 + RADIUS + 1)):\\n                if before[r][cc] != after[r][cc]:\\n                    self.obs[(c, r - r0, cc - c0)][(before[r][cc], after[r][cc])] += 1; local += 1\\n        if local == 0:\\n            self.far[c] += 1\\n\\n    def fit(self, min_support: float = 0.5) -> None:\\n        self.rules = {}\\n        for (c, dr, dc), cnt in self.obs.items():\\n            n = self.clicks[c]\\n            mapping: dict[int, int] = {}\\n            ok = True\\n            by_before: dict[int, Counter] = defaultdict(Counter)\\n            for (b, a), k in cnt.items():\\n                by_before[b][a] += k\\n            total = sum(cnt.values())\\n            for b, outs in by_before.items():\\n                a, k = outs.most_common(1)[0]\\n                if k < 0.7 * sum(outs.values()):\\n                    ok = False; break\\n                mapping[b] = a\\n            if ok and total >= max(1, min_support * n):\\n                self.rules.setdefault(c, {})[(dr, dc)] = mapping\\n\\n    def predict(self, board, r0: int, c0: int):\\n        c = board[r0][c0]\\n        rules = self.rules.get(c)\\n        if not rules:\\n            return None\\n        out = [row[:] for row in board]; changed = 0\\n        for (dr, dc), mapping in rules.items():\\n            r, cc = r0 + dr, c0 + dc\\n            if 0 <= r < 64 and 0 <= cc < 64 and board[r][cc] in mapping:\\n                out[r][cc] = mapping[board[r][cc]]; changed += 1\\n        return out if changed else None\\n\\n    def summary(self) -> str:\\n        rows = []\\n        for c, rules in sorted(self.rules.items()):\\n            maps = Counter()\\n            for m in rules.values():\\n                for b, a in m.items():\\n                    maps[(b, a)] += 1\\n            rows.append(f\\\"click colour {c}: {len(rules)} offsets change, transitions {dict(maps.most_common(4))} ({self.clicks[c]} clicks, {self.far[c]} HUD-only)\\\")\\n        return \\\"\\\\n\\\".join(rows) if rows else \\\"(no consistent click effects)\\\"\\n\\n\\ndef _counts(board, hud_rows: int = 2) -> Counter:\\n    cnt: Counter = Counter()\\n    for r in range(hud_rows, 64 - hud_rows):\\n        for v in board[r][hud_rows:64 - hud_rows]:\\n            cnt[v] += 1\\n    return cnt\\n\\n\\ndef learn_goal(attempt: list) -> dict:\\n    \\\"\\\"\\\"attempt: transitions of the winning attempt (last one completes the level). Returns predicate candidates.\\\"\\\"\\\"\\n    clicks = [t for t in attempt if isinstance(t.action, dict)]\\n    if not clicks:\\n        return {}\\n    final = clicks[-1]\\n    first = attempt[0].before_frame.grid; pre = final.before_frame.grid\\n    c0, c1 = _counts(first), _counts(pre)\\n    zero = sorted(c for c, n in c0.items() if n >= 3 and c1.get(c, 0) == 0)\\n    submit_colour = pre[final.action[\\\"row\\\"]][final.action[\\\"col\\\"]]\\n    return {\\\"zero_colours\\\": zero, \\\"submit_colour\\\": submit_colour, \\\"pre_counts\\\": dict(c1)}\\n\\n\\ndef satisfied(board, goal: dict) -> bool:\\n    if not goal or not goal.get(\\\"zero_colours\\\"):\\n        return False\\n    cnt = _counts(board)\\n    return all(cnt.get(c, 0) == 0 for c in goal[\\\"zero_colours\\\"])\\n\\n\\ndef plan_clicks(model: ClickModel, board, goal: dict, candidates: list[tuple[int, int]], max_depth: int = 6, max_nodes: int = 20000) -> Optional[list[tuple[int, int]]]:\\n    \\\"\\\"\\\"BFS over click sequences (candidate positions) on the learned model until the goal predicate holds.\\\"\\\"\\\"\\n    from collections import deque\\n    key = lambda b: hash(tuple(tuple(r) for r in b))\\n    start = [row[:] for row in board]\\n    if satisfied(start, goal):\\n        return []\\n    # monotone removal goals ('every colour-k cell gone'): greedy \\u2014 click each candidate whose predicted effect removes colour-k cells\\n    zero = set(goal.get(\\\"zero_colours\\\") or [])\\n    if zero:\\n        b = start; plan = []\\n        for _ in range(60):\\n            cnt = _counts(b); remaining = sum(cnt.get(c, 0) for c in zero)\\n            if remaining == 0:\\n                return plan\\n            best = None\\n            for (r, c) in candidates:\\n                nb = model.predict(b, r, c)\\n                if nb is None:\\n                    continue\\n                rem = sum(_counts(nb).get(cc, 0) for cc in zero)\\n                if rem < remaining and (best is None or rem < best[0]):\\n                    best = (rem, (r, c), nb)\\n            if best is None:\\n                break\\n            plan.append(best[1]); b = best[2]\\n        if plan and satisfied(b, goal):\\n            return plan\\n    q = deque([(start, [])]); seen = {key(start)}; n = 0\\n    while q and n < max_nodes:\\n        b, path = q.popleft(); n += 1\\n        if len(path) >= max_depth:\\n            continue\\n        for (r, c) in candidates:\\n            nb = model.predict(b, r, c)\\n            if nb is None:\\n                continue\\n            k = key(nb)\\n            if k in seen:\\n                continue\\n            seen.add(k)\\n            if satisfied(nb, goal):\\n                return path + [(r, c)]\\n            q.append((nb, path + [(r, c)]))\\n    return None\\n\\n\\ndef run_click_model(session, model: ClickModel, goal: dict, *, max_actions: int = 60, log=print) -> dict:\\n    \\\"\\\"\\\"Level solver for click games: plan on the model, execute click by click, re-fit the model on surprises and re-plan;\\n    when the predicate holds, press the submit colour (if any). Bounded by max_actions.\\\"\\\"\\\"\\n    level0 = session.level; start = session.actions_used; replans = 0\\n    while session.actions_used - start < max_actions and session.level == level0 and replans < 6:\\n        board = session.frame.grid\\n        nodes = [n for n in session.frame.segmentation[\\\"nodes\\\"] if not n[\\\"hud\\\"] and n[\\\"color\\\"] in model.rules]\\n        candidates = [(n[\\\"center\\\"][0], n[\\\"center\\\"][1]) for n in nodes][:40]\\n        if satisfied(board, goal):\\n            sub = goal.get(\\\"submit_colour\\\")\\n            btn = [n for n in session.frame.segmentation[\\\"nodes\\\"] if not n[\\\"hud\\\"] and n[\\\"color\\\"] == sub]\\n            if btn:\\n                res = session.execute([{\\\"action\\\": \\\"MOUSE\\\", \\\"row\\\": btn[0][\\\"center\\\"][0], \\\"col\\\": btn[0][\\\"center\\\"][1]}])\\n                return {\\\"completed\\\": session.level > level0, \\\"actions\\\": session.actions_used - start, \\\"reason\\\": \\\"predicate met, submit pressed\\\"}\\n            return {\\\"completed\\\": False, \\\"actions\\\": session.actions_used - start, \\\"reason\\\": \\\"predicate met but no submit button\\\"}\\n        plan = plan_clicks(model, board, goal, candidates)\\n        if not plan:\\n            return {\\\"completed\\\": False, \\\"actions\\\": session.actions_used - start, \\\"reason\\\": f\\\"no plan on the model ({len(candidates)} candidates)\\\"}\\n        log(f\\\"click-model: plan of {len(plan)} clicks\\\")\\n        for (r, c) in plan:\\n            predicted = model.predict(session.frame.grid, r, c)\\n            before = session.frame.grid\\n            res = session.execute([{\\\"action\\\": \\\"MOUSE\\\", \\\"row\\\": r, \\\"col\\\": c}])\\n            if session.level > level0:\\n                return {\\\"completed\\\": True, \\\"actions\\\": session.actions_used - start, \\\"reason\\\": \\\"level completed during the plan\\\"}\\n            if res[\\\"game_over\\\"]:\\n                return {\\\"completed\\\": False, \\\"actions\\\": session.actions_used - start, \\\"reason\\\": \\\"game over\\\"}\\n            model.observe(before, session.frame.grid, r, c)\\n            if predicted is None or predicted != session.frame.grid:\\n                model.fit(); replans += 1\\n                log(\\\"click-model: surprise -> re-fit and re-plan\\\")\\n                break\\n    return {\\\"completed\\\": session.level > level0, \\\"actions\\\": session.actions_used - start, \\\"reason\\\": \\\"budget/replans exhausted\\\"}\\n\", \"explorer.py\": \"\\\"\\\"\\\"Layer 0: model-free graph exploration of one game level.\\n\\nThe environment is deterministic and cheap; the model is slow and expensive. This module therefore explores\\nwithout any model: every visited board (HUD masked) is a node, every macro action tried from it is an edge,\\nand the next thing to try is always the highest-priority macro that has not been tried at the current node.\\nWhen the current node has nothing left, the explorer walks (over known edges) to the nearest node that has,\\nand when nothing reachable is left it resets the level (the start node is always known).\\n\\nMacro actions (all generic, no game knowledge):\\n  goto+interact   walk to an unvisited target object and press the interaction key there (SPACE / ACTION7)\\n  interact        press an interaction key where the avatar stands\\n  click           click an object (one per 4x4 cell; smallest objects and one colour per round first)\\n  frontier        walk to the nearest unexplored walkable block\\n  key / key-run   press a movement key once / until the board stops changing (max 8)\\n\\nThe result of a level is the macro path that completed it (feeds goals.py / rules.py) plus the graph itself.\\n\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport time\\nfrom collections import deque\\nfrom dataclasses import dataclass, field\\nfrom typing import Optional\\n\\nfrom .frame import masked_ascii, summarize_diff\\nfrom .nav import NavHelper\\n\\nINTERACT_KEYS = (\\\"SPACE\\\", \\\"ACTION7\\\")\\nMOVE_KEYS = (\\\"UP\\\", \\\"DOWN\\\", \\\"LEFT\\\", \\\"RIGHT\\\")\\nCHUNK = 24   # GameSession.execute caps one call at MAX_ACTIONS_PER_CALL\\n\\n\\n@dataclass\\nclass Macro:\\n    kind: str                 # goto+interact | interact | click | frontier | key | key-run\\n    actions: list[dict]       # concrete engine actions\\n    label: str\\n    priority: int             # lower runs first\\n\\n    def key(self) -> str:\\n        return self.label\\n\\n\\n@dataclass\\nclass Node:\\n    state: str\\n    level: int\\n    tried: set = field(default_factory=set)          # macro labels tried here\\n    edges: dict = field(default_factory=dict)        # macro label -> (actions, result state | \\\"GAME_OVER\\\" | \\\"LEVEL\\\")\\n    macros_cache: Optional[list] = None\\n\\n\\nclass Explorer:\\n    def __init__(self, session, *, log=print, max_actions_per_level: int = 300, max_seconds: float = 300.0):\\n        self.s = session\\n        self.log = log\\n        self.max_actions_per_level = max_actions_per_level\\n        self.max_seconds = max_seconds\\n        self.nodes: dict[str, Node] = {}\\n        self.hazard_labels: set[str] = set()     # (state, label) pairs that ended in game over\\n        self.solutions: dict[int, list[str]] = {}\\n        self.trace: list[str] = []\\n        self.label_hist: dict[tuple[int, str], list[int]] = {}   # (level, label) -> [tried, produced-same-state]\\n        self.frontier_actions: dict[int, int] = {}\\n        self.inv_tried: set = set()   # (level, inventory signature, label): a macro is tried once per world state, not once per avatar position\\n        self.effect_stats: dict[tuple[str, int], list[int]] = {}   # (kind, colour) -> [tries, effective]: which kinds of touches did anything, across levels\\n\\n    def inv_sig(self) -> str:\\n        \\\"\\\"\\\"World inventory: non-HUD objects except the avatar's own colours. Moving the avatar does not change it; collecting,\\n        toggling or opening something does. Novelty is judged against this, so 'goto(target)' is retried only after the world changed.\\\"\\\"\\\"\\n        nav = self._nav(); skip_cols = set()\\n        try:\\n            av = nav.avatar() if nav else None\\n            skip_cols |= set(av[\\\"colors\\\"]) if av else set()\\n            g = nav.gauge() if nav else None\\n            if g:\\n                skip_cols.add(g[\\\"color\\\"])\\n        except Exception:\\n            pass\\n        items = []\\n        for n in self.s.frame.segmentation[\\\"nodes\\\"]:\\n            if n[\\\"hud\\\"] or n[\\\"color\\\"] in skip_cols:\\n                continue\\n            r0, c0, r1, c1 = n[\\\"bbox\\\"]\\n            thin = (r1 - r0 <= 2) or (c1 - c0 <= 2)\\n            if thin and (r0 <= 1 or r1 >= 62 or c0 <= 1 or c1 >= 62):\\n                continue   # gauge / counter pieces along the frame edge (their emptied part is not flagged as HUD)\\n            items.append((n[\\\"color\\\"], n[\\\"pixels\\\"], (r0, c0, r1, c1)))\\n        return f\\\"L{self.s.level}:\\\" + str(hash(tuple(sorted(items))))\\n\\n    KIND_RANK = {\\\"goto+interact\\\": 0, \\\"goto\\\": 0, \\\"interact\\\": 1, \\\"click\\\": 2, \\\"frontier\\\": 3, \\\"key\\\": 4, \\\"key-run\\\": 5}\\n    KIND_RANK_MOVEMENT = {\\\"goto+interact\\\": 0, \\\"goto\\\": 0, \\\"interact\\\": 1, \\\"frontier\\\": 2, \\\"key\\\": 3, \\\"key-run\\\": 4, \\\"click\\\": 6}\\n\\n    def goal_colors(self) -> dict[int, int]:\\n        \\\"\\\"\\\"Colours that mattered in earlier levels' wins (from goals.py hypotheses), with a rank: the goal colour first,\\n        collectibles / clicked colours next. Soft transfer: they are tried first on the new level, nothing is replayed blindly.\\\"\\\"\\\"\\n        out: dict[int, int] = {}\\n        for h in self.s.goal_hypotheses:\\n            if h.get(\\\"level\\\", 0) >= self.s.level:\\n                continue\\n            for c in ([h.get(\\\"reach\\\")] if h.get(\\\"reach\\\") is not None else []):\\n                out.setdefault(int(c), 0)\\n            for c in ([h.get(\\\"collect\\\")] if h.get(\\\"collect\\\") is not None else []):\\n                out.setdefault(int(c), 1)\\n            for i, c in enumerate(dict.fromkeys(h.get(\\\"sequence\\\") or [])):\\n                out.setdefault(int(c), 2 + i)\\n        return out\\n\\n    def _macro_color(self, m: Macro) -> Optional[int]:\\n        import re\\n        mt = re.match(r\\\"(?:goto|click)\\\\((\\\\d+)@\\\", m.label)\\n        return int(mt.group(1)) if mt else None\\n\\n    def _rank(self, level: int, m: Macro) -> tuple:\\n        \\\"\\\"\\\"Colours that won earlier levels first, then goal-directed kinds (after the world changed, re-trying a target is worth\\n        more than mapping another corridor), then never-tried labels before tried ones, dead labels (same state twice) last.\\\"\\\"\\\"\\n        h = self.label_hist.get((level, m.label), [0, 0])\\n        ranks = self.KIND_RANK_MOVEMENT if self._movement_game() else self.KIND_RANK\\n        dead = h[1] >= (1 if m.kind == \\\"click\\\" else 2)   # a click that changed nothing at all (HUD included) once is a dead button\\n        gc = self.goal_colors(); c = self._macro_color(m)\\n        goal_rank = gc.get(c, 99) if c is not None else 99\\n        eff = self._effect_ratio(m)   # colours whose touches did something on earlier levels first; known no-ops last\\n        return (1 if dead else 0, 0 if goal_rank < 99 else 1, goal_rank, ranks.get(m.kind, 9), 1 if eff < 0.2 else 0, -round(eff, 1), 1 if h[0] > 0 else 0, m.priority)\\n\\n    def _movement_game(self) -> bool:\\n        nav = self._nav()\\n        return bool(nav and nav.moves and any(a in self.s.valid_actions for a in MOVE_KEYS))\\n\\n    # \\u2500\\u2500 state \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\n    def state(self) -> str:\\n        return f\\\"L{self.s.level}:\\\" + str(hash(masked_ascii(self.s.frame)))\\n\\n    def node(self) -> Node:\\n        st = self.state()\\n        n = self.nodes.get(st)\\n        if n is None:\\n            n = Node(st, self.s.level); self.nodes[st] = n\\n        return n\\n\\n    # \\u2500\\u2500 macro generation \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\n    def _nav(self) -> Optional[NavHelper]:\\n        key = (len(self.s.host_transitions), self.s.attempt_start_index, id(self.s.frame))\\n        cached = getattr(self, \\\"_nav_cache\\\", None)\\n        if cached and cached[0] == key:\\n            return cached[1]\\n        cur = [t for t in self.s.host_transitions[self.s.attempt_start_index:]\\n               if t.before_frame.level == t.after_frame.level == self.s.level]\\n        try:\\n            nav = NavHelper(cur, self.s.frame)\\n        except Exception:\\n            nav = None\\n        self._nav_cache = (key, nav)\\n        return nav\\n\\n    def macros(self, node: Node) -> list[Macro]:\\n        valid = list(self.s.valid_actions)\\n        inter = [a for a in INTERACT_KEYS if a in valid]\\n        keys = [a for a in MOVE_KEYS if a in valid]\\n        out: list[Macro] = []\\n        nav = self._nav()\\n        cur = [t for t in self.s.host_transitions[self.s.attempt_start_index:] if t.after_frame.level == self.s.level]\\n        if nav and keys and nav.moves and nav.avatar():\\n            # targets: unvisited first, nearest first\\n            tg = [t for t in nav.targets(max_n=14) if t.get(\\\"path_len\\\") is not None]\\n            tg.sort(key=lambda t: (bool(t.get(\\\"visited\\\")), t[\\\"path_len\\\"]))\\n            for i, t in enumerate(tg[:8]):\\n                path = nav.path_to(t[\\\"row\\\"], t[\\\"col\\\"])\\n                if path is None or len(path) > 30:\\n                    continue\\n                acts = [{\\\"action\\\": a} for a in path]\\n                if inter:\\n                    out.append(Macro(\\\"goto+interact\\\", acts + [{\\\"action\\\": inter[0]}],\\n                                     f\\\"goto({t['color']}@{t['row']},{t['col']})+{inter[0]}\\\", 10 + i + (40 if t.get(\\\"visited\\\") else 0)))\\n                else:\\n                    out.append(Macro(\\\"goto\\\", acts, f\\\"goto({t['color']}@{t['row']},{t['col']})\\\", 10 + i + (40 if t.get(\\\"visited\\\") else 0)))\\n            fr = nav.frontier()\\n            if fr is not None:\\n                path = nav.path_to(fr[0], fr[1])\\n                if path:\\n                    out.append(Macro(\\\"frontier\\\", [{\\\"action\\\": a} for a in path], f\\\"frontier({fr[0]},{fr[1]})\\\", 30))\\n        for a in inter:\\n            out.append(Macro(\\\"interact\\\", [{\\\"action\\\": a}], f\\\"{a}\\\", 20))\\n        if \\\"MOUSE\\\" in valid and self.s.frame is not None:\\n            clicked = {(t.action[\\\"row\\\"] // 4, t.action[\\\"col\\\"] // 4) for t in cur if isinstance(t.action, dict)}\\n            nodes = []\\n            for n in self.s.frame.segmentation[\\\"nodes\\\"]:\\n                if n[\\\"hud\\\"]:\\n                    continue\\n                r0, c0, r1, c1 = n[\\\"bbox\\\"]\\n                thin = (r1 - r0 <= 2) or (c1 - c0 <= 2)\\n                if thin and (r0 <= 1 or r1 >= 62 or c0 <= 1 or c1 >= 62):\\n                    continue   # edge strips (gauge / counters) are never click targets\\n                if n[\\\"pixels\\\"] <= 2 and keys:\\n                    continue   # 1-2 px specks in a movement game are marks, not buttons\\n                nodes.append(n)\\n            nodes.sort(key=lambda n: n[\\\"pixels\\\"])\\n            click_base = 25 if not (keys and nav and nav.moves) else 70   # in a movement game clicking is the last resort\\n            seen_colors: set = set(); seen_cells: set = set()\\n            for j, n in enumerate(nodes[:60]):\\n                cell = (n[\\\"center\\\"][0] // 4, n[\\\"center\\\"][1] // 4)\\n                if cell in seen_cells:\\n                    continue\\n                seen_cells.add(cell)\\n                first_of_color = n[\\\"color\\\"] not in seen_colors\\n                seen_colors.add(n[\\\"color\\\"])\\n                pr = click_base + (0 if first_of_color else 10) + (20 if cell in clicked else 0) + j // 8\\n                out.append(Macro(\\\"click\\\", [{\\\"action\\\": \\\"MOUSE\\\", \\\"row\\\": n[\\\"center\\\"][0], \\\"col\\\": n[\\\"center\\\"][1]}],\\n                                 f\\\"click({n['color']}@{n['center'][0]},{n['center'][1]})\\\", pr))\\n        for a in keys:\\n            out.append(Macro(\\\"key\\\", [{\\\"action\\\": a}], f\\\"{a}\\\", 50))\\n        for a in keys:\\n            out.append(Macro(\\\"key-run\\\", [{\\\"action\\\": a}] * 8, f\\\"{a}x8\\\", 60))\\n        out.sort(key=lambda m: m.priority)\\n        return out\\n\\n    # \\u2500\\u2500 execution \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\n    def _execute(self, actions: list[dict]) -> dict:\\n        \\\"\\\"\\\"Execute in chunks; stop at level completion / game over / invalid action.\\\"\\\"\\\"\\n        res_all = {\\\"executed_count\\\": 0, \\\"board_changed\\\": False, \\\"level_completed\\\": False, \\\"game_over\\\": False, \\\"stopped_reason\\\": None}\\n        for i in range(0, len(actions), CHUNK):\\n            r = self.s.execute(actions[i:i + CHUNK])\\n            res_all[\\\"executed_count\\\"] += r[\\\"executed_count\\\"]; res_all[\\\"board_changed\\\"] |= r[\\\"board_changed\\\"]\\n            res_all[\\\"level_completed\\\"] |= r[\\\"level_completed\\\"]; res_all[\\\"game_over\\\"] |= r[\\\"game_over\\\"]\\n            res_all[\\\"stopped_reason\\\"] = r[\\\"stopped_reason\\\"]\\n            if r[\\\"level_completed\\\"] or r[\\\"game_over\\\"] or (r[\\\"stopped_reason\\\"] and \\\"only the first\\\" not in r[\\\"stopped_reason\\\"]):\\n                break\\n            if r[\\\"executed_count\\\"] == 0:\\n                break\\n        return res_all\\n\\n    def _effect_ratio(self, m: Macro) -> float:\\n        c = self._macro_color(m)\\n        if c is None:\\n            return 0.5\\n        st = self.effect_stats.get((m.kind, c))\\n        return 0.5 if not st or st[0] == 0 else st[1] / st[0]\\n\\n    def _run_macro(self, node: Node, m: Macro) -> dict:\\n        actions = m.actions\\n        full_before = self.s.frame.ascii if self.s.frame is not None else None\\n        masked_before = masked_ascii(self.s.frame) if self.s.frame is not None else None\\n        if m.kind == \\\"key-run\\\":   # press until the board stops changing\\n            done = 0; last = None\\n            for _ in range(8):\\n                before = masked_ascii(self.s.frame)\\n                r = self.s.execute([actions[0]]); done += r[\\\"executed_count\\\"]\\n                last = r\\n                if r[\\\"level_completed\\\"] or r[\\\"game_over\\\"] or masked_ascii(self.s.frame) == before or r[\\\"executed_count\\\"] == 0:\\n                    break\\n            r = dict(last or {\\\"executed_count\\\": 0, \\\"board_changed\\\": False, \\\"level_completed\\\": False, \\\"game_over\\\": False, \\\"stopped_reason\\\": None})\\n            r[\\\"executed_count\\\"] = done\\n        else:\\n            r = self._execute(actions)\\n        node.tried.add(m.label)\\n        result = \\\"LEVEL\\\" if r[\\\"level_completed\\\"] else \\\"GAME_OVER\\\" if r[\\\"game_over\\\"] else self.state()\\n        node.edges[m.label] = (actions, result)\\n        h = self.label_hist.setdefault((node.level, m.label), [0, 0]); h[0] += 1\\n        # effect grade: 2 = the masked world changed, 1 = only the HUD/counter changed (progress can hide there: vc33 pumps), 0 = nothing\\n        full_changed = self.s.frame is not None and full_before is not None and self.s.frame.ascii != full_before\\n        world_changed = self.s.frame is not None and masked_before is not None and masked_ascii(self.s.frame) != masked_before\\n        grade = 2 if (r[\\\"level_completed\\\"] or world_changed) else 1 if full_changed else 0\\n        c = self._macro_color(m)\\n        if c is not None:\\n            st = self.effect_stats.setdefault((m.kind, c), [0, 0]); st[0] += 2; st[1] += grade   # ratio in [0, 1]\\n        if grade == 0:\\n            h[1] += 1   # nothing at all changed: a dead button\\n        if r[\\\"game_over\\\"]:\\n            self.hazard_labels.add((node.state, m.label))\\n        self.trace.append(f\\\"{m.label} -> {result if result in ('LEVEL', 'GAME_OVER') else ('same' if result == node.state else 'new' if result not in self.nodes else 'known')}\\\")\\n        return r\\n\\n    # \\u2500\\u2500 search for something untried \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\n    def _plan_to_untried(self) -> Optional[list[tuple[str, list[dict], str]]]:\\n        \\\"\\\"\\\"BFS over known edges to the nearest node that still has an untried macro. Returns the edge path.\\\"\\\"\\\"\\n        start = self.state()\\n        prev: dict[str, tuple[Optional[str], Optional[str]]] = {start: (None, None)}\\n        q = deque([start])\\n        while q:\\n            st = q.popleft()\\n            n = self.nodes.get(st)\\n            if n is None:\\n                continue\\n            if st != start:\\n                ms = self.macros(n) if st == self.state() else None   # macros depend on the live frame: only evaluable at the live node\\n                # for non-live nodes fall back to the cached list generated when we were there\\n                ms = ms if ms is not None else (n.macros_cache or [])\\n                if any(m.label not in n.tried for m in ms):\\n                    path = []; cur = st\\n                    while prev[cur][0] is not None:\\n                        p, lab = prev[cur]; path.append((p, self.nodes[p].edges[lab][0], lab)); cur = p\\n                    return list(reversed(path))\\n            for lab, (acts, res) in n.edges.items():\\n                if res in (\\\"LEVEL\\\", \\\"GAME_OVER\\\") or res in prev:\\n                    continue\\n                prev[res] = (st, lab); q.append(res)\\n        return None\\n\\n    # \\u2500\\u2500 main loop \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\n    def play_level(self) -> dict:\\n        level = self.s.level\\n        t0 = time.time(); a0 = self.s.actions_used; macros_run = 0; resets = 0\\n        path_labels: list[str] = []\\n        while self.s.level == level and self.s.state != \\\"WIN\\\":\\n            if self.s.actions_used - a0 >= self.max_actions_per_level or time.time() - t0 > self.max_seconds:\\n                return {\\\"level\\\": level, \\\"completed\\\": False, \\\"actions\\\": self.s.actions_used - a0, \\\"macros\\\": macros_run, \\\"resets\\\": resets, \\\"reason\\\": \\\"budget\\\"}\\n            node = self.node()\\n            ms = self.macros(node); node.macros_cache = ms\\n            inv = self.inv_sig()   # world inventory (no avatar, gauge or edge strips): empirically better than the raw board for click games too (r11l, vc33)\\n            untried = [m for m in ms if m.label not in node.tried and (node.state, m.label) not in self.hazard_labels\\n                       and (level, inv, m.label) not in self.inv_tried]\\n            if self.frontier_actions.get(level, 0) > 0.4 * self.max_actions_per_level:\\n                untried = [m for m in untried if m.kind != \\\"frontier\\\"] or untried\\n            if untried:\\n                m = min(untried, key=lambda m: self._rank(level, m))   # never-tried-on-this-level labels first, dead labels last\\n                self.inv_tried.add((level, inv, m.label))\\n                r = self._run_macro(node, m); macros_run += 1\\n                if m.kind == \\\"frontier\\\":\\n                    self.frontier_actions[level] = self.frontier_actions.get(level, 0) + r[\\\"executed_count\\\"]\\n                if r[\\\"level_completed\\\"]:\\n                    path_labels.append(m.label)\\n                    self.solutions[level] = path_labels\\n                    return {\\\"level\\\": level, \\\"completed\\\": True, \\\"actions\\\": self.s.actions_used - a0, \\\"macros\\\": macros_run, \\\"resets\\\": resets, \\\"path\\\": path_labels}\\n                if r[\\\"game_over\\\"]:\\n                    path_labels = []; resets += 1\\n                    continue\\n                if r[\\\"executed_count\\\"] == 0:\\n                    node.tried.add(m.label)   # invalid here: never again at this node\\n                    continue\\n                path_labels.append(m.label)\\n                continue\\n            plan = self._plan_to_untried()\\n            if plan:\\n                ok = True\\n                for st, acts, lab in plan:\\n                    if self.state() != st:\\n                        ok = False; break\\n                    r = self._execute(acts)\\n                    path_labels.append(lab)\\n                    if r[\\\"level_completed\\\"]:\\n                        self.solutions[level] = path_labels\\n                        return {\\\"level\\\": level, \\\"completed\\\": True, \\\"actions\\\": self.s.actions_used - a0, \\\"macros\\\": macros_run, \\\"resets\\\": resets, \\\"path\\\": path_labels}\\n                    if r[\\\"game_over\\\"]:\\n                        ok = False; path_labels = []; resets += 1; break\\n                if not ok:\\n                    self.trace.append(\\\"replay diverged\\\")\\n                continue\\n            # nothing reachable is untried: reset the level and continue from the start node (its edges are known)\\n            if resets >= 3:\\n                return {\\\"level\\\": level, \\\"completed\\\": False, \\\"actions\\\": self.s.actions_used - a0, \\\"macros\\\": macros_run, \\\"resets\\\": resets, \\\"reason\\\": \\\"exhausted\\\"}\\n            self.s.reset(); resets += 1; path_labels = []\\n            self.s.attempt_start_index = len(self.s.host_transitions)\\n            self.trace.append(\\\"reset (nothing untried reachable)\\\")\\n            start = self.node()\\n            if all(m.label in start.tried for m in self.macros(start)):\\n                return {\\\"level\\\": level, \\\"completed\\\": False, \\\"actions\\\": self.s.actions_used - a0, \\\"macros\\\": macros_run, \\\"resets\\\": resets, \\\"reason\\\": \\\"exhausted\\\"}\\n        return {\\\"level\\\": level, \\\"completed\\\": self.s.level > level, \\\"actions\\\": self.s.actions_used - a0, \\\"macros\\\": macros_run, \\\"resets\\\": resets, \\\"path\\\": path_labels}\\n\\n    def _layer1(self) -> Optional[dict]:\\n        \\\"\\\"\\\"Before exploring a new level, replay what won the previous one (goal hypotheses from goals.py): reach/collect\\n        via the nav planner, click games via the recorded colour sequence. Cheap, deterministic, and it is exactly the\\n        knowledge transfer the model never managed.\\\"\\\"\\\"\\n        from . import autopilot\\n        level0 = self.s.level; a0 = self.s.actions_used\\n        # rule-based autopilots (two-body merge when a mirrored body is confirmed) need a few observed moves first\\n        keys = [a for a in MOVE_KEYS if a in self.s.valid_actions]\\n        if keys and not [t for t in self.s.host_transitions[self.s.attempt_start_index:] if t.after_frame.level == level0]:\\n            self._execute([{\\\"action\\\": a} for a in keys])   # one probe per key: learns controls and feeds rule induction\\n            if self.s.level > level0:\\n                return {\\\"level\\\": level0, \\\"completed\\\": True, \\\"actions\\\": self.s.actions_used - a0, \\\"macros\\\": 0, \\\"resets\\\": 0, \\\"path\\\": [\\\"probe\\\"]}\\n        saved_hyps = self.s.goal_hypotheses\\n        try:\\n            self.s.goal_hypotheses = []   # only the rule-based (two-body) branch here; goal hypotheses go through the gauge-aware goal search below\\n            for _ in range(2):\\n                if self.s._maybe_autopilot() and self.s.level > level0:\\n                    self.trace.append(\\\"layer1 rules-autopilot completed the level\\\")\\n                    return {\\\"level\\\": level0, \\\"completed\\\": True, \\\"actions\\\": self.s.actions_used - a0, \\\"macros\\\": 0, \\\"resets\\\": 0, \\\"path\\\": [\\\"layer1:rules-autopilot\\\"]}\\n        except Exception as e:\\n            self.trace.append(f\\\"layer1 autopilot error {type(e).__name__}\\\")\\n        finally:\\n            self.s.goal_hypotheses = saved_hyps\\n        hyps = [h for h in self.s.goal_hypotheses if h.get(\\\"level\\\", 0) < self.s.level]\\n        if keys and self.s.level > 1:\\n            # generic movement solver on induced knowledge from earlier levels: goal colours + collectibles + refills + hazards\\n            from .rules import induce\\n            goal_c, coll_c, refill_c, hazard_c = [], [], [], set()\\n            for h in hyps:\\n                if h.get(\\\"reach\\\") is not None and h[\\\"reach\\\"] not in goal_c:\\n                    goal_c.append(h[\\\"reach\\\"])\\n                if h.get(\\\"collect\\\") is not None and h[\\\"collect\\\"] not in coll_c:\\n                    coll_c.append(h[\\\"collect\\\"])\\n            prev = [t for t in self.s.host_transitions if t.before_frame.level == t.after_frame.level < self.s.level]\\n            try:\\n                rules, _ = induce(prev[-400:], prev[-1].after_frame) if prev else ([], [])\\n                for r_ in rules:\\n                    if r_.kind == \\\"collect\\\" and r_.params[\\\"color\\\"] not in coll_c:\\n                        coll_c.append(r_.params[\\\"color\\\"])\\n                    if r_.kind == \\\"refill\\\" and r_.params[\\\"color\\\"] not in refill_c:\\n                        refill_c.append(r_.params[\\\"color\\\"])\\n                    if r_.kind == \\\"hazard\\\":\\n                        hazard_c.add(r_.params[\\\"color\\\"])\\n            except Exception as e:\\n                self.trace.append(f\\\"layer1 induce error {type(e).__name__}\\\")\\n            if goal_c or coll_c:\\n                amounts: dict = {}\\n                for attempt in range(2):   # attempt 1 learns (refills, doors); a game over resets the level and attempt 2 replans with what was learned\\n                    r = autopilot.run_goal_search(self.s, goal_colors=goal_c, collect_colors=coll_c, refill_colors=refill_c, hazard_colors=hazard_c,\\n                                                  max_actions=120, log=lambda m: self.trace.append(m), refill_amounts=amounts)\\n                    self.trace.append(f\\\"layer1 goal-search #{attempt + 1}(goals={goal_c}, collect={coll_c}, refill={refill_c}): {r.get('reason')} ({r.get('actions')} actions)\\\")\\n                    if r.get(\\\"completed\\\") or self.s.level > level0:\\n                        return {\\\"level\\\": level0, \\\"completed\\\": True, \\\"actions\\\": self.s.actions_used - a0, \\\"macros\\\": 0, \\\"resets\\\": 0, \\\"path\\\": [\\\"layer1:goal-search\\\"]}\\n                    refill_c = r.get(\\\"refill_colors\\\", refill_c); amounts = r.get(\\\"refill_amounts\\\", amounts)\\n                    if attempt == 0 and (r.get(\\\"reason\\\") == \\\"game over\\\" or not r.get(\\\"refill_colors\\\")):\\n                        if r.get(\\\"reason\\\") != \\\"game over\\\":\\n                            break   # nothing new was learned: a second pass would repeat the first\\n                        self.s.attempt_start_index = len(self.s.host_transitions)\\n                        self._execute([{\\\"action\\\": a} for a in keys])   # relearn controls on the fresh attempt\\n        if not hyps:\\n            return None\\n        for h in hyps[:3]:\\n            try:\\n                if h.get(\\\"type\\\") in (\\\"reach\\\", \\\"collect_reach\\\", \\\"collect_all\\\"):\\n                    r = autopilot.run_reach(self.s, h, max_actions=80, log=lambda m: None)\\n                elif h.get(\\\"type\\\") == \\\"click_sequence\\\":\\n                    r = autopilot.run_click_sequence(self.s, h, max_actions=40, log=lambda m: None)\\n                else:\\n                    continue\\n            except Exception as e:\\n                self.trace.append(f\\\"layer1 error {type(e).__name__}\\\"); continue\\n            self.trace.append(f\\\"layer1 {h.get('type')}: {r.get('reason')} ({r.get('actions')} actions)\\\")\\n            if r.get(\\\"completed\\\") or self.s.level > level0:\\n                return {\\\"level\\\": level0, \\\"completed\\\": True, \\\"actions\\\": self.s.actions_used - a0, \\\"macros\\\": 0, \\\"resets\\\": 0, \\\"path\\\": [f\\\"layer1:{h.get('type')}\\\"]}\\n        return None\\n\\n    def play(self, max_levels: int = 3, client=None, advisor_rounds: int = 6) -> list[dict]:\\n        out = []\\n        for _ in range(max_levels):\\n            if self.s.state == \\\"WIN\\\":\\n                break\\n            r = self._layer1()\\n            if r is None and client is not None and self.s.level > 1:\\n                from .advisor import advise_level\\n                level0 = self.s.level; a0 = self.s.actions_used\\n                adv = advise_level(self, client, rounds=advisor_rounds, log=self.log)\\n                if adv.get(\\\"completed\\\"):\\n                    r = {\\\"level\\\": level0, \\\"completed\\\": True, \\\"actions\\\": self.s.actions_used - a0, \\\"macros\\\": 0, \\\"resets\\\": 0, \\\"path\\\": [f\\\"advisor:{adv.get('hypothesis', '')[:60]}\\\"]}\\n            if r is None:\\n                r = self.play_level()\\n            out.append(r)\\n            self.log(f\\\"[explorer] level {r['level']}: {'COMPLETED' if r['completed'] else 'not completed'} in {r['actions']} actions, {r['macros']} macros, {r['resets']} resets\\\" + (f\\\" \\u2014 {r.get('reason')}\\\" if not r[\\\"completed\\\"] else f\\\" \\u2014 path {r.get('path')}\\\"))\\n            if not r[\\\"completed\\\"]:\\n                break\\n        return out\\n\", \"frame.py\": \"\\\"\\\"\\\"Observation encoding: grid -> ascii, connected-component segmentation,\\nobject hashes, containment and adjacency. Pure python, no dependencies.\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nfrom collections import Counter\\nfrom dataclasses import dataclass, field\\nfrom typing import Any, Optional\\n\\n# 16 ARC colours as single characters (hex digits keep the mapping obvious)\\nCOLOR_CHARS = \\\"0123456789abcdef\\\"\\nCOLOR_NAMES = {0: \\\"black\\\", 1: \\\"blue\\\", 2: \\\"red\\\", 3: \\\"green\\\", 4: \\\"yellow\\\", 5: \\\"grey\\\", 6: \\\"magenta\\\",\\n               7: \\\"orange\\\", 8: \\\"sky\\\", 9: \\\"brown\\\", 10: \\\"white\\\", 11: \\\"purple\\\", 12: \\\"cyan\\\",\\n               13: \\\"lime\\\", 14: \\\"pink\\\", 15: \\\"teal\\\"}\\n\\n\\ndef grid_to_ascii(grid: list[list[int]]) -> str:\\n    return \\\"\\\\n\\\".join(\\\"\\\".join(COLOR_CHARS[v & 15] for v in row) for row in grid)\\n\\n\\ndef _shape_hash(cells: list[tuple[int, int]], color: int) -> str:\\n    y0 = min(r for r, _ in cells); x0 = min(c for _, c in cells)\\n    norm = tuple(sorted((r - y0, c - x0) for r, c in cells))\\n    return f\\\"{color:x}:{abs(hash(norm)) % 10**8:08d}\\\"\\n\\n\\n@dataclass\\nclass Frame:\\n    grid: list[list[int]]\\n    step: int = 0\\n    level: int = 1\\n    _ascii: Optional[str] = field(default=None, repr=False)\\n    _seg: Optional[dict] = field(default=None, repr=False)\\n\\n    @property\\n    def shape(self) -> tuple[int, int]:\\n        return (len(self.grid), len(self.grid[0]) if self.grid else 0)\\n\\n    @property\\n    def ascii(self) -> str:\\n        if self._ascii is None:\\n            self._ascii = grid_to_ascii(self.grid)\\n        return self._ascii\\n\\n    @property\\n    def background(self) -> int:\\n        return Counter(v for row in self.grid for v in row).most_common(1)[0][0]\\n\\n    @property\\n    def segmentation(self) -> dict:\\n        \\\"\\\"\\\"{'nodes': [...], 'adjacency': [[i, j], ...], 'background': color}.\\n        Node: id, color, pixels, bbox (r0, c0, r1, c1), center (row, col), hash,\\n        children (ids fully inside this node's bbox), hud (edge strip heuristic).\\\"\\\"\\\"\\n        if self._seg is None:\\n            self._seg = segment(self.grid)\\n        return self._seg\\n\\n    def to_payload(self) -> dict:\\n        return {\\\"grid\\\": self.grid, \\\"step\\\": self.step, \\\"level\\\": self.level}\\n\\n    @staticmethod\\n    def from_payload(p: dict) -> \\\"Frame\\\":\\n        return Frame(grid=[list(map(int, r)) for r in p[\\\"grid\\\"]], step=int(p.get(\\\"step\\\", 0)), level=int(p.get(\\\"level\\\", 1)))\\n\\n\\ndef segment(grid: list[list[int]], max_nodes: int = 60) -> dict:\\n    h, w = len(grid), len(grid[0])\\n    bg = Counter(v for row in grid for v in row).most_common(1)[0][0]\\n    comp = [[-1] * w for _ in range(h)]\\n    nodes: list[dict] = []\\n    for sr in range(h):\\n        for sc in range(w):\\n            if comp[sr][sc] >= 0 or grid[sr][sc] == bg:\\n                continue\\n            color = grid[sr][sc]; nid = len(nodes)\\n            stack, cells = [(sr, sc)], []\\n            comp[sr][sc] = nid\\n            while stack:\\n                r, c = stack.pop(); cells.append((r, c))\\n                for nr, nc in ((r - 1, c), (r + 1, c), (r, c - 1), (r, c + 1)):\\n                    if 0 <= nr < h and 0 <= nc < w and comp[nr][nc] < 0 and grid[nr][nc] == color:\\n                        comp[nr][nc] = nid; stack.append((nr, nc))\\n            rs = [r for r, _ in cells]; cs = [c for _, c in cells]\\n            r0, r1, c0, c1 = min(rs), max(rs), min(cs), max(cs)\\n            hud = ((r1 - r0 <= 2 and c1 - c0 >= 24 and (r0 <= 1 or r1 >= h - 2)) or\\n                   (c1 - c0 <= 2 and r1 - r0 >= 24 and (c0 <= 1 or c1 >= w - 2)))\\n            nodes.append({\\\"id\\\": nid, \\\"color\\\": color, \\\"pixels\\\": len(cells), \\\"bbox\\\": (r0, c0, r1, c1),\\n                          \\\"center\\\": ((r0 + r1) // 2, (c0 + c1) // 2), \\\"hash\\\": _shape_hash(cells, color),\\n                          \\\"children\\\": [], \\\"hud\\\": hud})\\n    # adjacency (edge-sharing components, ignoring background)\\n    adj: set[tuple[int, int]] = set()\\n    for r in range(h):\\n        for c in range(w):\\n            a = comp[r][c]\\n            if a < 0:\\n                continue\\n            for nr, nc in ((r + 1, c), (r, c + 1)):\\n                if nr < h and nc < w:\\n                    b = comp[nr][nc]\\n                    if b >= 0 and b != a:\\n                        adj.add((min(a, b), max(a, b)))\\n    # containment by bbox (small inside larger)\\n    for n in nodes:\\n        r0, c0, r1, c1 = n[\\\"bbox\\\"]\\n        for m in nodes:\\n            if m is n:\\n                continue\\n            mr0, mc0, mr1, mc1 = m[\\\"bbox\\\"]\\n            if r0 < mr0 and c0 < mc0 and mr1 < r1 and mc1 < c1:\\n                n[\\\"children\\\"].append(m[\\\"id\\\"])\\n    keep = sorted(nodes, key=lambda n: -n[\\\"pixels\\\"])[:max_nodes]\\n    keep_ids = {n[\\\"id\\\"] for n in keep}\\n    return {\\\"nodes\\\": sorted(keep, key=lambda n: n[\\\"id\\\"]),\\n            \\\"adjacency\\\": sorted([list(p) for p in adj if p[0] in keep_ids and p[1] in keep_ids]),\\n            \\\"background\\\": bg}\\n\\n\\ndef summarize_diff(before: Frame, after: Frame, max_items: int = 6) -> dict:\\n    \\\"\\\"\\\"Compact description of what changed between two frames.\\\"\\\"\\\"\\n    changed = [(r, c) for r in range(len(before.grid)) for c in range(len(before.grid[0]))\\n               if before.grid[r][c] != after.grid[r][c]]\\n    out: dict[str, Any] = {\\\"changed_cells\\\": len(changed)}\\n    if not changed:\\n        return out\\n    rs = [r for r, _ in changed]; cs = [c for _, c in changed]\\n    out[\\\"bbox\\\"] = (min(rs), min(cs), max(rs), max(cs))\\n    bs = {n[\\\"hash\\\"]: n for n in before.segmentation[\\\"nodes\\\"]}\\n    as_ = {n[\\\"hash\\\"]: n for n in after.segmentation[\\\"nodes\\\"]}\\n    moved, appeared, gone = [], [], []\\n    for hsh, n in as_.items():\\n        if hsh in bs:\\n            b = bs[hsh]\\n            if b[\\\"center\\\"] != n[\\\"center\\\"]:\\n                moved.append({\\\"color\\\": n[\\\"color\\\"], \\\"pixels\\\": n[\\\"pixels\\\"], \\\"from\\\": b[\\\"center\\\"], \\\"to\\\": n[\\\"center\\\"]})\\n        else:\\n            appeared.append({\\\"color\\\": n[\\\"color\\\"], \\\"pixels\\\": n[\\\"pixels\\\"], \\\"center\\\": n[\\\"center\\\"]})\\n    for hsh, b in bs.items():\\n        if hsh not in as_:\\n            gone.append({\\\"color\\\": b[\\\"color\\\"], \\\"pixels\\\": b[\\\"pixels\\\"], \\\"center\\\": b[\\\"center\\\"]})\\n    out[\\\"moved\\\"] = moved[:max_items]; out[\\\"appeared\\\"] = appeared[:max_items]; out[\\\"disappeared\\\"] = gone[:max_items]\\n    return out\\n\\n\\ndef masked_ascii(frame: \\\"Frame\\\") -> str:\\n    \\\"\\\"\\\"Board text with HUD strips (edge counters/gauges) blanked out, for change and cycle detection.\\\"\\\"\\\"\\n    rows = [list(r) for r in frame.ascii.splitlines()]\\n    for n in frame.segmentation[\\\"nodes\\\"]:\\n        if n.get(\\\"hud\\\"):\\n            r0, c0, r1, c1 = n[\\\"bbox\\\"]\\n            for r in range(r0, r1 + 1):\\n                for c in range(c0, c1 + 1):\\n                    rows[r][c] = \\\".\\\"\\n    return \\\"\\\\n\\\".join(\\\"\\\".join(r) for r in rows)\\n\\n\\ndef infer_cell_grid(grid: list[list[int]], min_n: int = 6, max_n: int = 32, tolerance: float = 0.06):\\n    \\\"\\\"\\\"Recover the game's logical N x N cell grid from a 64 x 64 nearest-neighbour render.\\n    Returns (n, cells) with cells[r][c] = the dominant colour of block (r, c), or None if no N fits.\\\"\\\"\\\"\\n    h = len(grid)\\n    best = None\\n    for n in range(min_n, max_n + 1):\\n        bounds = [int(i * h / n) for i in range(n + 1)]\\n        bad = 0; cells = []\\n        for r in range(n):\\n            row = []\\n            for c in range(n):\\n                vals = Counter(grid[i][j] for i in range(bounds[r], bounds[r + 1]) for j in range(bounds[c], bounds[c + 1]))\\n                colour, cnt = vals.most_common(1)[0]\\n                total = sum(vals.values())\\n                if cnt < total:\\n                    bad += 1\\n                row.append(colour)\\n            cells.append(row)\\n        if bad <= tolerance * n * n:\\n            return n, cells\\n    return None\\n\\n\\ndef cell_bounds(n: int, size: int = 64) -> list[int]:\\n    return [int(i * size / n) for i in range(n + 1)]\\n\\n\\ndef cell_lattice(grid: list[list[int]], hud_rows: set | None = None, hud_cols: set | None = None, min_votes: int = 2):\\n    \\\"\\\"\\\"Recover the cell lattice of a rendered game grid from colour-change positions.\\n    Returns (row_bounds, col_bounds): boundary indices (a cell spans [b[k], b[k+1])). Works for any scaling.\\\"\\\"\\\"\\n    h, w = len(grid), len(grid[0])\\n    hud_rows = hud_rows or set(); hud_cols = hud_cols or set()\\n    col_votes = [0] * (w + 1); row_votes = [0] * (h + 1)\\n    for i in range(h):\\n        if i in hud_rows:\\n            continue\\n        for j in range(1, w):\\n            if grid[i][j] != grid[i][j - 1]:\\n                col_votes[j] += 1\\n    for j in range(w):\\n        if j in hud_cols:\\n            continue\\n        for i in range(1, h):\\n            if grid[i][j] != grid[i - 1][j]:\\n                row_votes[i] += 1\\n    col_b = [0] + [j for j in range(1, w) if col_votes[j] >= min_votes] + [w]\\n    row_b = [0] + [i for i in range(1, h) if row_votes[i] >= min_votes] + [h]\\n    return row_b, col_b\\n\\n\\ndef to_cells(grid: list[list[int]], row_b: list[int], col_b: list[int]) -> list[list[int]]:\\n    \\\"\\\"\\\"Dominant colour per lattice cell.\\\"\\\"\\\"\\n    out = []\\n    for r in range(len(row_b) - 1):\\n        row = []\\n        for c in range(len(col_b) - 1):\\n            vals = Counter(grid[i][j] for i in range(row_b[r], row_b[r + 1]) for j in range(col_b[c], col_b[c + 1]))\\n            row.append(vals.most_common(1)[0][0])\\n        out.append(row)\\n    return out\\n\\n\\ndef hud_lines(frame) -> tuple[set, set]:\\n    rows, cols = set(), set()\\n    for n in frame.segmentation[\\\"nodes\\\"]:\\n        if n[\\\"hud\\\"]:\\n            r0, c0, r1, c1 = n[\\\"bbox\\\"]\\n            if r1 - r0 <= 2:\\n                rows |= set(range(r0, r1 + 1))\\n            if c1 - c0 <= 2:\\n                cols |= set(range(c0, c1 + 1))\\n    return rows, cols\\n\\n\\ndef texture_colors(grid: list[list[int]]) -> set:\\n    \\\"\\\"\\\"Colours that occur mostly as isolated 1-px runs (checkerboard textures inside cells).\\\"\\\"\\\"\\n    stats: dict = {}\\n    for row in grid:\\n        prev = row[0]; k = 0\\n        for v in row + [None]:\\n            if v == prev:\\n                k += 1\\n            else:\\n                s = stats.setdefault(prev, [0, 0]); s[0] += 1; s[1] += (k == 1)\\n                prev = v; k = 1\\n    return {c for c, (n, ones) in stats.items() if n >= 8 and ones / n > 0.8}\\n\\n\\ndef find_lattice(grid: list[list[int]], ignore: set | None = None, n_range=(6, 24), origins=(0, 1, 2, 3, 4)):\\n    \\\"\\\"\\\"Find (n, origin, bounds) such that colour-change positions (ignoring texture colours) fall on the lattice\\n    round(origin + k*(64-2*origin)/n). Returns (coverage, n, origin, bounds) or None.\\\"\\\"\\\"\\n    ignore = ignore or set()\\n    h, w = len(grid), len(grid[0])\\n    col_changes: Counter = Counter(); row_changes: Counter = Counter()\\n    for i in range(h):\\n        for j in range(1, w):\\n            if grid[i][j] != grid[i][j - 1] and grid[i][j] not in ignore and grid[i][j - 1] not in ignore:\\n                col_changes[j] += 1\\n    for j in range(w):\\n        for i in range(1, h):\\n            if grid[i][j] != grid[i - 1][j] and grid[i][j] not in ignore and grid[i - 1][j] not in ignore:\\n                row_changes[i] += 1\\n    total = sum(col_changes.values()) + sum(row_changes.values())\\n    if not total:\\n        return None\\n    best = None\\n    for n in range(n_range[0], n_range[1] + 1):\\n        for o in origins:\\n            step = (w - 2 * o) / n\\n            bounds = [round(o + k * step) for k in range(n + 1)]\\n            bs = set(bounds)\\n            hit = sum(v for j, v in col_changes.items() if j in bs) + sum(v for i, v in row_changes.items() if i in bs)\\n            cov = hit / total\\n            if best is None or cov > best[0] + 1e-9 or (abs(cov - best[0]) < 1e-9 and n < best[1]):\\n                best = (cov, n, o, bounds)\\n    return best\\n\\n\\ndef lattice_cells(grid: list[list[int]], bounds: list[int]) -> list[list[Counter]]:\\n    \\\"\\\"\\\"Colour histogram per lattice cell (cells outside the lattice margin are ignored).\\\"\\\"\\\"\\n    out = []\\n    for r in range(len(bounds) - 1):\\n        row = []\\n        for c in range(len(bounds) - 1):\\n            row.append(Counter(grid[i][j] for i in range(bounds[r], bounds[r + 1]) for j in range(bounds[c], bounds[c + 1])))\\n        out.append(row)\\n    return out\\n\\n\\n# ARC colour palette (index -> RGB), used for the optional image observation\\nPALETTE = [(0, 0, 0), (0, 116, 217), (255, 65, 54), (46, 204, 64), (255, 220, 0), (170, 170, 170), (240, 18, 190), (255, 133, 27),\\n           (127, 219, 255), (135, 12, 37), (255, 255, 255), (177, 13, 201), (0, 200, 200), (140, 240, 60), (255, 160, 180), (0, 140, 120)]\\n\\n\\ndef grid_to_png_b64(grid: list[list[int]], scale: int = 6, gridlines: bool = True) -> str:\\n    \\\"\\\"\\\"Render the board as a PNG (base64) with a thin grid every cell so the model can count cells.\\\"\\\"\\\"\\n    import base64, io\\n    from PIL import Image, ImageDraw\\n    h, w = len(grid), len(grid[0])\\n    img = Image.new(\\\"RGB\\\", (w * scale, h * scale))\\n    d = ImageDraw.Draw(img)\\n    for r in range(h):\\n        for c in range(w):\\n            d.rectangle([c * scale, r * scale, (c + 1) * scale - 1, (r + 1) * scale - 1], fill=PALETTE[grid[r][c] & 15])\\n    if gridlines and scale >= 4:\\n        for k in range(0, w * scale, scale * 8):\\n            d.line([(k, 0), (k, h * scale - 1)], fill=(60, 60, 60), width=1)\\n        for k in range(0, h * scale, scale * 8):\\n            d.line([(0, k), (w * scale - 1, k)], fill=(60, 60, 60), width=1)\\n    buf = io.BytesIO(); img.save(buf, format=\\\"PNG\\\")\\n    return base64.b64encode(buf.getvalue()).decode()\\n\", \"goals.py\": \"\\\"\\\"\\\"Win-condition inference from a completed level (harness side, no model).\\n\\nGiven the transitions of the winning attempt (same level, ending with the transition that entered the next\\nlevel), produce ranked goal hypotheses that planners can act on in later levels:\\n  reach:          the final move entered a cell of colour G (a goal tile / door)\\n  collect_reach:  all objects of colour C disappeared during the attempt, then the avatar reached G\\n  collect_all:    the last object of colour C disappeared on the final action (no separate goal tile)\\n  click_sequence: (click games) the ordered colours clicked in the winning attempt\\n\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nfrom collections import Counter\\nfrom typing import Any\\n\\nfrom .nav import NavHelper, extract_objects\\n\\n\\ndef _objs(frame):\\n    return extract_objects(frame.grid)[0]\\n\\n\\ndef infer(attempt: list, level: int, movement_keys: bool = True) -> list[dict]:\\n    \\\"\\\"\\\"attempt: transitions of the winning attempt on `level` (last one enters level+1).\\\"\\\"\\\"\\n    if not attempt:\\n        return []\\n    same = [t for t in attempt if t.before_frame.level == level]\\n    if not same:\\n        return []\\n    first, last = same[0].before_frame, same[-1].before_frame   # last board seen on this level\\n    final = same[-1]\\n    hyps: list[dict] = []\\n    # colours that vanished completely during the attempt (collectibles), ignoring the avatar's own colours\\n    nav = NavHelper(same[:-1] or same, last)\\n    av = nav.avatar(); avatar_cols = set(av[\\\"colors\\\"]) if av else set()\\n    for (c, _), _ in nav.player_votes.most_common(4):\\n        avatar_cols.add(c)\\n    bg = nav.background\\n    def counts(frame):\\n        cnt = Counter()\\n        for o in _objs(frame):\\n            if o[\\\"size\\\"] <= 200 and o[\\\"color\\\"] != bg and o[\\\"color\\\"] not in avatar_cols and o[\\\"color\\\"] not in nav.floor_colors:\\n                cnt[o[\\\"color\\\"]] += 1\\n        return cnt\\n    c0, c1 = counts(first), counts(last)\\n    vanished = {c: n for c, n in c0.items() if n >= 1 and c1.get(c, 0) == 0}\\n    # what did the final move enter? (movement games)\\n    action = final.action if isinstance(final.action, str) else \\\"MOUSE\\\"\\n    entered = None\\n    if action in nav.moves and av:\\n        dx, dy = nav.moves[action]\\n        r, c = av[\\\"row\\\"] + dy, av[\\\"col\\\"] + dx\\n        if 0 <= r < 64 and 0 <= c < 64:\\n            col = last.grid[r][c]\\n            if col != bg and col not in nav.floor_colors and col not in avatar_cols:\\n                entered = col\\n    if entered is not None:\\n        if vanished:\\n            for c, n in sorted(vanished.items(), key=lambda x: -x[1]):\\n                hyps.append({\\\"type\\\": \\\"collect_reach\\\", \\\"collect\\\": c, \\\"collect_count\\\": n, \\\"reach\\\": entered,\\n                             \\\"text\\\": f\\\"collect all colour-{c} objects ({n} on level {level}) and then step onto a colour-{entered} object\\\"})\\n        hyps.append({\\\"type\\\": \\\"reach\\\", \\\"reach\\\": entered, \\\"text\\\": f\\\"step onto a colour-{entered} object (the final move of level {level} entered one)\\\"})\\n    elif vanished:\\n        for c, n in sorted(vanished.items(), key=lambda x: -x[1]):\\n            hyps.append({\\\"type\\\": \\\"collect_all\\\", \\\"collect\\\": c, \\\"collect_count\\\": n, \\\"text\\\": f\\\"make every colour-{c} object disappear ({n} on level {level}); the level ended when the last one vanished\\\"})\\n    # click games: the winning click order by colour\\n    clicks = [t.action for t in same if isinstance(t.action, dict)]\\n    if clicks and not movement_keys:\\n        seq = []\\n        for t in same:\\n            if isinstance(t.action, dict):\\n                r, c = t.action[\\\"row\\\"], t.action[\\\"col\\\"]; seq.append(t.before_frame.grid[r][c])\\n        hyps.append({\\\"type\\\": \\\"click_sequence\\\", \\\"sequence\\\": seq[-12:], \\\"text\\\": f\\\"level {level} was won by clicking colours in this order: {seq[-12:]}\\\"})\\n    # two bodies of the avatar colour became one on the final move (merge games)\\n    if not hyps and avatar_cols:\\n        def n_bodies(frame):\\n            return sum(1 for o in _objs(frame) if o[\\\"color\\\"] in avatar_cols and o[\\\"size\\\"] >= 4)\\n        bodies_last = [o for o in _objs(last) if o[\\\"color\\\"] in avatar_cols and o[\\\"size\\\"] >= 4]\\n        adjacent = len(bodies_last) == 2 and abs(bodies_last[0][\\\"center\\\"][0] - bodies_last[1][\\\"center\\\"][0]) + abs(bodies_last[0][\\\"center\\\"][1] - bodies_last[1][\\\"center\\\"][1]) <= 12\\n        if n_bodies(first) >= 2 and (len(bodies_last) <= 1 or adjacent):\\n            hyps.append({\\\"type\\\": \\\"merge\\\", \\\"colors\\\": sorted(avatar_cols), \\\"text\\\": f\\\"bring the two colour-{sorted(avatar_cols)} bodies onto the same cell (they merged on the final move of level {level})\\\"})\\n    if not hyps:\\n        hyps.append({\\\"type\\\": \\\"unknown\\\", \\\"text\\\": f\\\"level {level} ended after {action}; no clear goal pattern (avatar colours {sorted(avatar_cols)}, vanished {dict(vanished)})\\\"})\\n    for h in hyps:\\n        h[\\\"level\\\"] = level\\n    return hyps\\n\\n\\ndef summary(hyps: list[dict]) -> str:\\n    if not hyps:\\n        return \\\"\\\"\\n    return \\\"GOAL HYPOTHESES (inferred by the harness from how earlier levels were won; the same rule usually applies):\\\\n\\\" + \\\\\\n        \\\"\\\\n\\\".join(f\\\"  {i + 1}. {h['text']}\\\" for i, h in enumerate(hyps[:3]))\\n\", \"llm.py\": \"\\\"\\\"\\\"Minimal OpenAI-compatible chat client (tool calling, reasoning field),\\ndependency-free so it runs inside the Kaggle rerun container.\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport json\\nimport time\\nimport urllib.error\\nimport urllib.request\\nfrom dataclasses import dataclass, field\\nfrom typing import Any, Optional\\n\\n\\n@dataclass\\nclass ChatResult:\\n    message: dict\\n    usage: dict = field(default_factory=dict)\\n    reasoning: str = \\\"\\\"\\n    latency: float = 0.0\\n    finish_reason: str = \\\"\\\"\\n\\n\\nclass ContextLengthError(RuntimeError):\\n    pass\\n\\n\\nclass ChatClient:\\n    def __init__(self, base_url: str = \\\"http://127.0.0.1:1234/v1\\\", model: str = \\\"local-qwen\\\", api_key: str = \\\"x\\\",\\n                 temperature: float = 0.6, top_p: float = 0.95, max_tokens: int = 4096, timeout: float = 600.0,\\n                 extra_body: Optional[dict] = None):\\n        self.base_url, self.model, self.api_key = base_url.rstrip(\\\"/\\\"), model, api_key\\n        self.temperature, self.top_p, self.max_tokens, self.timeout = temperature, top_p, max_tokens, timeout\\n        self.extra_body = extra_body or {}\\n        self.prompt_tokens = self.completion_tokens = 0\\n\\n    def chat(self, messages: list[dict], tools: Optional[list[dict]] = None, *, tool_choice: Any = \\\"auto\\\", retries: int = 3,\\n             override: Optional[dict] = None) -> ChatResult:\\n        body: dict[str, Any] = {\\\"model\\\": self.model, \\\"messages\\\": messages, \\\"temperature\\\": self.temperature, \\\"top_p\\\": self.top_p,\\n                                \\\"max_tokens\\\": self.max_tokens, **self.extra_body, **(override or {})}\\n        if tools:\\n            body[\\\"tools\\\"] = tools; body[\\\"tool_choice\\\"] = tool_choice\\n        data = json.dumps(body).encode()\\n        last: Exception = RuntimeError(\\\"no attempt\\\")\\n        for attempt in range(retries):\\n            t0 = time.time()\\n            req = urllib.request.Request(self.base_url + \\\"/chat/completions\\\", data=data,\\n                                         headers={\\\"Content-Type\\\": \\\"application/json\\\", \\\"Authorization\\\": f\\\"Bearer {self.api_key}\\\"})\\n            try:\\n                with urllib.request.urlopen(req, timeout=self.timeout) as resp:\\n                    out = json.loads(resp.read())\\n            except urllib.error.HTTPError as e:\\n                text = e.read().decode(errors=\\\"replace\\\")\\n                if e.code == 400 and (\\\"context length\\\" in text or \\\"maximum context\\\" in text or \\\"too long\\\" in text):\\n                    raise ContextLengthError(text[:300])\\n                last = RuntimeError(f\\\"HTTP {e.code}: {text[:300]}\\\")\\n            except Exception as e:  # network / timeout\\n                last = e\\n            else:\\n                choice = out[\\\"choices\\\"][0]; msg = choice[\\\"message\\\"]\\n                usage = out.get(\\\"usage\\\") or {}\\n                self.prompt_tokens += int(usage.get(\\\"prompt_tokens\\\", 0)); self.completion_tokens += int(usage.get(\\\"completion_tokens\\\", 0))\\n                reasoning = msg.pop(\\\"reasoning_content\\\", None) or msg.pop(\\\"reasoning\\\", None) or \\\"\\\"\\n                msg.setdefault(\\\"content\\\", \\\"\\\")\\n                if msg.get(\\\"content\\\") is None:\\n                    msg[\\\"content\\\"] = \\\"\\\"\\n                return ChatResult(message=msg, usage=usage, reasoning=reasoning, latency=time.time() - t0, finish_reason=choice.get(\\\"finish_reason\\\", \\\"\\\"))\\n            time.sleep(min(30, 2 ** attempt))\\n        raise last\\n\", \"nav.py\": \"\\\"\\\"\\\"Navigation helper (`nav`): learned from the transition history, rebuilt every tool call.\\n\\nAvatar detection from co-moving objects, cell-size estimate, coarse map,\\nfloor/wall/closed-target learning from actual avatar moves, BFS path_to,\\ntargets, frontier and action-gauge detection. Coordinates are (row, col).\\n\\nTypical use inside the tool:\\n    print(nav.summary())                      # avatar, moves, targets, gauge\\n    seq = nav.path_to(row, col)               # list of action names or None\\n    if seq: action(seq)\\n\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nfrom collections import Counter, deque\\nfrom typing import Any, Optional\\n\\nHEX = \\\"0123456789abcdef\\\"\\n_BIG = 400          # objects larger than this are never the avatar\\n_MAX_HISTORY = 300  # transitions considered (recent)\\n\\n\\ndef _grid_of(frame: Any) -> Optional[list[list[int]]]:\\n    if frame is None:\\n        return None\\n    g = getattr(frame, \\\"_grid\\\", None)\\n    if g is None and isinstance(frame, dict):\\n        g = frame.get(\\\"grid\\\")\\n    if g is None:\\n        g = getattr(frame, \\\"grid\\\", None)\\n    if not g:\\n        return None\\n    return [list(map(int, r)) for r in g]\\n\\n\\n_OBJ_CACHE: dict[int, tuple[object, tuple]] = {}   # id(grid) -> (grid, result); grids live in host_transitions, so identity is stable\\n\\n\\ndef extract_objects(grid: list[list[int]], max_objects: int = 40) -> list[dict]:\\n    \\\"\\\"\\\"Memoised: NavHelper is rebuilt every turn over the whole history, so the same grids are segmented again and again.\\\"\\\"\\\"\\n    hit = _OBJ_CACHE.get(id(grid))\\n    if hit is not None and hit[0] is grid and max_objects == 40:\\n        return hit[1]\\n    res = _extract_objects(grid, max_objects)\\n    if max_objects == 40:\\n        if len(_OBJ_CACHE) > 6000:\\n            _OBJ_CACHE.clear()\\n        _OBJ_CACHE[id(grid)] = (grid, res)\\n    return res\\n\\n\\ndef _extract_objects(grid: list[list[int]], max_objects: int = 40) -> list[dict]:\\n    h, w = len(grid), len(grid[0])\\n    background = Counter(c for row in grid for c in row).most_common(1)[0][0]\\n    seen = [[False] * w for _ in range(h)]\\n    objects = []\\n    for sy in range(h):\\n        for sx in range(w):\\n            if seen[sy][sx] or grid[sy][sx] == background:\\n                continue\\n            color = grid[sy][sx]\\n            stack, cells = [(sy, sx)], []\\n            seen[sy][sx] = True\\n            while stack:\\n                y, x = stack.pop()\\n                cells.append((y, x))\\n                for ny, nx in ((y - 1, x), (y + 1, x), (y, x - 1), (y, x + 1)):\\n                    if 0 <= ny < h and 0 <= nx < w and not seen[ny][nx] and grid[ny][nx] == color:\\n                        seen[ny][nx] = True\\n                        stack.append((ny, nx))\\n            ys = [c[0] for c in cells]; xs = [c[1] for c in cells]\\n            objects.append({\\\"color\\\": color, \\\"size\\\": len(cells),\\n                            \\\"bbox\\\": (min(xs), min(ys), max(xs), max(ys)),          # x0,y0,x1,y1\\n                            \\\"center\\\": (sum(xs) // len(xs), sum(ys) // len(ys))})    # x,y\\n    objects.sort(key=lambda o: -o[\\\"size\\\"])\\n    return objects[:max_objects], background\\n\\n\\ndef _moved(prev: list[dict], cur: list[dict]) -> list[tuple[dict, int, int]]:\\n    \\\"\\\"\\\"(new object, dx, dy) for objects with identical color+size that shifted.\\\"\\\"\\\"\\n    pool = list(prev)\\n    out = []\\n    for c in cur:\\n        best, best_d = None, None\\n        for i, p in enumerate(pool):\\n            if p is None or p[\\\"color\\\"] != c[\\\"color\\\"] or p[\\\"size\\\"] != c[\\\"size\\\"]:\\n                continue\\n            if (p[\\\"bbox\\\"][2] - p[\\\"bbox\\\"][0], p[\\\"bbox\\\"][3] - p[\\\"bbox\\\"][1]) != (c[\\\"bbox\\\"][2] - c[\\\"bbox\\\"][0], c[\\\"bbox\\\"][3] - c[\\\"bbox\\\"][1]):\\n                continue   # a translation keeps the bbox shape: a shrinking gauge strip must not be matched to a same-size block elsewhere (ar25)\\n            d = abs(p[\\\"center\\\"][0] - c[\\\"center\\\"][0]) + abs(p[\\\"center\\\"][1] - c[\\\"center\\\"][1])\\n            if best_d is None or d < best_d:\\n                best, best_d = i, d\\n        if best is not None:\\n            p = pool[best]; pool[best] = None\\n            if best_d:\\n                out.append((c, c[\\\"center\\\"][0] - p[\\\"center\\\"][0], c[\\\"center\\\"][1] - p[\\\"center\\\"][1]))\\n    return out\\n\\n\\ndef _gcd(a: int, b: int) -> int:\\n    while b:\\n        a, b = b, a % b\\n    return a\\n\\n\\nclass NavHelper:\\n    def __init__(self, transitions: list, current_frame: Any):\\n        self.grid = _grid_of(current_frame)\\n        self.objects, self.background = extract_objects(self.grid) if self.grid else ([], None)\\n        self.move_log: dict[str, list[tuple[int, int]]] = {}\\n        self.player_votes: Counter = Counter()\\n        self.player_parts: set[tuple[int, int]] = set()\\n        self.gauge_hist: dict[int, list[int]] = {}\\n        self.last_avatar_center: Optional[tuple[int, int]] = None\\n        self._steps: list[tuple[str, list[list[int]], list[list[int]]]] = []\\n        for t in (transitions or [])[-_MAX_HISTORY:]:\\n            b, a = _grid_of(getattr(t, \\\"before_frame\\\", None)), _grid_of(getattr(t, \\\"after_frame\\\", None))\\n            name = str(getattr(t, \\\"action\\\", \\\"\\\") or \\\"\\\").strip().upper()\\n            if b is None or a is None or not name:\\n                continue\\n            self._steps.append((name, b, a))\\n        self._learn()\\n        self.md = self._map_data()\\n        self.floor_votes: Counter = Counter()\\n        self.walls: set[tuple[int, int]] = set()\\n        self.visited: set[tuple[int, int]] = set()\\n        self.last_visit: dict[tuple[int, int], int] = {}   # block -> step index of last visit\\n        self.blocked: dict[tuple[int, int], int] = {}      # non-floor block we tried to enter and failed -> step\\n        self._learn_terrain()\\n\\n    # \\u2500\\u2500 learning \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\n    def _learn(self) -> None:\\n        prev_objs = None\\n        for name, b, a in self._steps:\\n            po, _ = extract_objects(b); co, _ = extract_objects(a)\\n            moved = [m for m in _moved(po, co) if m[0][\\\"size\\\"] <= _BIG]\\n            if moved and not name.startswith(\\\"MOUSE\\\"):\\n                near = [m for m in moved if abs(m[1]) + abs(m[2]) <= 12] or moved   # one key press moves a body a few pixels; a 20px jump is a mismatch\\n                c, dx, dy = max(near, key=lambda m: (m[0][\\\"size\\\"], abs(m[1]) + abs(m[2])))   # the biggest moving body is the player, not the farthest-jumping one\\n                self.move_log.setdefault(name, []).append((dx, dy))\\n                self.player_votes[(c[\\\"color\\\"], c[\\\"size\\\"])] += 1\\n                self.player_parts = {(m[0][\\\"color\\\"], m[0][\\\"size\\\"]) for m in moved if (m[1], m[2]) == (dx, dy)}\\n                self.last_avatar_center = c[\\\"center\\\"]   # disambiguates look-alike objects (trails, copies)\\n            # gauge: same-color object whose size changed a little near the same spot\\n            for o in co:\\n                for p in po:\\n                    if p[\\\"color\\\"] == o[\\\"color\\\"] and p[\\\"size\\\"] != o[\\\"size\\\"] and \\\\\\n                       abs(p[\\\"center\\\"][0] - o[\\\"center\\\"][0]) + abs(p[\\\"center\\\"][1] - o[\\\"center\\\"][1]) <= 3 and \\\\\\n                       o[\\\"size\\\"] > 8:\\n                        h = self.gauge_hist.setdefault(o[\\\"color\\\"], [])\\n                        if not h:\\n                            h.append(p[\\\"size\\\"])\\n                        h.append(o[\\\"size\\\"])\\n                        break\\n\\n    def move_delta(self, name: str) -> Optional[tuple[int, int]]:\\n        log = self.move_log.get(name.upper()) or []\\n        if not log:\\n            return None\\n        if len(log) == 1:\\n            d = log[0]\\n            return d if abs(d[0]) + abs(d[1]) <= 8 else None\\n        (delta, n), = Counter(log).most_common(1)\\n        return delta if n >= 2 and n * 2 > len(log) else None\\n\\n    @property\\n    def moves(self) -> dict[str, tuple[int, int]]:\\n        \\\"\\\"\\\"action name -> (dx, dy) in pixels (x=col, y=row).\\\"\\\"\\\"\\n        out = {}\\n        for a in self.move_log:\\n            d = self.move_delta(a)\\n            if d:\\n                out[a] = d\\n        # opposite keys are assumed to move the opposite way until observed (a wrong guess costs one probe)\\n        for a, b in ((\\\"UP\\\", \\\"DOWN\\\"), (\\\"DOWN\\\", \\\"UP\\\"), (\\\"LEFT\\\", \\\"RIGHT\\\"), (\\\"RIGHT\\\", \\\"LEFT\\\")):\\n            if a in out and b not in out and b not in self.move_log:\\n                out[b] = (-out[a][0], -out[a][1])\\n        return out\\n\\n    def avatar(self) -> Optional[dict]:\\n        main = None\\n        last = getattr(self, \\\"last_avatar_center\\\", None)\\n        for (color, size), _ in self.player_votes.most_common(3):\\n            cands = [o for o in self.objects if o[\\\"color\\\"] == color and o[\\\"size\\\"] == size]\\n            if cands:\\n                main = min(cands, key=lambda o: abs(o[\\\"center\\\"][0] - last[0]) + abs(o[\\\"center\\\"][1] - last[1])) if last else cands[0]\\n                break\\n        if main is None:\\n            return None\\n        x0, y0, x1, y1 = main[\\\"bbox\\\"]; size = main[\\\"size\\\"]; colors = {main[\\\"color\\\"]}\\n        for o in self.objects:\\n            if o is main or (o[\\\"color\\\"], o[\\\"size\\\"]) not in self.player_parts:\\n                continue\\n            bx0, by0, bx1, by1 = o[\\\"bbox\\\"]\\n            if bx0 > x1 + 1 or bx1 < x0 - 1 or by0 > y1 + 1 or by1 < y0 - 1:\\n                continue\\n            x0, y0, x1, y1 = min(x0, bx0), min(y0, by0), max(x1, bx1), max(y1, by1)\\n            size += o[\\\"size\\\"]; colors.add(o[\\\"color\\\"])\\n        return {\\\"colors\\\": sorted(colors), \\\"size\\\": size, \\\"bbox_xyxy\\\": (x0, y0, x1, y1),\\n                \\\"row\\\": (y0 + y1) // 2, \\\"col\\\": (x0 + x1) // 2}\\n\\n    def cell_size(self) -> int:\\n        g = 0\\n        for d in self.moves.values():\\n            for v in (abs(d[0]), abs(d[1])):\\n                if v:\\n                    g = _gcd(g, v)\\n        return g if 2 <= g <= 8 else 4\\n\\n    # \\u2500\\u2500 coarse map \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\n    def _map_data(self) -> Optional[dict]:\\n        if not self.grid:\\n            return None\\n        cell = self.cell_size(); av = self.avatar()\\n        ox = oy = 0\\n        if av:\\n            ox, oy = av[\\\"bbox_xyxy\\\"][0] % cell, av[\\\"bbox_xyxy\\\"][1] % cell\\n        colors, y = [], oy\\n        while y < 64 and len(colors) < 16:\\n            row, x = [], ox\\n            while x < 64 and len(row) < 16:\\n                block = [self.grid[yy][xx] for yy in range(y, min(64, y + cell)) for xx in range(x, min(64, x + cell))]\\n                row.append(Counter(block).most_common(1)[0][0]); x += cell\\n            colors.append(row); y += cell\\n        pb = ((av[\\\"bbox_xyxy\\\"][0] - ox) // cell, (av[\\\"bbox_xyxy\\\"][1] - oy) // cell) if av else None\\n        return {\\\"cell\\\": cell, \\\"ox\\\": ox, \\\"oy\\\": oy, \\\"colors\\\": colors, \\\"player\\\": pb,\\n                \\\"ncols\\\": len(colors[0]), \\\"nrows\\\": len(colors)}\\n\\n    def block_of(self, row: int, col: int) -> tuple[int, int]:\\n        md = self.md\\n        return (max(0, min(md[\\\"ncols\\\"] - 1, (col - md[\\\"ox\\\"]) // md[\\\"cell\\\"])),\\n                max(0, min(md[\\\"nrows\\\"] - 1, (row - md[\\\"oy\\\"]) // md[\\\"cell\\\"])))\\n\\n    def block_moves(self) -> dict[str, tuple[int, int]]:\\n        md = self.md; out = {}\\n        for a, (dx, dy) in self.moves.items():\\n            if dx % md[\\\"cell\\\"] == 0 and dy % md[\\\"cell\\\"] == 0:\\n                out[a] = (dx // md[\\\"cell\\\"], dy // md[\\\"cell\\\"])\\n        return out\\n\\n    def _learn_terrain(self) -> None:\\n        \\\"\\\"\\\"Replay the history through the current map geometry: blocks the avatar\\n        actually moved from are floor; moves that did not move it mark walls.\\\"\\\"\\\"\\n        if not self.md:\\n            return\\n        md = self.md; bm = self.block_moves()\\n        if not bm:\\n            return\\n        prev_block = None\\n        sig = None\\n        for step_i, (name, b, a) in enumerate(self._steps):\\n            objs_a, _ = extract_objects(a)\\n            # locate avatar in the after-frame by main color/size votes\\n            av = None\\n            for (color, size), _ in self.player_votes.most_common(3):\\n                for o in objs_a:\\n                    if o[\\\"color\\\"] == color and o[\\\"size\\\"] == size:\\n                        av = o; break\\n                if av:\\n                    break\\n            blk = ((av[\\\"bbox\\\"][0] - md[\\\"ox\\\"]) // md[\\\"cell\\\"], (av[\\\"bbox\\\"][1] - md[\\\"oy\\\"]) // md[\\\"cell\\\"]) if av else None\\n            if blk is not None:\\n                self.visited.add(blk)\\n                self.last_visit[blk] = step_i\\n            mv = bm.get(name)\\n            if prev_block is not None and blk is not None and mv is not None:\\n                if blk == prev_block:\\n                    tgt = (prev_block[0] + mv[0], prev_block[1] + mv[1])\\n                    tc, tr = tgt\\n                    colour = None\\n                    if 0 <= tr < md[\\\"nrows\\\"] and 0 <= tc < md[\\\"ncols\\\"]:\\n                        cell = md[\\\"cell\\\"]\\n                        x0, y0 = md[\\\"ox\\\"] + tc * cell, md[\\\"oy\\\"] + tr * cell\\n                        block = [a[yy][xx] for yy in range(y0, min(64, y0 + cell)) for xx in range(x0, min(64, x0 + cell))]\\n                        colour = Counter(block).most_common(1)[0][0]\\n                    # a failed step into a non-floor block is a closed target (door/lock), not a wall\\n                    if colour is not None and colour not in self.floor_colors and colour != self.background \\\\\\n                            and self.floor_votes:\\n                        self.blocked[tgt] = step_i\\n                    else:\\n                        self.walls.add(tgt)\\n                elif blk == (prev_block[0] + mv[0], prev_block[1] + mv[1]):\\n                    c, r = prev_block\\n                    if 0 <= r < md[\\\"nrows\\\"] and 0 <= c < md[\\\"ncols\\\"]:\\n                        # colour of the block we just left, in the after-frame\\n                        cell = md[\\\"cell\\\"]\\n                        x0, y0 = md[\\\"ox\\\"] + c * cell, md[\\\"oy\\\"] + r * cell\\n                        block = [a[yy][xx] for yy in range(y0, min(64, y0 + cell)) for xx in range(x0, min(64, x0 + cell))]\\n                        self.floor_votes[Counter(block).most_common(1)[0][0]] += 1\\n            prev_block = blk\\n        if self.md[\\\"player\\\"] is not None:\\n            self.visited.add(self.md[\\\"player\\\"])\\n            self.last_visit[self.md[\\\"player\\\"]] = len(self._steps)\\n\\n    @property\\n    def floor_colors(self) -> set[int]:\\n        strong = {c for c, n in self.floor_votes.items() if n >= 2}\\n        if strong:\\n            return strong\\n        return {self.floor_votes.most_common(1)[0][0]} if self.floor_votes else set()\\n\\n    def walkable(self, b: tuple[int, int]) -> bool:\\n        md = self.md; c, r = b\\n        if not (0 <= r < md[\\\"nrows\\\"] and 0 <= c < md[\\\"ncols\\\"]) or b in self.walls:\\n            return False\\n        return md[\\\"colors\\\"][r][c] in self.floor_colors or b == md[\\\"player\\\"]\\n\\n    # \\u2500\\u2500 planning \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\n    def path_to(self, row: int, col: int, max_len: int = 40, ignore_wall_target: bool = False) -> Optional[list[str]]:\\n        \\\"\\\"\\\"BFS over known-walkable blocks to the block containing pixel (row, col).\\n        Returns action names (e.g. ['LEFT','LEFT','UP']) or None if unreachable\\n        / navigation not learned yet. The final step may enter any colour.\\\"\\\"\\\"\\n        if not self.md or self.md[\\\"player\\\"] is None:\\n            return None\\n        bm = self.block_moves()\\n        if not bm:\\n            return None\\n        start, target = self.md[\\\"player\\\"], self.block_of(row, col)\\n        if start == target:\\n            return []\\n        if target in self.walls and not ignore_wall_target:   # a door bumped before the key was collected is still a valid goal later\\n            return None\\n        prev = {start: (start, \\\"\\\")}\\n        q = deque([start])\\n        while q:\\n            cur = q.popleft()\\n            for a, (dc, dr) in bm.items():\\n                nb = (cur[0] + dc, cur[1] + dr)\\n                if nb in prev:\\n                    continue\\n                if nb == target or self.walkable(nb):\\n                    prev[nb] = (cur, a)\\n                    if nb == target:\\n                        q.clear(); break\\n                    q.append(nb)\\n        if target not in prev:\\n            return None\\n        acts, b = [], target\\n        while b != start:\\n            b, a = prev[b]; acts.append(a)\\n        acts.reverse()\\n        return acts[:max_len]\\n\\n    def targets(self, max_n: int = 10) -> list[dict]:\\n        av = self.avatar()\\n        \\\"\\\"\\\"Non-floor, non-background small objects with (row, col) and path length.\\\"\\\"\\\"\\n        if not self.md or self.md[\\\"player\\\"] is None:\\n            return []\\n        av = self.avatar(); pb = av[\\\"bbox_xyxy\\\"] if av else None\\n        out, seen = [], set()\\n        for o in self.objects:\\n            if o[\\\"size\\\"] > 120 or o[\\\"color\\\"] == self.background or o[\\\"color\\\"] in self.floor_colors:\\n                continue\\n            bx0, by0, bx1, by1 = o[\\\"bbox\\\"]\\n            if ((bx1 - bx0 <= 2 and by1 - by0 >= 8 and (bx0 <= 1 or bx1 >= 62)) or (by1 - by0 <= 2 and bx1 - bx0 >= 8 and (by0 <= 1 or by1 >= 62))):\\n                continue   # thin LONG strip hugging an edge (gauge / counter pieces), never a target; small squares at the edge stay (ls20 goal squares)\\n            # small marks drawn inside a larger non-floor object are parts of that object, not targets\\n            ox, oy = o[\\\"center\\\"]; contained = False\\n            for big in self.objects:\\n                if big is o or big[\\\"size\\\"] <= o[\\\"size\\\"] * 3 or big[\\\"color\\\"] == self.background or big[\\\"color\\\"] in self.floor_colors:\\n                    continue\\n                bx0, by0, bx1, by1 = big[\\\"bbox\\\"]\\n                if bx0 < ox < bx1 and by0 < oy < by1:\\n                    contained = True; break\\n            if contained:\\n                continue\\n            if False:\\n                continue\\n            x0, y0, x1, y1 = o[\\\"bbox\\\"]\\n            if pb and not (x0 > pb[2] or x1 < pb[0] or y0 > pb[3] or y1 < pb[1]):\\n                continue\\n            b = self.block_of(o[\\\"center\\\"][1], o[\\\"center\\\"][0])\\n            if b in seen or b == self.md[\\\"player\\\"]:\\n                continue\\n            seen.add(b)\\n            p = self.path_to(o[\\\"center\\\"][1], o[\\\"center\\\"][0])\\n            near = [(b[0] + dc, b[1] + dr) for dc in (-1, 0, 1) for dr in (-1, 0, 1)]\\n            touched = [self.last_visit[q] for q in near if q in self.last_visit]\\n            if b in self.blocked:\\n                touched.append(self.blocked[b])\\n            out.append({\\\"color\\\": o[\\\"color\\\"], \\\"colour\\\": o[\\\"color\\\"], \\\"size\\\": o[\\\"size\\\"], \\\"row\\\": o[\\\"center\\\"][1], \\\"col\\\": o[\\\"center\\\"][0],\\n                        \\\"path_len\\\": None if p is None else len(p),\\n                        \\\"visited\\\": bool(touched) or b in self.walls,\\n                        \\\"last_visit\\\": max(touched) if touched else -1})\\n        out.sort(key=lambda t: (t[\\\"path_len\\\"] is None, t[\\\"path_len\\\"] or 0))\\n        return out[:max_n]\\n\\n    def frontier(self) -> Optional[tuple[int, int]]:\\n        \\\"\\\"\\\"Nearest unvisited walkable block as (row, col) pixel centre, else an\\n        unknown-colour block adjacent to the reachable area.\\\"\\\"\\\"\\n        if not self.md or self.md[\\\"player\\\"] is None:\\n            return None\\n        bm = self.block_moves()\\n        if not bm:\\n            return None\\n        md = self.md; start = md[\\\"player\\\"]\\n        seen, q, unknown = {start}, deque([start]), []\\n        while q:\\n            cur = q.popleft()\\n            for dc, dr in bm.values():\\n                nb = (cur[0] + dc, cur[1] + dr)\\n                if nb in seen:\\n                    continue\\n                seen.add(nb)\\n                if self.walkable(nb):\\n                    if nb not in self.visited:\\n                        return self._block_center(nb)\\n                    q.append(nb)\\n                elif nb not in self.walls and 0 <= nb[1] < md[\\\"nrows\\\"] and 0 <= nb[0] < md[\\\"ncols\\\"]:\\n                    unknown.append(nb)\\n        return self._block_center(unknown[0]) if unknown else None\\n\\n    def _block_center(self, b: tuple[int, int]) -> tuple[int, int]:\\n        md = self.md; c, r = b\\n        return (md[\\\"oy\\\"] + r * md[\\\"cell\\\"] + md[\\\"cell\\\"] // 2, md[\\\"ox\\\"] + c * md[\\\"cell\\\"] + md[\\\"cell\\\"] // 2)\\n\\n    def gauge(self) -> Optional[dict]:\\n        for color, hist in self.gauge_hist.items():\\n            if len(hist) < 3:\\n                continue\\n            steps = [b - a for a, b in zip(hist, hist[1:])]\\n            s = steps[-1]; k = 0\\n            while k < len(steps) and steps[-1 - k] == s:\\n                k += 1\\n            if s != 0 and k >= 2:\\n                return {\\\"color\\\": color, \\\"size\\\": hist[-1], \\\"per_action\\\": s,\\n                        \\\"actions_left\\\": hist[-1] // abs(s) if s < 0 else None}\\n            # monotonic but uneven steps (e.g. -1, -2, -1): still a counter; report the average step\\n            recent = steps[-4:]\\n            if len(recent) >= 3 and all(x < 0 for x in recent) or (len(recent) >= 3 and all(x > 0 for x in recent)):\\n                avg = sum(recent) / len(recent)\\n                return {\\\"color\\\": color, \\\"size\\\": hist[-1], \\\"per_action\\\": round(avg, 1),\\n                        \\\"actions_left\\\": int(hist[-1] / abs(avg)) if avg < 0 else None}\\n        return None\\n\\n    # \\u2500\\u2500 text \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\n    def map(self) -> str:\\n        if not self.md:\\n            return \\\"\\\"\\n        md = self.md; pb = md[\\\"player\\\"]\\n        av = self.avatar(); pw = ph = 1\\n        if av:\\n            pw = max(1, (av[\\\"bbox_xyxy\\\"][2] - av[\\\"bbox_xyxy\\\"][0] + md[\\\"cell\\\"]) // md[\\\"cell\\\"])\\n            ph = max(1, (av[\\\"bbox_xyxy\\\"][3] - av[\\\"bbox_xyxy\\\"][1] + md[\\\"cell\\\"]) // md[\\\"cell\\\"])\\n        rows = []\\n        for r, row in enumerate(md[\\\"colors\\\"]):\\n            line = []\\n            for c, col in enumerate(row):\\n                if pb and pb[0] <= c < pb[0] + pw and pb[1] <= r < pb[1] + ph:\\n                    line.append(\\\"P\\\")\\n                elif (c, r) in self.walls:\\n                    line.append(\\\"#\\\")\\n                else:\\n                    line.append(HEX[col])\\n            rows.append(f\\\"{r:2d} {''.join(line)}\\\")\\n        return (f\\\"cell={md['cell']}px origin(row={md['oy']},col={md['ox']}); char=(col,row) block majority colour hex, \\\"\\n                f\\\"P=avatar, #=known wall; pixel row=r*{md['cell']}+{md['oy']}, col=c*{md['cell']}+{md['ox']}\\\\n\\\" + \\\"\\\\n\\\".join(rows))\\n\\n    def summary(self) -> str:\\n        av = self.avatar(); mv = self.moves\\n        lines = []\\n        if av:\\n            lines.append(f\\\"avatar: colours {av['colors']} size {av['size']} at row {av['row']}, col {av['col']}\\\")\\n        else:\\n            lines.append(\\\"avatar: not identified yet (take each movement action once or twice)\\\")\\n        if mv:\\n            lines.append(\\\"moves (dx=cols,dy=rows): \\\" + \\\", \\\".join(f\\\"{a}=({dx:+d},{dy:+d})\\\" for a, (dx, dy) in mv.items()))\\n            lines.append(f\\\"floor colours {sorted(self.floor_colors) or '?'}, walls known {len(self.walls)}, blocks visited {len(self.visited)}\\\")\\n        g = self.gauge()\\n        if g:\\n            lines.append(f\\\"gauge: colour {g['color']} {g['size']} cells, {g['per_action']:+d}/action\\\"\\n                         + (f\\\", ~{g['actions_left']} actions left\\\" if g[\\\"actions_left\\\"] is not None else \\\"\\\"))\\n        ts = self.targets()\\n        if ts:\\n            lines.append(\\\"targets (row,col colour size path visited):\\\")\\n            for t in ts:\\n                lines.append(f\\\"  ({t['row']},{t['col']}) c{t['color']} s{t['size']} \\\"\\n                             + (f\\\"{t['path_len']} moves\\\" if t[\\\"path_len\\\"] is not None else \\\"unreachable\\\")\\n                             + (\\\" visited\\\" if t[\\\"visited\\\"] else \\\"\\\"))\\n        fr = self.frontier()\\n        if fr:\\n            lines.append(f\\\"frontier: nearest unexplored block at row {fr[0]}, col {fr[1]}\\\")\\n        return \\\"\\\\n\\\".join(lines)\\n\\n\\ndef build_nav(transitions: list, current_frame: Any) -> Optional[NavHelper]:\\n    try:\\n        return NavHelper(transitions, current_frame)\\n    except Exception:\\n        return None\\n\", \"planner.py\": \"\\\"\\\"\\\"Rule-based planners that run on confirmed rules instead of the model's arithmetic.\\n\\ntwo_body_merge: joint-state BFS for games where a second body moves with the avatar under a fixed\\ntransform (mirror), walls stop each body separately, hazards reset, and the goal is to bring both\\nbodies onto the same cell (or adjacent-swap). Cells are the movement lattice (cell size from nav).\\n\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nfrom collections import deque\\nfrom typing import Optional\\n\\n\\ndef _cell_color(grid, r, c):\\n    if 0 <= r < len(grid) and 0 <= c < len(grid[0]):\\n        return grid[r][c]\\n    return None\\n\\n\\ndef _body_blocked(grid, r, c, size, wall_colors, body_colors):\\n    \\\"\\\"\\\"Would a size x size body with top-left (r, c) overlap a wall colour or leave the grid?\\\"\\\"\\\"\\n    h, w = len(grid), len(grid[0])\\n    if r < 0 or c < 0 or r + size > h or c + size > w:\\n        return True\\n    for i in range(r, r + size):\\n        for j in range(c, c + size):\\n            v = grid[i][j]\\n            if v in wall_colors and v not in body_colors:\\n                return True\\n    return False\\n\\n\\ndef _touches(grid, r, c, size, colors):\\n    for i in range(r, r + size):\\n        for j in range(c, c + size):\\n            if 0 <= i < len(grid) and 0 <= j < len(grid[0]) and grid[i][j] in colors:\\n                return True\\n    return False\\n\\n\\ndef two_body_merge(grid, body_a, body_b, moves: dict, transform: str, wall_colors, hazard_colors=(), body_colors=(),\\n                   max_nodes: int = 200000) -> Optional[list[str]]:\\n    \\\"\\\"\\\"body_a/body_b: (top_row, left_col, size). moves: action -> (dx, dy) in pixels. transform: mirror_x|mirror_y|mirror_xy|same.\\n    Returns the action list that brings both bodies to the same top-left, avoiding hazards, or None.\\\"\\\"\\\"\\n    (ra, ca, sa), (rb, cb, sb) = body_a, body_b\\n    walls = set(wall_colors); hazards = set(hazard_colors); bcol = set(body_colors)\\n    tf = {\\\"same\\\": (1, 1), \\\"mirror_x\\\": (-1, 1), \\\"mirror_y\\\": (1, -1), \\\"mirror_xy\\\": (-1, -1)}[transform]\\n    start = (ra, ca, rb, cb)\\n    prev = {start: None}; q = deque([start]); n = 0\\n    while q and n < max_nodes:\\n        s = q.popleft(); n += 1\\n        r1, c1, r2, c2 = s\\n        if (r1, c1) == (r2, c2):\\n            path = []\\n            while prev[s] is not None:\\n                s, a = prev[s]; path.append(a)\\n            return path[::-1]\\n        for a, (dx, dy) in moves.items():\\n            n1 = (r1 + dy, c1 + dx); n2 = (r2 + tf[1] * dy, c2 + tf[0] * dx)\\n            if _body_blocked(grid, n1[0], n1[1], sa, walls, bcol):\\n                n1 = (r1, c1)\\n            if _body_blocked(grid, n2[0], n2[1], sb, walls, bcol):\\n                n2 = (r2, c2)\\n            if n1 == (r1, c1) and n2 == (r2, c2):\\n                continue\\n            if hazards and (_touches(grid, n1[0], n1[1], sa, hazards) or _touches(grid, n2[0], n2[1], sb, hazards)):\\n                continue\\n            # swap across adjacent cells also counts as a merge\\n            if n1 == (r2, c2) and n2 == (r1, c1):\\n                path = [a]\\n                t = s\\n                while prev[t] is not None:\\n                    t, b = prev[t]; path.append(b)\\n                return path[::-1]\\n            ns = (n1[0], n1[1], n2[0], n2[1])\\n            if ns not in prev:\\n                prev[ns] = (s, a); q.append(ns)\\n    return None\\n\\n\\n# ---------------------------------------------------------------- cell-space version (lattice from the body itself)\\ndef lattice_from_body(bbox_xyxy, size: int = 64):\\n    \\\"\\\"\\\"A one-cell body at pixel bbox (x0, y0, x1, y1) fixes the lattice: step = body width, origin = x0 mod step.\\\"\\\"\\\"\\n    x0, y0, x1, y1 = bbox_xyxy\\n    step = max(x1 - x0 + 1, y1 - y0 + 1)\\n    ox, oy = x0 % step, y0 % step\\n    cols = [ox + k * step for k in range(0, (size - ox) // step + 1)]\\n    rows = [oy + k * step for k in range(0, (size - oy) // step + 1)]\\n    return rows, cols\\n\\n\\ndef cell_of(bbox_xyxy, rows, cols):\\n    x0, y0, x1, y1 = bbox_xyxy\\n    r = max(i for i, b in enumerate(rows) if b <= y0) if any(b <= y0 for b in rows) else 0\\n    c = max(i for i, b in enumerate(cols) if b <= x0) if any(b <= x0 for b in cols) else 0\\n    return r, c\\n\\n\\ndef cell_colors(grid, rows, cols):\\n    \\\"\\\"\\\"Per cell: (dominant colour, set of colours).\\\"\\\"\\\"\\n    out = []\\n    for r in range(len(rows) - 1):\\n        row = []\\n        for c in range(len(cols) - 1):\\n            vals = {}\\n            for i in range(rows[r], min(rows[r + 1], len(grid))):\\n                for j in range(cols[c], min(cols[c + 1], len(grid[0]))):\\n                    vals[grid[i][j]] = vals.get(grid[i][j], 0) + 1\\n            dom = max(vals, key=vals.get) if vals else None\\n            row.append((dom, set(vals)))\\n        out.append(row)\\n    return out\\n\\n\\ndef two_body_merge_cells(cells, a, b, transform: str, passable, hazard_colors=(), max_nodes: int = 200000, forbidden=(), blocked=()) -> Optional[list[str]]:\\n    \\\"\\\"\\\"cells[r][c] = (dominant, colour set). a, b = (row, col) cell coords. Moves are one cell.\\n    A move is blocked for a body if the target cell's dominant colour is not passable (or off-grid).\\n    Hazard cells (containing a hazard colour) are never entered.\\\"\\\"\\\"\\n    H, W = len(cells), len(cells[0])\\n    MOVES = {\\\"UP\\\": (-1, 0), \\\"DOWN\\\": (1, 0), \\\"LEFT\\\": (0, -1), \\\"RIGHT\\\": (0, 1)}\\n    tf = {\\\"same\\\": (1, 1), \\\"mirror_x\\\": (1, -1), \\\"mirror_y\\\": (-1, 1), \\\"mirror_xy\\\": (-1, -1)}[transform]   # (row sign, col sign)\\n    hz = set(hazard_colors); fb = set(forbidden); blocked = set(blocked); passable = set(passable)\\n\\n    def step(pos, dr, dc):\\n        r, c = pos[0] + dr, pos[1] + dc\\n        if not (0 <= r < H and 0 <= c < W):\\n            return pos\\n        dom, cols = cells[r][c]\\n        if cols & hz:            # hazard-textured cells are enterable (and deadly), never walls\\n            return (r, c)\\n        if not cols <= passable or (r, c) in blocked:   # any unknown sprite in the cell blocks (gates, doors)\\n            return pos\\n        return (r, c)\\n\\n    start = (a, b); prev = {start: None}; q = deque([start]); n = 0\\n    while q and n < max_nodes:\\n        s = q.popleft(); n += 1\\n        pa, pb = s\\n        if pa == pb:\\n            path = []\\n            while prev[s] is not None:\\n                s, act = prev[s]; path.append(act)\\n            return path[::-1]\\n        for act, (dr, dc) in MOVES.items():\\n            na = step(pa, dr, dc); nb = step(pb, tf[0] * dr, tf[1] * dc)\\n            if na == pa and nb == pb:\\n                continue\\n            if hz and (cells[na[0]][na[1]][1] & hz or cells[nb[0]][nb[1]][1] & hz):\\n                continue\\n            if fb and (na in fb or nb in fb):   # cells that caused a reset before\\n                continue\\n            if na == pb and nb == pa:   # swap counts as a merge\\n                path = [act]; t = s\\n                while prev[t] is not None:\\n                    t, x = prev[t]; path.append(x)\\n                return path[::-1]\\n            ns = (na, nb)\\n            if ns not in prev:\\n                prev[ns] = (s, act); q.append(ns)\\n    return None\\n\\n\\ndef plan_resources(nav, *, goal_cells: set, collect_cells: set, refill_cells: dict, hazard_colors: set,\\n                   gauge_left: Optional[int], require_collect: bool = True, max_states: int = 300000) -> Optional[list[str]]:\\n    \\\"\\\"\\\"Resource-aware route on the nav block grid: state = (block, gauge actions left, collected set, refills used).\\n    Moves cost one gauge unit; entering a refill block adds refill_cells[block] actions (once); entering a collect block collects it;\\n    the goal is any goal block, entered after every collectible (when require_collect) \\u2014 the final step may enter a non-floor block\\n    (doors). Cells come from object centres (small items do not dominate a block's majority colour). Returns the action list or None.\\\"\\\"\\\"\\n    from collections import deque\\n    md = nav.md\\n    if not md or md[\\\"player\\\"] is None or not goal_cells:\\n        return None\\n    bm = nav.block_moves()\\n    if not bm:\\n        return None\\n    colors = md[\\\"colors\\\"]; nrows, ncols = md[\\\"nrows\\\"], md[\\\"ncols\\\"]\\n    def col_of(b):\\n        c, r = b\\n        return colors[r][c] if 0 <= r < nrows and 0 <= c < ncols else None\\n    cl = sorted(collect_cells); idx = {b: i for i, b in enumerate(cl)}; full = (1 << len(cl)) - 1\\n    start_b = md[\\\"player\\\"]; g0 = gauge_left if gauge_left is not None else 10 ** 6\\n    prev = {(start_b, 0, frozenset()): None}; best = {(start_b, 0, frozenset()): g0}\\n    q = deque([(start_b, g0, 0, frozenset())]); n = 0\\n    while q and n < max_states:\\n        b, g, mask, used = q.popleft(); n += 1\\n        for a, (dc, dr) in bm.items():\\n            nb = (b[0] + dc, b[1] + dr); cc = col_of(nb)\\n            if cc is None or nb in nav.walls or cc in hazard_colors:\\n                continue\\n            is_goal = nb in goal_cells\\n            if not is_goal and not (cc in nav.floor_colors or nb in collect_cells or nb in refill_cells or nb == start_b):\\n                continue\\n            ng = g - 1\\n            if ng < 0:\\n                continue\\n            nmask = mask | (1 << idx[nb]) if nb in idx else mask\\n            nused = used\\n            if nb in refill_cells and nb not in used:\\n                ng += int(refill_cells[nb]); nused = used | {nb}\\n            if is_goal:\\n                if require_collect and nmask != full:\\n                    continue\\n                acts = [a]; cur = (b, mask, used)\\n                while prev.get(cur) is not None:\\n                    pb, pm, pu, pa = prev[cur]; acts.append(pa); cur = (pb, pm, pu)\\n                return list(reversed(acts))\\n            key = (nb, nmask, nused)\\n            if best.get(key, -1) >= ng:\\n                continue\\n            best[key] = ng; prev[key] = (b, mask, used, a)\\n            q.append((nb, ng, nmask, nused))\\n    return None\\n\", \"prompts.py\": \"\\\"\\\"\\\"System prompt and per-turn text for the arcnav agent (our wording).\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nPYTHON_TOOL = {\\n    \\\"type\\\": \\\"function\\\",\\n    \\\"function\\\": {\\n        \\\"name\\\": \\\"python\\\",\\n        \\\"description\\\": \\\"Run Python code in the persistent game sandbox. Use action(...) inside it to play. Print what you need to see.\\\",\\n        \\\"parameters\\\": {\\\"type\\\": \\\"object\\\",\\n                       \\\"properties\\\": {\\\"code\\\": {\\\"type\\\": \\\"string\\\", \\\"description\\\": \\\"Python source to execute\\\"},\\n                                      \\\"goal\\\": {\\\"type\\\": \\\"string\\\", \\\"description\\\": \\\"Your current goal hypothesis for this game (repeat or update it every call)\\\"},\\n                                      \\\"plan\\\": {\\\"type\\\": \\\"string\\\", \\\"description\\\": \\\"The concrete next step this call performs and what comes after\\\"},\\n                                      \\\"roles\\\": {\\\"type\\\": \\\"string\\\", \\\"description\\\": \\\"Object roles learned so far, e.g. '5=door, 8=key, 11=gauge'\\\"}},\\n                       \\\"required\\\": [\\\"code\\\", \\\"goal\\\", \\\"plan\\\"]},\\n    },\\n}\\n\\nPROPOSE_TOOL = {\\n    \\\"type\\\": \\\"function\\\",\\n    \\\"function\\\": {\\n        \\\"name\\\": \\\"propose_solver\\\",\\n        \\\"description\\\": \\\"Store a persistent solver. `code` is Python source defining `def solve():` (and optionally `def predict(before_frame, action_name)`) \\\"\\n                       \\\"that reads current_frame, transitions, level_transitions, nav, valid_actions and returns the next actions (a list) or [].\\\",\\n        \\\"parameters\\\": {\\\"type\\\": \\\"object\\\", \\\"properties\\\": {\\\"code\\\": {\\\"type\\\": \\\"string\\\", \\\"description\\\": \\\"Python source defining def solve(): ...\\\"}},\\n                       \\\"required\\\": [\\\"code\\\"]},\\n    },\\n}\\n\\nSYSTEM_PROMPT = \\\"\\\"\\\"You are an autonomous agent playing an unknown grid puzzle game (ARC-AGI-3). The board is a 64x64 grid of colour\\nindices 0-15, shown as ascii rows using the hex digits 0-9a-f (one character per cell, row 0 at the top). Nobody tells you the rules:\\ndiscover them by acting, then complete as many levels as possible using as FEW actions as possible (the score per level is\\n(baseline_actions/your_actions)^2, so wasted actions cost score; unfinished levels score 0).\\n\\nYou act only through the `python` tool. The sandbox is a persistent Python process with these variables and helpers:\\n- `current_frame`: Frame with `.ascii` (the board as text), `.grid` (list of 64 rows of ints), `.level`, `.step`,\\n  `.segmentation` = {'nodes': [{id, color, pixels, bbox(r0,c0,r1,c1), center(row,col), hash, children, hud}], 'adjacency': [[i,j]...], 'background'}.\\n  `hud` marks thin strips along an edge (usually a status bar / action counter, not a clickable object).\\n- `transitions`: list of Transition(action, before_frame, after_frame, changed) for every action taken so far (newest last);\\n  `level_transitions` is the part that belongs to the current level.\\n- `valid_actions`: names allowed now, among UP, DOWN, LEFT, RIGHT, SPACE, ACTION7 (a game-specific extra key: probe it once) and\\n  MOUSE (a click: {'action':'MOUSE','row':r,'col':c}).\\n- `action(x)`: execute one action or a list of actions, e.g. action('UP') or action(['LEFT','LEFT']) or action({'action':'MOUSE','row':10,'col':20}).\\n  It returns {'executed_count','board_changed','level_completed','game_over','stopped_reason'} and refreshes every variable above.\\n  Execution stops early when a level completes or the game ends, and at most 24 actions run per call. After a game over the harness resets the game for you; the level restarts.\\n- `nav`: navigation helper rebuilt from `transitions` on every call (None until the first frame). `nav.summary()` gives the avatar\\n  (the object your movement keys move), learned move deltas, floor/wall knowledge, an action gauge if the game has one, candidate targets\\n  with (row, col) and BFS path length, and the nearest unexplored block. `nav.path_to(row, col)` returns the list of moves to reach the block\\n  containing that pixel (pass it straight to action), `nav.frontier()` the nearest unexplored walkable block (row, col) or None, `nav.targets()` a list of dicts with keys\\n  row, col, color, size, path_len (None if unreachable), visited, last_visit,\\n  `nav.map()` a coarse map (P = avatar, # = wall). It learns only from real moves, so press each movement key once or twice first.\\n- `goal_hypotheses`: list of dicts the harness inferred from how earlier levels were won (type reach / collect_reach /\\n  collect_all / click_sequence with colours); the same win rule usually holds on later levels \\u2014 plan for it.\\n- `rules_text`: rules the harness induced from this level's recorded transitions (movement deltas, walls, gauge cost,\\n  refills, collectibles, click effects) with support counts; CONFIRMED rules fit every transition. Build on them.\\n- `level_recaps`: harness-written summaries of how each earlier level was won (winning action sequence, collected objects,\\n  gauge refills). Levels of one game share the rules, so reuse the strategy on the new layout.\\n- `checklist`: a dict you share with the harness: `goal` (str), `roles` (dict colour -> role), `plan` (str), `tried` (list of\\n  'what + outcome'). The harness fills the facts (controls, avatar, limits, object counts) and shows the whole checklist at the top\\n  of every turn with the first unresolved item. `goal`, `plan` and `roles` are arguments of the python tool (fill them on every\\n  call); `tried` is written by the harness from what each turn did and what happened.\\n- `notes`: a string you own. Keep the rules learned so far in it (what each key does, objects and their roles, the goal\\n  hypothesis, the next plan). It persists and is shown to you at every turn, so update it instead of re-analysing the board.\\n- `propose_solver(code)`: store a persistent solver. `code` is a string defining `def solve():` that reads the variables above and returns the\\n  next actions (a list, or [] when undecided). Optionally define `def predict(before_frame, action_name)` returning the expected next\\n  board as an ascii string; it is scored against the recorded transitions and proposals with accuracy below 0.5 are rejected.\\n  A solver whose predict() reaches accuracy >= 0.8 on at least 6 recorded transitions is VERIFIED: the harness then runs solve() on its\\n  own every turn until it returns [], raises, stops changing the board, causes a game over, or the level stops advancing. A solver\\n  without predict() (or below 0.8) is kept as a DRAFT: its suggestion is shown to you each turn, but you decide what to execute. Never define your own\\n  `propose_solver` or `action`.\\n\\nHow to work \\u2014 START FROM THE CHECKLIST, NOT FROM ZERO:\\n0. ACT EVERY TURN: each python call should execute at least one action unless the previous turn's result is still unread.\\n   Inspection alone is only acceptable on the very first turn. Every turn begins with the CHECKLIST. Do not re-derive settled items; go straight to the first unresolved one (the header names it)\\n   and spend the turn resolving it: untried key -> press it; role unknown -> touch/click that object once; goal missing -> state a\\n   hypothesis and test it; plan present -> execute it. Record outcomes in `checklist['tried']` so they are never repeated.\\n1. Look first: print `current_frame.ascii` (or parts of it) and `current_frame.segmentation` summaries. Identify the avatar, walls,\\n   collectables, doors, counters, and the HUD.\\n2. Probe cheaply: one action per movement key, one click per distinct object type, and compare `transitions[-1].before_frame.ascii`\\n   with `.after_frame.ascii` (or print `nav.summary()`). Write down what each action does.\\n3. Form a hypothesis about the goal (reach a target, collect items in some order, match a pattern, toggle objects), then act on it with\\n   short scripted sequences, not one action per turn. Use `nav.path_to` for movement instead of hand-written step lists.\\n4. When the rules are clear or a level was just completed, encode them in `propose_solver(code)` so the following levels are played\\n   programmatically and cheaply. Keep solve() deterministic and short; prefer search over hand-written sequences.\\n5. A thin strip along an edge whose length changes by a fixed amount every action is an action counter (gauge). It is never the\\n   goal: do not try to fill or empty it, and never click it. When it runs out the level restarts (game over), so plan within it.\\n6. Every tool call should either gain information or make progress. Do not repeat a probe whose result you already know. If the board\\n   stops changing, change strategy (different key, different object, the SPACE key, a different order).\\n\\nReply format \\u2014 every turn, before the tool call, write exactly these five short lines (one sentence each, no more):\\nWorld model: <what the objects are and how the board works, as far as you know>\\nGoal model: <what you believe ends the level>\\nAction model: <what each key/click does, with deltas>\\nRecent findings: <what the last turn taught you>\\nPlan: <the concrete actions this call will execute>\\nThen make exactly one tool call (`python`, or `propose_solver` when you are ready to store a solver). Except for the very first\\nturn (inspect + probe), every call must execute at least one action; prefer a whole route or a probe batch over a single step.\\nDo not restate the analysis at length: the five lines are your memory, update them and act.\\n\\\"\\\"\\\"\\n\\nNAV_TEMPLATE = '''def solve():\\n    if nav is None:\\n        return []\\n    recent = [t.action for t in level_transitions[-8:]]\\n    for a in ['UP', 'DOWN', 'LEFT', 'RIGHT']:   # learn every movement key once before routing\\n        if a in valid_actions and a not in nav.moves and a not in recent:\\n            return [a]\\n    for t in nav.targets():\\n        if not t['visited'] and t['path_len']:\\n            return nav.path_to(t['row'], t['col']) or []\\n    reach = [t for t in nav.targets() if t['path_len']]   # all visited: revisit the least recently visited\\n    if reach:\\n        t = min(reach, key=lambda t: t['last_visit'])\\n        return nav.path_to(t['row'], t['col']) or []\\n    fr = nav.frontier()\\n    return (nav.path_to(fr[0], fr[1]) or []) if fr else []\\n'''\\n\\nCLICK_TEMPLATE = '''def solve():\\n    nodes = [n for n in current_frame.segmentation['nodes'] if not n['hud'] and n['pixels'] <= 600]\\n    nodes.sort(key=lambda n: n['pixels'])\\n    clicked = {}\\n    for t in transitions:\\n        a = t.action if isinstance(t.action, dict) else None\\n        if a and a.get('action') == 'MOUSE':\\n            key = (a['row'] // 4, a['col'] // 4)\\n            clicked.setdefault(key, [0, 0]); clicked[key][0] += 1; clicked[key][1] += int(t.changed)\\n    key = lambda n: (n['center'][0] // 4, n['center'][1] // 4)\\n    fresh = [n for n in nodes if key(n) not in clicked]\\n    pool = fresh or [n for n in nodes if clicked.get(key(n), [0, 0])[1] > 0] or nodes\\n    if not pool:\\n        return []\\n    r, c = pool[0]['center']\\n    return [{'action': 'MOUSE', 'row': r, 'col': c}]\\n'''\\n\\nSOLVER_GUIDE = f\\\"\\\"\\\"\\nSolver templates (adapt the target choice to the rules you inferred):\\n- Navigation game (visit the nearest unvisited target, else explore):\\npropose_solver('''\\n{NAV_TEMPLATE}''')\\n- Click game (no avatar, MOUSE only: sweep objects, re-click the ones that changed the board, skip HUD strips):\\npropose_solver('''\\n{CLICK_TEMPLATE}''')\\n\\\"\\\"\\\"\\n\\n\\ndef system_prompt() -> str:\\n    return SYSTEM_PROMPT + SOLVER_GUIDE\\n\\n\\ndef turn_header(*, level: int, levels_total: int, actions_used: int, level_actions: int, valid_actions: list[str], budget_line: str) -> str:\\n    return (f\\\"Level {level}/{levels_total}. Actions used: {actions_used} total, {level_actions} on this level. \\\"\\n            f\\\"Valid actions now: {', '.join(valid_actions) or 'none'}. {budget_line}\\\")\\n\", \"rules.py\": \"\\\"\\\"\\\"Symbolic rule induction over recorded transitions (object level, exact fit).\\n\\nEach rule is a parametric hypothesis fitted to the transitions of the current level. A rule reports\\n`support` (transitions it explains) and `counter` (transitions it contradicts); rules with counter == 0\\nand support >= 2 are 'confirmed'. The harness shows confirmed rules and unexplained transitions to the\\nmodel, and planners may rely on confirmed rules.\\n\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nfrom collections import Counter, defaultdict\\nfrom dataclasses import dataclass, field\\nfrom typing import Any, Optional\\n\\nfrom .frame import masked_ascii\\nfrom .nav import NavHelper, extract_objects\\n\\n\\n@dataclass\\nclass Rule:\\n    kind: str\\n    params: dict\\n    support: int = 0\\n    counter: int = 0\\n    examples: list = field(default_factory=list)\\n\\n    @property\\n    def confirmed(self) -> bool:\\n        return self.counter == 0 and self.support >= 2\\n\\n    def text(self) -> str:\\n        p = self.params\\n        if self.kind == \\\"move\\\":\\n            return f\\\"{p['action']} moves the avatar by (dx={p['dx']}, dy={p['dy']}) [{self.support} ok, {self.counter} blocked]\\\"\\n        if self.kind == \\\"wall\\\":\\n            return f\\\"the avatar cannot enter colour {p['color']} (blocked {self.support}x)\\\"\\n        if self.kind == \\\"gauge\\\":\\n            return f\\\"the gauge (colour {p['color']}) drops by {p['per_action']} per action\\\"\\n        if self.kind == \\\"refill\\\":\\n            return f\\\"touching colour {p['color']} refills the gauge (+{p['amount']}) and consumes it\\\"\\n        if self.kind == \\\"collect\\\":\\n            return f\\\"touching colour {p['color']} objects makes them disappear (collected, {self.support}x)\\\"\\n        if self.kind == \\\"hazard\\\":\\n            return f\\\"touching colour {p['color']} resets the avatar to its start position ({self.support}x)\\\"\\n        if self.kind == \\\"click\\\":\\n            return f\\\"clicking colour {p['color']} changes colour(s) {sorted(p['changes'])} ({self.support}x)\\\" if p[\\\"changes\\\"] else f\\\"clicking colour {p['color']} does nothing ({self.support}x)\\\"\\n        if self.kind == \\\"mirror\\\":\\n            how = {\\\"mirror_x\\\": \\\"horizontally mirrored (-dx, dy)\\\", \\\"mirror_y\\\": \\\"vertically mirrored (dx, -dy)\\\", \\\"mirror_xy\\\": \\\"point-mirrored (-dx, -dy)\\\"}[p[\\\"how\\\"]]\\n            return f\\\"a second body (colour {p['color']}, {p['size']}px) moves with every move, {how} \\u2014 walls stop each body separately ({self.support}x)\\\"\\n        if self.kind == \\\"noop\\\":\\n            return f\\\"{p['action']} changed nothing ({self.support}x)\\\"\\n        return f\\\"{self.kind} {p}\\\"\\n\\n\\ndef _objs(frame):\\n    objs, bg = extract_objects(frame.grid)\\n    return objs, bg\\n\\n\\ndef _count(frame, color):\\n    return sum(1 for row in frame.grid for v in row if v == color)\\n\\n\\ndef induce(transitions: list, current_frame) -> tuple[list[Rule], list[str]]:\\n    \\\"\\\"\\\"transitions: objects with .action (str or dict), .before_frame, .after_frame (same level).\\n    Returns (rules, unexplained) where unexplained lists transitions no rule accounts for.\\\"\\\"\\\"\\n    rules: list[Rule] = []\\n    if not transitions:\\n        return rules, []\\n    nav = NavHelper(transitions, current_frame)\\n    moves = nav.moves\\n    # 1. movement + walls\\n    per_action_ok: dict[str, int] = Counter(); per_action_blocked: dict[str, int] = Counter()\\n    wall_hits: Counter = Counter()\\n    avatar_positions = []\\n    for t in transitions:\\n        a = t.action if isinstance(t.action, str) else \\\"MOUSE\\\"\\n        if a not in moves:\\n            continue\\n        nb = NavHelper([t], t.after_frame)\\n        av_b = NavHelper(transitions, t.before_frame).avatar(); av_a = NavHelper(transitions, t.after_frame).avatar()\\n        if not av_b or not av_a:\\n            continue\\n        moved = (av_a[\\\"row\\\"] - av_b[\\\"row\\\"], av_a[\\\"col\\\"] - av_b[\\\"col\\\"]) != (0, 0)\\n        if moved:\\n            per_action_ok[a] += 1\\n        else:\\n            per_action_blocked[a] += 1\\n            dx, dy = moves[a]; r, c = av_b[\\\"row\\\"] + dy, av_b[\\\"col\\\"] + dx\\n            if 0 <= r < 64 and 0 <= c < 64:\\n                wall_hits[t.before_frame.grid[r][c]] += 1\\n        avatar_positions.append((av_b[\\\"row\\\"], av_b[\\\"col\\\"], av_a[\\\"row\\\"], av_a[\\\"col\\\"], a, t))\\n    for a, (dx, dy) in moves.items():\\n        rules.append(Rule(\\\"move\\\", {\\\"action\\\": a, \\\"dx\\\": dx, \\\"dy\\\": dy}, support=per_action_ok[a], counter=0))\\n    for color, n in wall_hits.items():\\n        rules.append(Rule(\\\"wall\\\", {\\\"color\\\": color}, support=n))\\n    # 1b. mirrored / co-moving second body: another object that moves whenever the avatar moves, with a fixed transform\\n    if moves:\\n        pairs: Counter = Counter(); decisive: Counter = Counter(); seen_moves = 0\\n        for t in transitions:\\n            a = t.action if isinstance(t.action, str) else \\\"MOUSE\\\"\\n            if a not in moves:\\n                continue\\n            po, _ = _objs(t.before_frame); co, _ = _objs(t.after_frame)\\n            from .nav import _moved\\n            mv = [(o, dx, dy) for o, dx, dy in _moved(po, co) if 4 <= o[\\\"size\\\"] <= 400]   # ignore 1-3 px specks (ar25 false trigger)\\n            if len(mv) < 2:\\n                continue\\n            seen_moves += 1\\n            adx, ady = moves[a]\\n            for o, dx, dy in mv:\\n                # a pure vertical move is consistent with a horizontal mirror (and vice versa): it counts as support but is not decisive;\\n                # a rule needs at least one decisive observation (otherwise co-moving avatar parts / trails would look mirrored: ls20, sk48)\\n                if (dx, dy) == (-adx, ady) and not (adx == 0 and ady == 0):\\n                    pairs[(\\\"mirror_x\\\", o[\\\"color\\\"], o[\\\"size\\\"])] += 1\\n                    if adx != 0:\\n                        decisive[(\\\"mirror_x\\\", o[\\\"color\\\"], o[\\\"size\\\"])] += 1\\n                if (dx, dy) == (adx, -ady) and not (adx == 0 and ady == 0):\\n                    pairs[(\\\"mirror_y\\\", o[\\\"color\\\"], o[\\\"size\\\"])] += 1\\n                    if ady != 0:\\n                        decisive[(\\\"mirror_y\\\", o[\\\"color\\\"], o[\\\"size\\\"])] += 1\\n                if (dx, dy) == (-adx, -ady) and adx != 0 and ady != 0:\\n                    pairs[(\\\"mirror_xy\\\", o[\\\"color\\\"], o[\\\"size\\\"])] += 1\\n                    decisive[(\\\"mirror_xy\\\", o[\\\"color\\\"], o[\\\"size\\\"])] += 1\\n        for (kind, color, size), n in pairs.items():\\n            if n >= 2 and n * 2 >= seen_moves and decisive[(kind, color, size)] >= 1:   # moves in at least half of the observed moves, with a decisive one\\n                rules.append(Rule(\\\"mirror\\\", {\\\"how\\\": kind, \\\"color\\\": color, \\\"size\\\": size}, support=n, counter=max(0, seen_moves - n)))\\n    # 2. gauge and refills\\n    g = nav.gauge()\\n    av0 = nav.avatar(); avatar_cols = set(av0[\\\"colors\\\"]) if av0 else set()\\n    for (color, size), _ in nav.player_votes.most_common(5):\\n        avatar_cols.add(color)\\n    if g:\\n        rules.append(Rule(\\\"gauge\\\", {\\\"color\\\": g[\\\"color\\\"], \\\"per_action\\\": g[\\\"per_action\\\"]}, support=len(transitions)))\\n        color = g[\\\"color\\\"]\\n        refills: Counter = Counter(); amounts: dict = {}\\n        for t in transitions:\\n            b, af = _count(t.before_frame, color), _count(t.after_frame, color)\\n            if af > b + 2:\\n                # a small non-avatar object that vanished where the avatar arrived\\n                av_a = NavHelper(transitions, t.after_frame).avatar()\\n                after_objs = _objs(t.after_frame)[0]\\n                gone = [o for o in _objs(t.before_frame)[0] if o[\\\"size\\\"] <= 40 and o[\\\"color\\\"] not in avatar_cols and o[\\\"color\\\"] != nav.background\\n                        and o[\\\"color\\\"] not in nav.floor_colors\\n                        and not any(p[\\\"color\\\"] == o[\\\"color\\\"] and p[\\\"center\\\"] == o[\\\"center\\\"] for p in after_objs)\\n                        and (not av_a or (abs(o[\\\"center\\\"][1] - av_a[\\\"row\\\"]) <= 6 and abs(o[\\\"center\\\"][0] - av_a[\\\"col\\\"]) <= 6))]\\n                for o in gone[:1]:\\n                    refills[o[\\\"color\\\"]] += 1; amounts[o[\\\"color\\\"]] = af - b\\n        for c, n in refills.items():\\n            rules.append(Rule(\\\"refill\\\", {\\\"color\\\": c, \\\"amount\\\": amounts[c]}, support=n))\\n    # 3. collectibles and hazards (movement games)\\n    if moves:\\n        vanish: Counter = Counter(); reset_colors: Counter = Counter()\\n        av = nav.avatar(); avatar_colors = set(av[\\\"colors\\\"]) if av else set()\\n        for (color, size), _ in nav.player_votes.most_common(5):\\n            avatar_colors.add(color)\\n        start = avatar_positions[0][:2] if avatar_positions else None\\n        for rb, cb, ra, ca, a, t in avatar_positions:\\n            before_objs, _ = _objs(t.before_frame); after_objs, _ = _objs(t.after_frame)\\n            after_keys = {(o[\\\"color\\\"], o[\\\"center\\\"]) for o in after_objs}\\n            for o in before_objs:\\n                if o[\\\"size\\\"] <= 60 and o[\\\"color\\\"] not in avatar_colors and (o[\\\"color\\\"], o[\\\"center\\\"]) not in after_keys:\\n                    # object vanished; did the avatar arrive at it?\\n                    if abs(o[\\\"center\\\"][1] - ra) <= 4 and abs(o[\\\"center\\\"][0] - ca) <= 4:\\n                        vanish[o[\\\"color\\\"]] += 1\\n            if start and (ra, ca) == start and (rb, cb) != start and abs(rb - ra) + abs(cb - ca) > 6:\\n                dx, dy = moves[a]; r, c = rb + dy, cb + dx\\n                if 0 <= r < 64 and 0 <= c < 64:\\n                    reset_colors[t.before_frame.grid[r][c]] += 1\\n        for color, n in vanish.items():\\n            if color != nav.background and color not in avatar_colors and color not in nav.floor_colors:\\n                rules.append(Rule(\\\"collect\\\", {\\\"color\\\": color}, support=n))\\n        for color, n in reset_colors.items():\\n            rules.append(Rule(\\\"hazard\\\", {\\\"color\\\": color}, support=n))\\n    # 4. click effects\\n    clicks: dict[int, Counter] = defaultdict(Counter); click_n: Counter = Counter()\\n    for t in transitions:\\n        if not isinstance(t.action, dict):\\n            continue\\n        r, c = t.action[\\\"row\\\"], t.action[\\\"col\\\"]\\n        color = t.before_frame.grid[r][c]\\n        click_n[color] += 1\\n        changed = {t.before_frame.grid[i][j] for i in range(64) for j in range(64)\\n                   if masked_ascii(t.before_frame).splitlines()[i][j] != masked_ascii(t.after_frame).splitlines()[i][j]}\\n        clicks[color].update(changed)\\n    for color, n in click_n.items():\\n        rules.append(Rule(\\\"click\\\", {\\\"color\\\": color, \\\"changes\\\": set(clicks[color])}, support=n))\\n    # 5. no-ops\\n    noop: Counter = Counter()\\n    for t in transitions:\\n        a = t.action if isinstance(t.action, str) else \\\"MOUSE\\\"\\n        if a != \\\"MOUSE\\\" and masked_ascii(t.before_frame) == masked_ascii(t.after_frame):\\n            noop[a] += 1\\n    for a, n in noop.items():\\n        if a not in moves:\\n            rules.append(Rule(\\\"noop\\\", {\\\"action\\\": a}, support=n))\\n    # unexplained: transitions with a masked change that no rule kind covers\\n    unexplained = []\\n    for i, t in enumerate(transitions):\\n        a = t.action if isinstance(t.action, str) else \\\"MOUSE\\\"\\n        if a == \\\"MOUSE\\\" or a in moves or a in noop:\\n            continue\\n        if masked_ascii(t.before_frame) != masked_ascii(t.after_frame):\\n            unexplained.append(f\\\"transition {i}: {a} changed the board but matches no known rule\\\")\\n    return rules, unexplained\\n\\n\\ndef summary(rules: list[Rule], unexplained: list[str]) -> str:\\n    conf = [r for r in rules if r.confirmed]; weak = [r for r in rules if not r.confirmed and r.support >= 1]\\n    lines = [\\\"CONFIRMED RULES (fit every recorded transition):\\\"] + [f\\\"  - {r.text()}\\\" for r in conf] if conf else [\\\"CONFIRMED RULES: none yet\\\"]\\n    if weak:\\n        lines.append(\\\"TENTATIVE (seen once or with counter-examples): \\\" + \\\"; \\\".join(r.text() for r in weak[:6]))\\n    if unexplained:\\n        lines.append(\\\"UNEXPLAINED: \\\" + \\\"; \\\".join(unexplained[:4]))\\n    return \\\"\\\\n\\\".join(lines)\\n\", \"runner.py\": \"\\\"\\\"\\\"Play a list of games (thread pool) and write an experiments-style result JSON.\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport datetime as dt\\nimport json\\nimport logging\\nimport os\\nimport subprocess\\nimport sys\\nimport threading\\nfrom concurrent.futures import ThreadPoolExecutor\\nfrom pathlib import Path\\nfrom typing import Optional\\n\\nROOT = Path(__file__).resolve().parents[1]\\n\\n# default = no chain-of-thought (set C: 2.30 vs 0.9 with thinking on the 4-game set; turns are 2x cheaper); configs/think.json re-enables it\\nDEFAULT_CONFIG = {\\\"base_url\\\": \\\"http://127.0.0.1:1234/v1\\\", \\\"model\\\": \\\"local-qwen\\\", \\\"temperature\\\": 0.7, \\\"top_p\\\": 0.8, \\\"max_tokens\\\": 4096,\\n                  \\\"extra_body\\\": {\\\"chat_template_kwargs\\\": {\\\"enable_thinking\\\": False}},\\n                  \\\"max_minutes\\\": 20, \\\"max_actions\\\": 3000, \\\"max_model_turns\\\": 400, \\\"tool_timeout\\\": 30, \\\"context_tokens\\\": 32768, \\\"jobs\\\": 2}\\n\\n\\ndef _git_info() -> dict:\\n    try:\\n        rev = subprocess.check_output([\\\"git\\\", \\\"rev-parse\\\", \\\"--short\\\", \\\"HEAD\\\"], cwd=ROOT, text=True).strip()\\n        dirty = bool(subprocess.check_output([\\\"git\\\", \\\"status\\\", \\\"--porcelain\\\", \\\"arcnav\\\"], cwd=ROOT, text=True).strip())\\n        return {\\\"commit\\\": rev, \\\"dirty\\\": dirty}\\n    except Exception:\\n        return {}\\n\\n\\ndef make_arcade(environments_dir: Optional[str] = None, *, competition: bool = False, base_url: Optional[str] = None):\\n    import arc_agi\\n    from arc_agi import OperationMode\\n    os.environ.setdefault(\\\"ARC_API_KEY\\\", \\\"local-dev\\\")\\n    if competition:\\n        return arc_agi.Arcade(operation_mode=OperationMode.COMPETITION, arc_base_url=base_url or os.environ.get(\\\"ARC_BASE_URL\\\", \\\"http://gateway:8001/\\\"))\\n    return arc_agi.Arcade(operation_mode=OperationMode.OFFLINE, environments_dir=environments_dir or str(ROOT / \\\"environment_files\\\"))\\n\\n\\ndef play_game(arc, game_id: str, cfg: dict, log_dir: Path, *, lock: threading.Lock) -> dict:\\n    from .agent import GameSession\\n    from .llm import ChatClient\\n    client = None if cfg.get(\\\"no_model\\\") else ChatClient(cfg[\\\"base_url\\\"], cfg[\\\"model\\\"], temperature=cfg[\\\"temperature\\\"], top_p=cfg[\\\"top_p\\\"],\\n                                                          max_tokens=cfg[\\\"max_tokens\\\"], extra_body=cfg.get(\\\"extra_body\\\") or {})\\n    with lock:\\n        env = arc.make(game_id)\\n    if env is None:\\n        return {\\\"game_id\\\": game_id, \\\"error\\\": \\\"could not create env\\\"}\\n    sess = GameSession(env, game_id, client, log_dir=log_dir, max_minutes=cfg[\\\"max_minutes\\\"], max_actions=cfg[\\\"max_actions\\\"],\\n                       max_model_turns=cfg[\\\"max_model_turns\\\"], tool_timeout=cfg[\\\"tool_timeout\\\"], context_tokens=cfg[\\\"context_tokens\\\"],\\n                       verbose=cfg.get(\\\"verbose\\\", True), deadline=cfg.get(\\\"deadline\\\"), think_first_turns=int(cfg.get(\\\"think_first_turns\\\", 0)), tool_choice_required=bool(cfg.get(\\\"tool_choice_required\\\", False)),\\n                       oracle_rules=(cfg.get(\\\"oracle_rules\\\") or {}).get(game_id.split(\\\"-\\\")[0], \\\"\\\"), image_context=bool(cfg.get(\\\"image_context\\\", False)))\\n    try:\\n        return sess.play()\\n    except Exception as e:\\n        logging.exception(\\\"game %s crashed\\\", game_id)\\n        sess.sandbox.close()\\n        return {\\\"game_id\\\": game_id, \\\"error\\\": f\\\"{type(e).__name__}: {e}\\\", \\\"actions\\\": sess.actions_used, \\\"levels_completed\\\": sess.level - 1}\\n\\n\\ndef run(game_ids: list[str], cfg: dict, *, out_dir: Path, tag: str = \\\"\\\", environments_dir: Optional[str] = None, arc=None) -> dict:\\n    run_id = dt.datetime.now(dt.timezone.utc).strftime(\\\"%Y%m%d-%H%M%S\\\") + f\\\"-{os.getpid()}\\\"\\n    log_dir = out_dir / \\\"logs\\\" / run_id\\n    arc = arc or make_arcade(environments_dir)\\n    lock = threading.Lock()\\n    started = dt.datetime.now(dt.timezone.utc)\\n    with ThreadPoolExecutor(max_workers=int(cfg.get(\\\"jobs\\\", 2))) as ex:\\n        games = list(ex.map(lambda g: play_game(arc, g, cfg, log_dir, lock=lock), game_ids))\\n    try:\\n        sc = arc.get_scorecard(); scd = sc.model_dump() if hasattr(sc, \\\"model_dump\\\") else {}\\n    except Exception as e:  # the competition gateway scores on its own side\\n        scd = {\\\"error\\\": repr(e)}\\n    by_id = {}\\n    for e in scd.get(\\\"environments\\\", []) or []:\\n        if e.get(\\\"runs\\\"):\\n            by_id[e[\\\"id\\\"].split(\\\"-\\\")[0]] = e[\\\"runs\\\"][-1]\\n    for g in games:\\n        r = by_id.get(g[\\\"game_id\\\"].split(\\\"-\\\")[0])\\n        if r:\\n            g[\\\"score\\\"] = r.get(\\\"score\\\"); g[\\\"level_scores\\\"] = r.get(\\\"level_scores\\\"); g[\\\"level_baseline_actions\\\"] = r.get(\\\"level_baseline_actions\\\")\\n    scores = [g.get(\\\"score\\\") or 0.0 for g in games]\\n    result = {\\\"schema\\\": 1, \\\"experiment\\\": \\\"arcnav\\\", \\\"run_id\\\": run_id, \\\"started_at\\\": started.isoformat(),\\n              \\\"finished_at\\\": dt.datetime.now(dt.timezone.utc).isoformat(), \\\"git\\\": _git_info(), \\\"tag\\\": tag,\\n              \\\"config\\\": {\\\"description\\\": \\\"arcnav: tool-using LLM harness with nav helper and solver synthesis\\\", \\\"games\\\": game_ids, \\\"params\\\": cfg},\\n              \\\"aggregate\\\": {\\\"score\\\": sum(scores) / max(1, len(scores)), \\\"levels_completed\\\": sum(g.get(\\\"levels_completed\\\", 0) for g in games),\\n                            \\\"games_level2plus\\\": sum(1 for g in games if g.get(\\\"levels_completed\\\", 0) >= 2),\\n                            \\\"actions\\\": sum(g.get(\\\"actions\\\", 0) for g in games), \\\"games_played\\\": len(games)},\\n              \\\"games\\\": games, \\\"scorecard\\\": scd}\\n    out_dir.mkdir(parents=True, exist_ok=True)\\n    out = out_dir / f\\\"run-{run_id}.json\\\"\\n    out.write_text(json.dumps(result, indent=2, default=str))\\n    print(f\\\"\\\\n========= arcnav {tag} =========\\\")\\n    for g in games:\\n        print(f\\\"  {g['game_id']:8} levels={g.get('levels_completed', '?'):>3}/{g.get('levels_total', '?')} actions={g.get('actions', '?'):>5} \\\"\\n              f\\\"score={g.get('score', 0) or 0:.3f} turns={g.get('model_turns', '?')} solver={g.get('solver_stored', '?')} stop={g.get('stop_reason', g.get('error'))}\\\")\\n    print(f\\\"Aggregate score: {result['aggregate']['score']:.4f} | games with level>=2: {result['aggregate']['games_level2plus']}/{len(games)}\\\\nResult saved: {out}\\\")\\n    return result\\n\\n\\ndef main(argv=None) -> None:\\n    ap = argparse.ArgumentParser(description=\\\"Play ARC-AGI-3 games with the arcnav agent\\\")\\n    ap.add_argument(\\\"--games\\\", default=\\\"ls20\\\", help=\\\"comma-separated game ids or 'all'\\\")\\n    ap.add_argument(\\\"--config\\\", help=\\\"JSON file overriding DEFAULT_CONFIG\\\")\\n    ap.add_argument(\\\"--out\\\", default=str(ROOT / \\\"experiments\\\" / \\\"arcnav\\\" / \\\"results\\\"))\\n    ap.add_argument(\\\"--tag\\\", default=\\\"\\\")\\n    ap.add_argument(\\\"--minutes\\\", type=float); ap.add_argument(\\\"--jobs\\\", type=int); ap.add_argument(\\\"--max-actions\\\", type=int)\\n    ap.add_argument(\\\"--no-model\\\", action=\\\"store_true\\\", help=\\\"only the environment/sandbox path (smoke test)\\\")\\n    ap.add_argument(\\\"--quiet\\\", action=\\\"store_true\\\")\\n    a = ap.parse_args(argv)\\n    logging.basicConfig(level=logging.WARNING)\\n    cfg = dict(DEFAULT_CONFIG)\\n    if a.config:\\n        cfg.update(json.loads(Path(a.config).read_text()))\\n    if a.minutes: cfg[\\\"max_minutes\\\"] = a.minutes\\n    if a.jobs: cfg[\\\"jobs\\\"] = a.jobs\\n    if a.max_actions: cfg[\\\"max_actions\\\"] = a.max_actions\\n    cfg[\\\"no_model\\\"] = a.no_model; cfg[\\\"verbose\\\"] = not a.quiet\\n    arc = make_arcade()\\n    if a.games == \\\"all\\\":\\n        games = sorted(e.game_id.split(\\\"-\\\")[0] for e in arc.get_environments())\\n    else:\\n        games = [g.strip() for g in a.games.split(\\\",\\\") if g.strip()]\\n    run(games, cfg, out_dir=Path(a.out), tag=a.tag, arc=arc)\\n\\n\\nif __name__ == \\\"__main__\\\":\\n    main()\\n\", \"sandbox.py\": \"\\\"\\\"\\\"Python tool sandbox: runs model-written code in a separate `python -I`\\nprocess with a JSON-lines protocol. Inside the child the code sees\\n`current_frame`, `history`, `transitions`, `valid_actions`, `nav`,\\n`action(...)` (executes actions on the host and refreshes state) and\\n`propose_solver(code)` (stores a persistent `solve()` on the host).\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport json\\nimport os\\nimport subprocess\\nimport sys\\nimport threading\\nimport time\\nfrom pathlib import Path\\nfrom typing import Any, Callable, Optional\\n\\n_PKG = Path(__file__).resolve().parent\\n_SANDBOX_SRC_FILES = (\\\"frame.py\\\", \\\"nav.py\\\")\\n\\nCHILD_PROGRAM = r'''\\nimport ast, json, os, resource, signal, sys, traceback, types\\n_out = os.fdopen(os.dup(1), \\\"w\\\"); _in = sys.stdin; _err = sys.stderr\\n_devnull = os.open(os.devnull, os.O_WRONLY); os.dup2(_devnull, 1)   # fd 1 is no longer the protocol channel\\nsys.stdout = open(os.devnull, \\\"w\\\")\\n\\ndef _send(obj):\\n    _out.write(json.dumps(obj) + \\\"\\\\n\\\"); _out.flush()\\n\\ndef _recv():\\n    line = _in.readline()\\n    if not line:\\n        raise SystemExit(0)\\n    return json.loads(line)\\n\\n_init = _recv()\\n_mods = {}\\nfor _name, _src in _init[\\\"sources\\\"].items():\\n    _m = types.ModuleType(\\\"arcnav.\\\" + _name); sys.modules[\\\"arcnav.\\\" + _name] = _m\\n    exec(compile(_src, _name + \\\".py\\\", \\\"exec\\\"), _m.__dict__); _mods[_name] = _m\\nos.dup2(_devnull, 2); sys.stderr = open(os.devnull, \\\"w\\\")\\nFrame = _mods[\\\"frame\\\"].Frame\\nNavHelper = _mods[\\\"nav\\\"].NavHelper\\n\\nclass Transition:\\n    \\\"\\\"\\\"One executed action: `action` (name or MOUSE(row=r, col=c)), `before_frame`, `after_frame`, `changed`.\\\"\\\"\\\"\\n    def __init__(self, action, before_frame, after_frame, result=None):\\n        self.action, self.before_frame, self.after_frame, self.result = action, before_frame, after_frame, result or {}\\n    @property\\n    def changed(self):\\n        return self.before_frame.ascii != self.after_frame.ascii\\n    def __repr__(self):\\n        return f\\\"Transition({self.action}, changed={self.changed}, level={self.after_frame.level})\\\"\\n\\nG = {\\\"__name__\\\": \\\"__main__\\\", \\\"Frame\\\": Frame, \\\"NavHelper\\\": NavHelper, \\\"Transition\\\": Transition}\\n\\ndef _refresh(state):\\n    G[\\\"current_frame\\\"] = Frame.from_payload(state[\\\"frame\\\"]) if state.get(\\\"frame\\\") else None\\n    G[\\\"valid_actions\\\"] = list(state.get(\\\"valid_actions\\\") or [])\\n    G[\\\"level\\\"] = int(state.get(\\\"level\\\", 1)); G[\\\"levels_total\\\"] = int(state.get(\\\"levels_total\\\", 0))\\n    trans = []\\n    for t in state.get(\\\"transitions\\\") or []:\\n        trans.append(Transition(t[\\\"action\\\"], Frame.from_payload(t[\\\"before\\\"]), Frame.from_payload(t[\\\"after\\\"]), t.get(\\\"result\\\")))\\n    G[\\\"transitions\\\"] = trans\\n    G[\\\"history\\\"] = trans  # alias\\n    G[\\\"last_action_result\\\"] = state.get(\\\"last_action_result\\\")\\n    G[\\\"level_recaps\\\"] = list(state.get(\\\"level_recaps\\\") or [])\\n    G[\\\"rules_text\\\"] = state.get(\\\"rules_text\\\") or \\\"\\\"\\n    G[\\\"goal_hypotheses\\\"] = list(state.get(\\\"goal_hypotheses\\\") or [])\\n    if \\\"notes\\\" not in G or not G[\\\"notes\\\"]:\\n        G[\\\"notes\\\"] = state.get(\\\"notes\\\") or \\\"\\\"\\n    if not isinstance(G.get(\\\"checklist\\\"), dict) or not G[\\\"checklist\\\"]:\\n        G[\\\"checklist\\\"] = dict(state.get(\\\"checklist\\\") or {\\\"goal\\\": \\\"\\\", \\\"roles\\\": {}, \\\"plan\\\": \\\"\\\", \\\"tried\\\": []})\\n    try:\\n        cur = [t for t in trans if t.before_frame.level == t.after_frame.level == G[\\\"level\\\"]]\\n        G[\\\"level_transitions\\\"] = cur\\n        G[\\\"nav\\\"] = NavHelper(cur, G[\\\"current_frame\\\"]) if G[\\\"current_frame\\\"] is not None else None\\n        G[\\\"nav_error\\\"] = None\\n    except Exception as e:  # pragma: no cover\\n        G[\\\"nav\\\"], G[\\\"nav_error\\\"] = None, repr(e)\\n\\ndef _normalize(actions):\\n    if isinstance(actions, (str, dict)):\\n        actions = [actions]\\n    if not isinstance(actions, (list, tuple)) or not actions:\\n        raise ValueError(\\\"action(actions) expects an action or a non-empty list of actions\\\")\\n    out = []\\n    for a in actions:\\n        if isinstance(a, str):\\n            if a.strip().upper() in (\\\"MOUSE\\\", \\\"CLICK\\\", \\\"ACTION6\\\"):\\n                raise ValueError(\\\"MOUSE needs coordinates: use {'action': 'MOUSE', 'row': r, 'col': c}\\\")\\n            out.append({\\\"action\\\": a.strip().upper()})\\n        elif isinstance(a, dict):\\n            d = {k: v for k, v in a.items()}; d[\\\"action\\\"] = str(d.get(\\\"action\\\", \\\"\\\")).strip().upper()\\n            if d[\\\"action\\\"] in (\\\"MOUSE\\\", \\\"CLICK\\\", \\\"ACTION6\\\"):\\n                d[\\\"action\\\"] = \\\"MOUSE\\\"\\n                if \\\"row\\\" not in d or \\\"col\\\" not in d:\\n                    raise ValueError(\\\"MOUSE needs row and col (0-63)\\\")\\n                d[\\\"row\\\"], d[\\\"col\\\"] = int(d[\\\"row\\\"]), int(d[\\\"col\\\"])\\n                if not (0 <= d[\\\"row\\\"] <= 63 and 0 <= d[\\\"col\\\"] <= 63):\\n                    raise ValueError(f\\\"MOUSE coordinates must be within 0..63 (got row={d['row']}, col={d['col']})\\\")\\n            out.append(d)\\n        else:\\n            raise ValueError(f\\\"unsupported action {a!r}\\\")\\n    return out\\n\\ndef action(actions):\\n    \\\"\\\"\\\"Execute one or more actions on the game and refresh all runtime variables.\\n    Returns {'executed_count', 'board_changed', 'level_completed', 'game_over', 'stopped_reason', 'results'}.\\\"\\\"\\\"\\n    acts = _normalize(actions)\\n    _send({\\\"type\\\": \\\"action\\\", \\\"actions\\\": acts})\\n    reply = _recv()\\n    if reply.get(\\\"type\\\") != \\\"action_result\\\":\\n        raise RuntimeError(\\\"protocol error\\\")\\n    _refresh(reply[\\\"state\\\"])\\n    return reply[\\\"result\\\"]\\n\\ndef propose_solver(code, verify_last=12):\\n    \\\"\\\"\\\"Store a persistent solver. `code` must define `def solve():` returning the next actions (list) or [].\\n    Optional `def predict(before_frame, action_name)` -> ascii string is checked against recent transitions.\\\"\\\"\\\"\\n    if not isinstance(code, str):\\n        raise TypeError(\\\"propose_solver(code) expects a string of python code\\\")\\n    report = {\\\"ok\\\": False}\\n    try:\\n        tree = ast.parse(code)\\n    except SyntaxError as e:\\n        report[\\\"reason\\\"] = f\\\"syntax error: {e}\\\"; _send({\\\"type\\\": \\\"solver\\\", \\\"code\\\": code, \\\"report\\\": report}); return _recv().get(\\\"result\\\", report)\\n    names = {n.name for n in tree.body if isinstance(n, ast.FunctionDef)}\\n    if \\\"solve\\\" not in names:\\n        report[\\\"reason\\\"] = \\\"code must define `def solve():`\\\"; _send({\\\"type\\\": \\\"solver\\\", \\\"code\\\": code, \\\"report\\\": report}); return _recv().get(\\\"result\\\", report)\\n    if \\\"propose_solver\\\" in names or \\\"action\\\" in names:\\n        report[\\\"reason\\\"] = \\\"do not redefine the built-ins `propose_solver` / `action`\\\"; _send({\\\"type\\\": \\\"solver\\\", \\\"code\\\": code, \\\"report\\\": report}); return _recv().get(\\\"result\\\", report)\\n    ns = dict(G)\\n    try:\\n        exec(compile(code, \\\"solver.py\\\", \\\"exec\\\"), ns)\\n        dry = ns[\\\"solve\\\"]()\\n        dry = list(dry or [])\\n        _normalize(dry) if dry else None\\n        report[\\\"dry_run_actions\\\"] = [a if isinstance(a, str) else dict(a) for a in dry[:8]]\\n    except Exception as e:\\n        report[\\\"reason\\\"] = f\\\"solve() raised on dry run: {type(e).__name__}: {e}\\\"\\n        _send({\\\"type\\\": \\\"solver\\\", \\\"code\\\": code, \\\"report\\\": report}); return _recv().get(\\\"result\\\", report)\\n    if \\\"predict\\\" in names and G.get(\\\"transitions\\\"):\\n        checked = G[\\\"transitions\\\"][-verify_last:]; hits = 0; mism = []\\n        for t in checked:\\n            try:\\n                p = ns[\\\"predict\\\"](t.before_frame, str(t.action))\\n            except Exception as e:\\n                p = None; mism.append(f\\\"predict raised {type(e).__name__}: {e}\\\")\\n            if p is None:\\n                continue\\n            if str(p).strip() == t.after_frame.ascii.strip():\\n                hits += 1\\n            else:\\n                exp, got = t.after_frame.ascii.splitlines(), str(p).strip().splitlines()\\n                bad = [i for i in range(min(len(exp), len(got))) if exp[i] != got[i]][:3]\\n                mism.append(f\\\"{t.action}: rows differ at {bad}\\\")\\n        report[\\\"predict_accuracy\\\"] = hits / max(1, len(checked)); report[\\\"predict_mismatches\\\"] = mism[:5]\\n        if report[\\\"predict_accuracy\\\"] < 0.5:\\n            report[\\\"reason\\\"] = f\\\"predict() accuracy {report['predict_accuracy']:.2f} < 0.5 \\u2014 fix the world model first\\\"\\n            _send({\\\"type\\\": \\\"solver\\\", \\\"code\\\": code, \\\"report\\\": report}); return _recv().get(\\\"result\\\", report)\\n    report[\\\"ok\\\"] = True\\n    report[\\\"verified\\\"] = bool(\\\"predict\\\" in names and report.get(\\\"predict_accuracy\\\", 0) >= 0.8 and len(G.get(\\\"transitions\\\") or []) >= 6)\\n    exec(compile(code, \\\"solver.py\\\", \\\"exec\\\"), G)   # accepted: solve()/predict() live in the runtime namespace\\n    _send({\\\"type\\\": \\\"solver\\\", \\\"code\\\": code, \\\"report\\\": report})\\n    return _recv().get(\\\"result\\\", report)\\n\\nG[\\\"action\\\"] = action; G[\\\"propose_solver\\\"] = propose_solver\\n_refresh(_init[\\\"state\\\"])\\n\\ndef _run(code):\\n    tree = ast.parse(code)\\n    if tree.body and isinstance(tree.body[-1], ast.Expr):\\n        last = ast.Expression(tree.body.pop().value)\\n        exec(compile(tree, \\\"<tool>\\\", \\\"exec\\\"), G)\\n        v = eval(compile(last, \\\"<tool>\\\", \\\"eval\\\"), G)\\n        if v is not None:\\n            print(repr(v), file=_cap)\\n    else:\\n        exec(compile(tree, \\\"<tool>\\\", \\\"exec\\\"), G)\\n\\nimport io\\n_cap = io.StringIO()\\nresource.setrlimit(resource.RLIMIT_AS, (4 << 30, 4 << 30))\\nwhile True:\\n    msg = _recv()\\n    if msg.get(\\\"type\\\") == \\\"state\\\":\\n        _refresh(msg[\\\"state\\\"]); continue\\n    if msg.get(\\\"type\\\") != \\\"run\\\":\\n        break\\n    _cap = io.StringIO(); sys.stdout = sys.stderr = _cap\\n    signal.signal(signal.SIGALRM, lambda *_: (_ for _ in ()).throw(TimeoutError(\\\"tool timeout\\\")))\\n    signal.alarm(int(msg.get(\\\"timeout\\\", 30)))\\n    err = None\\n    try:\\n        _run(msg[\\\"code\\\"])\\n    except SystemExit:\\n        pass\\n    except BaseException as e:\\n        tb = traceback.format_exception_only(type(e), e)\\n        err = \\\"\\\".join(tb).strip()\\n        if isinstance(e, TimeoutError):\\n            err = \\\"TimeoutError: tool timeout\\\"\\n    finally:\\n        signal.alarm(0)\\n    sys.stdout = sys.stderr = open(os.devnull, \\\"w\\\")\\n    try:\\n        _ck = json.loads(json.dumps(G.get(\\\"checklist\\\"), default=str))[:1] if isinstance(G.get(\\\"checklist\\\"), list) else json.loads(json.dumps(G.get(\\\"checklist\\\"), default=str))\\n    except Exception:\\n        _ck = {}\\n    _send({\\\"type\\\": \\\"done\\\", \\\"stdout\\\": _cap.getvalue(), \\\"error\\\": err, \\\"notes\\\": str(G.get(\\\"notes\\\") or \\\"\\\")[:3000], \\\"checklist\\\": _ck if isinstance(_ck, dict) else {}})\\n'''\\n\\n\\ndef _sources() -> dict[str, str]:\\n    return {name[:-3]: (_PKG / name).read_text() for name in _SANDBOX_SRC_FILES}\\n\\n\\nclass Sandbox:\\n    \\\"\\\"\\\"A persistent child interpreter for one game. `action_handler(actions) -> (result, state)`;\\n    `solver_handler(code, report) -> result`; `state_provider() -> state payload`.\\\"\\\"\\\"\\n\\n    def __init__(self, *, action_handler: Callable[[list[dict]], tuple[dict, dict]],\\n                 solver_handler: Callable[[str, dict], dict], state_provider: Callable[[], dict],\\n                 timeout: int = 30, max_output_chars: int = 6000):\\n        self.action_handler, self.solver_handler, self.state_provider = action_handler, solver_handler, state_provider\\n        self.timeout, self.max_output_chars = timeout, max_output_chars\\n        self.proc: Optional[subprocess.Popen] = None\\n\\n    def _start(self) -> None:\\n        env = {\\\"PATH\\\": os.environ.get(\\\"PATH\\\", \\\"\\\"), \\\"PYTHONHASHSEED\\\": \\\"0\\\", \\\"HOME\\\": os.environ.get(\\\"HOME\\\", \\\"/tmp\\\")}\\n        self.proc = subprocess.Popen([sys.executable, \\\"-I\\\", \\\"-c\\\", CHILD_PROGRAM], stdin=subprocess.PIPE, stdout=subprocess.PIPE,\\n                                     stderr=subprocess.DEVNULL, text=True, env=env, start_new_session=True)\\n        self._send({\\\"sources\\\": _sources(), \\\"state\\\": self.state_provider()})\\n\\n    def _send(self, obj: dict) -> None:\\n        assert self.proc and self.proc.stdin\\n        self.proc.stdin.write(json.dumps(obj) + \\\"\\\\n\\\"); self.proc.stdin.flush()\\n\\n    def _recv(self) -> dict:\\n        assert self.proc and self.proc.stdout\\n        line = self.proc.stdout.readline()\\n        if not line:\\n            raise RuntimeError(\\\"sandbox died\\\")\\n        return json.loads(line)\\n\\n    def close(self) -> None:\\n        if self.proc:\\n            try:\\n                os.killpg(self.proc.pid, 9)\\n            except Exception:\\n                pass\\n            self.proc = None\\n\\n    def run(self, code: str, *, timeout: Optional[int] = None) -> dict:\\n        \\\"\\\"\\\"Run tool code; returns {'stdout', 'error', 'actions_executed', 'proposals': [...]}\\\"\\\"\\\"\\n        if self.proc is None or self.proc.poll() is not None:\\n            self.close(); self._start()\\n        else:\\n            self._send({\\\"type\\\": \\\"state\\\", \\\"state\\\": self.state_provider()})\\n        self._send({\\\"type\\\": \\\"run\\\", \\\"code\\\": code, \\\"timeout\\\": timeout or self.timeout})\\n        executed, proposals = 0, []\\n        deadline = time.time() + (timeout or self.timeout) + 15\\n        while True:\\n            try:\\n                msg = self._recv()\\n            except RuntimeError:\\n                self.close()\\n                return {\\\"stdout\\\": \\\"\\\", \\\"error\\\": \\\"RuntimeError: python tool process died (memory/time limit?)\\\", \\\"actions_executed\\\": executed, \\\"proposals\\\": proposals}\\n            t = msg.get(\\\"type\\\")\\n            if t == \\\"action\\\":\\n                result, state = self.action_handler(msg[\\\"actions\\\"])\\n                executed += int(result.get(\\\"executed_count\\\", 0))\\n                self._send({\\\"type\\\": \\\"action_result\\\", \\\"result\\\": result, \\\"state\\\": state})\\n            elif t == \\\"solver\\\":\\n                res = self.solver_handler(msg[\\\"code\\\"], msg[\\\"report\\\"]); proposals.append(res)\\n                self._send({\\\"type\\\": \\\"solver_result\\\", \\\"result\\\": res})\\n            elif t == \\\"done\\\":\\n                out = msg.get(\\\"stdout\\\", \\\"\\\")\\n                if len(out) > self.max_output_chars:\\n                    out = out[: self.max_output_chars // 2] + \\\"\\\\n...[truncated]...\\\\n\\\" + out[-self.max_output_chars // 2:]\\n                return {\\\"stdout\\\": out, \\\"error\\\": msg.get(\\\"error\\\"), \\\"actions_executed\\\": executed, \\\"proposals\\\": proposals, \\\"notes\\\": msg.get(\\\"notes\\\", \\\"\\\"),\\n                        \\\"checklist\\\": msg.get(\\\"checklist\\\") or {}}\\n            if time.time() > deadline:\\n                self.close()\\n                return {\\\"stdout\\\": \\\"\\\", \\\"error\\\": \\\"TimeoutError: tool exceeded its time limit\\\", \\\"actions_executed\\\": executed, \\\"proposals\\\": proposals}\\n\", \"solver.py\": \"\\\"\\\"\\\"Persistent-solver policy: when the model has stored a `solve()`, the host\\nruns it turn after turn without the model, and stops on clear failure signals.\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport json\\nfrom typing import Optional\\n\\nMAX_ACTIONS_PER_TURN = 12\\nNOOP_TURN_LIMIT = 2          # consecutive solver turns without a board change\\nNO_PROGRESS_ACTIONS = 80     # solver actions without a level advance\\nCYCLE_WINDOW = 6             # identical board seen this many times -> cycling\\nDEMAND_EVERY_TURNS: Optional[int] = None   # periodic proposal demand disabled (hurt free play in v012e/f)\\n\\nRUN_SNIPPET = \\\"\\\"\\\"\\nexec(compile(__solver_code, \\\"solver.py\\\", \\\"exec\\\"), globals())\\n__acts = list(solve() or [])\\nif not __acts:\\n    print(\\\"SOLVER_EMPTY\\\")\\nelse:\\n    __r = action(__acts[:__solver_budget])\\n    print(\\\"SOLVER_RAN\\\", len(__acts), __r.get(\\\"executed_count\\\"), __r.get(\\\"board_changed\\\"), __r.get(\\\"level_completed\\\"), __r.get(\\\"game_over\\\"))\\n\\\"\\\"\\\"\\n\\n\\ndef run_snippet(code: str, budget: int = MAX_ACTIONS_PER_TURN) -> str:\\n    import json as _j\\n    return f\\\"__solver_code = {_j.dumps(code)}\\\\n__solver_budget = {int(budget)}\\\\n\\\" + RUN_SNIPPET\\n\\n\\nVERIFY_TO_AUTORUN = True   # iter2: only solvers whose predict() scores >= 0.8 on >= 6 transitions run without the model\\n\\n\\ndef new_solver(code: str, report: dict) -> dict:\\n    status = \\\"active\\\" if (report.get(\\\"verified\\\") or not VERIFY_TO_AUTORUN) else \\\"draft\\\"\\n    return {\\\"code\\\": code, \\\"report\\\": report, \\\"status\\\": status, \\\"turns\\\": 0, \\\"actions_run\\\": 0, \\\"noop_turns\\\": 0,\\n            \\\"no_progress_actions\\\": 0, \\\"reason\\\": None, \\\"failure_shown\\\": 0, \\\"board_seen\\\": {}}\\n\\n\\ndef status_lines(solver: Optional[dict], *, model_turns: int, level_just_completed: bool) -> list[str]:\\n    if not solver:\\n        base = \\\"Solver: none stored yet.\\\"\\n        if level_just_completed:\\n            return [base + \\\" You just completed a level, so you know the rules: in THIS turn call propose_solver(code) with a solve() that \\\"\\n                    \\\"reproduces what worked (see the templates in the system prompt) and let the harness play the new level.\\\"]\\n        if DEMAND_EVERY_TURNS is not None and model_turns >= DEMAND_EVERY_TURNS:\\n            return [base + f\\\" {model_turns} thinking turns spent: encode your current plan in propose_solver(code) now; the harness reports failures.\\\"]\\n        return [base + \\\" When the rules are clear, call propose_solver(code) so the harness can play on without you.\\\"]\\n    st = solver.get(\\\"status\\\")\\n    if st == \\\"draft\\\":\\n        return [f\\\"Solver: a DRAFT is stored (not verified, so the harness does not run it by itself). Its suggestion for the current board: \\\"\\n                f\\\"{solver.get('suggestion', '?')}. To let it run automatically, add `def predict(before_frame, action_name)` that returns the \\\"\\n                \\\"expected next board (ascii) and re-propose: accuracy >= 0.8 on the recorded transitions makes it verified. Or execute the suggestion yourself.\\\"]\\n    if st == \\\"failed\\\":\\n        shown = solver.get(\\\"failure_shown\\\", 0); solver[\\\"failure_shown\\\"] = shown + 1\\n        head = (f\\\"Solver: your stored solver FAILED ({solver.get('reason')}) after {solver.get('actions_run', 0)} actions. \\\"\\n                \\\"Act manually for a few turns to learn what was wrong; when you know, repair the code and call propose_solver(code) again.\\\")\\n        return [head, f\\\"Failed solver code:\\\\n{solver.get('code', '')[:1500]}\\\"] if shown == 0 else [head]\\n    if st == \\\"rejected\\\":\\n        return [f\\\"Solver: your last proposal was rejected: {solver.get('reason')}. Report: {json.dumps(solver.get('report', {}))[:600]}\\\"]\\n    return []\\n\"}, \"rulebook\": {\"__init__.py\": \"\\\"\\\"\\\"rulebook \\u2014 a rulebook-driven ARC-AGI-3 agent (new line, 2026-09-27).\\n\\nThe agent keeps an explicit rulebook of hypotheses (environment facts, rules, win conditions). A deterministic predictor\\nderived from the recorded transitions tells, for every candidate action, what should happen; the model only chooses among\\nthe candidates; every executed action is checked against its prediction and a mismatch triggers a rulebook review.\\n\\\"\\\"\\\"\\n\", \"agent.py\": \"\\\"\\\"\\\"The rulebook loop.\\n\\n  level 1 start \\u2500 INIT: the model (thinking on) writes the rulebook from the first board.\\n  every decision \\u2500 the program lists candidate actions and PREDICTS each deterministically from the record;\\n                   the model (thinking off) picks one; the program executes it one action at a time, checking\\n                   every action against its prediction.\\n  mismatch / level end / game over \\u2500 the program re-induces its facts, scores the rulebook, then the model\\n                   (thinking on) REVIEWS the rulebook (refute / add / edit) and the plan.\\n\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport json\\nimport time\\nfrom pathlib import Path\\nfrom typing import Optional\\n\\nfrom arcnav.frame import masked_ascii\\n\\nfrom .book import Rulebook\\nfrom .candidates import Candidate, build, resolve, state_key\\nfrom .env import Game, action_label\\nfrom .llm_io import Model\\nfrom .predict import Evidence\\nfrom . import goals2\\nfrom .entities import build_scene\\nfrom .plans import make_plans\\nfrom .repl import Repl\\n\\nDEFAULTS = {\\\"max_minutes\\\": 20.0, \\\"level_actions\\\": 200, \\\"max_actions\\\": 2000, \\\"reviews_per_level\\\": 8, \\\"max_levels\\\": 10,\\n            \\\"review_think\\\": \\\"level\\\",   # thinking during reviews: \\\"level\\\" = only after a level completion, \\\"always\\\", \\\"never\\\"\\n            \\\"init_think\\\": True,        # thinking for the initial rulebook (off on Kaggle: the FP8 27B thinks past the token budget)\\n            \\\"mode\\\": \\\"choose\\\"}          # \\\"choose\\\": the model picks a candidate label; \\\"coder\\\": the model writes python against the library\\n\\n\\nclass RulebookAgent:\\n    def __init__(self, game: Game, model: Optional[Model], *, log_dir: Path, cfg: Optional[dict] = None, deadline: Optional[float] = None):\\n        self.g, self.model = game, model\\n        self.cfg = {**DEFAULTS, **(cfg or {})}\\n        self.log_dir = Path(log_dir); self.log_dir.mkdir(parents=True, exist_ok=True)\\n        self.book = Rulebook()\\n        self.tried: set = set()\\n        self.outcomes: list[str] = []\\n        self.mismatches = self.reviews = self.observations = 0\\n        self.level_stats: list[dict] = []\\n        self.deadline = deadline\\n        self.t0 = time.time()\\n        self._log_f = open(self.log_dir / f\\\"{game.game_id}.log\\\", \\\"a\\\")\\n        game.record_path = self.log_dir / f\\\"{game.game_id}.actions.jsonl\\\"\\n        self.last_choice: list[str] = []\\n        self.last_cell: Optional[tuple] = None   # cell of the last executed click: the fallback never clicks it again right away (toggle undo)\\n        self.last_verdict = None; self.last_essential: list = []; self.pending_mismatch = None\\n        self.custom_rules: list = []      # (name, fn, accuracy, n) model-written transition rules verified by replay\\n        self.idle_turns = 0\\n        from collections import Counter as _C\\n        self.metrics = _C()               # counters for the evaluation report (prediction kinds, verdicts, plans, fallbacks ...)\\n        self.attempt_start_actions = 0    # actions_used when the current attempt (level or reset) began\\n        self.plan_ignored = 0             # consecutive decisions in which the model ignored an offered plan\\n        self.pending_then: list = []      # follow-up labels from the last decision's \\\"then\\\" list\\n        self.plan_fail: dict = {}         # (level, plan label) -> executions that ended in a mismatch / game over\\n        self.refused: dict = {}           # (level, label) -> times the model asked for a label that is not a candidate\\n        self.game_over_at: dict = {}      # level -> [actions into the attempt at which a game over happened]\\n        from collections import Counter as _C\\n        self.metrics = _C()               # counters for the evaluation report (prediction kinds, verdicts, plans, fallbacks ...)\\n        self.attempt_start_actions = 0    # actions_used when the current attempt (level or reset) began\\n        self.plan_ignored = 0             # consecutive decisions in which the model ignored an offered plan\\n        self.pending_then: list = []      # follow-up labels from the last decision's \\\"then\\\" list\\n        self.plan_fail: dict = {}         # (level, plan label) -> executions that ended in a mismatch / game over\\n        self.refused: dict = {}           # (level, label) -> times the model asked for a label that is not a candidate\\n        self.game_over_at: dict = {}      # level -> [actions into the attempt at which a game over happened]\\n        self.repl = None\\n        self.distrust: dict = {}          # (level, click colour) -> failed board predictions; the predictor stops predicting after 2\\n        self.preds: dict = {}             # rulebook entry id -> goals2.Predicate (win predicates learned from completed levels)\\n        self.level_start_frame = None     # first board of the current attempt (goal inference compares against it)\\n\\n    # \\u2500\\u2500 logging \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\n    def log(self, text: str) -> None:\\n        self._log_f.write(f\\\"[{time.time() - self.t0:6.0f}s a{self.g.actions_used:4d} L{self.g.level}] {text}\\\\n\\\"); self._log_f.flush()\\n\\n    def save_book(self) -> None:\\n        self.book.save(self.log_dir / f\\\"{self.g.game_id}.rulebook.json\\\")\\n\\n    def _out_of_time(self) -> bool:\\n        if self.deadline and time.time() > self.deadline:\\n            return True\\n        return time.time() - self.t0 > self.cfg[\\\"max_minutes\\\"] * 60 or self.g.actions_used >= self.cfg[\\\"max_actions\\\"]\\n\\n    # \\u2500\\u2500 evidence \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\n    def evidence(self, full: bool = False) -> Evidence:\\n        lv = self.g.level\\n        return Evidence(self.g, distrust={key: n for (l, key), n in self.distrust.items() if l == lv}, custom_rules=self.custom_rules)\\n\\n    def sync(self, ev: Evidence) -> list[str]:\\n        changes = self.book.sync(ev.facts(), level=self.g.level)\\n        if changes:\\n            self.log(\\\"rulebook <- evidence: \\\" + \\\" | \\\".join(changes)); self.save_book()\\n        return changes\\n\\n    # \\u2500\\u2500 win predicates \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\n    def live_preds(self) -> list:\\n        # win entries of known kinds written by the model (or by the old goal inference) get predicate objects too\\n        for e in self.book.section(\\\"win\\\"):\\n            if e.id not in self.preds and e.status != \\\"refuted\\\":\\n                p = goals2.from_entry(e)\\n                if p is not None:\\n                    self.preds[e.id] = p\\n        return [(eid, p) for eid, p in self.preds.items() if (e := self.book.get(eid)) is not None and e.status != \\\"refuted\\\"]\\n\\n    def goal_state(self, ev: Evidence) -> dict:\\n        \\\"\\\"\\\"Evaluate every live win predicate on the current board: progress lines for the model, and whether a submit is due.\\\"\\\"\\\"\\n        sc = ev.scene; lines = []; held = []; missing = []; submit = None; has_submit = False\\n        for eid, p in self.live_preds():\\n            if isinstance(p, goals2.Pressed):\\n                has_submit = True; b = p.button(sc)\\n                if b is not None:\\n                    submit = b\\n                lines.append(f\\\"{eid} [submit] {p.evaluate(sc, ev)[1]}\\\"); continue\\n            ok, txt, _ = p.evaluate(sc, ev)\\n            (held if ok else missing).append(eid)\\n            lines.append(f\\\"{eid} [{p.kind}] {'HOLDS' if ok else 'not yet'}: {txt}\\\")\\n        conds = [eid for eid, p in self.live_preds() if not isinstance(p, goals2.Pressed)]\\n        return {\\\"lines\\\": lines, \\\"held\\\": held, \\\"missing\\\": missing, \\\"all_hold\\\": bool(conds) and not missing, \\\"submit\\\": submit if has_submit else None,\\n                \\\"has_submit\\\": has_submit, \\\"conds\\\": conds}\\n\\n    def refute_preds(self, ids: list, note: str) -> list:\\n        out = []\\n        for eid in ids:\\n            r = self.book.refute(eid, note)\\n            if r:\\n                out.append(r)\\n        if out:\\n            self.log(\\\"win predicates refuted: \\\" + \\\" | \\\".join(out)); self.save_book()\\n        return out\\n\\n    # \\u2500\\u2500 model steps \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\n    def init_book(self) -> None:\\n        if self.model is None:\\n            self.book.plan = \\\"no model: try untested actions, then act on harness-induced rules\\\"; return\\n        obj = self.model.init_rulebook(self.g, think=bool(self.cfg.get(\\\"init_think\\\", True)))\\n        if not obj:\\n            self.log(\\\"INIT failed; starting with an empty rulebook\\\"); return\\n        for sec in (\\\"env\\\", \\\"rules\\\", \\\"win\\\"):\\n            for e in (obj.get(sec) or [])[:10]:\\n                if isinstance(e, dict) and e.get(\\\"text\\\"):\\n                    self.book.add(sec, str(e[\\\"text\\\"]), kind=str(e.get(\\\"kind\\\") or \\\"other\\\"), params=e.get(\\\"params\\\") if isinstance(e.get(\\\"params\\\"), dict) else {}, level=self.g.level)\\n                elif isinstance(e, str):\\n                    self.book.add(sec, e, level=self.g.level)\\n        self.book.plan = str(obj.get(\\\"plan\\\", \\\"\\\"))[:600]\\n        self.log(\\\"INIT rulebook:\\\\n\\\" + self.book.render()); self.save_book()\\n\\n    def review(self, event: str, changes: list[str], *, before=None, bbox=None, level_event: bool = False, extra: str = \\\"\\\") -> None:\\n        if self.model is None:\\n            return\\n        self.reviews += 1\\n        mode = self.cfg.get(\\\"review_think\\\", \\\"level\\\")\\n        think = mode == \\\"always\\\" or (mode == \\\"level\\\" and level_event)\\n        obj = self.model.review(self.g, self.book, event, changes, self.outcomes, before=before, bbox=bbox, think=think, extra=extra)\\n        if not obj:\\n            return\\n        done = self.book.apply_edits(obj.get(\\\"edits\\\") or [], level=self.g.level)\\n        self.apply_roles(obj.get(\\\"roles\\\"))\\n        if obj.get(\\\"procedure\\\"):\\n            proc = str(obj[\\\"procedure\\\"])[:500]\\n            old = next((e for e in self.book.section(\\\"win\\\") if e.text.startswith(\\\"PROCEDURE:\\\")), None)\\n            if old is not None:\\n                old.text = \\\"PROCEDURE: \\\" + proc; old.level = self.g.level\\n            else:\\n                self.book.add(\\\"win\\\", \\\"PROCEDURE: \\\" + proc, level=self.g.level)\\n        if obj.get(\\\"plan\\\"):\\n            self.book.plan = str(obj[\\\"plan\\\"])[:600]\\n        self.log(f\\\"REVIEW ({event[:60]}): {done}\\\\n\\\" + self.book.render()); self.save_book()\\n\\n    def apply_roles(self, roles) -> None:\\n        if not isinstance(roles, dict):\\n            return\\n        for key, role in list(roles.items())[:12]:\\n            key, role = str(key)[:40], str(role)[:60]\\n            text = f\\\"role of {key}: {role}\\\"\\n            old = next((e for e in self.book.section(\\\"env\\\") if e.text.startswith(f\\\"role of {key}:\\\")), None)\\n            if old is not None:\\n                old.text = text; old.level = self.g.level\\n            else:\\n                self.book.add(\\\"env\\\", text, level=self.g.level)\\n        self.save_book()\\n\\n    def _same_cell(self, c: Candidate) -> bool:\\n        a = c.actions[0] if c.actions else None\\n        return isinstance(a, dict) and self.last_cell is not None and (a[\\\"row\\\"] // 3, a[\\\"col\\\"] // 3) == self.last_cell\\n\\n    def choose(self, cands: list[Candidate], budget_text: str, ev: Evidence, goal: Optional[dict] = None) -> Candidate:\\n        cands = [c for c in cands if c.kind != \\\"info\\\"] or cands\\n        fallback = next((c for c in cands if not self._same_cell(c)), cands[0])   # untested first, then by priority; never undo the last toggle\\n        if self.model is None or not cands:\\n            return fallback\\n        extra = (\\\"WIN CONDITION PROGRESS (program-evaluated on the current board):\\\\n  \\\" + \\\"\\\\n  \\\".join(goal[\\\"lines\\\"])) if goal and goal.get(\\\"lines\\\") else \\\"\\\"\\n        bad = [lab for (lv, lab), n in self.refused.items() if lv == self.g.level and n >= 2]\\n        if bad:\\n            extra = (f\\\"NOTE: there is NO candidate named {', '.join(repr(b) for b in bad[:3])} in this game (asked {sum(self.refused[(self.g.level, b)] for b in bad[:3])}x, refused). \\\"\\n                     \\\"Choose an exact label from CANDIDATES.\\\\n\\\") + extra\\n        obj = self.model.decide(self.g, self.book, cands, self.outcomes, budget_text, extra=extra)\\n        if not obj:\\n            return fallback\\n        done = self.book.apply_edits(obj.get(\\\"edits\\\") or [], level=self.g.level)\\n        self.apply_roles(obj.get(\\\"roles\\\"))\\n        if done:\\n            self.log(f\\\"DECIDE edits: {done}\\\"); self.save_book()\\n        label = str(obj.get(\\\"choice\\\", \\\"\\\")).strip()\\n        c = resolve(label, cands, self.g, ev)\\n        if c is not None and c.label != label:\\n            self.log(f\\\"DECIDE: label {label!r} resolved to {c.label}\\\")\\n        if c is None:\\n            self.metrics[\\\"fallback\\\"] += 1\\n            self.refused[(self.g.level, label)] = self.refused.get((self.g.level, label), 0) + 1\\n            self.log(f\\\"DECIDE: label {label!r} not a candidate -> fallback {fallback.label}\\\")\\n            self.outcomes.append(f\\\"(program refused {label}: not a candidate \\u2014 hidden objects did nothing twice on this level; ran {fallback.label} instead)\\\")\\n            return fallback\\n        # wandering guards: a candidate already tried in this world state that is predicted to change nothing (or only a counter)\\n        # is a wasted action; so is the same label three times in a row\\n        self.last_choice.append(c.label)\\n        repeat = len(self.last_choice) >= 3 and len(set(self.last_choice[-3:])) == 1\\n        recent = self.last_choice[-3:-1].count(c.label) >= 1\\n        no_effect = c.pred_kind in (\\\"noop\\\", \\\"hud\\\", \\\"blocked\\\")\\n        rl = getattr(self, \\\"recent_labels\\\", [])\\n        monotone = len(rl) >= 12 and rl.count(c.label) >= 8 and c.kind != \\\"submit\\\"\\n        if monotone:\\n            alt = next((x for x in cands if not x.tested and x.label != c.label and x.pred_kind not in (\\\"noop\\\", \\\"hud\\\", \\\"blocked\\\") and not self._same_cell(x)), None)\\n            if alt:\\n                self.log(f\\\"DECIDE: {c.label} chosen {rl.count(c.label)} of the last 12 decisions without finishing the level -> {alt.label}\\\")\\n                self.outcomes.append(f\\\"(program vetoed {c.label}: {rl.count(c.label)}/12 recent decisions, no level progress; ran {alt.label} instead)\\\")\\n                self.metrics[\\\"veto_monotone\\\"] += 1\\n                return alt\\n        # veto only wasted repeats: a repeat that keeps changing the world (pressing a conveyor button again) is legitimate\\n        if (c.tested and (no_effect or repeat)) or (recent and no_effect):\\n            alt = next((x for x in cands if not x.tested and x.label != c.label and x.pred_kind not in (\\\"noop\\\", \\\"hud\\\", \\\"blocked\\\") and not self._same_cell(x)), None) or \\\\\\n                next((x for x in cands if not x.tested and x.label != c.label and not self._same_cell(x)), None)\\n            if alt:\\n                self.log(f\\\"DECIDE: {c.label} already tried here and predicted '{c.pred_kind}'{' (3x in a row)' if repeat else ''} -> {alt.label}\\\")\\n                self.outcomes.append(f\\\"(program vetoed {c.label}: already tried here, no world change expected; ran {alt.label} instead)\\\")\\n                return alt\\n        self.log(f\\\"DECIDE: {c.label} (expect: {str(obj.get('expect', ''))[:120]})\\\")\\n        self.metrics[\\\"chosen_\\\" + (\\\"plan\\\" if c.kind == \\\"plan\\\" else c.kind)] += 1\\n        then = obj.get(\\\"then\\\")\\n        self.pending_then = [str(x) for x in then][:3] if isinstance(then, list) else []\\n        return c\\n\\n    # \\u2500\\u2500 main loop \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\n    def play(self) -> dict:\\n        g = self.g\\n        g.reset()\\n        self.log(f\\\"start: {g.levels_total} levels, valid {g.valid_actions}\\\")\\n        self.init_book()\\n        stop = \\\"done\\\"\\n        while g.state != \\\"WIN\\\" and g.level <= self.cfg[\\\"max_levels\\\"]:\\n            r = self.play_level()\\n            self.level_stats.append(r)\\n            self.log(f\\\"level {r['level']}: {'COMPLETED' if r['completed'] else 'not completed'} in {r['actions']} actions ({r['reason']})\\\")\\n            if not r[\\\"completed\\\"]:\\n                stop = r[\\\"reason\\\"]; break\\n        res = {\\\"game_id\\\": g.game_id, \\\"levels_completed\\\": g.level - 1 if g.state != \\\"WIN\\\" else g.levels_total, \\\"levels_total\\\": g.levels_total,\\n               \\\"actions\\\": g.actions_used, \\\"level_action_log\\\": g.level_action_log, \\\"stop_reason\\\": stop, \\\"seconds\\\": round(time.time() - self.t0, 1),\\n               \\\"mismatches\\\": self.mismatches, \\\"reviews\\\": self.reviews, \\\"observations\\\": self.observations, \\\"rulebook\\\": self.book.stats(),\\n               \\\"rulebook_version\\\": self.book.version, \\\"levels\\\": self.level_stats,\\n               \\\"model_calls\\\": self.model.calls if self.model else {}, \\\"model_seconds\\\": round(self.model.seconds, 1) if self.model else 0,\\n               \\\"metrics\\\": dict(self.metrics)}\\n        self.log(\\\"result: \\\" + json.dumps(res)); self.save_book(); self._log_f.close()\\n        return res\\n\\n    def play_level(self) -> dict:\\n        g = self.g; level0 = g.level; a0 = g.actions_used; reviews0 = self.reviews; decisions = 0\\n        self.level_start_frame = g.frame; self.attempt_start_actions = g.actions_used\\n        while g.level == level0 and g.state != \\\"WIN\\\":\\n            if self._out_of_time():\\n                return self._lv(level0, a0, decisions, \\\"time/actions budget\\\")\\n            if g.actions_used - a0 >= self.cfg[\\\"level_actions\\\"]:\\n                return self._lv(level0, a0, decisions, \\\"level action cap\\\")\\n            ev = self.evidence(); self.sync(ev)\\n            goal = self.goal_state(ev)\\n            cands = build(g, ev, self.tried, goal=goal)\\n            plans = make_plans(g, ev, self.live_preds())\\n            kept = []\\n            for c in plans:\\n                nf = self.plan_fail.get((g.level, c.label), 0)\\n                if c.kind == \\\"plan\\\" and nf >= 3:   # three failures on this level: the plan is wrong for this board, stop offering it\\n                    c = Candidate(c.label, \\\"info\\\", [], f\\\"withdrawn after {nf} failed runs on this level: {c.prediction[:120]}\\\", priority=999)\\n                elif c.kind == \\\"plan\\\" and nf >= 2:\\n                    c.priority = 40; c.prediction = f\\\"[failed {nf}x on this level] \\\" + c.prediction\\n                kept.append(c)\\n            plans = kept\\n            self.metrics[\\\"plans_offered\\\"] += sum(1 for c in plans if c.kind == \\\"plan\\\")\\n            for c in plans:\\n                if c.kind == \\\"plan\\\" and c.actions:\\n                    k = ev.predict(c.actions[0]).kind\\n                    if k in (\\\"noop\\\", \\\"hud\\\", \\\"blocked\\\"):\\n                        c.pred_kind = k; c.prediction = f\\\"[first step predicted {k}] \\\" + c.prediction\\n                self.log(f\\\"{'plan candidate' if c.kind == 'plan' else 'plan info'}: {c.label} -> {c.prediction[:160]}\\\")\\n            cands = [c for c in plans if c.kind == \\\"plan\\\"] + cands + [c for c in plans if c.kind != \\\"plan\\\"]\\n            if not [c for c in cands if c.kind != \\\"info\\\"]:\\n                return self._lv(level0, a0, decisions, \\\"no candidates\\\")\\n            budget = f\\\"actions used on this level {g.actions_used - a0} (cap {self.cfg['level_actions']}), total {g.actions_used}\\\"\\n            lim = self.attempt_limit()\\n            if lim:\\n                budget += f\\\"; THIS ATTEMPT ENDS IN A GAME OVER AFTER ~{lim} ACTIONS (seen {len(self.game_over_at.get(g.level, []))}x): {lim - (g.actions_used - self.attempt_start_actions)} left\\\"\\n            if goal[\\\"lines\\\"]:\\n                self.log(\\\"goal state: \\\" + \\\" | \\\".join(goal[\\\"lines\\\"]))\\n            if self.cfg.get(\\\"mode\\\") == \\\"coder\\\" and self.model is not None:\\n                decisions += 1\\n                res = self.code_turn(ev, cands, plans, goal, budget, level0, reviews0, a0)\\n                if res == \\\"won\\\":\\n                    return self._lv(level0, a0, decisions, \\\"completed\\\")\\n                continue\\n            cand = self.choose(cands, budget, ev, goal); decisions += 1\\n            sk = state_key(g, ev)\\n            live_plans = [c for c in plans if c.kind == \\\"plan\\\" and self.plan_fail.get((g.level, c.label), 0) < 2 and c.pred_kind not in (\\\"noop\\\", \\\"hud\\\", \\\"blocked\\\")\\n                          and (sk, c.label) not in self.tried]\\n            if live_plans and cand.kind != \\\"plan\\\":\\n                self.plan_ignored += 1\\n                if self.plan_ignored >= 3:   # the model keeps ignoring a program-made plan for a live win condition: the program runs it\\n                    cand = live_plans[0]; self.plan_ignored = 0; self.metrics[\\\"plan_autorun\\\"] += 1\\n                    self.log(f\\\"DECIDE: plan ignored 3x -> program runs {cand.label}\\\")\\n                    self.outcomes.append(f\\\"(program ran {cand.label} because the plan was ignored three times)\\\")\\n            elif cand.kind == \\\"plan\\\":\\n                self.plan_ignored = 0\\n            self.tried.add((state_key(g, ev), cand.label))\\n            progress0 = self._progress_score(ev) if cand.kind == \\\"plan\\\" else None\\n            res = self.execute(cand, level0, reviews0)\\n            # batched follow-ups the model listed in \\\"then\\\": run while nothing surprising happens (labels re-resolved on the fresh board)\\n            for lab in list(getattr(self, \\\"pending_then\\\", []) or []):\\n                if res != \\\"ok\\\" or g.level != level0 or self._out_of_time():\\n                    break\\n                ev2 = self.evidence(); cands2 = build(g, ev2, self.tried, goal=self.goal_state(ev2))\\n                c2 = resolve(lab, cands2, g, ev2)\\n                if c2 is None or c2.kind in (\\\"info\\\", \\\"submit\\\") or c2.pred_kind in (\\\"noop\\\", \\\"hud\\\", \\\"blocked\\\") or (state_key(g, ev2), c2.label) in self.tried:\\n                    self.log(f\\\"THEN: {lab!r} skipped\\\"); continue\\n                self.log(f\\\"THEN: {c2.label}\\\"); self.metrics[\\\"then_run\\\"] += 1\\n                self.tried.add((state_key(g, ev2), c2.label))\\n                res = self.execute(c2, level0, reviews0)\\n            self.pending_then = []\\n            if cand.kind == \\\"plan\\\" and res not in (\\\"won\\\", \\\"level\\\"):\\n                progress1 = self._progress_score(self.evidence())\\n                if res in (\\\"mismatch\\\", \\\"over\\\") or progress1 <= progress0:   # a plan that ran without moving any win condition forward failed\\n                    self.plan_fail[(g.level, cand.label)] = self.plan_fail.get((g.level, cand.label), 0) + 1\\n                    self.log(f\\\"plan {cand.label}: no progress ({progress0:.2f} -> {progress1:.2f}), failures {self.plan_fail[(g.level, cand.label)]}\\\")\\n            # monotony guard: one label for most of the last 12 decisions without progress -> next decision must differ\\n            self.recent_labels = (getattr(self, \\\"recent_labels\\\", []) + [cand.label])[-12:]\\n            if res == \\\"won\\\":\\n                return self._lv(level0, a0, decisions, \\\"completed\\\")\\n        return self._lv(level0, a0, decisions, \\\"completed\\\" if g.level > level0 else \\\"left\\\")\\n\\n    def code_turn(self, ev, cands, plans, goal, budget: str, level0: int, reviews0: int, a0: int) -> str:\\n        \\\"\\\"\\\"Coder mode: one model call -> python code -> run against the library with verified actions.\\\"\\\"\\\"\\n        g = self.g\\n        if self.repl is None:\\n            self.repl = Repl(self)\\n        nudge = \\\"\\\"\\n        if g.level_action_log and g.actions_used - a0 > 2 * max(g.level_action_log[-1], 15):\\n            nudge = (f\\\"You have spent {g.actions_used - a0} actions on this level (the previous level took {g.level_action_log[-1]}). \\\"\\n                     \\\"Stop exploring: replay the procedure that won the previous level adapted to this board, or test one new hypothesis with a verified rule.\\\")\\n        ns = self.repl.namespace(ev, cands, plans, goal, level0, reviews0)\\n        code, text = self.model.code_turn(g, self.book, cands, plans, goal, self.outcomes, budget, self.repl.notes, self.repl_outputs if hasattr(self, \\\"repl_outputs\\\") else [], nudge)\\n        if not hasattr(self, \\\"repl_outputs\\\"):\\n            self.repl_outputs = []\\n        if not code:\\n            fb = next((c for c in cands if c.kind not in (\\\"info\\\",) and not self._same_cell(c)), None)\\n            self.log(f\\\"CODE: no code returned ({text[:100]!r}) -> fallback {fb.label if fb else 'none'}\\\")\\n            if fb is None:\\n                return \\\"ok\\\"\\n            self.tried.add((state_key(g, ev), fb.label))\\n            return self.execute(fb, level0, reviews0)\\n        self.log(\\\"CODE:\\\\n\\\" + code[:1500])\\n        self.pending_mismatch = None\\n        out = self.repl.run(code, ns)\\n        self.log(\\\"CODE OUTPUT:\\\\n\\\" + out[:1500])\\n        # runaway guard: turns that execute nothing cost model time but no game time; after two in a row the program acts\\n        self.idle_turns = (self.idle_turns + 1) if self.repl.turn_actions == 0 else 0\\n        if self.idle_turns >= 2:\\n            fb = next((c for c in cands if c.kind != \\\"info\\\" and not self._same_cell(c)), None)\\n            if fb is not None:\\n                self.log(f\\\"CODE: {self.idle_turns} calls without an action -> program runs {fb.label}\\\")\\n                self.repl_outputs.append(f\\\"--- program note: your last {self.idle_turns} calls executed no action, so the program ran {fb.label} itself ---\\\")\\n                self.idle_turns = 0\\n                self.tried.add((state_key(g, ev), fb.label))\\n                return self.execute(fb, level0, reviews0)\\n        self.repl_outputs = (self.repl_outputs + [f\\\"--- call (actions {self.repl.turn_actions}) ---\\\\n{code[:700]}\\\\n>>> output:\\\\n{out[:900]}\\\"])[-3:]\\n        if self.pending_mismatch is not None and self.reviews - reviews0 < self.cfg[\\\"reviews_per_level\\\"]:\\n            act, label, ptxt, vtxt, before, bbox, changes = self.pending_mismatch\\n            self.review(f\\\"MISMATCH on {action_label(act)} (inside model code). Predicted: {ptxt}. Observed: {vtxt}. Which rulebook entry was wrong, and what rule explains the observation?\\\",\\n                        changes, before=before, bbox=bbox)\\n            self.pending_mismatch = None\\n        if self.repl.turn_status in (\\\"won\\\", \\\"level\\\", \\\"over\\\"):\\n            return self.repl.turn_status\\n        return \\\"ok\\\"\\n\\n    def _progress_score(self, ev) -> float:\\n        \\\"\\\"\\\"Sum of the live win predicates' progress numbers (0..1 each) on the current board.\\\"\\\"\\\"\\n        sc = ev.scene; tot = 0.0\\n        for eid, p_ in self.live_preds():\\n            try:\\n                tot += float(p_.evaluate(sc, ev)[2])\\n            except Exception:\\n                pass\\n        return tot\\n\\n    def attempt_limit(self) -> Optional[int]:\\n        \\\"\\\"\\\"If game overs on this level happened at (nearly) the same action count, that count is the attempt's action limit.\\\"\\\"\\\"\\n        xs = self.game_over_at.get(self.g.level, [])\\n        if len(xs) >= 2 and max(xs) - min(xs) <= 3:\\n            return round(sum(xs) / len(xs))\\n        return None\\n\\n    def _lv(self, level0, a0, decisions, reason) -> dict:\\n        return {\\\"level\\\": level0, \\\"completed\\\": self.g.level > level0 or self.g.state == \\\"WIN\\\", \\\"actions\\\": self.g.actions_used - a0, \\\"decisions\\\": decisions, \\\"reason\\\": reason}\\n\\n    def execute(self, cand: Candidate, level0: int, reviews0: int) -> str:\\n        \\\"\\\"\\\"Run the candidate's actions one by one, each checked against its prediction. Returns 'won' | 'level' | 'over' | 'mismatch' | 'ok'.\\\"\\\"\\\"\\n        for i, act in enumerate(cand.actions):\\n            st = self._step(act, cand.label, level0, reviews0, kind=cand.kind)\\n            if st in (\\\"won\\\", \\\"level\\\", \\\"over\\\", \\\"mismatch\\\", \\\"invalid\\\"):\\n                return \\\"ok\\\" if st == \\\"invalid\\\" else st\\n            if cand.kind == \\\"key-run\\\" and not self.g.transitions[-1].before_frame.ascii != self.g.transitions[-1].after_frame.ascii:\\n                break\\n        return \\\"ok\\\"\\n\\n    def _step(self, act, label: str, level0: int, reviews0: int, *, kind: str = \\\"\\\", review_mismatch: bool = True) -> str:\\n        \\\"\\\"\\\"One verified action: predict, execute, judge, update the book and win predicates, review on surprises.\\n        Returns 'won' | 'level' | 'over' | 'mismatch' | 'ok' | 'invalid'.\\\"\\\"\\\"\\n        g = self.g\\n        ev = self.evidence()\\n        pred = ev.predict(act)\\n        before = g.frame\\n        res = g.step(act)\\n        if res.get(\\\"invalid\\\"):\\n            self.outcomes.append(f\\\"{label}: {action_label(act)} invalid now\\\"); return \\\"invalid\\\"\\n        t = g.transitions[-1]\\n        if isinstance(act, dict):   # only a click that changed the clicked cell itself (a toggle) is protected from an immediate undo\\n            self.last_cell = (act[\\\"row\\\"] // 3, act[\\\"col\\\"] // 3) if before.grid[act[\\\"row\\\"]][act[\\\"col\\\"]] != g.frame.grid[act[\\\"row\\\"]][act[\\\"col\\\"]] else None\\n        else:\\n            self.last_cell = None\\n        v = ev.check(pred, t, res)\\n        self.last_verdict = (pred, v, res)\\n        self.metrics[f\\\"pred_{pred.kind}\\\"] += 1; self.metrics[\\\"verdict_ok\\\" if v.ok else \\\"verdict_mismatch\\\" if v.ok is False else \\\"verdict_obs\\\"] += 1\\n        tag = \\\"OK \\\" if v.ok else \\\"MISMATCH\\\" if v.ok is False else \\\"obs\\\"\\n        self.log(f\\\"{tag}: {action_label(act)} [{label}] predicted: {pred.text} | actual: {v.text}\\\")\\n        self.outcomes.append(f\\\"a{g.actions_used} {action_label(act)} ({label}): predicted '{pred.text[:70]}' -> {v.text[:110]}\\\")\\n        self.outcomes = self.outcomes[-14:]\\n        if res[\\\"level_completed\\\"] or res[\\\"won\\\"]:\\n            ev2 = self.evidence(); changes = self.sync(ev2)\\n            wins = self.book.sync(ev2.win_facts(level0), level=level0)\\n            attempt = [x for x in g.level_transitions(level0) if x.attempt == t.attempt]\\n            try:\\n                post = ev.synth_board(pred, act)\\n                new_preds = goals2.infer(self.level_start_frame or attempt[0].before_frame, before, post, t.action, ev)\\n            except Exception as e:\\n                self.log(f\\\"goal inference failed: {e!r}\\\"); new_preds = []\\n            pfacts = [p_.fact() for p_ in new_preds]\\n            wins += self.book.sync(pfacts, level=level0)\\n            for p_ in new_preds:\\n                e = self.book.find(p_.kind, p_.fact()[\\\"params\\\"], exact=True)\\n                if e is not None:\\n                    self.preds[e.id] = p_\\n            if wins:\\n                self.log(\\\"win facts: \\\" + \\\" | \\\".join(wins)); self.save_book()\\n            try:\\n                essential = goals2.essential_sequence(attempt)\\n            except Exception as e:\\n                essential = [f\\\"(sequence unavailable: {e!r})\\\"]\\n            self.log(\\\"essential winning sequence: \\\" + \\\" ; \\\".join(essential))\\n            self.last_essential = essential\\n            seq = [action_label(x.action) for x in attempt]\\n            if g.state != \\\"WIN\\\":\\n                extra = (\\\"ESSENTIAL WINNING SEQUENCE (program-filtered: actions that changed nothing in the playfield removed; object ids refer to the old board):\\\\n  \\\"\\n                         + \\\"\\\\n  \\\".join(essential[-24:]) +\\n                         \\\"\\\\nWIN PREDICATES THE PROGRAM FOUND TRUE AT COMPLETION (each is verified/refuted automatically on this level):\\\\n  \\\"\\n                         + (\\\"\\\\n  \\\".join(f\\\"{e.id}: {e.text}\\\" for e in self.book.section(\\\"win\\\") if e.id in self.preds) or \\\"(none)\\\"))\\n                self.review(f\\\"LEVEL {level0} COMPLETED after {g.level_action_log[-1]} actions on the last attempt ({len(seq)} actions, {len(essential)} essential). \\\"\\n                            f\\\"The last action was {action_label(act)} ({label}). Level {g.level} starts now (new board below). Write the PROCEDURE that won, in terms of \\\"\\n                            f\\\"object roles (which objects to click in which order and why), so that it can be repeated on the new board; state which win condition proved \\\"\\n                            f\\\"true; and write the plan for the new level.\\\", changes + wins, level_event=True, extra=extra)\\n            return \\\"won\\\" if g.state == \\\"WIN\\\" else \\\"level\\\"\\n        if res[\\\"game_over\\\"]:\\n            n_att = g.actions_used - self.attempt_start_actions\\n            self.game_over_at.setdefault(g.level, []).append(n_att)\\n            g.reset(); self.level_start_frame = g.frame; self.attempt_start_actions = g.actions_used\\n            self.metrics[\\\"game_over\\\"] += 1\\n            ev2 = self.evidence(); changes = self.sync(ev2)\\n            lim = self.attempt_limit()\\n            if lim:\\n                changes = changes + [f\\\"the attempt ends in a GAME OVER after about {lim} actions on this level (seen {len(self.game_over_at[g.level])}x): the level must be finished within that budget\\\"]\\n            self.review(f\\\"GAME OVER after {action_label(act)} ({label}): the level was reset. Predicted: {pred.text}. Observed: {v.text}. \\\"\\n                        \\\"Add the rule that explains the game over (hazard / limit) and adjust the plan.\\\", changes, before=before, bbox=v.diff.get(\\\"bbox\\\"))\\n            return \\\"over\\\"\\n        # win predicates: a submit that did not end the level refutes the conditions that held and the button; a condition that\\n        # holds in a game without a submit button, while the level goes on, is refuted too\\n        if self.preds:\\n            gs = self.goal_state(self.evidence())\\n            if kind == \\\"submit\\\":\\n                self.refute_preds(gs[\\\"held\\\"] + [eid for eid, p_ in self.live_preds() if isinstance(p_, goals2.Pressed)],\\n                                  \\\"the submit button was pressed while these held, but the level did not end\\\")\\n            elif not gs[\\\"has_submit\\\"] and gs[\\\"held\\\"]:\\n                self.refute_preds(gs[\\\"held\\\"], \\\"held on the board but the level did not end (and there is no submit button)\\\")\\n        if v.ok is False:\\n            self.mismatches += 1\\n            if isinstance(act, dict) and pred.kind in (\\\"board\\\", \\\"objects\\\", \\\"cursor\\\"):\\n                key = (g.level, ev.click_key(act[\\\"row\\\"], act[\\\"col\\\"])[0]); self.distrust[key] = self.distrust.get(key, 0) + 1\\n            ev2 = self.evidence(); changes = self.sync(ev2)\\n            if review_mismatch and self.reviews - reviews0 < self.cfg[\\\"reviews_per_level\\\"]:\\n                self.review(f\\\"MISMATCH on {action_label(act)} ({label}). Predicted: {pred.text}. Observed: {v.text}. \\\"\\n                            \\\"Which rulebook entry was wrong, and what rule explains the observation?\\\", changes, before=before, bbox=v.diff.get(\\\"bbox\\\"))\\n            elif not review_mismatch:\\n                self.pending_mismatch = (act, label, pred.text, v.text, before, v.diff.get(\\\"bbox\\\"), changes)\\n            else:\\n                self.log(\\\"review cap reached for this level; programmatic update only\\\")\\n            return \\\"mismatch\\\"\\n        if v.ok is None:\\n            self.observations += 1\\n            if kind in (\\\"click\\\", \\\"key\\\", \\\"interact\\\", \\\"code\\\"):\\n                ev2 = self.evidence(); self.sync(ev2)   # the harness scores the book; the model sees the changes on its next decision\\n        return \\\"ok\\\"\\n\", \"book.py\": \"\\\"\\\"\\\"The rulebook: an explicit, editable list of hypotheses about the game.\\n\\nThree sections \\u2014 env (facts about the board/HUD/avatar), rules (what actions do), win (how a level is completed) \\u2014 plus a\\nfree-text plan. Every entry has a status: hypothesis / confirmed / refuted. Entries whose `kind` is machine-checkable carry\\n`params` so the harness can score them against the recorded transitions (`sync`); the others are text-only and are judged by\\nthe model during reviews. The model edits the book through a small edit language (`apply_edits`).\\n\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport json\\nfrom dataclasses import dataclass, field, asdict\\nfrom pathlib import Path\\nfrom typing import Optional\\n\\nSECTIONS = (\\\"env\\\", \\\"rules\\\", \\\"win\\\")\\nPREFIX = {\\\"env\\\": \\\"E\\\", \\\"rules\\\": \\\"R\\\", \\\"win\\\": \\\"W\\\"}\\nSTATUSES = (\\\"hypothesis\\\", \\\"confirmed\\\", \\\"refuted\\\")\\n\\n# machine-checkable kinds and the params that identify one rule of that kind\\nKIND_KEYS = {\\n    \\\"move\\\": (\\\"action\\\",), \\\"wall\\\": (\\\"color\\\",), \\\"noop\\\": (\\\"action\\\",), \\\"gauge\\\": (\\\"color\\\",), \\\"refill\\\": (\\\"color\\\",),\\n    \\\"collect\\\": (\\\"color\\\",), \\\"hazard\\\": (\\\"color\\\",), \\\"click\\\": (\\\"color\\\", \\\"region\\\"), \\\"mirror\\\": (\\\"color\\\",),\\n    \\\"button\\\": (\\\"color\\\", \\\"region\\\", \\\"bbox\\\"), \\\"coupled\\\": (\\\"marker\\\", \\\"color\\\"),\\n    # win kinds (from goals.infer)\\n    \\\"reach\\\": (\\\"reach\\\",), \\\"collect_reach\\\": (\\\"collect\\\", \\\"reach\\\"), \\\"collect_all\\\": (\\\"collect\\\",), \\\"click_sequence\\\": (), \\\"merge\\\": (),\\n    # win predicates (goals2)\\n    \\\"gone\\\": (\\\"color\\\", \\\"region\\\"), \\\"inside\\\": (\\\"color\\\", \\\"frame\\\"), \\\"equal\\\": (\\\"r1\\\", \\\"r2\\\"), \\\"pose\\\": (\\\"color\\\", \\\"target\\\"), \\\"pressed\\\": (\\\"color\\\", \\\"region\\\"),\\n    # env kinds\\n    \\\"avatar\\\": (),\\n}\\nWIN_KINDS = (\\\"reach\\\", \\\"collect_reach\\\", \\\"collect_all\\\", \\\"click_sequence\\\", \\\"merge\\\", \\\"gone\\\", \\\"inside\\\", \\\"equal\\\", \\\"pose\\\", \\\"pressed\\\")\\n\\n\\n@dataclass\\nclass Entry:\\n    id: str\\n    section: str\\n    text: str\\n    kind: str = \\\"other\\\"\\n    params: dict = field(default_factory=dict)\\n    status: str = \\\"hypothesis\\\"\\n    support: int = 0\\n    counter: int = 0\\n    source: str = \\\"llm\\\"        # llm | harness\\n    note: str = \\\"\\\"             # latest evidence / reason for the status\\n    level: int = 0             # level on which the entry was written\\n\\n    def line(self) -> str:\\n        mark = {\\\"hypothesis\\\": \\\"?\\\", \\\"confirmed\\\": \\\"OK\\\", \\\"refuted\\\": \\\"X\\\"}[self.status]\\n        ev = f\\\" [{self.support} for / {self.counter} against]\\\" if (self.support or self.counter) else \\\"\\\"\\n        kp = f\\\" <{self.kind} {json.dumps(self.params, separators=(',', ':'))}>\\\" if self.kind != \\\"other\\\" and self.params else (f\\\" <{self.kind}>\\\" if self.kind != \\\"other\\\" else \\\"\\\")\\n        note = f\\\" \\u2014 {self.note}\\\" if self.note else \\\"\\\"\\n        return f\\\"[{self.id}][{mark}] {self.text}{kp}{ev}{note}\\\"\\n\\n\\ndef normalize_kind(kind: str, params: dict) -> tuple[str, dict]:\\n    \\\"\\\"\\\"Models often put the section name in `kind` and the real kind in params ('action'/'type'). Repair that.\\\"\\\"\\\"\\n    kind = str(kind or \\\"other\\\").strip().lower(); params = dict(params or {})\\n    if kind in KIND_KEYS:\\n        return kind, params\\n    for key in (\\\"type\\\", \\\"kind\\\", \\\"action\\\"):\\n        v = str(params.get(key, \\\"\\\")).strip().lower()\\n        if v in KIND_KEYS and not (v in (\\\"move\\\", \\\"noop\\\") and key == \\\"action\\\"):\\n            params.pop(key); return v, params\\n    return (\\\"other\\\" if kind in (\\\"env\\\", \\\"rules\\\", \\\"rule\\\", \\\"win\\\", \\\"\\\", \\\"none\\\") else kind), params\\n\\n\\nclass Rulebook:\\n    def __init__(self):\\n        self.entries: list[Entry] = []\\n        self.plan: str = \\\"\\\"\\n        self.history: list[str] = []      # one line per edit (audit trail)\\n        self.version = 0\\n\\n    # \\u2500\\u2500 access \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\n    def section(self, name: str) -> list[Entry]:\\n        return [e for e in self.entries if e.section == name]\\n\\n    def get(self, id_: str) -> Optional[Entry]:\\n        for e in self.entries:\\n            if e.id == id_:\\n                return e\\n        return None\\n\\n    def next_id(self, section: str) -> str:\\n        n = 1 + max([int(e.id[1:]) for e in self.entries if e.section == section] or [0])\\n        return f\\\"{PREFIX[section]}{n}\\\"\\n\\n    def find(self, kind: str, params: dict, exact: bool = False) -> Optional[Entry]:\\n        \\\"\\\"\\\"Entry of the same kind and identifying params. An entry agreeing on every given param (exact) is preferred over one\\n        that agrees only on the identifying keys; with exact=True only the former counts.\\\"\\\"\\\"\\n        keys = KIND_KEYS.get(kind)\\n        if keys is None:\\n            return None\\n        loose = None\\n        for e in self.entries:\\n            # a key the entry does not carry is a wildcard (a model-written 'click colour 1' matches every region)\\n            if e.kind == kind and all(k not in e.params or str(e.params.get(k)) == str(params.get(k)) for k in keys):\\n                if all(str(e.params.get(k)) == str(v) for k, v in params.items() if k in e.params):\\n                    return e\\n                loose = loose or e\\n        return None if exact else loose\\n\\n    # \\u2500\\u2500 edits \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\n    def add(self, section: str, text: str, *, kind: str = \\\"other\\\", params: Optional[dict] = None, source: str = \\\"llm\\\",\\n            status: str = \\\"hypothesis\\\", note: str = \\\"\\\", level: int = 0, support: int = 0, counter: int = 0) -> Entry:\\n        if section not in SECTIONS:\\n            section = \\\"rules\\\"\\n        kind, params = normalize_kind(kind, params or {})\\n        e = Entry(self.next_id(section), section, text.strip()[:300], kind, params, status,\\n                  support, counter, source, note[:200], level)\\n        self.entries.append(e); self.version += 1\\n        self.history.append(f\\\"+{e.id} ({source}) {e.text[:80]}\\\")\\n        return e\\n\\n    def apply_edits(self, edits: list, *, level: int = 0, source: str = \\\"llm\\\") -> list[str]:\\n        \\\"\\\"\\\"Edit ops: {op: add, section, text, kind?, params?} | {op: confirm|refute|remove, id, note?} | {op: edit, id, text?, kind?, params?, note?}\\n        Machine-checkable entries keep their harness-decided status: the model may only add/edit/remove them.\\\"\\\"\\\"\\n        done = []\\n        for ed in edits or []:\\n            if not isinstance(ed, dict):\\n                continue\\n            op = str(ed.get(\\\"op\\\", \\\"\\\")).lower()\\n            try:\\n                if op == \\\"add\\\" and ed.get(\\\"text\\\"):\\n                    e = self.add(str(ed.get(\\\"section\\\", \\\"rules\\\")), str(ed[\\\"text\\\"]), kind=str(ed.get(\\\"kind\\\") or \\\"other\\\"),\\n                                 params=ed.get(\\\"params\\\") if isinstance(ed.get(\\\"params\\\"), dict) else {}, source=source, level=level)\\n                    done.append(f\\\"added {e.id}\\\")\\n                elif op in (\\\"confirm\\\", \\\"refute\\\") and self.get(str(ed.get(\\\"id\\\", \\\"\\\"))):\\n                    e = self.get(str(ed[\\\"id\\\"]))\\n                    if e.kind in KIND_KEYS and e.source == \\\"harness\\\":\\n                        done.append(f\\\"{e.id}: status is decided by the harness for kind {e.kind}\\\"); continue\\n                    e.status = \\\"confirmed\\\" if op == \\\"confirm\\\" else \\\"refuted\\\"; e.note = str(ed.get(\\\"note\\\", \\\"\\\"))[:200]\\n                    self.version += 1; self.history.append(f\\\"{op} {e.id}: {e.note[:60]}\\\"); done.append(f\\\"{op} {e.id}\\\")\\n                elif op == \\\"remove\\\" and self.get(str(ed.get(\\\"id\\\", \\\"\\\"))):\\n                    e = self.get(str(ed[\\\"id\\\"])); self.entries.remove(e); self.version += 1\\n                    self.history.append(f\\\"-{e.id}\\\"); done.append(f\\\"removed {e.id}\\\")\\n                elif op == \\\"edit\\\" and self.get(str(ed.get(\\\"id\\\", \\\"\\\"))):\\n                    e = self.get(str(ed[\\\"id\\\"]))\\n                    if ed.get(\\\"text\\\"):\\n                        e.text = str(ed[\\\"text\\\"]).strip()[:300]\\n                    if ed.get(\\\"kind\\\") or isinstance(ed.get(\\\"params\\\"), dict):\\n                        e.kind, e.params = normalize_kind(ed.get(\\\"kind\\\") or e.kind, ed.get(\\\"params\\\") if isinstance(ed.get(\\\"params\\\"), dict) else e.params)\\n                    if ed.get(\\\"note\\\"):\\n                        e.note = str(ed[\\\"note\\\"])[:200]\\n                    if ed.get(\\\"status\\\") in STATUSES and not (e.kind in KIND_KEYS and e.source == \\\"harness\\\"):\\n                        e.status = ed[\\\"status\\\"]\\n                    self.version += 1; self.history.append(f\\\"edit {e.id}: {e.text[:60]}\\\"); done.append(f\\\"edited {e.id}\\\")\\n            except Exception as ex:  # a malformed edit never breaks play\\n                done.append(f\\\"edit failed: {type(ex).__name__}\\\")\\n        return done\\n\\n    # \\u2500\\u2500 harness evidence \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\n    def sync(self, facts: list[dict], *, level: int) -> list[str]:\\n        \\\"\\\"\\\"facts: [{kind, params, support, counter, text}] induced from the transitions. Matching entries (same kind + key params)\\n        get the counts and a status; contradicting entries of the same kind (e.g. a different delta for the same key) are refuted;\\n        unmatched facts with support are added as harness entries. Returns change descriptions.\\\"\\\"\\\"\\n        changes = []\\n        for f in facts:\\n            kind, params = f[\\\"kind\\\"], f.get(\\\"params\\\", {})\\n            section = \\\"win\\\" if kind in WIN_KINDS else \\\"env\\\" if kind in (\\\"avatar\\\",) else \\\"rules\\\"\\n            e = self.find(kind, params)\\n            support, counter = int(f.get(\\\"support\\\", 0)), int(f.get(\\\"counter\\\", 0))\\n            status = \\\"confirmed\\\" if counter == 0 and support >= 2 else \\\"refuted\\\" if counter >= 1 and counter >= support else \\\"hypothesis\\\"\\n            if e is not None:\\n                # same identity: does the model's parametrisation agree with the evidence?\\n                agree = all(str(e.params.get(k)) == str(params.get(k)) for k in params if k in e.params)\\n                if not agree and e.source == \\\"llm\\\":\\n                    if e.status != \\\"refuted\\\":\\n                        e.status = \\\"refuted\\\"; e.note = f\\\"evidence says {json.dumps(params, separators=(',', ':'))}\\\"; self.version += 1\\n                        changes.append(f\\\"{e.id} refuted: {e.note}\\\")\\n                    if self.find(kind, params, exact=True) is None:   # add the harness version next to it (once)\\n                        n = self.add(section, f[\\\"text\\\"], kind=kind, params=params, source=\\\"harness\\\", status=status, level=level,\\n                                     support=support, counter=counter, note=\\\"induced from transitions\\\")\\n                        changes.append(f\\\"{n.id} added (harness): {n.text}\\\")\\n                    continue\\n                new_status = status if e.source == \\\"harness\\\" or e.kind in KIND_KEYS else e.status\\n                if (e.support, e.counter, e.status) != (support, counter, new_status):\\n                    if e.status != new_status:\\n                        changes.append(f\\\"{e.id} {e.status} -> {new_status} ({support} for / {counter} against)\\\")\\n                    e.support, e.counter, e.status = support, counter, new_status; self.version += 1\\n                if e.source == \\\"harness\\\":\\n                    e.text = f[\\\"text\\\"]\\n                    for k, v in params.items():\\n                        e.params[k] = v\\n            elif support >= 1:\\n                n = self.add(section, f[\\\"text\\\"], kind=kind, params=params, source=\\\"harness\\\", status=status, level=level,\\n                             support=support, counter=counter, note=\\\"induced from transitions\\\")\\n                changes.append(f\\\"{n.id} added (harness): {n.text}\\\")\\n        # a refuted model entry that a confirmed harness entry of the same identity supersedes is noise: drop it (history keeps it)\\n        for e in list(self.entries):\\n            if e.source == \\\"llm\\\" and e.status == \\\"refuted\\\" and e.kind in KIND_KEYS:\\n                keys = KIND_KEYS[e.kind]\\n                if any(h is not e and h.source == \\\"harness\\\" and h.kind == e.kind and h.status == \\\"confirmed\\\"\\n                       and all(str(h.params.get(k)) == str(e.params.get(k)) for k in keys) for h in self.entries):\\n                    self.entries.remove(e); self.version += 1; self.history.append(f\\\"-{e.id} (superseded)\\\")\\n                    changes.append(f\\\"{e.id} removed (superseded by the verified rule)\\\")\\n        return changes\\n\\n    def refute(self, id_: str, note: str) -> Optional[str]:\\n        \\\"\\\"\\\"Harness-decided refutation (e.g. a win predicate held but the level did not end).\\\"\\\"\\\"\\n        e = self.get(id_)\\n        if e is None or e.status == \\\"refuted\\\":\\n            return None\\n        e.status = \\\"refuted\\\"; e.counter += 1; e.note = note[:200]; self.version += 1\\n        self.history.append(f\\\"refute {e.id} (harness): {note[:60]}\\\")\\n        return f\\\"{e.id} refuted: {note}\\\"\\n\\n    # \\u2500\\u2500 text / io \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\n    def render(self, *, include_refuted: bool = True, compact: bool = False) -> str:\\n        \\\"\\\"\\\"compact: refuted entries only as a one-line list, at most 14 entries per section (newest first beyond the confirmed ones),\\n        notes shortened \\u2014 for the per-decision prompt on a busy GPU.\\\"\\\"\\\"\\n        out = []\\n        titles = {\\\"env\\\": \\\"ENVIRONMENT (what is on the board)\\\", \\\"rules\\\": \\\"RULES (what actions do)\\\", \\\"win\\\": \\\"WIN CONDITIONS (how a level is completed)\\\"}\\n        for s in SECTIONS:\\n            es = [e for e in self.section(s) if include_refuted or e.status != \\\"refuted\\\"]\\n            out.append(titles[s] + \\\":\\\")\\n            if compact:\\n                live = [e for e in es if e.status != \\\"refuted\\\"]; dead = [e for e in es if e.status == \\\"refuted\\\"]\\n                if len(live) > 14:\\n                    conf = [e for e in live if e.status == \\\"confirmed\\\"]; hyp = [e for e in live if e.status != \\\"confirmed\\\"]\\n                    live = (conf[-8:] if len(conf) > 8 else conf) + hyp[-(14 - min(8, len(conf))):]\\n                for e in live:\\n                    ln = e.line()\\n                    out.append(\\\"  \\\" + (ln[:220] if len(ln) > 220 else ln))\\n                if dead:\\n                    out.append(\\\"  refuted: \\\" + \\\", \\\".join(f\\\"{e.id} ({e.text[:40]})\\\" for e in dead[-6:]) + (f\\\" +{len(dead) - 6} more\\\" if len(dead) > 6 else \\\"\\\"))\\n                if not live and not dead:\\n                    out.append(\\\"  (none)\\\")\\n                continue\\n            out.extend(\\\"  \\\" + e.line() for e in es) if es else out.append(\\\"  (none)\\\")\\n        out.append(\\\"PLAN: \\\" + (self.plan or \\\"(none)\\\"))\\n        return \\\"\\\\n\\\".join(out)\\n\\n    def stats(self) -> dict:\\n        return {s: {st: sum(1 for e in self.section(s) if e.status == st) for st in STATUSES} for s in SECTIONS}\\n\\n    def to_json(self) -> dict:\\n        return {\\\"version\\\": self.version, \\\"plan\\\": self.plan, \\\"entries\\\": [asdict(e) for e in self.entries], \\\"history\\\": self.history[-200:]}\\n\\n    def save(self, path: Path) -> None:\\n        Path(path).write_text(json.dumps(self.to_json(), indent=1))\\n\\n    @staticmethod\\n    def from_json(d: dict) -> \\\"Rulebook\\\":\\n        b = Rulebook(); b.plan = d.get(\\\"plan\\\", \\\"\\\"); b.version = int(d.get(\\\"version\\\", 0)); b.history = list(d.get(\\\"history\\\", []))\\n        b.entries = [Entry(**e) for e in d.get(\\\"entries\\\", [])]\\n        return b\\n\", \"candidates.py\": \"\\\"\\\"\\\"Candidate actions for one decision: every valid key, a run of a key, clicks on distinct objects, walks to targets.\\nEach candidate gets its deterministic prediction (per first step, and for walks the simulated end position).\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nfrom collections import Counter\\nfrom dataclasses import dataclass, field\\nfrom typing import Optional\\n\\nfrom arcnav.frame import masked_ascii\\n\\nfrom .env import Game, MOVE_KEYS, INTERACT_KEYS\\nfrom .predict import Evidence, Prediction\\n\\n\\n@dataclass\\nclass Candidate:\\n    label: str\\n    kind: str                     # key | key-run | interact | click | goto | frontier\\n    actions: list                 # engine actions (str or dict)\\n    prediction: str\\n    color: Optional[int] = None\\n    tested: bool = False          # tried before in this board state (masked) on this level\\n    priority: int = 50            # fallback policy order (lower first)\\n    pred_kind: str = \\\"unknown\\\"    # predictor kind of the first step\\n\\n    def line(self) -> str:\\n        return f\\\"{self.label:<26} -> {self.prediction}\\\" + (\\\"  [tried here]\\\" if self.tested else \\\"\\\")\\n\\n\\ndef state_key(game: Game, ev: Optional[Evidence] = None) -> str:\\n    \\\"\\\"\\\"Level + world inventory (non-HUD objects away from the edge, avatar excluded) + avatar position. Edge counters/gauges are\\n    ignored so that an action that only ticks a counter does not make every candidate look new again.\\\"\\\"\\\"\\n    skip = set(ev.avatar[\\\"colors\\\"]) if ev and ev.avatar else set()\\n    items = []\\n    for n in game.frame.segmentation[\\\"nodes\\\"]:\\n        if n[\\\"hud\\\"] or n[\\\"color\\\"] in skip:\\n            continue\\n        r0, c0, r1, c1 = n[\\\"bbox\\\"]\\n        if (r1 - r0 <= 2 or c1 - c0 <= 2) and (r0 <= 2 or r1 >= 61 or c0 <= 2 or c1 >= 61):\\n            continue\\n        items.append((n[\\\"color\\\"], n[\\\"pixels\\\"], r0, c0, r1, c1))\\n    av = tuple(ev.avatar[\\\"bbox_xyxy\\\"]) if ev and ev.avatar else ()\\n    return f\\\"L{game.level}:{hash((tuple(sorted(items)), av))}\\\"\\n\\n\\ndef build(game: Game, ev: Evidence, tried: set, *, max_clicks: int = 24, goal: Optional[dict] = None) -> list[Candidate]:\\n    valid = list(game.valid_actions); out: list[Candidate] = []\\n    if goal and goal.get(\\\"submit\\\") is not None and \\\"MOUSE\\\" in valid:\\n        b = goal[\\\"submit\\\"]; r, c = b.center\\n        if game.frame.grid[r][c] != b.color:\\n            r, c = (b.cells or [(r, c)])[0]\\n        held = goal.get(\\\"held\\\", [])\\n        out.append(Candidate(\\\"submit\\\", \\\"submit\\\", [{\\\"action\\\": \\\"MOUSE\\\", \\\"row\\\": r, \\\"col\\\": c}],\\n                             (f\\\"click the submit button #{b.id}: LEVEL COMPLETES if the win conditions {held} are right (they hold now)\\\" if goal.get(\\\"all_hold\\\")\\n                              else f\\\"click the submit button #{b.id}: the win conditions do not hold yet ({goal.get('missing', '')}), so probably nothing\\\"),\\n                             color=b.color, priority=0 if goal.get(\\\"all_hold\\\") else 90, pred_kind=\\\"level\\\" if goal.get(\\\"all_hold\\\") else \\\"hud\\\"))\\n    st = state_key(game, ev)\\n    keys = [a for a in MOVE_KEYS if a in valid]; inter = [a for a in INTERACT_KEYS if a in valid]\\n    nav = ev.nav\\n    movement = bool(nav and ev.moves and ev.avatar)\\n    # walks to targets (movement games with a known avatar)\\n    if movement:\\n        tg = [t for t in nav.targets(max_n=14) if t.get(\\\"path_len\\\") is not None]\\n        tg.sort(key=lambda t: (bool(t.get(\\\"visited\\\")), t[\\\"path_len\\\"]))\\n        for i, t in enumerate(tg[:8]):\\n            path = nav.path_to(t[\\\"row\\\"], t[\\\"col\\\"])\\n            if path is None or len(path) > 30:\\n                continue\\n            if not path:\\n                continue\\n            lab = f\\\"goto({t['color']}@{t['row']},{t['col']})\\\"\\n            pred = _walk_prediction(ev, path, t)\\n            out.append(Candidate(lab, \\\"goto\\\", list(path), pred, color=t[\\\"color\\\"], priority=10 + i + (30 if t.get(\\\"visited\\\") else 0), pred_kind=\\\"move\\\"))\\n        fr = nav.frontier()\\n        if fr is not None:\\n            path = nav.path_to(fr[0], fr[1])\\n            if path:\\n                out.append(Candidate(f\\\"frontier({fr[0]},{fr[1]})\\\", \\\"frontier\\\", list(path), f\\\"walk {len(path)} steps into unexplored floor at ({fr[0]},{fr[1]})\\\", priority=35))\\n    for a in keys:\\n        p = ev.predict(a)\\n        out.append(Candidate(a, \\\"key\\\", [a], p.text, priority=40 if p.kind == \\\"unknown\\\" else 45, pred_kind=p.kind))\\n    if movement:\\n        for a in keys:\\n            p = ev.predict(a)\\n            if p.kind == \\\"move\\\":\\n                out.append(Candidate(f\\\"{a}x8\\\", \\\"key-run\\\", [a] * 8, f\\\"repeat {a} until blocked (up to 8): first step {p.text}\\\", priority=48))\\n    for a in inter:\\n        p = ev.predict(a)\\n        out.append(Candidate(a, \\\"interact\\\", [a], p.text, priority=20 if p.kind == \\\"unknown\\\" else 42, pred_kind=p.kind))\\n    if \\\"MOUSE\\\" in valid and game.frame is not None:\\n        sc = ev.scene\\n        objs = []\\n        dead = 0; inert_groups = set()\\n        for o in sc.objs:\\n            if o.hud:\\n                continue\\n            h = ev.cs.hist.get((o.color, o.region))\\n            if h and h.get(\\\"world\\\", 0) == 0 and sum(h.values()) >= 3 and len({k for k in ev.cs.clicked_at if k[0] == o.region}) >= 3:\\n                inert_groups.add((o.color, o.region)); dead += 1; continue   # three different objects of this colour/area did nothing: the whole group is inert\\n            r0, c0, r1, c1 = o.bbox\\n            thin = (r1 - r0 <= 2) or (c1 - c0 <= 2)\\n            if thin and (r0 <= 1 or r1 >= 62 or c0 <= 1 or c1 >= 62):\\n                continue\\n            if o.size <= 2 and keys:\\n                continue\\n            if ev.cs.is_dead(o.key):\\n                dead += 1; continue   # clicked twice on this level with no effect in the playfield: not offered again\\n            objs.append(o)\\n        # round-robin over (colour, region) groups, smallest objects first, so that no group crowds out the others\\n        groups: dict = {}\\n        for o in sorted(objs, key=lambda o: o.size):\\n            groups.setdefault((o.color, o.region), []).append(o)\\n        ordered = []\\n        for k in range(max((len(v) for v in groups.values()), default=0)):\\n            for key in sorted(groups, key=lambda k_: groups[k_][0].size):\\n                if k < len(groups[key]) and k < 8:\\n                    ordered.append(groups[key][k])\\n        seen_cells: set = set(); seen_groups: Counter = Counter()\\n        base = 25 if not movement else 60\\n        for o in ordered:\\n            r, c = o.center\\n            if game.frame.grid[r][c] != o.color:   # centre of a hollow shape: pick one of its own cells\\n                r, c = (o.cells or [(r, c)])[0]\\n            cell = (r // 4, c // 4)\\n            if cell in seen_cells:\\n                continue\\n            seen_cells.add(cell)\\n            act = {\\\"action\\\": \\\"MOUSE\\\", \\\"row\\\": r, \\\"col\\\": c}\\n            p = ev.predict(act)\\n            lab = f\\\"click({o.color}@{r},{c})\\\"\\n            g = (o.color, o.region)\\n            again = ev.cs.clicked_at.get((o.region, o.bbox), 0)   # a toggle-type object already touched on this level goes last (sweep before re-toggling); buttons are exempt\\n            vt = ev.cs.vanish.get((o.color, o.region))\\n            toggler = bool(vt) and vt.most_common(1)[0][0] != -1\\n            again = again if toggler else 0\\n            out.append(Candidate(lab, \\\"click\\\", [act], f\\\"[#{o.id} {o.size}px\\\" + (f\\\" in P{o.region}\\\" if o.region >= 0 else \\\"\\\") + (f\\\", clicked {again}x\\\" if again else \\\"\\\") + f\\\"] {p.text}\\\", color=o.color,\\n                                 priority=base + (0 if seen_groups[g] == 0 else 8) + min(seen_groups[g], 6) + (10 if p.kind in (\\\"noop\\\", \\\"hud\\\") else 0) + (20 if again else 0), pred_kind=p.kind))\\n            seen_groups[g] += 1\\n            if len([c for c in out if c.kind == \\\"click\\\"]) >= max_clicks:\\n                break\\n        if dead:\\n            out.append(Candidate(\\\"(hidden)\\\", \\\"info\\\", [], f\\\"{dead} objects hidden: clicked with no effect on this level\\\" +\\n                                 (\\\"; inert groups: \\\" + \\\", \\\".join(f\\\"colour {c} in P{r}\\\" for c, r in sorted(inert_groups)) if inert_groups else \\\"\\\"), priority=999))\\n    for c in out:\\n        c.tested = (st, c.label) in tried\\n    out.sort(key=lambda c: (c.tested, c.priority))\\n    return out\\n\\n\\ndef _walk_prediction(ev: Evidence, path: list, t: dict) -> str:\\n    av = ev.avatar; x0, y0, x1, y1 = av[\\\"bbox_xyxy\\\"]\\n    for a in path:\\n        dx, dy = ev.moves.get(a, (0, 0)); x0 += dx; y0 += dy\\n    gauge = f\\\"; gauge {ev.gauge['per_action'] * len(path):+}\\\" if ev.gauge else \\\"\\\"\\n    return f\\\"walk {len(path)} steps ({''.join(a[0] for a in path)}) to colour {t['color']} object at ({t['row']},{t['col']}); avatar ends near ({y0},{x0}){gauge}\\\"\\n\\n\\ndef resolve(label: str, cands: list, game: Game, ev: Evidence) -> Optional[Candidate]:\\n    \\\"\\\"\\\"Map a model-written label to a candidate: exact label first; then click(c@r,c) / goto(c@r,c) with coordinates the\\n    model read off the board (any cell of that colour is accepted as an ad-hoc click; goto goes to the nearest listed target of\\n    that colour); bare key names.\\\"\\\"\\\"\\n    import re\\n    label = label.strip()\\n    by = {c.label: c for c in cands}\\n    if label in by:\\n        return by[label]\\n    m = re.match(r\\\"(click|goto)\\\\((\\\\d+)@(\\\\d+),(\\\\d+)\\\\)\\\", label)\\n    if m:\\n        kind, col, r, c = m.group(1), int(m.group(2)), int(m.group(3)), int(m.group(4))\\n        same = [x for x in cands if x.kind == kind and x.color == col]\\n        if kind == \\\"click\\\" and 0 <= r < 64 and 0 <= c < 64 and game.frame.grid[r][c] == col and \\\"MOUSE\\\" in game.valid_actions:\\n            o = ev.scene.obj_around(r, c)\\n            if o is not None and ev.cs.is_dead(o.key):\\n                return None   # an object that did nothing twice on this level is not clickable again, whatever the model writes\\n            act = {\\\"action\\\": \\\"MOUSE\\\", \\\"row\\\": r, \\\"col\\\": c}; p = ev.predict(act)\\n            return Candidate(f\\\"click({col}@{r},{c})\\\", \\\"click\\\", [act], p.text, color=col, pred_kind=p.kind)\\n        if same:\\n            return min(same, key=lambda x: abs(_coord(x.label)[0] - r) + abs(_coord(x.label)[1] - c))\\n        return None\\n    up = label.upper()\\n    return by.get(up) or next((x for x in cands if x.label.upper() == up), None)\\n\\n\\ndef _coord(label: str) -> tuple:\\n    import re\\n    m = re.search(r\\\"@(\\\\d+),(\\\\d+)\\\", label)\\n    return (int(m.group(1)), int(m.group(2))) if m else (0, 0)\\n\", \"clicks.py\": \"\\\"\\\"\\\"Click records and statistics keyed by (colour, region) and by button object.\\n\\nOne ClickRecord per click transition (cached on the transition): what was clicked (object / region), the change class,\\nwhether the clicked object vanished, which object moved onto the click point (marker / selection), and the displacement of\\nevery matched object. ClickStats aggregates records into the tables the predictor uses:\\n  hist[(colour, region)]     change classes            vanish[(colour, region)]  colour the object turned into\\n  cursor[(colour, region)]   marker colour             succ[button key]          old bbox -> new bbox of objects that moved\\n  coupled[(marker, colour)]  displacement ratio        dead[object key]          clicks that changed nothing / HUD only\\n\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nfrom collections import Counter, defaultdict\\nfrom dataclasses import dataclass, field\\nfrom fractions import Fraction\\nfrom typing import Optional\\n\\nfrom arcnav.frame import Frame, masked_ascii\\n\\nfrom .entities import Obj, Scene, build_scene, match\\n\\n\\ndef _edge_only_diff(a, b, margin: int = 3) -> bool:\\n    for i in range(64):\\n        for j in range(64):\\n            if a[i][j] != b[i][j] and margin <= i < 64 - margin and margin <= j < 64 - margin:\\n                return False\\n    return True\\n\\n\\ndef change_class(before: Frame, after: Frame) -> str:\\n    \\\"\\\"\\\"'world' = something inside the playfield changed; 'hud' = only cells within 3px of the board edge or HUD strips changed; 'none'.\\\"\\\"\\\"\\n    if before.ascii == after.ascii:\\n        return \\\"none\\\"\\n    if masked_ascii(before) == masked_ascii(after) or _edge_only_diff(before.grid, after.grid):\\n        return \\\"hud\\\"\\n    return \\\"world\\\"\\n\\n\\n@dataclass\\nclass ClickRecord:\\n    row: int\\n    col: int\\n    color: int\\n    region: int\\n    obj: Optional[Obj]                 # clicked object in the before scene (None = clicked a region's floor)\\n    cls: str                           # none | hud | world\\n    vanish_to: Optional[int]           # colour the clicked object's cells became (all of them, nothing else in the playfield changed)\\n    marker: Optional[Obj]              # object that moved onto the click point (before-scene object)\\n    marker_to: Optional[tuple] = None  # its new bbox\\n    marker_sel: bool = False           # it landed on the clicked object (selection) rather than on floor\\n    moved: list = field(default_factory=list)      # [(oa, ob, dr, dc)]\\n    appeared: list = field(default_factory=list)\\n    disappeared: list = field(default_factory=list)\\n    occ_before: Optional[dict] = None              # bbox -> (colour, shape) of every non-HUD object (buttons only)\\n    occ_after: Optional[dict] = None\\n\\n\\ndef record(t) -> ClickRecord:\\n    \\\"\\\"\\\"Compute (and cache on the transition) the click record.\\\"\\\"\\\"\\n    rec = getattr(t, \\\"_rec\\\", None)\\n    if rec is not None:\\n        return rec\\n    r, c = t.action[\\\"row\\\"], t.action[\\\"col\\\"]\\n    sa, sb = build_scene(t.before_frame), build_scene(t.after_frame)\\n    colour = t.before_frame.grid[r][c]\\n    obj = sa.obj_around(r, c); region = obj.region if obj else sa.label[r][c]\\n    cls = change_class(t.before_frame, t.after_frame)\\n    m = match(sa, sb) if cls == \\\"world\\\" else {\\\"pairs\\\": [], \\\"moved\\\": [], \\\"appeared\\\": [], \\\"disappeared\\\": []}\\n    vanish = None\\n    hit = sa.obj_at(r, c)\\n    if cls == \\\"world\\\" and hit is not None and hit.cells and any(o.id == hit.id for o in m[\\\"disappeared\\\"]):\\n        obj_v = hit\\n        after = t.after_frame.grid; cs = set(obj_v.cells)\\n        cols = {after[rr][cc] for rr, cc in obj_v.cells}\\n        if len(cols) == 1 and all(t.before_frame.grid[i][j] == after[i][j] or (i, j) in cs for i in range(3, 61) for j in range(3, 61)):\\n            vanish = cols.pop()\\n    marker = None; marker_to = None; sel = False\\n    for oa, ob, dr, dc in sorted(m[\\\"moved\\\"], key=lambda x: -x[0].size):\\n        if oa.size >= 4 and ob.bbox[0] <= r <= ob.bbox[2] and ob.bbox[1] <= c <= ob.bbox[3]:\\n            marker, marker_to = oa, ob.bbox\\n            sel = obj is not None and obj.size >= 4 and obj.id != oa.id\\n            break\\n    occ_b = occ_a = None\\n    def _occ(sc: Scene, skip: int = -1) -> dict:\\n        return {o.bbox: (o.color, o.shape) for o in sc.objs\\n                if not o.hud and o.id != skip and o.size >= 2 and 1 < o.center[0] < 62 and 1 < o.center[1] < 62}\\n    if cls == \\\"world\\\" and obj is not None and obj.size >= 4 and marker is None and m[\\\"moved\\\"]:\\n        occ_b = _occ(sa, obj.id); occ_a = _occ(sb)\\n    rec = ClickRecord(r, c, colour, region, obj, cls, vanish, marker, marker_to, sel, m[\\\"moved\\\"], m[\\\"appeared\\\"], m[\\\"disappeared\\\"], occ_b, occ_a)\\n    try:\\n        t._rec = rec\\n    except Exception:\\n        pass\\n    return rec\\n\\n\\ndef _ratio(a: int, b: int) -> Optional[Fraction]:\\n    if b == 0:\\n        return None if a != 0 else Fraction(0)\\n    return Fraction(a, b)\\n\\n\\nclass ClickStats:\\n    def __init__(self):\\n        self.hist: dict = defaultdict(Counter)\\n        self.vanish: dict = defaultdict(Counter)\\n        self.cursor: dict = defaultdict(Counter)        # (colour, region) -> Counter(marker colour | -1)\\n        self.select: dict = defaultdict(Counter)        # (colour, region) -> Counter(True/False): marker landed on the clicked object\\n        self.presses: dict = defaultdict(list)          # button key -> [(occ_before, occ_after)] with occ = {bbox: (colour, shape)}\\n        self.succ_n: Counter = Counter()                # button key -> clicks observed\\n        self._sigma_cache: dict = {}\\n        self.coupled: dict = defaultdict(Counter)       # (marker colour, other colour) -> Counter((ratio_r, ratio_c))\\n        self.coupled_obs: dict = defaultdict(list)      # (marker colour, other colour) -> [((marker dr, dc), (other dr, dc))]\\n        self.dead: Counter = Counter()                  # object key -> none/hud clicks\\n        self.clicked_at: Counter = Counter()            # (region, bbox) -> clicks on this level, whatever the object's colour was\\n        self.alive: Counter = Counter()                 # object key -> clicks that changed the playfield\\n        self.n = 0\\n\\n    def add(self, rec: ClickRecord) -> None:\\n        self.n += 1\\n        key = (rec.color, rec.region)\\n        self.hist[key][rec.cls] += 1\\n        self.vanish[key][rec.vanish_to if rec.vanish_to is not None else -1] += 1\\n        self.cursor[key][rec.marker.color if rec.marker else -1] += 1\\n        if rec.marker:\\n            self.select[key][rec.marker_sel] += 1\\n        if rec.obj is not None:\\n            (self.dead if rec.cls != \\\"world\\\" else self.alive)[rec.obj.key] += 1\\n            self.clicked_at[(rec.obj.region, rec.obj.bbox)] += 1\\n        if rec.obj is not None and rec.moved and rec.marker is None and rec.occ_before is not None:\\n            b = rec.obj.key; self.succ_n[b] += 1\\n            self.presses[b].append((rec.occ_before, rec.occ_after)); self._sigma_cache.pop(b, None)\\n        if rec.marker is not None and not rec.marker_sel:\\n            mdr, mdc = rec.marker_to[0] - rec.marker.bbox[0], rec.marker_to[1] - rec.marker.bbox[1]\\n            for oa, ob, dr, dc in rec.moved:\\n                if oa.id == rec.marker.id or oa.size < 4 or oa.shape == rec.marker.shape or min(oa.bbox[2] - oa.bbox[0], oa.bbox[3] - oa.bbox[1]) < 2:\\n                    continue\\n                self.coupled_obs[(rec.marker.color, oa.color)].append(((mdr, mdc), (dr, dc)))\\n                rr, rc = _ratio(dr, mdr), _ratio(dc, mdc)\\n                if rr is not None and rc is not None:\\n                    self.coupled[(rec.marker.color, oa.color)][(rr, rc)] += 1\\n\\n    # \\u2500\\u2500 lookups with colour-only fallback \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\n    def _get(self, table: dict, colour: int, region: int):\\n        v = table.get((colour, region))\\n        if v:\\n            return v, True\\n        agg: Counter = Counter()\\n        for (c, rg), cnt in table.items():\\n            if c == colour:\\n                agg.update(cnt)\\n        return (agg if agg else None), False\\n\\n    def hist_of(self, colour: int, region: int):\\n        return self._get(self.hist, colour, region)\\n\\n    def vanish_of(self, colour: int, region: int):\\n        return self._get(self.vanish, colour, region)\\n\\n    def cursor_of(self, colour: int, region: int):\\n        return self._get(self.cursor, colour, region)\\n\\n    def coupling(self, marker_colour: int, other_colour: int):\\n        obs = self.coupled_obs.get((marker_colour, other_colour))\\n        if not obs or len(obs) < 3:\\n            return None\\n        best = None\\n        for ratio in self.coupled[(marker_colour, other_colour)]:\\n            rr, rc = ratio\\n            n = sum(1 for (mdr, mdc), (dr, dc) in obs if abs(round(float(rr) * mdr) - dr) <= 1 and abs(round(float(rc) * mdc) - dc) <= 1)\\n            if best is None or n > best[1]:\\n                best = (ratio, n)\\n        return best if best and best[1] * 10 >= 6 * len(obs) else None\\n\\n    def sigma(self, button_key) -> dict:\\n        \\\"\\\"\\\"Slot permutation of a button: slot -> next slot, solved from every recorded press by constraint propagation.\\n        Constraints: an occupied slot p sends its occupant to sigma(p) (same colour+shape there afterwards); an empty slot\\n        sends 'empty'. Identical blocks are no problem because only occupancy is used, not object identity.\\\"\\\"\\\"\\n        if button_key in self._sigma_cache:\\n            return self._sigma_cache[button_key]\\n        presses = self.presses.get(button_key) or []\\n        slots: set = set()\\n        for occ_b, occ_a in presses:\\n            slots |= set(occ_b) | set(occ_a)\\n        cand = {}\\n        for p in slots:\\n            cs = set()\\n            for q in slots:\\n                ok = True\\n                for occ_b, occ_a in presses:\\n                    if occ_b.get(p) != occ_a.get(q):\\n                        ok = False; break\\n                if ok:\\n                    cs.add(q)\\n            cand[p] = cs\\n        changed = True; fixed = {}; self._tentative = getattr(self, \\\"_tentative\\\", {})\\n        while changed:\\n            changed = False\\n            for p, cs in cand.items():\\n                if p in fixed:\\n                    continue\\n                if len(cs) == 1:\\n                    q = next(iter(cs)); fixed[p] = q; changed = True\\n                    for p2, cs2 in cand.items():\\n                        if p2 != p and q in cs2 and p2 not in fixed:\\n                            cs2.discard(q)\\n        self._sigma_cache[button_key] = fixed\\n        # tie-break (tentative, never used for verdicts): among the remaining candidates the nearest slot when it is unique and close\\n        tentative = {}\\n        for p, cs in cand.items():\\n            if p in fixed or not cs:\\n                continue\\n            free = [q for q in cs if q not in fixed.values()]\\n            if not free:\\n                continue\\n            free.sort(key=lambda q: abs(q[0] - p[0]) + abs(q[1] - p[1]))\\n            d0 = abs(free[0][0] - p[0]) + abs(free[0][1] - p[1])\\n            if d0 <= 8 and (len(free) == 1 or abs(free[1][0] - p[0]) + abs(free[1][1] - p[1]) > d0):\\n                tentative[p] = free[0]\\n        self._sigma_cache[button_key] = fixed\\n        self._tentative[button_key] = tentative\\n        return fixed\\n\\n    def tentative(self, button_key) -> dict:\\n        self.sigma(button_key)\\n        return self._tentative.get(button_key, {})\\n\\n    def is_dead(self, key) -> bool:\\n        return self.dead.get(key, 0) >= 2 and self.alive.get(key, 0) == 0\\n\\n    def successor(self, button_key, colour: int, bbox: tuple):\\n        sg = self.sigma(button_key)\\n        return sg.get(bbox)\\n\\n    def slots(self, button_key) -> set:\\n        out: set = set()\\n        for occ_b, occ_a in self.presses.get(button_key) or []:\\n            out |= set(occ_b) | set(occ_a)\\n        return out\\n\", \"entities.py\": \"\\\"\\\"\\\"Entities: regions (panels / floors / walls = large connected components of one colour), objects (the other components)\\nwith the region that encloses them, and object matching between two frames (displacements, appearances, disappearances).\\n\\nEverything the rulebook says about a click is keyed by (colour, region) instead of colour alone, so that the same colour\\nplaying two roles in two panels (tn36's read-only template marks vs editable marks) yields two rules, not one contradiction.\\n\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nfrom collections import Counter, defaultdict\\nfrom dataclasses import dataclass, field\\nfrom typing import Optional\\n\\nfrom arcnav.frame import Frame\\n\\nREGION_MIN = 150     # px: components at least this big are regions, smaller ones are objects\\nEDGE = 3             # px: a thin strip within this distance of the board edge is HUD\\n\\n\\n@dataclass\\nclass Obj:\\n    id: int\\n    color: int\\n    size: int\\n    bbox: tuple            # (r0, c0, r1, c1)\\n    center: tuple          # (r, c)\\n    shape: int             # hash of the cell pattern relative to bbox\\n    region: int            # enclosing region id, -1 if none\\n    hud: bool\\n    cells: Optional[list] = None\\n\\n    @property\\n    def key(self) -> tuple:\\n        \\\"\\\"\\\"Identity within a level while the object does not move: (colour, region, bbox).\\\"\\\"\\\"\\n        return (self.color, self.region, self.bbox)\\n\\n    def label(self) -> str:\\n        return f\\\"#{self.id} colour {self.color} {self.size}px @({self.center[0]},{self.center[1]})\\\" + (f\\\" in P{self.region}\\\" if self.region >= 0 else \\\"\\\")\\n\\n\\n@dataclass\\nclass Region:\\n    id: int\\n    color: int\\n    size: int\\n    bbox: tuple\\n    edge: bool             # touches the board edge (border / frame)\\n\\n    def label(self) -> str:\\n        r0, c0, r1, c1 = self.bbox\\n        return f\\\"P{self.id} (colour {self.color} area rows {r0}-{r1} cols {c0}-{c1})\\\"\\n\\n\\n@dataclass\\nclass Scene:\\n    frame: Frame\\n    objs: list\\n    regions: list\\n    label: list            # 64x64 region id per cell (-1 = object / none)\\n    comp: list             # 64x64 component index per cell (objects only, -1 otherwise)\\n\\n    def obj_at(self, r: int, c: int) -> Optional[Obj]:\\n        i = self.comp[r][c]\\n        return self.objs[i] if i >= 0 else None\\n\\n    def obj_around(self, r: int, c: int, min_size: int = 4) -> Optional[Obj]:\\n        \\\"\\\"\\\"The clicked object, or \\u2014 when the click hit a speck (<= 2 px) \\u2014 the smallest real object whose bbox contains it.\\\"\\\"\\\"\\n        o = self.obj_at(r, c)\\n        if o is not None and o.size > 2:\\n            return o\\n        best = None\\n        for x in self.objs:\\n            if x.hud or x.size < min_size or (o is not None and x.id == o.id):\\n                continue\\n            r0, c0, r1, c1 = x.bbox\\n            if r0 <= r <= r1 and c0 <= c <= c1 and (best is None or x.size < best.size):\\n                best = x\\n        return best or o\\n\\n    def region_at(self, r: int, c: int) -> int:\\n        o = self.obj_at(r, c)\\n        return o.region if o else self.label[r][c]\\n\\n    def region_named(self, rid: int) -> Optional[Region]:\\n        return next((x for x in self.regions if x.id == rid), None)\\n\\n    def by_region(self) -> dict:\\n        out: dict = defaultdict(list)\\n        for o in self.objs:\\n            out[o.region].append(o)\\n        return out\\n\\n\\ndef _shape_hash(cells, r0, c0) -> int:\\n    return hash(tuple(sorted((r - r0, c - c0) for r, c in cells)))\\n\\n\\ndef build_scene(frame: Frame) -> Scene:\\n    cached = getattr(frame, \\\"_scene\\\", None)\\n    if cached is not None:\\n        return cached\\n    grid = frame.grid; h, w = len(grid), len(grid[0])\\n    seen = [[-1] * w for _ in range(h)]\\n    comps = []\\n    for sr in range(h):\\n        for sc in range(w):\\n            if seen[sr][sc] >= 0:\\n                continue\\n            color = grid[sr][sc]; cid = len(comps); stack = [(sr, sc)]; cells = []\\n            seen[sr][sc] = cid\\n            while stack:\\n                r, c = stack.pop(); cells.append((r, c))\\n                for nr, nc in ((r - 1, c), (r + 1, c), (r, c - 1), (r, c + 1)):\\n                    if 0 <= nr < h and 0 <= nc < w and seen[nr][nc] < 0 and grid[nr][nc] == color:\\n                        seen[nr][nc] = cid; stack.append((nr, nc))\\n            rs = [r for r, _ in cells]; cs = [c for _, c in cells]\\n            comps.append((color, cells, (min(rs), min(cs), max(rs), max(cs))))\\n    label = [[-1] * w for _ in range(h)]; comp = [[-1] * w for _ in range(h)]\\n    regions: list[Region] = []; objs: list[Obj] = []\\n    for color, cells, bbox in comps:\\n        r0, c0, r1, c1 = bbox\\n        thin = (r1 - r0 <= 2) or (c1 - c0 <= 2)\\n        hud = thin and (r0 < EDGE or r1 >= h - EDGE or c0 < EDGE or c1 >= w - EDGE) and len(cells) >= 8\\n        if len(cells) >= REGION_MIN and not hud:\\n            rid = len(regions)\\n            regions.append(Region(rid, color, len(cells), bbox, r0 == 0 or c0 == 0 or r1 == h - 1 or c1 == w - 1))\\n            for r, c in cells:\\n                label[r][c] = rid\\n        else:\\n            oid = len(objs)\\n            objs.append(Obj(oid, color, len(cells), bbox, ((r0 + r1) // 2, (c0 + c1) // 2), _shape_hash(cells, r0, c0), -1, hud,\\n                            cells if len(cells) <= 400 else None))\\n            for r, c in cells:\\n                comp[r][c] = oid\\n    # enclosing region of an object: majority region label on the 1-px ring around its bbox\\n    for o in objs:\\n        r0, c0, r1, c1 = o.bbox; votes: Counter = Counter()\\n        for r in range(r0 - 1, r1 + 2):\\n            for c in (c0 - 1, c1 + 1):\\n                if 0 <= r < h and 0 <= c < w and label[r][c] >= 0:\\n                    votes[label[r][c]] += 1\\n        for c in range(c0, c1 + 1):\\n            for r in (r0 - 1, r1 + 1):\\n                if 0 <= r < h and 0 <= c < w and label[r][c] >= 0:\\n                    votes[label[r][c]] += 1\\n        if votes:\\n            o.region = votes.most_common(1)[0][0]\\n        else:   # ring is all objects (e.g. a mark inside a bigger object): inherit the neighbouring object's region later\\n            o.region = -1\\n    for o in objs:\\n        if o.region == -1:\\n            r0, c0, r1, c1 = o.bbox; votes = Counter()\\n            for r in range(r0 - 1, r1 + 2):\\n                for c in range(c0 - 1, c1 + 2):\\n                    if 0 <= r < h and 0 <= c < w and comp[r][c] >= 0 and comp[r][c] != o.id and objs[comp[r][c]].region >= 0:\\n                        votes[objs[comp[r][c]].region] += 1\\n            if votes:\\n                o.region = votes.most_common(1)[0][0]\\n    sc = Scene(frame, objs, regions, label, comp)\\n    try:\\n        frame._scene = sc\\n    except Exception:\\n        pass\\n    return sc\\n\\n\\ndef match(a: Scene, b: Scene, max_dist: int = 64) -> dict:\\n    \\\"\\\"\\\"Pair objects of scene a with objects of scene b: same colour and shape, nearest first (greedy on distance), so that\\n    identical blocks that all shift one slot are each paired with their nearest successor. Returns\\n    {'pairs': [(oa, ob)], 'moved': [(oa, ob, dr, dc)], 'appeared': [ob], 'disappeared': [oa]}.\\\"\\\"\\\"\\n    groups_b: dict = defaultdict(list)\\n    for o in b.objs:\\n        if not o.hud:\\n            groups_b[(o.color, o.size, o.shape)].append(o)\\n    cands = []\\n    for oa in a.objs:\\n        if oa.hud:\\n            continue\\n        for ob in groups_b.get((oa.color, oa.size, oa.shape), []):\\n            d = abs(oa.center[0] - ob.center[0]) + abs(oa.center[1] - ob.center[1])\\n            if d <= max_dist:\\n                cands.append((d, oa.id, ob.id))\\n    cands.sort()\\n    used_a: set = set(); used_b: set = set(); pairs = []\\n    for d, ia, ib in cands:\\n        if ia in used_a or ib in used_b:\\n            continue\\n        used_a.add(ia); used_b.add(ib); pairs.append((a.objs[ia], b.objs[ib]))\\n    moved = [(oa, ob, ob.bbox[0] - oa.bbox[0], ob.bbox[1] - oa.bbox[1]) for oa, ob in pairs if oa.bbox != ob.bbox]\\n    appeared = [o for o in b.objs if not o.hud and o.id not in used_b]\\n    disappeared = [o for o in a.objs if not o.hud and o.id not in used_a]\\n    return {\\\"pairs\\\": pairs, \\\"moved\\\": moved, \\\"appeared\\\": appeared, \\\"disappeared\\\": disappeared}\\n\\n\\ndef scene_text(sc: Scene, limit_per_region: int = 14) -> str:\\n    \\\"\\\"\\\"Compact entity table for prompts: regions, then objects grouped by region and colour.\\\"\\\"\\\"\\n    lines = []\\n    for rg in sc.regions:\\n        lines.append(f\\\"  {rg.label()}\\\" + (\\\" [border]\\\" if rg.edge else \\\"\\\"))\\n    by = sc.by_region()\\n    for rid in sorted(by, key=lambda k: (k < 0, k)):\\n        objs = [o for o in by[rid] if not o.hud]\\n        if not objs:\\n            continue\\n        cols: dict = defaultdict(list)\\n        for o in objs:\\n            cols[o.color].append(o)\\n        head = f\\\"  in P{rid}:\\\" if rid >= 0 else \\\"  (no region):\\\"\\n        parts = []\\n        for c, os_ in sorted(cols.items(), key=lambda kv: -len(kv[1])):\\n            os_.sort(key=lambda o: (o.size, o.center))\\n            pos = \\\", \\\".join(f\\\"#{o.id}@({o.center[0]},{o.center[1]}){o.size}px\\\" for o in os_[:limit_per_region])\\n            parts.append(f\\\"colour {c} x{len(os_)}: {pos}{' ...' if len(os_) > limit_per_region else ''}\\\")\\n        lines.append(head + \\\" \\\" + \\\"; \\\".join(parts))\\n    huds = [o for o in sc.objs if o.hud]\\n    if huds:\\n        lines.append(\\\"  HUD strips: \\\" + \\\", \\\".join(f\\\"colour {o.color} {o.size}px rows {o.bbox[0]}-{o.bbox[2]} cols {o.bbox[1]}-{o.bbox[3]}\\\" for o in huds[:6]))\\n    return \\\"\\\\n\\\".join(lines)\\n\", \"env.py\": \"\\\"\\\"\\\"Thin environment wrapper: step, reset, frames and the transition record. No model, no sandbox.\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport hashlib\\nimport json\\nimport time\\nfrom dataclasses import dataclass\\nfrom pathlib import Path\\nfrom typing import Optional, Union\\n\\nfrom arcengine import GameAction\\n\\nfrom arcnav.frame import Frame\\n\\nMODEL_TO_ENGINE = {\\\"UP\\\": \\\"ACTION1\\\", \\\"DOWN\\\": \\\"ACTION2\\\", \\\"LEFT\\\": \\\"ACTION3\\\", \\\"RIGHT\\\": \\\"ACTION4\\\", \\\"SPACE\\\": \\\"ACTION5\\\", \\\"MOUSE\\\": \\\"ACTION6\\\", \\\"ACTION7\\\": \\\"ACTION7\\\"}\\nENGINE_TO_MODEL = {v: k for k, v in MODEL_TO_ENGINE.items()}\\nMOVE_KEYS = (\\\"UP\\\", \\\"DOWN\\\", \\\"LEFT\\\", \\\"RIGHT\\\")\\nINTERACT_KEYS = (\\\"SPACE\\\", \\\"ACTION7\\\")\\n\\nAction = Union[str, dict]   # \\\"UP\\\" | {\\\"action\\\": \\\"MOUSE\\\", \\\"row\\\": r, \\\"col\\\": c}\\n\\n\\ndef action_label(act: Action) -> str:\\n    if isinstance(act, dict):\\n        return f\\\"MOUSE({act['row']},{act['col']})\\\"\\n    return act\\n\\n\\n@dataclass\\nclass Transition:\\n    action: Action          # str for keys, dict {\\\"action\\\": \\\"MOUSE\\\", \\\"row\\\", \\\"col\\\"} for clicks (what arcnav.rules expects)\\n    before_frame: Frame\\n    after_frame: Frame\\n    level: int\\n    attempt: int            # attempt index on that level (increments on every reset)\\n    step: int\\n\\n\\nclass Game:\\n    def __init__(self, env, game_id: str):\\n        self.env, self.game_id = env, game_id\\n        self.frame: Optional[Frame] = None\\n        self.level, self.levels_total, self.state = 1, 0, \\\"NOT_PLAYED\\\"\\n        self.valid_actions: list[str] = []\\n        self.actions_used = self.level_actions = self.step_no = 0\\n        self.attempt = 0\\n        self.transitions: list[Transition] = []\\n        self.level_action_log: list[int] = []\\n        self.t0 = time.time()\\n        self.record_path: Optional[Path] = None   # set by the agent: every reset/step is appended here as one JSON line (eval viewer replay)\\n\\n    def _record(self, rec: dict) -> None:\\n        if self.record_path is None:\\n            return\\n        rec[\\\"t\\\"] = round(time.time() - self.t0, 2)\\n        rec[\\\"hash\\\"] = hashlib.sha1(self.frame.ascii.encode()).hexdigest()[:12] if self.frame else None\\n        with open(self.record_path, \\\"a\\\") as f:\\n            f.write(json.dumps(rec) + \\\"\\\\n\\\")\\n\\n    # \\u2500\\u2500 engine \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\n    def _apply(self, raw) -> None:\\n        self.state = str(raw.state).split(\\\".\\\")[-1]\\n        self.levels_total = int(raw.win_levels or 0)\\n        self.level = int(raw.levels_completed) + 1\\n        self.valid_actions = [ENGINE_TO_MODEL.get(f\\\"ACTION{a}\\\", f\\\"ACTION{a}\\\") for a in (raw.available_actions or [])]\\n        grids = [g.tolist() if hasattr(g, \\\"tolist\\\") else g for g in raw.frame]\\n        if grids:\\n            self.frame = Frame(grids[-1], step=self.step_no, level=self.level)\\n\\n    def reset(self) -> None:\\n        self._apply(self.env.step(GameAction.RESET))\\n        self.attempt += 1\\n        self.level_actions = 0\\n        self._record({\\\"kind\\\": \\\"reset\\\", \\\"step\\\": self.step_no, \\\"level\\\": self.level, \\\"attempt\\\": self.attempt})\\n\\n    def step(self, act: Action) -> dict:\\n        \\\"\\\"\\\"Execute one action. Returns {changed, level_completed, game_over, won}. Never resets by itself.\\\"\\\"\\\"\\n        name = act[\\\"action\\\"] if isinstance(act, dict) else act\\n        if self.valid_actions and name not in self.valid_actions:\\n            return {\\\"changed\\\": False, \\\"level_completed\\\": False, \\\"game_over\\\": False, \\\"won\\\": False, \\\"invalid\\\": True}\\n        ga = GameAction.from_name(MODEL_TO_ENGINE[name]) if hasattr(GameAction, \\\"from_name\\\") else GameAction[MODEL_TO_ENGINE[name]]\\n        data = {\\\"x\\\": int(act[\\\"col\\\"]), \\\"y\\\": int(act[\\\"row\\\"])} if name == \\\"MOUSE\\\" else {}\\n        if data:\\n            ga.set_data(data)\\n        before, prev_level = self.frame, self.level\\n        raw = self.env.step(ga, data=ga.action_data.model_dump())\\n        self.step_no += 1; self.actions_used += 1; self.level_actions += 1\\n        self._apply(raw)\\n        label = {\\\"action\\\": \\\"MOUSE\\\", \\\"row\\\": int(act[\\\"row\\\"]), \\\"col\\\": int(act[\\\"col\\\"])} if name == \\\"MOUSE\\\" else name\\n        self.transitions.append(Transition(label, before, self.frame, prev_level, self.attempt, self.step_no))\\n        completed = self.level > prev_level or self.state == \\\"WIN\\\"\\n        if completed:\\n            self.level_action_log.append(self.level_actions); self.level_actions = 0; self.attempt += 1\\n        self._record({\\\"kind\\\": \\\"step\\\", \\\"step\\\": self.step_no, \\\"action\\\": label, \\\"level\\\": prev_level, \\\"attempt\\\": self.transitions[-1].attempt,\\n                      \\\"changed\\\": before.ascii != self.frame.ascii, \\\"level_completed\\\": completed, \\\"game_over\\\": self.state == \\\"GAME_OVER\\\", \\\"won\\\": self.state == \\\"WIN\\\"})\\n        return {\\\"changed\\\": before.ascii != self.frame.ascii, \\\"level_completed\\\": completed, \\\"game_over\\\": self.state == \\\"GAME_OVER\\\",\\n                \\\"won\\\": self.state == \\\"WIN\\\", \\\"invalid\\\": False}\\n\\n    # \\u2500\\u2500 views \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\n    def level_transitions(self, level: Optional[int] = None) -> list[Transition]:\\n        lv = self.level if level is None else level\\n        return [t for t in self.transitions if t.level == lv]\\n\\n    def attempt_transitions(self) -> list[Transition]:\\n        \\\"\\\"\\\"Transitions of the current attempt on the current level (since the last reset / level start).\\\"\\\"\\\"\\n        return [t for t in self.transitions if t.level == self.level and t.attempt == self.attempt]\\n\\n    def has_move_keys(self) -> bool:\\n        return any(a in self.valid_actions for a in MOVE_KEYS)\\n\", \"goals2.py\": \"\\\"\\\"\\\"Relational win predicates learned from a completed level and evaluated live on later levels.\\n\\nAt level completion the harness looks at the board just before the completing action (and, when the effect model can\\npredict it, the board just after) and keeps every predicate of a small family that holds there but did not hold when the\\nlevel started: gone(colour in a region), inside(objects of colour a in frames of colour f), equal(two panels' marks),\\npose(an object has the same shape as a target object), pressed(the final click was a submit button). On later levels\\neach predicate is re-grounded on the new board and reports whether it holds and how far it is from holding.\\n\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nfrom collections import Counter, defaultdict\\nfrom dataclasses import dataclass, field\\nfrom typing import Optional\\n\\nfrom arcnav.frame import Frame\\n\\nfrom .entities import Obj, Scene, build_scene\\n\\nFRAME_KINDS = (\\\"corners\\\", \\\"outline\\\")\\n\\n\\n@dataclass\\nclass FrameShape:\\n    color: int\\n    kind: str\\n    inner: tuple            # (r0, c0, r1, c1) the area a block must occupy\\n    dots: list              # object ids forming the frame\\n\\n    def contains(self, o: Obj) -> bool:\\n        r0, c0, r1, c1 = self.inner\\n        tol = 1 if self.kind == \\\"corners\\\" else 2\\n        ctol = 2 if self.kind == \\\"corners\\\" else 3\\n        return o.bbox[0] >= r0 - tol and o.bbox[1] >= c0 - tol and o.bbox[2] <= r1 + tol and o.bbox[3] <= c1 + tol and \\\\\\n            abs((o.bbox[0] + o.bbox[2]) / 2 - (r0 + r1) / 2) <= ctol and abs((o.bbox[1] + o.bbox[3]) / 2 - (c0 + c1) / 2) <= ctol\\n\\n\\ndef find_frames(sc: Scene) -> list:\\n    \\\"\\\"\\\"Frames drawn with small marks: four corner marks of one colour forming a rectangle, or a ring of dots.\\\"\\\"\\\"\\n    out: list[FrameShape] = []\\n    small: dict = defaultdict(list)\\n    for o in sc.objs:\\n        if not o.hud and o.size <= 4:\\n            small[o.color].append(o)\\n    for c, objs in small.items():\\n        used: set = set()\\n        # dotted outlines first: chain dots (<= 2 px) within distance 3\\n        dots = [o for o in objs if o.size <= 2]\\n        seen: set = set()\\n        for o in dots:\\n            if o.id in seen:\\n                continue\\n            cluster = [o]; seen.add(o.id); i = 0\\n            while i < len(cluster):\\n                cur = cluster[i]; i += 1\\n                for x in dots:\\n                    if x.id not in seen and abs(x.center[0] - cur.center[0]) + abs(x.center[1] - cur.center[1]) <= 3:\\n                        seen.add(x.id); cluster.append(x)\\n            if len(cluster) >= 6:\\n                rs = [x.center[0] for x in cluster]; cs = [x.center[1] for x in cluster]\\n                r0, c0, r1, c1 = min(rs), min(cs), max(rs), max(cs)\\n                # a ring, not a line: dots on at least three sides of the bbox\\n                sides = sum(1 for cond in (any(x.center[0] == r0 for x in cluster), any(x.center[0] == r1 for x in cluster),\\n                                           any(x.center[1] == c0 for x in cluster), any(x.center[1] == c1 for x in cluster)) if cond)\\n                if r1 - r0 >= 4 and c1 - c0 >= 4 and sides >= 3 and len(cluster) <= 40:\\n                    out.append(FrameShape(c, \\\"outline\\\", (r0, c0, r1, c1), [x.id for x in cluster])); used |= {x.id for x in cluster}\\n        # corner frames: four equal marks at the corners of a rectangle\\n        marks = [o for o in objs if o.id not in used]\\n        for a in marks:\\n            for b in marks:\\n                if b.id == a.id or abs(b.bbox[0] - a.bbox[0]) > 1 or b.bbox[1] <= a.bbox[3] + 1 or b.size != a.size:\\n                    continue\\n                for d in marks:\\n                    if d.id in (a.id, b.id) or abs(d.bbox[1] - a.bbox[1]) > 1 or d.bbox[0] <= a.bbox[2] + 1 or d.size != a.size:\\n                        continue\\n                    e = next((x for x in marks if x.id not in (a.id, b.id, d.id) and abs(x.bbox[0] - d.bbox[0]) <= 1 and abs(x.bbox[1] - b.bbox[1]) <= 1), None)\\n                    if e is None or a.id in used:\\n                        continue\\n                    inner = (a.bbox[2] + 1, a.bbox[3] + 1, d.bbox[0] - 1, b.bbox[1] - 1)\\n                    if inner[2] - inner[0] >= 1 and inner[3] - inner[1] >= 1 and inner[2] - inner[0] <= 12 and inner[3] - inner[1] <= 12:\\n                        out.append(FrameShape(c, \\\"corners\\\", inner, [a.id, b.id, d.id, e.id])); used |= {a.id, b.id, d.id, e.id}\\n    return out\\n\\n\\ndef region_color_map(sc: Scene) -> dict:\\n    \\\"\\\"\\\"region id -> (colour, index among regions of that colour ordered by position)\\\"\\\"\\\"\\n    by: dict = defaultdict(list)\\n    for rg in sc.regions:\\n        by[rg.color].append(rg)\\n    out = {}\\n    for c, rgs in by.items():\\n        rgs.sort(key=lambda r: (r.bbox[0], r.bbox[1]))\\n        for i, rg in enumerate(rgs):\\n            out[rg.id] = (c, i)\\n    return out\\n\\n\\ndef _region_by_spec(sc: Scene, spec: tuple) -> Optional[int]:\\n    inv = {v: k for k, v in region_color_map(sc).items()}\\n    return inv.get(tuple(spec))\\n\\n\\n# \\u2500\\u2500 predicates \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\n\\nclass Predicate:\\n    kind = \\\"pred\\\"\\n\\n    def fact(self) -> dict:\\n        raise NotImplementedError\\n\\n    def evaluate(self, sc: Scene) -> tuple:\\n        \\\"\\\"\\\"(holds, progress text, progress number: 0 = far, 1 = holds)\\\"\\\"\\\"\\n        raise NotImplementedError\\n\\n    @property\\n    def needs_submit(self) -> bool:\\n        return False\\n\\n\\n@dataclass\\nclass Gone(Predicate):\\n    color: int\\n    region: tuple           # (region colour, index)\\n    kind = \\\"gone\\\"\\n\\n    def fact(self) -> dict:\\n        return {\\\"kind\\\": \\\"gone\\\", \\\"params\\\": {\\\"color\\\": self.color, \\\"region\\\": list(self.region)}, \\\"support\\\": 1, \\\"counter\\\": 0,\\n                \\\"text\\\": f\\\"no colour-{self.color} object is left in the colour-{self.region[0]} area\\\"}\\n\\n    def evaluate(self, sc: Scene, ev=None) -> tuple:\\n        rid = _region_by_spec(sc, self.region)\\n        n = sum(1 for o in sc.objs if o.color == self.color and not o.hud and (rid is None or o.region == rid))\\n        return n == 0, (f\\\"holds (no colour-{self.color} objects left)\\\" if n == 0 else f\\\"{n} colour-{self.color} objects still there\\\"), 1.0 if n == 0 else 1.0 / (1 + n)\\n\\n\\n@dataclass\\nclass Inside(Predicate):\\n    color: int              # block colour\\n    frame: int              # frame colour\\n    fkind: str\\n    kind = \\\"inside\\\"\\n\\n    def fact(self) -> dict:\\n        same = self.color == self.frame\\n        return {\\\"kind\\\": \\\"inside\\\", \\\"params\\\": {\\\"color\\\": self.color, \\\"frame\\\": self.frame}, \\\"support\\\": 1, \\\"counter\\\": 0,\\n                \\\"text\\\": (f\\\"every block sits inside the frame of its own colour ({'corner marks' if self.fkind == 'corners' else 'dotted outline'}; seen with colour {self.color})\\\" if same\\n                         else f\\\"every colour-{self.color} block sits inside a frame drawn in colour {self.frame} ({'corner marks' if self.fkind == 'corners' else 'dotted outline'})\\\")}\\n\\n    def pairs(self, sc: Scene, frames: Optional[list] = None) -> list:\\n        \\\"\\\"\\\"[(colour of blocks, colour of frames)] this predicate covers on the board: one pair, or every same-colour pair.\\\"\\\"\\\"\\n        allf = frames if frames is not None else find_frames(sc)\\n        if self.color != self.frame:\\n            return [(self.color, self.frame)]\\n        return sorted({f.color for f in allf})\\n\\n    def evaluate(self, sc: Scene, ev=None, frames: Optional[list] = None) -> tuple:\\n        allf = frames if frames is not None else find_frames(sc)\\n        same = self.color == self.frame\\n        pairs = [(c, c) for c in sorted({f.color for f in allf})] if same else [(self.color, self.frame)]\\n        tot_k = tot_n = 0; notes = []\\n        for bc, fc in pairs:\\n            frames_c = [f for f in allf if f.color == fc]\\n            dot_ids = {i for f in frames_c for i in f.dots}\\n            blocks = [o for o in sc.objs if o.color == bc and not o.hud and o.size >= 4 and o.id not in dot_ids]\\n            if not frames_c or not blocks:\\n                continue\\n            inside = [o for o in blocks if any(f.contains(o) for f in frames_c)]\\n            k, n = len(inside), min(len(blocks), len(frames_c)); tot_k += k; tot_n += n\\n            if k < n:\\n                notes.append(f\\\"colour {bc}: {k}/{n} in frames; \\\" + \\\", \\\".join(f\\\"#{o.id}@({o.center[0]},{o.center[1]})\\\" for o in blocks if o not in inside) + \\\" outside, frames at \\\" + \\\", \\\".join(f\\\"({f.inner[0]},{f.inner[1]})\\\" for f in frames_c))\\n        if tot_n == 0:\\n            return False, f\\\"no colour-{self.frame} frame / colour-{self.color} block on this board\\\" if not same else \\\"no frame with same-colour blocks on this board\\\", 0.0\\n        holds = tot_k >= tot_n\\n        return holds, (f\\\"holds ({tot_k} block(s) in frames)\\\" if holds else \\\"; \\\".join(notes)), tot_k / tot_n\\n\\n\\n@dataclass\\nclass Equal(Predicate):\\n    r1: tuple               # (region colour, index) editable\\n    r2: tuple               # template\\n    kind = \\\"equal\\\"\\n\\n    def fact(self) -> dict:\\n        return {\\\"kind\\\": \\\"equal\\\", \\\"params\\\": {\\\"r1\\\": list(self.r1), \\\"r2\\\": list(self.r2)}, \\\"support\\\": 1, \\\"counter\\\": 0,\\n                \\\"text\\\": f\\\"the marks in the colour-{self.r1[0]} area match the marks in the colour-{self.r2[0]} area (same pattern)\\\"}\\n\\n    def evaluate(self, sc: Scene, ev=None) -> tuple:\\n        a, b = _region_by_spec(sc, self.r1), _region_by_spec(sc, self.r2)\\n        if a is None or b is None:\\n            return False, \\\"one of the areas is missing on this board\\\", 0.0\\n        ma, mb = _mask(sc, a), _mask(sc, b)\\n        diff = set(ma.items()) ^ set(mb.items())\\n        cells = len({k for k, v in diff})\\n        return cells == 0, (\\\"holds (patterns identical)\\\" if cells == 0 else f\\\"{cells} cells differ between the two areas\\\"), 1.0 / (1 + cells)\\n\\n\\n@dataclass\\nclass Pose(Predicate):\\n    color: int\\n    target: int\\n    kind = \\\"pose\\\"\\n\\n    def fact(self) -> dict:\\n        return {\\\"kind\\\": \\\"pose\\\", \\\"params\\\": {\\\"color\\\": self.color, \\\"target\\\": self.target}, \\\"support\\\": 1, \\\"counter\\\": 0,\\n                \\\"text\\\": f\\\"a colour-{self.color} object has exactly the shape of a colour-{self.target} object (the target shape)\\\"}\\n\\n    def evaluate(self, sc: Scene, ev=None) -> tuple:\\n        a = [o for o in sc.objs if o.color == self.color and not o.hud and o.size >= 6]\\n        t = [o for o in sc.objs if o.color == self.target and not o.hud and o.size >= 6]\\n        ok = any(x.shape == y.shape for x in a for y in t)\\n        return ok, (\\\"holds (shapes match)\\\" if ok else f\\\"no colour-{self.color} object matches a colour-{self.target} shape yet\\\"), 1.0 if ok else 0.0\\n\\n\\n@dataclass\\nclass Pressed(Predicate):\\n    color: int\\n    region: tuple\\n    size: int\\n    kind = \\\"pressed\\\"\\n\\n    def fact(self) -> dict:\\n        return {\\\"kind\\\": \\\"pressed\\\", \\\"params\\\": {\\\"color\\\": self.color, \\\"region\\\": list(self.region)}, \\\"support\\\": 1, \\\"counter\\\": 0,\\n                \\\"text\\\": f\\\"the level is submitted by clicking the colour-{self.color} button ({self.size}px) in the colour-{self.region[0]} area once the other conditions hold\\\"}\\n\\n    def button(self, sc: Scene) -> Optional[Obj]:\\n        rid = _region_by_spec(sc, self.region)\\n        cands = [o for o in sc.objs if o.color == self.color and not o.hud and o.size >= 8 and (rid is None or o.region == rid)]\\n        return min(cands, key=lambda o: abs(o.size - self.size)) if cands else None\\n\\n    def evaluate(self, sc: Scene, ev=None) -> tuple:\\n        b = self.button(sc)\\n        return b is not None, (f\\\"submit button = #{b.id} at ({b.center[0]},{b.center[1]})\\\" if b else \\\"no such button on this board\\\"), 1.0 if b else 0.0\\n\\n    @property\\n    def needs_submit(self) -> bool:\\n        return True\\n\\n\\n# \\u2500\\u2500 movement-game predicates (built from rulebook entries: model-written or harness win facts) \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\n\\ndef _avatar_box(ev):\\n    av = getattr(ev, \\\"avatar\\\", None) if ev is not None else None\\n    if not av:\\n        return None\\n    x0, y0, x1, y1 = av[\\\"bbox_xyxy\\\"]\\n    return (y0, x0, y1, x1)\\n\\n\\ndef _touching(box, o: Obj, gap: int = 1) -> bool:\\n    r0, c0, r1, c1 = box\\n    return not (o.bbox[0] > r1 + gap or o.bbox[2] < r0 - gap or o.bbox[1] > c1 + gap or o.bbox[3] < c0 - gap)\\n\\n\\n@dataclass\\nclass CollectAll(Predicate):\\n    color: int\\n    kind = \\\"collect_all\\\"\\n\\n    def fact(self) -> dict:\\n        return {\\\"kind\\\": \\\"collect_all\\\", \\\"params\\\": {\\\"collect\\\": self.color}, \\\"support\\\": 1, \\\"counter\\\": 0, \\\"text\\\": f\\\"every colour-{self.color} object has been collected\\\"}\\n\\n    def evaluate(self, sc: Scene, ev=None) -> tuple:\\n        box = _avatar_box(ev)\\n        left = [o for o in sc.objs if o.color == self.color and not o.hud and o.size <= 80 and not (box and _touching(box, o, 0))]\\n        n = len(left)\\n        return n == 0, (f\\\"holds (no colour-{self.color} objects left)\\\" if n == 0 else f\\\"{n} colour-{self.color} objects left: \\\" + \\\", \\\".join(f\\\"#{o.id}@({o.center[0]},{o.center[1]})\\\" for o in left[:5])), 1.0 / (1 + n)\\n\\n\\n@dataclass\\nclass Reach(Predicate):\\n    color: int\\n    kind = \\\"reach\\\"\\n\\n    def fact(self) -> dict:\\n        return {\\\"kind\\\": \\\"reach\\\", \\\"params\\\": {\\\"reach\\\": self.color}, \\\"support\\\": 1, \\\"counter\\\": 0, \\\"text\\\": f\\\"the avatar stands on / touches a colour-{self.color} object\\\"}\\n\\n    def evaluate(self, sc: Scene, ev=None) -> tuple:\\n        box = _avatar_box(ev)\\n        targets = [o for o in sc.objs if o.color == self.color and not o.hud]\\n        if box is None:\\n            return False, \\\"avatar not identified yet\\\", 0.0\\n        if not targets:\\n            return False, f\\\"no colour-{self.color} object on the board\\\", 0.0\\n        hit = [o for o in targets if _touching(box, o)]\\n        return bool(hit), (f\\\"holds (avatar touches #{hit[0].id})\\\" if hit else f\\\"avatar at ({box[0]},{box[1]}); nearest colour-{self.color} object at \\\" + \\\", \\\".join(f\\\"#{o.id}@({o.center[0]},{o.center[1]})\\\" for o in targets[:3])), 1.0 if hit else 0.0\\n\\n\\n@dataclass\\nclass CollectReach(Predicate):\\n    color: int\\n    reach: int\\n    kind = \\\"collect_reach\\\"\\n\\n    def fact(self) -> dict:\\n        return {\\\"kind\\\": \\\"collect_reach\\\", \\\"params\\\": {\\\"collect\\\": self.color, \\\"reach\\\": self.reach}, \\\"support\\\": 1, \\\"counter\\\": 0,\\n                \\\"text\\\": f\\\"collect every colour-{self.color} object, then touch a colour-{self.reach} object\\\"}\\n\\n    def evaluate(self, sc: Scene, ev=None) -> tuple:\\n        ok1, t1, p1 = CollectAll(self.color).evaluate(sc, ev)\\n        ok2, t2, p2 = Reach(self.reach).evaluate(sc, ev)\\n        return ok1 and ok2, (\\\"holds\\\" if ok1 and ok2 else (\\\"collected; \\\" + t2 if ok1 else t1)), (p1 + (p2 if ok1 else 0)) / 2\\n\\n\\nclass CustomGoal(Predicate):\\n    \\\"\\\"\\\"A model-written predicate fn(objects, board) -> bool, evaluated and refuted like the built-in ones.\\\"\\\"\\\"\\n    kind = \\\"custom\\\"\\n\\n    def __init__(self, name: str, fn, needs_submit: bool = False):\\n        self.name, self.fn, self._submit = name, fn, needs_submit\\n\\n    def fact(self) -> dict:\\n        return {\\\"kind\\\": \\\"custom\\\", \\\"params\\\": {\\\"name\\\": self.name}, \\\"support\\\": 0, \\\"counter\\\": 0, \\\"text\\\": f\\\"GOAL {self.name}\\\"}\\n\\n    def evaluate(self, sc: Scene, ev=None) -> tuple:\\n        objs = [{\\\"id\\\": o.id, \\\"color\\\": o.color, \\\"size\\\": o.size, \\\"bbox\\\": o.bbox, \\\"center\\\": o.center, \\\"region\\\": o.region, \\\"hud\\\": o.hud, \\\"cells\\\": o.cells} for o in sc.objs]\\n        try:\\n            ok = bool(self.fn(objs, sc.frame.grid))\\n        except Exception as e:\\n            return False, f\\\"predicate raised {type(e).__name__}: {e}\\\", 0.0\\n        return ok, (\\\"holds\\\" if ok else \\\"not yet\\\"), 1.0 if ok else 0.0\\n\\n    @property\\n    def needs_submit(self) -> bool:\\n        return self._submit\\n\\n\\ndef from_entry(entry) -> Optional[Predicate]:\\n    \\\"\\\"\\\"A predicate object for a rulebook win entry of a known kind (model-written or harness-written), else None.\\\"\\\"\\\"\\n    k, p = entry.kind, entry.params or {}\\n    try:\\n        if k == \\\"collect_all\\\" and p.get(\\\"collect\\\") is not None:\\n            return CollectAll(int(p[\\\"collect\\\"]))\\n        if k == \\\"reach\\\" and p.get(\\\"reach\\\") is not None:\\n            return Reach(int(p[\\\"reach\\\"]))\\n        if k == \\\"collect_reach\\\" and p.get(\\\"collect\\\") is not None and p.get(\\\"reach\\\") is not None:\\n            return CollectReach(int(p[\\\"collect\\\"]), int(p[\\\"reach\\\"]))\\n        if k == \\\"gone\\\" and p.get(\\\"color\\\") is not None and p.get(\\\"region\\\"):\\n            return Gone(int(p[\\\"color\\\"]), tuple(p[\\\"region\\\"]))\\n        if k == \\\"inside\\\" and p.get(\\\"color\\\") is not None and p.get(\\\"frame\\\") is not None:\\n            return Inside(int(p[\\\"color\\\"]), int(p[\\\"frame\\\"]), \\\"corners\\\")\\n        if k == \\\"pressed\\\" and p.get(\\\"color\\\") is not None and p.get(\\\"region\\\"):\\n            return Pressed(int(p[\\\"color\\\"]), tuple(p[\\\"region\\\"]), int(p.get(\\\"size\\\", 20)))\\n    except (TypeError, ValueError):\\n        return None\\n    return None\\n\\n\\ndef _mask(sc: Scene, rid: int) -> dict:\\n    rg = sc.region_named(rid); r0, c0 = rg.bbox[0], rg.bbox[1]\\n    out = {}\\n    for o in sc.objs:\\n        if o.region == rid and not o.hud and o.size <= 16 and o.cells:\\n            for r, c in o.cells:\\n                out[(r - r0, c - c0)] = o.color\\n    return out\\n\\n\\n# \\u2500\\u2500 inference at level completion \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\n\\ndef infer(start: Frame, pre: Frame, post_board: Optional[list], final_action, ev) -> list:\\n    \\\"\\\"\\\"Predicates that hold on the (predicted) completing board but not on the level's first board.\\\"\\\"\\\"\\n    s0, s1 = build_scene(start), build_scene(pre)\\n    s2 = build_scene(Frame(post_board, step=pre.step, level=pre.level)) if post_board is not None else None\\n    preds: list[Predicate] = []\\n    rc1 = region_color_map(s1)\\n    # gone(colour, region)\\n    for rid, spec in rc1.items():\\n        cols0 = {o.color for o in s0.objs if o.region == _region_by_spec(s0, spec) and not o.hud} if _region_by_spec(s0, spec) is not None else set()\\n        cols1 = {o.color for o in s1.objs if o.region == rid and not o.hud}\\n        for c in cols0 - cols1:\\n            preds.append(Gone(c, spec))\\n    # inside(colour, frame colour): on the predicted post board (what the final action produced) when available, else on pre;\\n    # a predicate the final action made true (false on pre, true on post) is the strongest evidence\\n    submit_like = isinstance(final_action, dict) and ev is not None and (lambda o: o is not None and ev.cs.alive.get(o.key, 0) == 0)(s1.obj_around(final_action[\\\"row\\\"], final_action[\\\"col\\\"]))\\n    boards = ([(\\\"post\\\", s2)] if s2 is not None else []) + [(\\\"pre\\\", s1)]\\n    pre_frames = find_frames(s1)\\n    # marker colours (the selection highlight that jumps to wherever one clicks) are never the block a frame is for\\n    marker_cols = set()\\n    if ev is not None:\\n        for tbl in (ev.cs.cursor, ev.cs_all.cursor):\\n            for key, cnt in tbl.items():\\n                k, n = cnt.most_common(1)[0]\\n                if k != -1 and n * 2 > sum(cnt.values()):\\n                    marker_cols.add(k)\\n    for tag, sc in boards:\\n        frames = pre_frames   # frames are static drawings; a block arriving on the post board may hide some of their dots\\n        if not frames:\\n            continue\\n        dot_ids = {i for f in frames for i in f.dots}\\n        for fc in {f.color for f in frames}:\\n            fs = [f for f in frames if f.color == fc]\\n            for bc in {o.color for o in sc.objs if o.size >= 4 and not o.hud and o.id not in dot_ids and o.color not in marker_cols}:\\n                blocks = [o for o in sc.objs if o.color == bc and o.size >= 4 and not o.hud and o.id not in dot_ids]\\n                k = sum(1 for o in blocks if any(f.contains(o) for f in fs))\\n                if k >= min(len(blocks), len(fs)) and k >= 1:\\n                    p = Inside(bc, fc, fs[0].kind)\\n                    if tag == \\\"post\\\" and p.evaluate(s1, None, pre_frames)[0] and not submit_like:\\n                        continue   # already true before the final action, which was not a submit: not what the action achieved\\n                    if tag == \\\"pre\\\" and s2 is not None and not submit_like:\\n                        continue   # the final action changed the board: judge on the post board only\\n                    if not p.evaluate(s0)[0] and not any(isinstance(q, Inside) and q.color == bc and q.frame == fc for q in preds):\\n                        preds.append(p)\\n    # equal(two areas of the same size)\\n    regs = [rg for rg in s1.regions if not rg.edge or rg.size < 1500]\\n    for a in regs:\\n        for b in regs:\\n            if a.id >= b.id or abs((a.bbox[2] - a.bbox[0]) - (b.bbox[2] - b.bbox[0])) > 1 or abs((a.bbox[3] - a.bbox[1]) - (b.bbox[3] - b.bbox[1])) > 1:\\n                continue\\n            ma, mb = _mask(s1, a.id), _mask(s1, b.id)\\n            if ma and ma == mb:\\n                p = Equal(rc1[a.id], rc1[b.id])\\n                if not p.evaluate(s0)[0]:\\n                    preds.append(p)\\n    # pose(colour a matches a colour-t shape)\\n    shapes: dict = defaultdict(set)\\n    for o in s1.objs:\\n        if not o.hud and o.size >= 6:\\n            shapes[o.shape].add(o.color)\\n    for sh, cols in shapes.items():\\n        if len(cols) >= 2:\\n            cols = sorted(cols)\\n            for a in cols:\\n                for t in cols:\\n                    if a != t:\\n                        p = Pose(a, t)\\n                        if not p.evaluate(s0)[0] and not any(isinstance(q, Pose) and q.color == a and q.target == t for q in preds):\\n                            preds.append(p)\\n    # pressed(submit button): the final click hit a sizeable object whose earlier clicks did nothing in the playfield\\n    if isinstance(final_action, dict):\\n        o = s1.obj_around(final_action[\\\"row\\\"], final_action[\\\"col\\\"])\\n        if o is not None and o.size >= 8 and o.region in rc1:\\n            dead_before = ev.cs.dead.get(o.key, 0) if ev is not None else 0\\n            alive_before = ev.cs.alive.get(o.key, 0) if ev is not None else 0\\n            same = [x for x in s1.objs if x.color == o.color and not x.hud and x.size >= 8]\\n            if alive_before == 0 and (dead_before >= 1 or len(same) == 1):\\n                preds.append(Pressed(o.color, rc1[o.region], o.size))\\n    return preds\\n\\n\\ndef essential_sequence(attempt: list) -> list:\\n    \\\"\\\"\\\"The winning attempt's actions minus those that changed nothing in the playfield (the last one is always kept),\\n    described at object level.\\\"\\\"\\\"\\n    from .clicks import record\\n    out = []\\n    for i, t in enumerate(attempt):\\n        last = i == len(attempt) - 1\\n        if isinstance(t.action, dict):\\n            rec = record(t)\\n            if rec.cls != \\\"world\\\" and not last:\\n                continue\\n            o = rec.obj\\n            what = \\\"level completed\\\" if last else (\\\"the object turned into colour %d\\\" % rec.vanish_to if rec.vanish_to is not None else\\n                                                     (\\\"the colour-%d marker moved here\\\" % rec.marker.color if rec.marker else f\\\"{len(rec.moved)} objects moved\\\"))\\n            out.append(f\\\"click {('#%d colour %d %dpx' % (o.id, o.color, o.size)) if o else 'floor colour %d' % rec.color}\\\" + (f\\\" in P{rec.region}\\\" if rec.region >= 0 else \\\"\\\") + f\\\" @({rec.row},{rec.col}) -> {what}\\\")\\n        else:\\n            if t.before_frame.ascii == t.after_frame.ascii and not last:\\n                continue\\n            out.append(f\\\"{t.action}\\\" + (\\\" -> level completed\\\" if last else \\\"\\\"))\\n    return out\\n\", \"llm_io.py\": \"\\\"\\\"\\\"Model I/O: three prompt types (INIT rulebook / DECIDE action / REVIEW rulebook) and tolerant JSON parsing.\\nEvery call sends a fresh, complete brief \\u2014 the rulebook and the record are the memory, not the chat history.\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport json\\nimport re\\nfrom typing import Optional\\n\\nfrom arcnav.frame import Frame\\nfrom arcnav.llm import ChatClient\\n\\nfrom .book import Rulebook\\n\\nKEY_HELP = (\\\"UP/DOWN/LEFT/RIGHT/SPACE/ACTION7 are keys (their meaning is unknown until tested; SPACE and ACTION7 are often 'interact' \\\"\\n            \\\"or 'rotate'); MOUSE clicks a cell (row, col). Each level restarts on GAME OVER. The score per level is (baseline_actions/\\\"\\n            \\\"your_actions)^2, so the fewest actions win; an unfinished level scores 0.\\\")\\n\\nSCHEMA_HELP = \\\"\\\"\\\"Entry format: {\\\"text\\\": \\\"<one sentence>\\\", \\\"kind\\\": \\\"<kind>\\\", \\\"params\\\": {...}}. Use a machine-checkable kind whenever it fits so\\nthe program can verify the entry against every recorded action (otherwise kind \\\"other\\\"):\\n  rules: move {\\\"action\\\":\\\"UP\\\",\\\"dx\\\":0,\\\"dy\\\":-4}  (dx = columns, dy = rows, in pixels; negative dy = up)\\n         wall {\\\"color\\\":k}  noop {\\\"action\\\":\\\"SPACE\\\"}  collect {\\\"color\\\":k}  hazard {\\\"color\\\":k}  gauge {\\\"color\\\":k,\\\"per_action\\\":-1}\\n         refill {\\\"color\\\":k,\\\"amount\\\":n}  click {\\\"color\\\":k}  (say in text what clicking that colour does)  other {}\\n  win:   reach {\\\"reach\\\":k}  collect_all {\\\"collect\\\":k}  collect_reach {\\\"collect\\\":k,\\\"reach\\\":k2}  click_sequence {\\\"sequence\\\":[...]}  other {}\\n  env:   kind \\\"other\\\" (avatar colour, walls, HUD/counters, distinct objects and their likely roles).\\\"\\\"\\\"\\n\\nINIT_SYSTEM = \\\"\\\"\\\"You write the initial RULEBOOK for a program that plays an unknown 64x64 grid puzzle game (ARC-AGI-3). Colours are 0-15,\\nshown as hex digits (row 0 at the top). Nobody knows the rules: write them as HYPOTHESES to be tested by the program, which will\\nconfirm or refute each one from real actions and ask you to revise. Be concrete (colours, positions, key names) and short.\\n\\\"\\\"\\\" + KEY_HELP + \\\"\\\\n\\\" + SCHEMA_HELP + \\\"\\\"\\\"\\nAnswer with JSON only:\\n{\\\"env\\\": [entry, ...], \\\"rules\\\": [entry, ...], \\\"win\\\": [entry, ...], \\\"plan\\\": \\\"<what to test first, then how to win>\\\"}\\nAt most 8 env, 10 rules, 4 win entries.\\\"\\\"\\\"\\n\\nDECIDE_SYSTEM = \\\"\\\"\\\"You choose the next action for a program playing a grid puzzle game. You see the RULEBOOK (hypotheses; [OK] confirmed,\\n[?] untested, [X] refuted), the ENTITIES (areas P0.. = panels/floors; objects #id with colour, size, position, grouped by area), the\\nboard, the WIN CONDITION PROGRESS the program evaluates on the current board, and the CANDIDATE actions with the program's\\ndeterministic prediction of what each will do ('never'/'unknown' = untested). 'plan:...' candidates are multi-action programs the\\nharness built to make a win condition true; 'submit' presses the submit button. Pick ONE candidate label. Prefer: (1) a plan or\\nsubmit candidate whose prediction says a win condition will hold, (2) an action the plan calls for that makes progress, (3) an action\\nthat tests an unverified rule or reveals an unknown effect cheaply, (4) never a candidate marked [tried here] or predicted to change\\nnothing unless the rulebook explains why it would differ now. Objects that did nothing twice are hidden. Actions cost score.\\nAnswer with JSON only: {\\\"choice\\\": \\\"<exact label>\\\", \\\"then\\\": [\\\"<label>\\\", \\\"<label>\\\"], \\\"expect\\\": \\\"<what you expect, one line>\\\",\\n\\\"roles\\\": {\\\"<#id or 'colour c in Pk'>\\\": \\\"<role>\\\"}, \\\"edits\\\": [<optional rulebook edits>]}. \\\"then\\\" (optional, up to 3 labels) lists further\\ncandidates to run right after the first one when the procedure is clear (e.g. the next marks to toggle); the program stops the batch at the\\nfirst surprise. Batching saves model calls, which are the scarce resource. 'roles' (optional) names what things are: button, submit, mark, template, piece, anchor, hole, frame, hud, wall.\\nEdit ops: {\\\"op\\\":\\\"add\\\",\\\"section\\\":\\\"env|rules|win\\\",\\\"text\\\":..,\\\"kind\\\":..,\\\"params\\\":{}} {\\\"op\\\":\\\"confirm|refute|remove\\\",\\\"id\\\":\\\"R3\\\",\\\"note\\\":\\\"..\\\"}\\n{\\\"op\\\":\\\"edit\\\",\\\"id\\\":\\\"R3\\\",\\\"text\\\":..}. Keep edits rare and factual.\\\"\\\"\\\"\\n\\nREVIEW_SYSTEM = \\\"\\\"\\\"You maintain the RULEBOOK of a program playing a grid puzzle game. Something did not go as the rulebook predicted, or a\\nlevel ended. Revise the rulebook so that it explains ALL recorded evidence: refute or edit wrong entries, add the rule that explains the\\nsurprise (machine-checkable kind when possible), update the win condition and the plan. Program-verified entries (source harness, with\\nfor/against counts) are facts; do not contradict them. Think in terms of OBJECT ROLES, not colours: the same colour can be an editable\\nmark in one area and a read-only template in another; small blobs can be anchors that move a piece; corner marks or dotted outlines are\\nframes/holes that a block or piece must fill; a button whose earlier clicks did nothing may be the submit button. Be concrete and short.\\n\\\"\\\"\\\" + SCHEMA_HELP + \\\"\\\"\\\"\\nAnswer with JSON only: {\\\"roles\\\": {\\\"<#id or 'colour c in Pk'>\\\": \\\"<role>\\\"}, \\\"procedure\\\": \\\"<the steps that win a level, in terms of roles>\\\",\\n\\\"edits\\\": [ {\\\"op\\\":\\\"add\\\",\\\"section\\\":\\\"rules\\\",\\\"text\\\":\\\"..\\\",\\\"kind\\\":\\\"..\\\",\\\"params\\\":{}}, {\\\"op\\\":\\\"refute\\\",\\\"id\\\":\\\"R2\\\",\\\"note\\\":\\\"..\\\"},\\n{\\\"op\\\":\\\"edit\\\",\\\"id\\\":\\\"W1\\\",\\\"text\\\":\\\"..\\\"}, {\\\"op\\\":\\\"remove\\\",\\\"id\\\":\\\"E4\\\"} ], \\\"plan\\\": \\\"<updated plan>\\\"}\\\"\\\"\\\"\\n\\n\\nPYTHON_TOOL = {\\\"type\\\": \\\"function\\\", \\\"function\\\": {\\\"name\\\": \\\"python\\\", \\\"description\\\": \\\"Run Python code against the game library. Use act()/click()/press() inside it to play; print what you need to see.\\\",\\n                                                 \\\"parameters\\\": {\\\"type\\\": \\\"object\\\", \\\"properties\\\": {\\\"code\\\": {\\\"type\\\": \\\"string\\\", \\\"description\\\": \\\"Python source to execute\\\"}}, \\\"required\\\": [\\\"code\\\"]}}}\\n\\nCODER_SYSTEM = \\\"\\\"\\\"You play an unknown 64x64 grid puzzle game (colours 0-15 as hex digits, row 0 at the top) by writing Python that runs against a\\ngame library. The score per level is (baseline_actions/your_actions)^2, so use few actions; unfinished levels score 0. Levels of one game\\nshare rules, so what won level N is the plan for level N+1 (adapted to the new board).\\n\\nVariables (rebuilt every call): board (64x64 ints), ascii, level, valid_actions, objects (dicts: id,color,size,bbox=(r0,c0,r1,c1),center=(r,c),\\nregion,hud,cells), regions [(id,colour,bbox)], entities (text), transitions (last 60: action, before, after, level), book (the RULEBOOK text:\\n[OK] confirmed / [?] hypothesis / [X] refuted), goal (win-condition progress lines), candidates [(label, program prediction)],\\nplans [(label, prediction, actions)], plan_actions {label: actions}, notes (your own memory).\\nFunctions: act(a) executes one action or a list \\u2014 a = 'UP'|'DOWN'|'LEFT'|'RIGHT'|'SPACE'|'ACTION7' or (row, col) for a click; every action\\nis checked against the program's prediction and the result is printed; act stops at a level end / game over / 30 actions per call.\\nclick(r,c), press(key, n), path_to(r,c) -> key list on the learned floor map, predict(a) -> what the program expects, scene() -> fresh board\\nand objects, check_rule(fn) scores fn(before_board, action)->after_board|None on every recorded transition (>=0.8 on >=4 makes it a\\nverified rule the predictor uses), define_goal(name, fn(objects, board)->bool, needs_submit=False) registers a win condition the program\\nevaluates and refutes automatically, note(text, section, kind, params) adds a rulebook entry, refute(id, why), set_plan(text), remember(text).\\n\\nHow to work: read book/goal/candidates/plans first. If a plan candidate says a win condition will hold, run it: act(plan_actions[label]).\\nOtherwise write the procedure as code (loops over objects, several actions per call, not one click at a time), verify a hypothesis about\\nan effect with check_rule before relying on it, and define the win condition with define_goal so the program can tell you when it holds.\\nEvery call must execute at least one action (act/click/press) unless it defines a rule or goal. Never repeat an action that changed\\nnothing; candidates already tried in this state are marked. Clicks are act((row, col)) \\u2014 coordinates from objects/candidates. Write ONE\\npython call per turn. Do your reasoning by ACTING, not by writing: at most two short comment lines, no analysis essays, no plans in\\ncomments; a call that only prints or reads state is wasted (the board, objects and predictions are already in this message). Batch\\nseveral actions per call when the procedure is clear. No prose outside the tool call.\\\"\\\"\\\"\\n\\n\\ndef objects_text(frame: Frame, limit: int = 10) -> str:\\n    by: dict[int, list] = {}\\n    hud = []\\n    for n in frame.segmentation[\\\"nodes\\\"]:\\n        (hud if n[\\\"hud\\\"] else by.setdefault(n[\\\"color\\\"], [])).append(n)\\n    rows = []\\n    for c, ns in sorted(by.items(), key=lambda kv: -len(kv[1])):\\n        ns.sort(key=lambda n: n[\\\"pixels\\\"])\\n        pos = \\\", \\\".join(f\\\"({n['center'][0]},{n['center'][1]}){n['pixels']}px\\\" for n in ns[:limit])\\n        rows.append(f\\\"  colour {c}: {len(ns)} object(s) at {pos}{' ...' if len(ns) > limit else ''}\\\")\\n    if hud:\\n        rows.append(\\\"  HUD strips (edge counters/gauges): \\\" + \\\", \\\".join(f\\\"colour {n['color']} {n['pixels']}px rows {n['bbox'][0]}-{n['bbox'][2]} cols {n['bbox'][1]}-{n['bbox'][3]}\\\" for n in hud[:6]))\\n    return \\\"\\\\n\\\".join(rows) or \\\"  (no objects)\\\"\\n\\n\\ndef entities_text(frame: Frame) -> str:\\n    from .entities import build_scene, scene_text\\n    try:\\n        return scene_text(build_scene(frame))\\n    except Exception:\\n        return objects_text(frame)\\n\\n\\ndef crop(frame: Frame, bbox, pad: int = 2) -> str:\\n    r0, c0, r1, c1 = bbox\\n    r0, c0, r1, c1 = max(0, r0 - pad), max(0, c0 - pad), min(63, r1 + pad), min(63, c1 + pad)\\n    lines = frame.ascii.splitlines()\\n    return f\\\"rows {r0}-{r1}, cols {c0}-{c1}:\\\\n\\\" + \\\"\\\\n\\\".join(f\\\"{r:2d} {lines[r][c0:c1 + 1]}\\\" for r in range(r0, r1 + 1))\\n\\n\\ndef parse_json(text: str) -> Optional[dict]:\\n    if not text:\\n        return None\\n    text = text.strip()\\n    m = re.search(r\\\"```(?:json)?\\\\s*(\\\\{.*?\\\\})\\\\s*```\\\", text, re.S)\\n    body = m.group(1) if m else None\\n    if body is None:\\n        i = text.find(\\\"{\\\")\\n        if i < 0:\\n            return None\\n        depth = 0; end = None\\n        for j, ch in enumerate(text[i:], i):\\n            if ch == \\\"{\\\":\\n                depth += 1\\n            elif ch == \\\"}\\\":\\n                depth -= 1\\n                if depth == 0:\\n                    end = j; break\\n        body = text[i:end + 1] if end else text[i:]\\n    for cand in (body, re.sub(r\\\",\\\\s*([\\\\]}])\\\", r\\\"\\\\1\\\", body), re.sub(r\\\",\\\\s*([\\\\]}])\\\", r\\\"\\\\1\\\", body).replace(\\\"'\\\", '\\\"')):\\n        try:\\n            obj = json.loads(cand)\\n            return obj if isinstance(obj, dict) else None\\n        except Exception:\\n            continue\\n    return None\\n\\n\\nclass Model:\\n    def __init__(self, client: ChatClient, *, think_tokens: int = 12000, decide_tokens: int = 1500, review_tokens: int = 6000, log=print, compact: bool = False):\\n        self.client, self.think_tokens, self.decide_tokens, self.review_tokens, self.log = client, think_tokens, decide_tokens, review_tokens, log\\n        self.compact = compact            # shorter prompts: board only when asked, abridged rulebook, fewer candidates/outcomes, short answers\\n        self.last_board = None; self.last_board_actions = -1\\n        self.calls = {\\\"init\\\": 0, \\\"decide\\\": 0, \\\"review\\\": 0, \\\"failed\\\": 0}\\n        self.seconds = 0.0\\n\\n    def _call(self, kind: str, system: str, user: str, *, think: bool) -> Optional[dict]:\\n        self.calls[kind] += 1\\n        msgs = [{\\\"role\\\": \\\"system\\\", \\\"content\\\": system}, {\\\"role\\\": \\\"user\\\", \\\"content\\\": user}]\\n        for attempt, th in enumerate([think, False] if think else [False, False]):\\n            budget = (self.think_tokens if kind == \\\"init\\\" else self.review_tokens) if th else (self.decide_tokens if kind == \\\"decide\\\" else (2000 if self.compact else 4000))\\n            override = {\\\"chat_template_kwargs\\\": {\\\"enable_thinking\\\": th}, \\\"max_tokens\\\": budget}\\n            if attempt == 1 and not think:   # the first no-think answer rambled past the budget: demand the bare object\\n                msgs = msgs[:2] + [{\\\"role\\\": \\\"user\\\", \\\"content\\\": \\\"Reply with ONLY the JSON object on one line. No explanation.\\\"}]\\n            try:\\n                r = self.client.chat(msgs, tools=None, tool_choice=\\\"none\\\", override=override)\\n            except Exception as e:\\n                self.log(f\\\"[model:{kind}] call failed: {e!r}\\\"); self.calls[\\\"failed\\\"] += 1\\n                return None\\n            self.seconds += r.latency\\n            content = r.message.get(\\\"content\\\", \\\"\\\") or \\\"\\\"\\n            obj = parse_json(content)\\n            self.log(f\\\"[model:{kind}] think={th} {r.latency:.0f}s finish={r.finish_reason} out={len(content)} chars reasoning={len(r.reasoning)} chars\\\"\\n                     + (\\\"\\\" if obj else \\\" \\u2014 NO JSON\\\") + (f\\\"\\\\n  reasoning: {r.reasoning[:600]!r}\\\" if r.reasoning else \\\"\\\") + f\\\"\\\\n  answer: {content[:1200]}\\\")\\n            if obj is not None:\\n                return obj\\n            if attempt == 0:\\n                self.log(f\\\"[model:{kind}] retry \\\" + (\\\"without thinking\\\" if think else \\\"with a JSON-only reminder\\\"))\\n        self.calls[\\\"failed\\\"] += 1\\n        return None\\n\\n    # \\u2500\\u2500 prompts \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\n    def init_rulebook(self, game, think: bool = True) -> Optional[dict]:\\n        f = game.frame\\n        user = (f\\\"GAME: {game.levels_total or '?'} levels; level {game.level}. Valid actions now: {game.valid_actions}.\\\\n\\\"\\n                f\\\"ENTITIES (areas P0.. = panels/floors/walls; objects #id colour size @(row,col), grouped by area):\\\\n{entities_text(f)}\\\\n\\\\n\\\"\\n                f\\\"BOARD (64x64 hex digits, row 0 at the top):\\\\n{f.ascii}\\\\n\\\\n\\\"\\n                \\\"Write the initial rulebook (hypotheses) and the plan. Name the role of each area and object group (env entries).\\\")\\n        return self._call(\\\"init\\\", INIT_SYSTEM, user, think=think)\\n\\n    def code_turn(self, game, book, cands: list, plans: list, goal: dict, outcomes: list[str], budget_text: str, notes: str, prev_outputs: list, nudge: str = \\\"\\\") -> tuple:\\n        \\\"\\\"\\\"One coder-mode turn: returns (code, text). Code comes from the python tool call, or from a ```python block in the content.\\\"\\\"\\\"\\n        f = game.frame\\n        parts = [book.render(), f\\\"GAME: level {game.level} of {game.levels_total or '?'}; {budget_text}; valid actions {game.valid_actions}.\\\"]\\n        if goal and goal.get(\\\"lines\\\"):\\n            parts.append(\\\"WIN CONDITION PROGRESS (program-evaluated):\\\\n  \\\" + \\\"\\\\n  \\\".join(goal[\\\"lines\\\"]))\\n        if nudge:\\n            parts.append(\\\"NOTE: \\\" + nudge)\\n        if notes:\\n            parts.append(\\\"YOUR NOTES:\\\\n\\\" + notes)\\n        if prev_outputs:\\n            parts.append(\\\"YOUR LAST CALLS AND THEIR OUTPUT:\\\\n\\\" + \\\"\\\\n\\\".join(prev_outputs[-2:]))\\n        if outcomes:\\n            parts.append(\\\"RECENT ACTIONS (newest last):\\\\n  \\\" + \\\"\\\\n  \\\".join(outcomes[-8:]))\\n        parts.append(f\\\"ENTITIES (areas P0.. and the objects in each; #id colour size @(row,col)):\\\\n{entities_text(f)}\\\")\\n        parts.append(f\\\"BOARD:\\\\n{f.ascii}\\\")\\n        parts.append(\\\"PLAN CANDIDATES (program-made; run with act(plan_actions[label])):\\\\n  \\\" + (\\\"\\\\n  \\\".join(f\\\"{c.label} -> {c.prediction[:200]}\\\" for c in plans if c.kind == \\\"plan\\\") or \\\"(none)\\\"))\\n        parts.append(\\\"CANDIDATE ACTIONS (label -> program prediction):\\\\n  \\\" + \\\"\\\\n  \\\".join(c.line() for c in cands[:32] if c.kind != \\\"info\\\"))\\n        parts.append(\\\"Write the python call for this turn.\\\")\\n        user = \\\"\\\\n\\\\n\\\".join(parts)\\n        self.calls[\\\"decide\\\"] += 1\\n        msgs = [{\\\"role\\\": \\\"system\\\", \\\"content\\\": CODER_SYSTEM}, {\\\"role\\\": \\\"user\\\", \\\"content\\\": user}]\\n        try:\\n            r = self.client.chat(msgs, tools=[PYTHON_TOOL], tool_choice=\\\"auto\\\", override={\\\"chat_template_kwargs\\\": {\\\"enable_thinking\\\": False}, \\\"max_tokens\\\": 1400})\\n        except Exception as e:\\n            self.log(f\\\"[model:code] call failed: {e!r}\\\"); self.calls[\\\"failed\\\"] += 1; return \\\"\\\", \\\"\\\"\\n        self.seconds += r.latency\\n        msg = r.message; content = msg.get(\\\"content\\\") or \\\"\\\"\\n        code = \\\"\\\"\\n        for call in msg.get(\\\"tool_calls\\\") or []:\\n            raw = call.get(\\\"function\\\", {}).get(\\\"arguments\\\") or \\\"\\\"\\n            try:\\n                args = json.loads(raw or \\\"{}\\\")\\n                if isinstance(args, dict) and args.get(\\\"code\\\"):\\n                    code = str(args[\\\"code\\\"]); break\\n                if isinstance(args, str) and args.strip():\\n                    code = args; break\\n            except Exception:\\n                m = re.search(r'\\\"code\\\"\\\\s*:\\\\s*\\\"(.*)\\\"\\\\s*}?\\\\s*$', raw, re.S)   # unterminated / badly escaped JSON: salvage the code string\\n                if m:\\n                    code = m.group(1).encode().decode(\\\"unicode_escape\\\", errors=\\\"ignore\\\"); break\\n                if any(k in raw for k in (\\\"act(\\\", \\\"click(\\\", \\\"press(\\\", \\\"print(\\\")):\\n                    code = raw; break\\n                self.log(f\\\"[model:code] unparsable tool arguments: {raw[:200]!r}\\\")\\n        if not code:\\n            m = re.search(r\\\"```(?:python)?\\\\s*(.*?)```\\\", content, re.S)\\n            if m:\\n                code = m.group(1)\\n            elif any(k in content for k in (\\\"act(\\\", \\\"click(\\\", \\\"press(\\\", \\\"print(\\\")) and not content.strip().startswith(\\\"{\\\"):\\n                code = content\\n        self.log(f\\\"[model:code] {r.latency:.0f}s finish={r.finish_reason} tool_calls={len(msg.get('tool_calls') or [])} code={len(code)} chars content={len(content)} chars\\\")\\n        if not code:\\n            self.calls[\\\"failed\\\"] += 1\\n        return code, content\\n\\n    def board_block(self, game, force: bool = False) -> str:\\n        \\\"\\\"\\\"Full board when asked (level start, after a review, every 8th decision), else a diff against the last full board shown.\\\"\\\"\\\"\\n        f = game.frame\\n        if not self.compact or force or self.last_board is None or self.last_board.level != f.level or game.actions_used - self.last_board_actions >= 8:\\n            self.last_board = f; self.last_board_actions = game.actions_used\\n            return f\\\"BOARD (64x64):\\\\n{f.ascii}\\\"\\n        from arcnav.frame import summarize_diff\\n        d = summarize_diff(self.last_board, f, max_items=6)\\n        if not d.get(\\\"changed_cells\\\"):\\n            return \\\"BOARD: unchanged since the last full board shown.\\\"\\n        parts = [f\\\"{d['changed_cells']} cells changed since the last full board (rows {d['bbox'][0]}-{d['bbox'][2]}, cols {d['bbox'][1]}-{d['bbox'][3]})\\\"]\\n        if d.get(\\\"moved\\\"):\\n            parts.append(\\\"moved \\\" + \\\", \\\".join(f\\\"colour {m['color']} {m['from']}->{m['to']}\\\" for m in d[\\\"moved\\\"][:4]))\\n        if d.get(\\\"disappeared\\\"):\\n            parts.append(\\\"vanished \\\" + \\\", \\\".join(f\\\"colour {x['color']}@{x['center']}\\\" for x in d[\\\"disappeared\\\"][:4]))\\n        if d.get(\\\"appeared\\\"):\\n            parts.append(\\\"appeared \\\" + \\\", \\\".join(f\\\"colour {x['color']}@{x['center']}\\\" for x in d[\\\"appeared\\\"][:4]))\\n        return \\\"BOARD: \\\" + \\\"; \\\".join(parts) + \\\" (the ENTITIES table below is current).\\\"\\n\\n    def decide(self, game, book: Rulebook, cands: list, outcomes: list[str], budget_text: str, extra: str = \\\"\\\", force_board: bool = False) -> Optional[dict]:\\n        f = game.frame\\n        nc, no = (24, 6) if self.compact else (40, 10)\\n        user = (f\\\"{book.render(compact=self.compact)}\\\\n\\\\nGAME: level {game.level} of {game.levels_total or '?'}; {budget_text}; valid actions {game.valid_actions}.\\\\n\\\"\\n                + (extra + \\\"\\\\n\\\" if extra else \\\"\\\")\\n                + ((\\\"RECENT ACTIONS (newest last):\\\\n  \\\" + \\\"\\\\n  \\\".join(outcomes[-no:]) + \\\"\\\\n\\\") if outcomes else \\\"\\\")\\n                + f\\\"\\\\nENTITIES (areas P0.. and the objects in each; #id colour size @(row,col)):\\\\n{entities_text(f)}\\\\n\\\\n{self.board_block(game, force_board)}\\\\n\\\\nCANDIDATES (label -> program prediction):\\\\n  \\\"\\n                + \\\"\\\\n  \\\".join(c.line() for c in cands[:nc]) + \\\"\\\\n\\\\nChoose one candidate label.\\\")\\n        return self._call(\\\"decide\\\", DECIDE_SYSTEM, user, think=False)\\n\\n    def review(self, game, book: Rulebook, event: str, evidence_changes: list[str], outcomes: list[str], *, before: Optional[Frame], bbox=None, think: bool = True, extra: str = \\\"\\\") -> Optional[dict]:\\n        f = game.frame\\n        parts = [book.render(), f\\\"GAME: level {game.level} of {game.levels_total or '?'}; valid actions {game.valid_actions}.\\\", \\\"EVENT: \\\" + event]\\n        if extra:\\n            parts.append(extra)\\n        if evidence_changes:\\n            parts.append(\\\"PROGRAM-VERIFIED CHANGES TO THE RULEBOOK (facts):\\\\n  \\\" + \\\"\\\\n  \\\".join(evidence_changes[-12:]))\\n        if outcomes:\\n            parts.append(\\\"RECENT ACTIONS (newest last):\\\\n  \\\" + \\\"\\\\n  \\\".join(outcomes[-12:]))\\n        if before is not None and bbox is not None:\\n            parts.append(\\\"BEFORE (changed region):\\\\n\\\" + crop(before, bbox) + \\\"\\\\nAFTER (same region):\\\\n\\\" + crop(f, bbox))\\n        if self.compact and before is not None and bbox is not None:\\n            parts.append(f\\\"ENTITIES NOW (areas P0.. and the objects in each):\\\\n{entities_text(f)}\\\")   # the crop above shows the change; the full board is skipped\\n        else:\\n            parts.append(f\\\"ENTITIES NOW (areas P0.. and the objects in each):\\\\n{entities_text(f)}\\\\n\\\\nBOARD NOW:\\\\n{f.ascii}\\\")\\n        if self.compact:\\n            self.last_board = None   # the next decision shows the full board again\\n        parts.append(\\\"Revise the rulebook (edits) and the plan.\\\")\\n        return self._call(\\\"review\\\", REVIEW_SYSTEM, \\\"\\\\n\\\\n\\\".join(parts), think=think)\\n\", \"plans.py\": \"\\\"\\\"\\\"Program-made plan candidates: short action sequences that make a live win predicate true on the learned effect model.\\n\\n  match:   Equal predicate     -> click every editable mark that differs from the template (toggle rule known)\\n  press:   Inside predicate    -> BFS over button presses on the solved slot permutations until the blocks sit in the frames\\n  marker:  Inside predicate    -> move the selected marker so that the coupled piece lands in the outline\\nThe model only chooses among them; every action is still verified one by one when executed.\\n\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nfrom collections import deque\\nfrom typing import Optional\\n\\nfrom .candidates import Candidate\\nfrom .entities import Scene\\nfrom .predict import Evidence\\nfrom . import goals2\\n\\n\\ndef make_plans(game, ev: Evidence, preds: list, *, max_depth: int = 24, max_states: int = 20000) -> list[Candidate]:\\n    out: list[Candidate] = []\\n    sc = ev.scene\\n    for eid, p in preds:\\n        try:\\n            if isinstance(p, goals2.Equal):\\n                c = _plan_match(ev, sc, p, eid)\\n            elif isinstance(p, goals2.Inside):\\n                c = _plan_press(ev, sc, p, eid, max_depth, max_states) or _plan_marker(ev, sc, p, eid)\\n            elif isinstance(p, (goals2.CollectAll, goals2.Reach, goals2.CollectReach)):\\n                c = _plan_move(ev, sc, p, eid)\\n            else:\\n                c = None\\n        except Exception as e:  # a planner bug must never stop play\\n            c = Candidate(f\\\"plan:{eid}\\\", \\\"info\\\", [], f\\\"planner error {type(e).__name__}\\\", priority=999)\\n        if c is not None:\\n            out.append(c)\\n    return out\\n\\n\\ndef _plan_match(ev: Evidence, sc: Scene, p: goals2.Equal, eid: str) -> Optional[Candidate]:\\n    a, b = goals2._region_by_spec(sc, p.r1), goals2._region_by_spec(sc, p.r2)\\n    if a is None or b is None:\\n        return None\\n    ma, mb = goals2._mask(sc, a), goals2._mask(sc, b)\\n    if ma == mb:\\n        return None\\n    ra, rb = sc.region_named(a), sc.region_named(b)\\n    diff = {k for k in set(ma) | set(mb) if ma.get(k) != mb.get(k)}\\n    # objects of the editable area that cover differing cells\\n    targets = []\\n    for o in sc.objs:\\n        if o.region != a or o.hud or not o.cells:\\n            continue\\n        if any((r - ra.bbox[0], c - ra.bbox[1]) in diff for r, c in o.cells):\\n            targets.append(o)\\n    if not targets:\\n        return None\\n    acts = []\\n    for o in targets[:24]:\\n        r, c = o.center\\n        if sc.frame.grid[r][c] != o.color:\\n            r, c = o.cells[0]\\n        acts.append({\\\"action\\\": \\\"MOUSE\\\", \\\"row\\\": r, \\\"col\\\": c})\\n    return Candidate(f\\\"plan:match({eid})\\\", \\\"plan\\\", acts, f\\\"click the {len(targets)} marks in P{a} that differ from the template P{b} (toggle each once) -> {eid} should hold\\\",\\n                     priority=5, pred_kind=\\\"plan\\\")\\n\\n\\ndef _plan_press(ev: Evidence, sc: Scene, p: goals2.Inside, eid: str, max_depth: int, max_states: int) -> Optional[Candidate]:\\n    allf = goals2.find_frames(sc)\\n    pairs = [(c, c) for c in sorted({f.color for f in allf})] if p.color == p.frame else [(p.color, p.frame)]\\n    frames = [f for f in allf if any(f.color == fc for _, fc in pairs)]\\n    dot_ids = {i for f in frames for i in f.dots}\\n    blocks = [o for o in sc.objs if any(o.color == bc for bc, _ in pairs) and not o.hud and o.size >= 4 and o.id not in dot_ids]\\n    if not frames or not blocks:\\n        return None\\n    buttons = [(key, ev.cs.sigma(key)) for key in ev.cs.presses]\\n    buttons = [(key, sg) for key, sg in buttons if any(q != q0 for q0, q in sg.items())]\\n    if not buttons:\\n        # nothing learned on this level yet: press a button that moved things on an earlier level (same colour), if any\\n        known_cols = {key[0] for key in ev.cs_all.presses}\\n        cand = [o for o in sc.objs if o.color in known_cols and not o.hud and o.size >= 8]\\n        if cand:\\n            b = min(cand, key=lambda o: ev.cs.succ_n.get(o.key, 0))\\n            return Candidate(f\\\"plan:learn({eid})\\\", \\\"plan\\\", [{\\\"action\\\": \\\"MOUSE\\\", \\\"row\\\": b.center[0], \\\"col\\\": b.center[1]}],\\n                             f\\\"press button #{b.id} (colour {b.color} buttons moved blocks on an earlier level) to learn its track on this level\\\", priority=7, pred_kind=\\\"plan\\\")\\n        return None\\n    button_objs = {}\\n    for key, _ in buttons:\\n        o = next((x for x in sc.objs if x.key == key), None)\\n        if o is not None:\\n            button_objs[key] = o\\n    buttons = [(key, sg) for key, sg in buttons if key in button_objs]\\n    if not buttons:\\n        return None\\n    movable = set()\\n    for _, sg in buttons:\\n        movable |= {q0 for q0, q in sg.items() if q != q0} | {q for q0, q in sg.items() if q != q0}\\n    if not any(o.bbox in movable for o in blocks):\\n        return None\\n    static = {o.bbox: (o.color, o.shape) for o in sc.objs if not o.hud and o.bbox not in movable}\\n    start = frozenset((o.bbox, o.color, o.shape) for o in sc.objs if not o.hud and o.bbox in movable)\\n    need = min(len(blocks), len(frames))\\n\\n    fcol = {bc: fc for bc, fc in pairs}\\n\\n    def goal(state) -> bool:\\n        k = 0\\n        for bbox, col, sh in state:\\n            if col in fcol and any(f.color == fcol[col] and f.inner[0] - 2 <= bbox[0] and f.inner[1] - 2 <= bbox[1] and bbox[2] <= f.inner[2] + 2 and bbox[3] <= f.inner[3] + 2\\n                                   and abs((bbox[0] + bbox[2]) / 2 - (f.inner[0] + f.inner[2]) / 2) <= 2 and abs((bbox[1] + bbox[3]) / 2 - (f.inner[1] + f.inner[3]) / 2) <= 2 for f in frames):\\n                k += 1\\n        return k >= need\\n\\n    def step(state, sg):\\n        new = set()\\n        for bbox, col, sh in state:\\n            q = sg.get(bbox)\\n            if q is None and bbox in movable and bbox in sg:\\n                return None\\n            if q is None:\\n                # not on this button's track (or unsolved): a slot on this track with unknown successor makes the press unpredictable\\n                if bbox in {q0 for q0 in sg}:\\n                    return None\\n                new.add((bbox, col, sh))\\n            else:\\n                new.add((q, col, sh))\\n        return frozenset(new)\\n\\n    if goal(start):\\n        return None\\n    q = deque([(start, [])]); seen = {start}; n = 0\\n    while q and n < max_states:\\n        state, path = q.popleft(); n += 1\\n        if len(path) >= max_depth:\\n            continue\\n        for key, sg in buttons:\\n            nxt = step(state, sg)\\n            if nxt is None or nxt in seen:\\n                continue\\n            seen.add(nxt)\\n            if goal(nxt):\\n                seq = path + [key]\\n                acts = [{\\\"action\\\": \\\"MOUSE\\\", \\\"row\\\": button_objs[k].center[0], \\\"col\\\": button_objs[k].center[1]} for k in seq]\\n                for a in acts:\\n                    if sc.frame.grid[a[\\\"row\\\"]][a[\\\"col\\\"]] != sc.obj_at(a[\\\"row\\\"], a[\\\"col\\\"]).color if sc.obj_at(a[\\\"row\\\"], a[\\\"col\\\"]) else True:\\n                        pass\\n                from collections import Counter\\n                cnt = Counter(f\\\"#{button_objs[k].id}\\\" for k in seq)\\n                return Candidate(f\\\"plan:press({eid})\\\", \\\"plan\\\", acts, f\\\"{len(seq)} button presses ({', '.join(f'{b}x{c}' for b, c in cnt.items())}) move the colour-{p.color} blocks into the colour-{p.frame} frames on the learned tracks -> {eid} should hold\\\",\\n                                 priority=5, pred_kind=\\\"plan\\\")\\n            q.append((nxt, path + [key]))\\n    # no sequence on what is solved: press the least-explored button-like object (same colours as known buttons) so that every track gets solved\\n    known_cols = {key[0] for key in ev.cs.presses} | {key[0] for key in ev.cs_all.presses}\\n    cand = [o for o in sc.objs if o.color in known_cols and not o.hud and o.size >= 8]\\n    if not cand:\\n        return None\\n    b = min(cand, key=lambda o: (ev.cs.succ_n.get(o.key, 0), o.center))\\n    return Candidate(f\\\"plan:learn({eid})\\\", \\\"plan\\\", [{\\\"action\\\": \\\"MOUSE\\\", \\\"row\\\": b.center[0], \\\"col\\\": b.center[1]}],\\n                     f\\\"press button #{b.id} ({ev.cs.succ_n.get(b.key, 0)} presses so far) to learn its track; no press sequence up to {max_depth} exists yet on the solved tracks ({n} states searched)\\\",\\n                     priority=7, pred_kind=\\\"plan\\\")\\n\\n\\ndef _plan_marker(ev: Evidence, sc: Scene, p: goals2.Inside, eid: str) -> Optional[Candidate]:\\n    allf = goals2.find_frames(sc)\\n    for bc, fc in ([(c, c) for c in sorted({f.color for f in allf})] if p.color == p.frame else [(p.color, p.frame)]):\\n        c = _plan_marker_pair(ev, sc, p, eid, bc, fc, allf)\\n        if c is not None and c.kind == \\\"plan\\\":\\n            return c\\n    return None\\n\\n\\ndef _plan_marker_pair(ev: Evidence, sc: Scene, p: goals2.Inside, eid: str, bc: int, fc: int, allf: list) -> Optional[Candidate]:\\n    frames = [f for f in allf if f.color == fc]\\n    dot_ids = {i for f in frames for i in f.dots}\\n    pieces = [o for o in sc.objs if o.color == bc and not o.hud and o.size >= 4 and o.id not in dot_ids and min(o.bbox[2] - o.bbox[0], o.bbox[3] - o.bbox[1]) >= 2\\n              and not any(f.contains(o) for f in frames)]\\n    if not frames or not pieces:\\n        return None\\n    # the marker colour that moves on clicks (cursor rule) and its coupling with the piece colour\\n    markers = []\\n    for (col, rg), cnt in ev.cs.cursor.items():\\n        k, n = cnt.most_common(1)[0]\\n        if k != -1 and n * 2 > sum(cnt.values()):\\n            markers.append(k)\\n    # objects whose click makes the marker jump onto them (selection): the other anchors\\n    selectable = []\\n    for (col, rg), cnt in ev.cs.select.items():\\n        if cnt.get(True, 0) > cnt.get(False, 0):\\n            selectable.extend(o for o in sc.objs if o.color == col and o.region == rg and not o.hud and o.size >= 4)\\n    for k in set(markers):\\n        cp = ev.cs.coupling(k, bc)\\n        if not cp:\\n            continue\\n        (rr, rc), _ = cp\\n        if float(rr) == 0 or float(rc) == 0:\\n            continue\\n        mks = [o for o in sc.objs if o.color == k and not o.hud and o.size >= 4]\\n        if not mks:\\n            continue\\n        mk = mks[0]\\n        piece = min(pieces, key=lambda o: min(abs(o.center[0] - (f.inner[0] + f.inner[2]) / 2) + abs(o.center[1] - (f.inner[1] + f.inner[3]) / 2) for f in frames))\\n        f = min(frames, key=lambda f: abs(piece.center[0] - (f.inner[0] + f.inner[2]) / 2) + abs(piece.center[1] - (f.inner[1] + f.inner[3]) / 2))\\n        need_r = (f.inner[0] + f.inner[2]) / 2 - piece.center[0]; need_c = (f.inner[1] + f.inner[3]) / 2 - piece.center[1]\\n        floor = sc.region_at(mk.center[0], mk.center[1])\\n        # option 1: move the selected marker itself; option 2: select another anchor first, then move it\\n        options = [(None, mk)] + [(a, a) for a in selectable if a.id != mk.id]\\n        blocked = []\\n        for sel, anchor in options:\\n            tr, tc = int(round(anchor.center[0] + need_r / float(rr))), int(round(anchor.center[1] + need_c / float(rc)))\\n            if not (0 <= tr < 64 and 0 <= tc < 64) or sc.region_at(tr, tc) != floor or sc.obj_at(tr, tc) is not None:\\n                blocked.append((anchor.id, tr, tc)); continue\\n            acts = ([{\\\"action\\\": \\\"MOUSE\\\", \\\"row\\\": sel.center[0], \\\"col\\\": sel.center[1]}] if sel is not None else []) + [{\\\"action\\\": \\\"MOUSE\\\", \\\"row\\\": tr, \\\"col\\\": tc}]\\n            how = f\\\"select anchor #{sel.id} then click ({tr},{tc})\\\" if sel is not None else f\\\"click ({tr},{tc})\\\"\\n            return Candidate(f\\\"plan:marker({eid})\\\", \\\"plan\\\", acts, f\\\"{how}: the colour-{k} marker goes to ({tr},{tc}) and piece #{piece.id} (coupled x{rr}) lands in the colour-{fc} frame at ({f.inner[0]},{f.inner[1]}) -> {eid} should hold\\\",\\n                             priority=5, pred_kind=\\\"plan\\\")\\n        if blocked:\\n            return Candidate(f\\\"plan:marker({eid})\\\", \\\"info\\\", [], f\\\"to put piece #{piece.id} in the frame an anchor would have to reach \\\" + \\\", \\\".join(f\\\"({r},{c})\\\" for _, r, c in blocked[:3]) + \\\" \\u2014 not on the floor (wall or occupied)\\\", priority=999)\\n    return None\\n\\n\\ndef _plan_move(ev: Evidence, sc: Scene, p, eid: str) -> Optional[Candidate]:\\n    \\\"\\\"\\\"Walk to the nearest object of the colour the predicate needs (collect first, then reach), on the learned floor map.\\\"\\\"\\\"\\n    nav = ev.nav\\n    if nav is None or not ev.moves or not ev.avatar:\\n        return None\\n    if isinstance(p, goals2.CollectReach):\\n        ok1 = goals2.CollectAll(p.color).evaluate(sc, ev)[0]\\n        colour, what = (p.reach, \\\"reach\\\") if ok1 else (p.color, \\\"collect\\\")\\n    elif isinstance(p, goals2.CollectAll):\\n        colour, what = p.color, \\\"collect\\\"\\n    else:\\n        colour, what = p.color, \\\"reach\\\"\\n    box = goals2._avatar_box(ev)\\n    targets = [o for o in sc.objs if o.color == colour and not o.hud and o.size <= 400 and not (box and goals2._touching(box, o, 0))]\\n    if not targets:\\n        return None\\n    best = None; skipped = []\\n    for o in sorted(targets, key=lambda o: abs(o.center[0] - box[0]) + abs(o.center[1] - box[1]))[:8]:\\n        enclosing = sc.obj_around(o.center[0], o.center[1])\\n        if enclosing is not None and enclosing.id != o.id and enclosing.color in ev.wall_colors:\\n            skipped.append(f\\\"#{o.id} is drawn inside a colour-{enclosing.color} wall object\\\"); continue   # a decoration on a door, not a collectible\\n        path = nav.path_to(o.center[0], o.center[1], ignore_wall_target=True)\\n        if path:\\n            first = ev.predict(path[0])\\n            if first.kind == \\\"blocked\\\":\\n                skipped.append(f\\\"#{o.id}: first step {path[0]} is blocked\\\"); continue\\n            if \\\"HAZARD\\\" in first.text:\\n                skipped.append(f\\\"#{o.id}: first step {path[0]} enters a hazard\\\"); continue\\n            if not all(isinstance(a, str) for a in path):\\n                skipped.append(f\\\"#{o.id}: path unusable\\\"); continue\\n            if best is None or len(path) < len(best[1]):\\n                best = (o, path)\\n    if best is None:\\n        return Candidate(f\\\"plan:{what}({eid})\\\", \\\"info\\\", [], f\\\"no usable path from the avatar to a colour-{colour} object yet\\\" + (\\\" (\\\" + \\\"; \\\".join(skipped[:3]) + \\\")\\\" if skipped else \\\" (explore the floor first)\\\"), priority=999)\\n    o, path = best\\n    gauge = f\\\"; gauge {ev.gauge['per_action'] * len(path):+}\\\" if ev.gauge else \\\"\\\"\\n    return Candidate(f\\\"plan:{what}({eid})\\\", \\\"plan\\\", list(path), f\\\"walk {len(path)} steps ({''.join(a[0] for a in path)}) to colour-{colour} object #{o.id} at ({o.center[0]},{o.center[1]}){gauge} -> {eid}: {'it should vanish' if what == 'collect' else 'the avatar touches it'}\\\",\\n                     priority=6, pred_kind=\\\"plan\\\")\\n\", \"predict.py\": \"\\\"\\\"\\\"Deterministic predictor: from the recorded transitions, say what each action should do on the current board.\\n\\nEvidence is rebuilt from the game record (no model): movement deltas / walls / gauge (arcnav.nav, arcnav.rules), a\\nposition-relative click-effect model (arcnav.effects), no-op keys, and win facts from completed levels (arcnav.goals).\\nThe same evidence is exported as `facts()` so the rulebook can score the model's hypotheses against it.\\n\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nfrom collections import Counter\\nfrom dataclasses import dataclass, field\\nfrom typing import Optional\\n\\nfrom arcnav.effects import ClickModel\\nfrom arcnav.frame import Frame, masked_ascii, summarize_diff\\nfrom arcnav.nav import NavHelper, extract_objects\\nfrom arcnav import goals as goal_inference\\n\\nfrom .env import Game, Transition, MOVE_KEYS, action_label\\nfrom .entities import Obj, Scene, build_scene, match\\nfrom .clicks import ClickStats, change_class, record, _edge_only_diff\\n\\n\\n@dataclass\\nclass Prediction:\\n    kind: str                 # move | blocked | noop | board | hud | unknown | level\\n    text: str\\n    avatar_to: Optional[tuple] = None      # (row, col) of the avatar's bbox top-left after a move\\n    board: Optional[list] = None           # predicted grid (board kind)\\n    rules: list = field(default_factory=list)   # (kind, params) the prediction relies on\\n    confidence: str = \\\"hypothesis\\\"         # confirmed | hypothesis | none\\n    moves: Optional[dict] = None           # objects kind: before-object id -> predicted new bbox\\n    coupled: Optional[list] = None         # cursor kind: [(colour, predicted (dr, dc))] for objects coupled to the marker\\n\\n\\n@dataclass\\nclass Verdict:\\n    ok: Optional[bool]        # True = as predicted, False = mismatch, None = no prediction to check (observation)\\n    text: str\\n    diff: dict = field(default_factory=dict)\\n\\n\\nclass Evidence:\\n    \\\"\\\"\\\"Everything the harness knows for sure at this moment, derived from the transitions.\\\"\\\"\\\"\\n\\n    def __init__(self, game: Game, distrust: Optional[dict] = None, custom_rules: Optional[list] = None):\\n        self.game = game\\n        self.custom_rules = custom_rules or []   # (name, fn, accuracy, n): model-written, replay-verified transition rules\\n        self.distrust = distrust or {}     # click colour -> number of failed board predictions on this level (agent-maintained)\\n        self.frame: Frame = game.frame\\n        cur = game.attempt_transitions()\\n        self.nav = NavHelper(cur, self.frame) if game.has_move_keys() else None   # click-only games need no movement model\\n        try:\\n            self.rules = induce(game, self.nav)\\n        except Exception:\\n            self.rules = []\\n        self.click = ClickModel()\\n        for t in game.transitions:\\n            if isinstance(t.action, dict):\\n                self.click.observe(t.before_frame.grid, t.after_frame.grid, t.action[\\\"row\\\"], t.action[\\\"col\\\"])\\n        self.click.fit()\\n        self.moves: dict = dict(self.nav.moves) if self.nav else {}\\n        self.avatar = self.nav.avatar() if self.nav else None\\n        self.wall_colors = {r[\\\"params\\\"][\\\"color\\\"] for r in self.rules if r[\\\"kind\\\"] == \\\"wall\\\" and r[\\\"support\\\"] > r[\\\"counter\\\"]}\\n        self.ambiguous_colors = {r[\\\"params\\\"][\\\"color\\\"]: (r[\\\"support\\\"], r[\\\"counter\\\"]) for r in self.rules if r[\\\"kind\\\"] == \\\"wall\\\" and r[\\\"counter\\\"] >= 1}\\n        # a key is a no-op only while every press did nothing; once it did something the effect is conditional (state-dependent)\\n        self.noop_keys = {r[\\\"params\\\"][\\\"action\\\"] for r in self.rules if r[\\\"kind\\\"] == \\\"noop\\\" and r[\\\"support\\\"] >= 1 and r[\\\"counter\\\"] == 0}\\n        self.conditional_keys = {r[\\\"params\\\"][\\\"action\\\"]: (r[\\\"support\\\"], r[\\\"counter\\\"]) for r in self.rules if r[\\\"kind\\\"] == \\\"noop\\\" and r[\\\"counter\\\"] >= 1}\\n        self.flaky_moves = {r[\\\"params\\\"][\\\"action\\\"]: (r[\\\"support\\\"], r[\\\"counter\\\"]) for r in self.rules if r[\\\"kind\\\"] == \\\"move\\\" and r[\\\"counter\\\"] >= max(2, r[\\\"support\\\"])}\\n        self.hazard_colors = {r[\\\"params\\\"][\\\"color\\\"] for r in self.rules if r[\\\"kind\\\"] == \\\"hazard\\\"}\\n        self.gauge = self.nav.gauge() if self.nav else None\\n        # click statistics keyed by (colour, region) and by button object: the current level's records first, all levels as fallback\\n        self.scene: Scene = build_scene(self.frame)\\n        self.distrust = {k if isinstance(k, tuple) else (k, None): v for k, v in self.distrust.items()}\\n        self.cs = ClickStats(); self.cs_all = ClickStats()\\n        for t in game.transitions:\\n            if isinstance(t.action, dict):\\n                rec = record(t)\\n                (self.cs if t.level == game.level else self.cs_all).add(rec)\\n        if game.level_transitions():   # the offset click-effect model too: refit on this level once it has clicks\\n            lvl_clicks = [t for t in game.level_transitions() if isinstance(t.action, dict)]\\n            if lvl_clicks:\\n                self.click = ClickModel()\\n                for t in lvl_clicks:\\n                    self.click.observe(t.before_frame.grid, t.after_frame.grid, t.action[\\\"row\\\"], t.action[\\\"col\\\"])\\n                self.click.fit(min_support=0.8)\\n\\n    def _stat(self, name: str, colour: int, region: int):\\n        \\\"\\\"\\\"(counter, exact_region?) from this level's stats, else from all levels.\\\"\\\"\\\"\\n        v, exact = getattr(self.cs, name)(colour, region)\\n        if v:\\n            return v, exact, True\\n        v, exact = getattr(self.cs_all, name)(colour, region)\\n        return v, exact, False\\n\\n    def click_key(self, r: int, c: int) -> tuple:\\n        o = self.scene.obj_around(r, c)\\n        return (o.color if o is not None else self.frame.grid[r][c], o.region if o else self.scene.label[r][c]), o\\n\\n    # \\u2500\\u2500 facts for the rulebook \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\n    def facts(self) -> list[dict]:\\n        out = [dict(r) for r in self.rules]\\n        if self.avatar:\\n            av = self.avatar; x0, y0, x1, y1 = av[\\\"bbox_xyxy\\\"]\\n            out.append({\\\"kind\\\": \\\"avatar\\\", \\\"params\\\": {\\\"colors\\\": av[\\\"colors\\\"]}, \\\"support\\\": 2, \\\"counter\\\": 0,\\n                        \\\"text\\\": f\\\"the avatar (what the movement keys move) is the colour-{av['colors']} block, {y1 - y0 + 1}x{x1 - x0 + 1} px, now at rows {y0}-{y1} cols {x0}-{x1}\\\"})\\n        # per (colour, region) click facts from this level (fallback: all levels)\\n        stats = self.cs if self.cs.n else self.cs_all\\n        for (c, rg), h in sorted(stats.hist.items()):\\n            n = sum(h.values()); where = f\\\" in P{rg}\\\" if rg >= 0 else \\\"\\\"\\n            rgl = self.scene.region_named(rg)\\n            rgtxt = f\\\" (P{rg} = the colour-{rgl.color} area)\\\" if rgl else \\\"\\\"\\n            if h.get(\\\"world\\\", 0) == 0:\\n                what = \\\"changes only the HUD/counter\\\" if h.get(\\\"hud\\\") else \\\"changes nothing\\\"\\n                out.append({\\\"kind\\\": \\\"click\\\", \\\"params\\\": {\\\"color\\\": c, \\\"region\\\": rg, \\\"changes\\\": []}, \\\"support\\\": n, \\\"counter\\\": 0,\\n                            \\\"text\\\": f\\\"clicking colour {c}{where}{rgtxt} {what} ({n} clicks)\\\"})\\n                continue\\n            parts = []\\n            vt = stats.vanish.get((c, rg)); cu = stats.cursor.get((c, rg))\\n            if vt and vt.most_common(1)[0][0] != -1 and vt.most_common(1)[0][1] * 2 > n:\\n                k, m = vt.most_common(1)[0]; parts.append(f\\\"the clicked object turns into colour {k} ({m}/{n})\\\")\\n            if cu and cu.most_common(1)[0][0] != -1 and cu.most_common(1)[0][1] * 2 > n:\\n                k, m = cu.most_common(1)[0]\\n                sel = stats.select.get((c, rg), Counter())\\n                parts.append(f\\\"the colour-{k} marker {'jumps onto the clicked object (selection)' if sel.get(True, 0) > sel.get(False, 0) else 'moves to the clicked cell'} ({m}/{n})\\\")\\n            if h.get(\\\"hud\\\", 0) + h.get(\\\"none\\\", 0):\\n                parts.append(f\\\"{h.get('hud', 0) + h.get('none', 0)}/{n} clicks changed nothing in the playfield\\\")\\n            if not parts:\\n                parts.append(f\\\"changes the playfield ({h.get('world', 0)}/{n}), effect not yet modelled\\\")\\n            out.append({\\\"kind\\\": \\\"click\\\", \\\"params\\\": {\\\"color\\\": c, \\\"region\\\": rg}, \\\"support\\\": n, \\\"counter\\\": 0,\\n                        \\\"text\\\": f\\\"clicking colour {c}{where}{rgtxt}: \\\" + \\\"; \\\".join(parts)})\\n        for b, n in stats.succ_n.items():\\n            sg = stats.sigma(b); moving = sum(1 for p_, q in sg.items() if p_ != q)\\n            out.append({\\\"kind\\\": \\\"button\\\", \\\"params\\\": {\\\"color\\\": b[0], \\\"region\\\": b[1], \\\"bbox\\\": list(b[2])}, \\\"support\\\": n, \\\"counter\\\": 0,\\n                        \\\"text\\\": f\\\"the colour-{b[0]} button at rows {b[2][0]}-{b[2][2]} cols {b[2][1]}-{b[2][3]} shifts every block on its track one slot \\\"\\n                                f\\\"({moving} slot transitions solved from {n} presses; blocks return to their slot after a full cycle)\\\"})\\n        for (mk, oc), cnt in stats.coupled.items():\\n            r = stats.coupling(mk, oc)\\n            if r:\\n                (rr, rc), m = r\\n                out.append({\\\"kind\\\": \\\"coupled\\\", \\\"params\\\": {\\\"marker\\\": mk, \\\"color\\\": oc, \\\"ratio\\\": [str(rr), str(rc)]}, \\\"support\\\": m, \\\"counter\\\": sum(cnt.values()) - m,\\n                            \\\"text\\\": f\\\"when the colour-{mk} marker moves by (dr, dc), each colour-{oc} object moves by ({rr}*dr, {rc}*dc) \\u2014 it sits at a fixed fraction (e.g. the centroid) of the markers\\\"})\\n        return out\\n\\n    def win_facts(self, level: int) -> list[dict]:\\n        \\\"\\\"\\\"Win hypotheses from the attempt that completed `level` (goals.infer), as rulebook facts.\\\"\\\"\\\"\\n        same = self.game.level_transitions(level)\\n        if not same:\\n            return []\\n        last_attempt = same[-1].attempt\\n        attempt = [t for t in same if t.attempt == last_attempt]\\n        try:\\n            hyps = goal_inference.infer(attempt, level, movement_keys=self.game.has_move_keys() or bool(self.moves))\\n        except Exception:\\n            hyps = []\\n        out = []\\n        for h in hyps:\\n            if h.get(\\\"type\\\") in (\\\"reach\\\", \\\"collect_reach\\\", \\\"collect_all\\\", \\\"click_sequence\\\", \\\"merge\\\"):\\n                p = {k: v for k, v in h.items() if k in (\\\"reach\\\", \\\"collect\\\", \\\"collect_count\\\", \\\"sequence\\\", \\\"colors\\\")}\\n                out.append({\\\"kind\\\": h[\\\"type\\\"], \\\"params\\\": p, \\\"support\\\": 1, \\\"counter\\\": 0, \\\"text\\\": h[\\\"text\\\"]})\\n        return out\\n\\n    # \\u2500\\u2500 prediction \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\n    def _block_colour(self, row: int, col: int) -> Optional[int]:\\n        if not (0 <= row < 64 and 0 <= col < 64):\\n            return None\\n        return self.frame.grid[row][col]\\n\\n    def predict(self, act) -> Prediction:\\n        grid = self.frame.grid\\n        for name, fn, acc, n in self.custom_rules:\\n            try:\\n                out = fn([row[:] for row in grid], act)\\n            except Exception:\\n                out = None\\n            if out is not None and len(out) == 64:\\n                changed = sum(1 for i in range(64) for j in range(64) if out[i][j] != grid[i][j])\\n                return Prediction(\\\"board\\\", f\\\"model rule `{name}` ({acc:.0%} on {n} replays): {changed} cells change\\\", board=out, rules=[(\\\"other\\\", {\\\"name\\\": name})], confidence=\\\"confirmed\\\" if acc >= 0.9 else \\\"hypothesis\\\")\\n        if isinstance(act, dict):\\n            r, c = act[\\\"row\\\"], act[\\\"col\\\"]; colour = grid[r][c]\\n            (colour, region), obj = self.click_key(r, c)\\n            where = f\\\" in P{region}\\\" if region >= 0 else \\\"\\\"\\n            rule = (\\\"click\\\", {\\\"color\\\": colour, \\\"region\\\": region})\\n            h, exact, this_level = self._stat(\\\"hist_of\\\", colour, region)\\n            n = sum(h.values()) if h else 0\\n            if h and h.get(\\\"world\\\", 0) == 0 and (exact or n >= 3):\\n                kind = \\\"hud\\\" if h.get(\\\"hud\\\") else \\\"noop\\\"\\n                return Prediction(kind, f\\\"colour {colour}{where} clicked {n}x before: \\\" + (\\\"only the HUD changed\\\" if kind == \\\"hud\\\" else \\\"nothing changed\\\"),\\n                                  rules=[rule], confidence=\\\"confirmed\\\" if n >= 2 else \\\"hypothesis\\\")\\n            if obj is not None and self.cs.is_dead(obj.key):\\n                return Prediction(\\\"hud\\\", f\\\"this object was clicked {self.cs.dead[obj.key]}x on this level with no effect in the playfield\\\", rules=[rule], confidence=\\\"confirmed\\\")\\n            if self.distrust.get((colour, region), 0) >= 2:\\n                return Prediction(\\\"unknown\\\", f\\\"colour {colour}{where}: the effect model was wrong {self.distrust[(colour, region)]}x on this level; effect not predictable yet (clicked {n}x)\\\", rules=[rule], confidence=\\\"none\\\")\\n            # button with learned slot successors (conveyor / permutation)\\n            if obj is not None and self.cs.succ_n.get(obj.key):\\n                moves = {}; known = 0; unknown = 0; slots = self.cs.slots(obj.key); tent = self.cs.tentative(obj.key); guesses = {}\\n                for o in self.scene.objs:\\n                    if o.hud or o.id == obj.id:\\n                        continue\\n                    nb = self.cs.successor(obj.key, o.color, o.bbox)\\n                    if nb is not None and nb != o.bbox:\\n                        moves[o.id] = nb; known += 1\\n                    elif nb is None and o.bbox in slots:\\n                        unknown += 1\\n                        if o.bbox in tent and tent[o.bbox] != o.bbox:\\n                            guesses[o.id] = tent[o.bbox]\\n                if moves and known * 2 >= known + unknown:\\n                    return Prediction(\\\"objects\\\", f\\\"button #{obj.id}: {known} objects shift one slot along their tracks (\\\" + \\\", \\\".join(f\\\"#{i}->({b[0]},{b[1]})\\\" for i, b in list(moves.items())[:4]) + (\\\" ...\\\" if len(moves) > 4 else \\\")\\\") + (f\\\"; {unknown} objects on slots not yet solved\\\" + (\\\", probably \\\" + \\\", \\\".join(f\\\"#{i}->({b[0]},{b[1]})\\\" for i, b in list(guesses.items())[:3]) if guesses else \\\"\\\") if unknown else \\\"\\\"),\\n                                      moves=moves, rules=[(\\\"button\\\", {\\\"color\\\": colour, \\\"region\\\": region})], confidence=\\\"confirmed\\\" if self.cs.succ_n[obj.key] >= 2 else \\\"hypothesis\\\")\\n                if moves:\\n                    return Prediction(\\\"unknown\\\", f\\\"button #{obj.id}: {known} objects would shift along known tracks but {unknown} sit on slots whose next slot is still unknown\\\", rules=[rule], confidence=\\\"none\\\")\\n                return Prediction(\\\"unknown\\\", f\\\"button #{obj.id}: moved objects before ({self.cs.succ_n[obj.key]} presses), track not yet solved\\\", rules=[rule], confidence=\\\"none\\\")\\n            cu, cu_exact, _ = self._stat(\\\"cursor_of\\\", colour, region)\\n            if cu and cu.most_common(1)[0][0] != -1 and cu.most_common(1)[0][1] * 2 > sum(cu.values()):\\n                k = cu.most_common(1)[0][0]\\n                markers = [o for o in self.scene.objs if o.color == k and not o.hud and o.size >= 4]\\n                here = [o for o in markers if o.bbox[0] <= r <= o.bbox[2] and o.bbox[1] <= c <= o.bbox[3]]\\n                if here:\\n                    return Prediction(\\\"unknown\\\", f\\\"the colour-{k} marker is already at ({r},{c}); clicking here again has an unknown effect\\\", confidence=\\\"none\\\")\\n                sel = self.cs.select.get((colour, region)) or self.cs_all.select.get((colour, region)) or Counter()\\n                if obj is not None and obj.size >= 4 and sel.get(True, 0) > sel.get(False, 0):\\n                    return Prediction(\\\"cursor\\\", f\\\"the colour-{k} marker jumps onto the clicked object #{obj.id} (selection)\\\", rules=[rule], confidence=\\\"confirmed\\\" if sum(cu.values()) >= 2 else \\\"hypothesis\\\", avatar_to=(r, c))\\n                coupled = []\\n                if markers:\\n                    mk = min(markers, key=lambda o: abs(o.center[0] - r) + abs(o.center[1] - c))\\n                    dr, dc = r - mk.center[0], c - mk.center[1]\\n                    for oc in {o.color for o in self.scene.objs if not o.hud and o.color != k}:\\n                        cp = self.cs.coupling(k, oc)\\n                        if cp:\\n                            (rr, rc), _ = cp; coupled.append((oc, (round(float(rr) * dr), round(float(rc) * dc))))\\n                txt = f\\\"the colour-{k} marker moves to the click point ({r},{c})\\\"\\n                if coupled:\\n                    txt += \\\"; coupled: \\\" + \\\", \\\".join(f\\\"colour {oc} objects move by ({d[0]:+},{d[1]:+})\\\" for oc, d in coupled)\\n                else:\\n                    txt += f\\\" (other cells may change too: {h.get('world', 0) if h else 0} world changes seen)\\\"\\n                return Prediction(\\\"cursor\\\", txt, rules=[rule], confidence=\\\"confirmed\\\" if sum(cu.values()) >= 2 else \\\"hypothesis\\\", avatar_to=(r, c), coupled=coupled or None)\\n            vt, _, _ = self._stat(\\\"vanish_of\\\", colour, region)\\n            hit = self.scene.obj_at(r, c)\\n            if vt and vt.most_common(1)[0][0] != -1 and vt.most_common(1)[0][1] * 2 > sum(vt.values()) and hit is not None and hit.cells:\\n                k = vt.most_common(1)[0][0]\\n                board = [row[:] for row in grid]\\n                for rr, cc in hit.cells:\\n                    board[rr][cc] = k\\n                return Prediction(\\\"board\\\", f\\\"the clicked colour-{colour} object #{hit.id}{where} ({hit.size} cells) turns into colour {k}\\\", board=board,\\n                                  rules=[rule], confidence=\\\"confirmed\\\" if sum(vt.values()) >= 2 else \\\"hypothesis\\\")\\n            board = self.click.predict(grid, r, c) if self.click.clicks[colour] >= 5 else None\\n            if board is not None:\\n                changed = sum(1 for i in range(64) for j in range(64) if board[i][j] != grid[i][j])\\n                pairs = Counter((grid[i][j], board[i][j]) for i in range(64) for j in range(64) if board[i][j] != grid[i][j])\\n                conf = \\\"confirmed\\\" if self.click.clicks[colour] >= 2 else \\\"hypothesis\\\"\\n                return Prediction(\\\"board\\\", f\\\"changes {changed} cells near ({r},{c}): \\\" + \\\", \\\".join(f\\\"{b}->{a}x{n}\\\" for (b, a), n in pairs.most_common(3)),\\n                                  board=board, rules=[rule], confidence=conf)\\n            if h:\\n                return Prediction(\\\"unknown\\\", f\\\"colour {colour}{where} clicked {n}x before with no consistent effect model\\\", rules=[rule], confidence=\\\"none\\\")\\n            return Prediction(\\\"unknown\\\", f\\\"colour {colour}{where} never clicked\\\" + (\\\"\\\" if this_level else \\\" on this level\\\"), confidence=\\\"none\\\")\\n        name = act\\n        if name in self.conditional_keys and name not in self.moves:\\n            n0, n1 = self.conditional_keys[name]\\n            return Prediction(\\\"unknown\\\", f\\\"{name}: changed nothing {n0}x but changed the board {n1}x \\u2014 the effect depends on a condition (state) not yet known\\\", confidence=\\\"none\\\")\\n        if name in self.flaky_moves:\\n            n0, n1 = self.flaky_moves[name]\\n            return Prediction(\\\"unknown\\\", f\\\"{name}: the learned move delta fit {n0}x but failed {n1}x \\u2014 the avatar or step size is not identified reliably\\\", confidence=\\\"none\\\")\\n        if name in self.noop_keys and name not in self.moves:\\n            return Prediction(\\\"noop\\\", f\\\"{name} changed nothing before\\\", rules=[(\\\"noop\\\", {\\\"action\\\": name})],\\n                              confidence=\\\"confirmed\\\" if any(r[\\\"kind\\\"] == \\\"noop\\\" and r[\\\"params\\\"][\\\"action\\\"] == name and r[\\\"support\\\"] >= 2 for r in self.rules) else \\\"hypothesis\\\")\\n        if name in self.moves and self.avatar:\\n            dx, dy = self.moves[name]\\n            x0, y0, x1, y1 = self.avatar[\\\"bbox_xyxy\\\"]\\n            nx0, ny0, nx1, ny1 = x0 + dx, y0 + dy, x1 + dx, y1 + dy\\n            if nx0 < 0 or ny0 < 0 or nx1 > 63 or ny1 > 63:\\n                return Prediction(\\\"blocked\\\", f\\\"{name}: avatar at the board edge, stays at ({y0},{x0})\\\", avatar_to=(y0, x0), rules=[(\\\"move\\\", {\\\"action\\\": name})], confidence=\\\"hypothesis\\\")\\n            # cells the avatar would newly cover\\n            new_cells = [(rr, cc) for rr in range(ny0, ny1 + 1) for cc in range(nx0, nx1 + 1) if not (x0 <= cc <= x1 and y0 <= rr <= y1)]\\n            cols = Counter(grid[rr][cc] for rr, cc in new_cells)\\n            hit = [c for c in cols if c in self.wall_colors]\\n            haz = [c for c in cols if c in self.hazard_colors]\\n            amb = [c for c in cols if c in self.ambiguous_colors and c not in self.wall_colors]\\n            if amb and not hit and not haz:\\n                b, k = self.ambiguous_colors[amb[0]]\\n                return Prediction(\\\"unknown\\\", f\\\"{name}: colour {amb[0]} ahead \\u2014 blocked {b}x but entered {k}x before; outcome undetermined (some condition decides)\\\", confidence=\\\"none\\\")\\n            gauge_txt = f\\\"; gauge {self.gauge['per_action']:+}\\\" if self.gauge else \\\"\\\"\\n            conf = \\\"confirmed\\\" if any(r[\\\"kind\\\"] == \\\"move\\\" and r[\\\"params\\\"][\\\"action\\\"] == name and r[\\\"support\\\"] >= 2 for r in self.rules) else \\\"hypothesis\\\"\\n            if haz:\\n                return Prediction(\\\"move\\\", f\\\"{name}: enters colour {haz[0]} = HAZARD (reset expected)\\\", avatar_to=(ny0, nx0), rules=[(\\\"move\\\", {\\\"action\\\": name}), (\\\"hazard\\\", {\\\"color\\\": haz[0]})], confidence=conf)\\n            if hit:\\n                return Prediction(\\\"blocked\\\", f\\\"{name}: blocked by wall colour {hit[0]}, avatar stays at ({y0},{x0})\\\", avatar_to=(y0, x0),\\n                                  rules=[(\\\"move\\\", {\\\"action\\\": name}), (\\\"wall\\\", {\\\"color\\\": hit[0]})], confidence=conf)\\n            # unknown terrain: the block ahead is neither known floor nor known wall\\n            ahead = cols.most_common(1)[0][0] if cols else None\\n            floor = self.nav.floor_colors if self.nav else set()\\n            if ahead is not None and ahead not in floor and ahead != self.nav.background:\\n                return Prediction(\\\"move\\\", f\\\"{name}: avatar to ({ny0},{nx0}) onto colour {ahead} (untested terrain: may block or trigger something){gauge_txt}\\\",\\n                                  avatar_to=(ny0, nx0), rules=[(\\\"move\\\", {\\\"action\\\": name})], confidence=\\\"hypothesis\\\")\\n            return Prediction(\\\"move\\\", f\\\"{name}: avatar moves ({dy:+},{dx:+}) to ({ny0},{nx0}){gauge_txt}\\\", avatar_to=(ny0, nx0), rules=[(\\\"move\\\", {\\\"action\\\": name})], confidence=conf)\\n        if name in self.moves:\\n            return Prediction(\\\"unknown\\\", f\\\"{name}: moves something by {self.moves[name]} (avatar not identified)\\\", confidence=\\\"none\\\")\\n        pressed = sum(1 for t in self.game.transitions if t.action == name)\\n        return Prediction(\\\"unknown\\\", f\\\"{name} never pressed\\\" if not pressed else f\\\"{name} pressed {pressed}x, effect not modelled\\\", confidence=\\\"none\\\")\\n\\n    # \\u2500\\u2500 synthesised post-board (for goal inference) \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\n    def synth_board(self, pred: Prediction, act) -> Optional[list]:\\n        \\\"\\\"\\\"The board a prediction implies: board kind as is; objects / cursor kinds by moving objects on a copy of the grid.\\\"\\\"\\\"\\n        if pred.kind == \\\"board\\\":\\n            return pred.board\\n        moves: dict = {}\\n        if pred.kind == \\\"objects\\\" and pred.moves:\\n            moves = dict(pred.moves)\\n        elif pred.kind == \\\"cursor\\\" and isinstance(act, dict):\\n            k = None\\n            for o in self.scene.objs:\\n                pass\\n            markers = [o for o in self.scene.objs if not o.hud and o.size >= 4 and (\\\"marker\\\" in pred.text) and f\\\"colour-{o.color} marker\\\" in pred.text]\\n            if markers:\\n                mk = min(markers, key=lambda o: abs(o.center[0] - act[\\\"row\\\"]) + abs(o.center[1] - act[\\\"col\\\"]))\\n                dr, dc = act[\\\"row\\\"] - mk.center[0], act[\\\"col\\\"] - mk.center[1]\\n                moves[mk.id] = (mk.bbox[0] + dr, mk.bbox[1] + dc, mk.bbox[2] + dr, mk.bbox[3] + dc)\\n                for oc, (pdr, pdc) in (pred.coupled or []):\\n                    for o in self.scene.objs:\\n                        if o.color == oc and not o.hud and o.size >= 4 and min(o.bbox[2] - o.bbox[0], o.bbox[3] - o.bbox[1]) >= 2:\\n                            moves[o.id] = (o.bbox[0] + pdr, o.bbox[1] + pdc, o.bbox[2] + pdr, o.bbox[3] + pdc)\\n        if not moves:\\n            return None\\n        grid = [row[:] for row in self.frame.grid]\\n        for i, nb in moves.items():\\n            o = self.scene.objs[i]\\n            fill = self.scene.region_named(o.region).color if o.region >= 0 and self.scene.region_named(o.region) else self.frame.background\\n            for r, c in (o.cells or []):\\n                grid[r][c] = fill\\n        for i, nb in moves.items():\\n            o = self.scene.objs[i]\\n            for r, c in (o.cells or []):\\n                rr, cc = r + nb[0] - o.bbox[0], c + nb[1] - o.bbox[1]\\n                if 0 <= rr < 64 and 0 <= cc < 64:\\n                    grid[rr][cc] = o.color\\n        return grid\\n\\n    # \\u2500\\u2500 checking \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\n    def check(self, pred: Prediction, t: Transition, res: dict) -> Verdict:\\n        d = summarize_diff(t.before_frame, t.after_frame)\\n        world = change_class(t.before_frame, t.after_frame) == \\\"world\\\"\\n        summary = _diff_text(d, world)\\n        if res.get(\\\"level_completed\\\"):\\n            return Verdict(None, \\\"LEVEL COMPLETED \\u2014 \\\" + summary, d)\\n        if res.get(\\\"game_over\\\"):\\n            return Verdict(False if pred.kind not in (\\\"unknown\\\",) and \\\"HAZARD\\\" not in pred.text else None, \\\"GAME OVER \\u2014 \\\" + summary, d)\\n        if pred.kind == \\\"unknown\\\":\\n            return Verdict(None, \\\"observed: \\\" + summary, d)\\n        if pred.kind in (\\\"move\\\", \\\"blocked\\\"):\\n            av_after = NavHelper(self.game.attempt_transitions(), t.after_frame).avatar()\\n            if av_after is None:\\n                return Verdict(None, \\\"avatar lost after the move; observed: \\\" + summary, d)\\n            got = (av_after[\\\"bbox_xyxy\\\"][1], av_after[\\\"bbox_xyxy\\\"][0])\\n            if \\\"HAZARD\\\" in pred.text:\\n                return Verdict(None, \\\"hazard step; observed: \\\" + summary, d)\\n            ok = got == tuple(pred.avatar_to)\\n            extra = \\\"\\\" if not world or pred.kind == \\\"blocked\\\" else \\\"\\\"\\n            # a move that also changed something else in the world (collect/toggle) is not a mismatch of the move rule, but worth noting\\n            side = [x for x in (d.get(\\\"disappeared\\\") or []) + (d.get(\\\"appeared\\\") or [])]\\n            note = f\\\"; side effect: {summary}\\\" if ok and side else \\\"\\\"\\n            return Verdict(ok, (\\\"as predicted\\\" if ok else f\\\"MISMATCH: avatar at {got}, predicted {tuple(pred.avatar_to)}\\\") + note, d)\\n        if pred.kind == \\\"cursor\\\":\\n            rec = record(t)\\n            ok = rec.marker is not None\\n            note = \\\"\\\"\\n            if ok and pred.coupled:\\n                mdr, mdc = rec.marker_to[0] - rec.marker.bbox[0], rec.marker_to[1] - rec.marker.bbox[1]\\n                for oc, (pdr, pdc) in pred.coupled:\\n                    got = [(dr, dc) for oa, ob, dr, dc in rec.moved if oa.color == oc and oa.size >= 4]\\n                    if not got:\\n                        ok = False; note += f\\\"; colour {oc} did not move (predicted ({pdr:+},{pdc:+}))\\\"\\n                    elif not any(abs(dr - pdr) <= 1 and abs(dc - pdc) <= 1 for dr, dc in got):\\n                        ok = False; note += f\\\"; colour {oc} moved {got[0]} (predicted ({pdr:+},{pdc:+}))\\\"\\n            return Verdict(ok, (\\\"as predicted (marker moved to the click\\\" + (\\\"; coupled objects as predicted\\\" if pred.coupled else \\\"\\\") + \\\"); \\\" if ok else \\\"MISMATCH: \\\" + (\\\"no marker moved to the click; \\\" if rec.marker is None else \\\"marker moved but\\\" + note + \\\"; \\\")) + summary, d)\\n        if pred.kind == \\\"objects\\\":\\n            sb, sa_ = build_scene(t.before_frame), build_scene(t.after_frame)\\n            occ_after = {o.bbox: (o.color, o.shape) for o in sa_.objs if not o.hud}\\n            arriving = set(pred.moves.values())\\n            wrong = []\\n            for i, q in pred.moves.items():\\n                o = sb.objs[i]\\n                if occ_after.get(q) != (o.color, o.shape):\\n                    wrong.append((i, q, \\\"slot not filled as predicted\\\"))\\n                elif o.bbox not in arriving and occ_after.get(o.bbox) == (o.color, o.shape):\\n                    wrong.append((i, q, \\\"did not leave its slot\\\"))\\n            if not wrong:\\n                return Verdict(True, f\\\"as predicted ({len(pred.moves)} objects shifted along their tracks)\\\", d)\\n            i, q, why = wrong[0]\\n            return Verdict(False, f\\\"MISMATCH: {len(wrong)}/{len(pred.moves)} objects not where predicted (e.g. #{i} colour {sb.objs[i].color} -> ({q[0]},{q[1]}): {why}); \\\" + summary, d)\\n        if pred.kind == \\\"noop\\\":\\n            ok = not world and not d.get(\\\"changed_cells\\\")\\n            return Verdict(ok, \\\"as predicted (nothing changed)\\\" if ok else \\\"MISMATCH: predicted no change but \\\" + summary, d)\\n        if pred.kind == \\\"hud\\\":\\n            ok = not world\\n            return Verdict(ok, \\\"as predicted (HUD only)\\\" if ok else \\\"MISMATCH: predicted HUD-only but \\\" + summary, d)\\n        if pred.kind == \\\"board\\\":\\n            pm = _mask_like(pred.board, t.after_frame); am = masked_ascii(t.after_frame)\\n            if pm == am or _edge_only_diff(pred.board, t.after_frame.grid):\\n                return Verdict(True, \\\"as predicted (board matches)\\\", d)\\n            # how far off?\\n            wrong = sum(1 for a, b in zip(pm, am) if a != b)\\n            return Verdict(False, f\\\"MISMATCH: {wrong} cells differ from the predicted board; \\\" + summary, d)\\n        return Verdict(None, \\\"observed: \\\" + summary, d)\\n\\n\\ndef _mask_like(board, frame: Frame) -> str:\\n    return masked_ascii(Frame(board, step=frame.step, level=frame.level))\\n\\n\\ndef _diff_text(d: dict, world: bool) -> str:\\n    if not d.get(\\\"changed_cells\\\"):\\n        return \\\"nothing changed\\\"\\n    parts = [f\\\"{d['changed_cells']} cells changed\\\" + (\\\"\\\" if world else \\\" (HUD only)\\\")]\\n    if d.get(\\\"moved\\\"):\\n        parts.append(\\\"moved \\\" + \\\", \\\".join(f\\\"colour {m['color']} {m['from']}->{m['to']}\\\" for m in d[\\\"moved\\\"][:3]))\\n    if d.get(\\\"disappeared\\\"):\\n        parts.append(\\\"vanished \\\" + \\\", \\\".join(f\\\"colour {x['color']}({x['pixels']}px)@{x['center']}\\\" for x in d[\\\"disappeared\\\"][:3]))\\n    if d.get(\\\"appeared\\\"):\\n        parts.append(\\\"appeared \\\" + \\\", \\\".join(f\\\"colour {x['color']}({x['pixels']}px)@{x['center']}\\\" for x in d[\\\"appeared\\\"][:3]))\\n    if d.get(\\\"bbox\\\"):\\n        parts.append(f\\\"in rows {d['bbox'][0]}-{d['bbox'][2]}, cols {d['bbox'][1]}-{d['bbox'][3]}\\\")\\n    return \\\"; \\\".join(parts)\\n\\n\\n\\n\\ndef _avatar_in(grid, nav: NavHelper) -> Optional[dict]:\\n    \\\"\\\"\\\"The avatar object (top player vote colour/size) in an arbitrary grid; None if not found.\\\"\\\"\\\"\\n    objs, _ = extract_objects(grid)\\n    for (color, size), _ in nav.player_votes.most_common(3):\\n        for o in objs:\\n            if o[\\\"color\\\"] == color and o[\\\"size\\\"] == size:\\n                return o\\n    return None\\n\\n\\ndef induce(game: Game, nav: Optional[NavHelper]) -> list[dict]:\\n    \\\"\\\"\\\"Cheap O(n) rule induction over the current level's transitions (all attempts): move deltas, walls, no-ops, gauge,\\n    collectibles, hazards, refills. Each fact: {kind, params, support, counter, text}.\\\"\\\"\\\"\\n    out: list[dict] = []\\n    trans = game.level_transitions()\\n    if not trans:\\n        return out\\n    moves = dict(nav.moves) if nav else {}\\n    ok: Counter = Counter(); blocked: Counter = Counter(); bad: Counter = Counter(); walls: Counter = Counter()\\n    noop: Counter = Counter(); pressed: Counter = Counter(); vanish: Counter = Counter(); hazard: Counter = Counter()\\n    entered: Counter = Counter()\\n    start_pos = None\\n    for t in trans:\\n        if isinstance(t.action, dict):\\n            continue\\n        a = t.action; pressed[a] += 1\\n        if t.before_frame.ascii == t.after_frame.ascii:\\n            noop[a] += 1\\n        if nav is None or a not in moves:\\n            continue\\n        ab, aa = _avatar_in(t.before_frame.grid, nav), _avatar_in(t.after_frame.grid, nav)\\n        if not ab or not aa:\\n            continue\\n        if start_pos is None:\\n            start_pos = ab[\\\"bbox\\\"][:2]\\n        dx, dy = aa[\\\"bbox\\\"][0] - ab[\\\"bbox\\\"][0], aa[\\\"bbox\\\"][1] - ab[\\\"bbox\\\"][1]\\n        if (dx, dy) == moves[a]:\\n            ok[a] += 1\\n            x0, y0, x1, y1 = ab[\\\"bbox\\\"]\\n            for yy in range(y0 + dy, y1 + dy + 1):\\n                for xx in range(x0 + dx, x1 + dx + 1):\\n                    if 0 <= yy < 64 and 0 <= xx < 64 and not (x0 <= xx <= x1 and y0 <= yy <= y1):\\n                        entered[t.before_frame.grid[yy][xx]] += 1\\n        elif (dx, dy) == (0, 0):\\n            blocked[a] += 1\\n            mx, my = moves[a]; x0, y0, x1, y1 = ab[\\\"bbox\\\"]\\n            cells = [(yy, xx) for yy in range(y0 + my, y1 + my + 1) for xx in range(x0 + mx, x1 + mx + 1)\\n                     if 0 <= yy < 64 and 0 <= xx < 64 and not (x0 <= xx <= x1 and y0 <= yy <= y1)]\\n            if cells:\\n                walls[Counter(t.before_frame.grid[yy][xx] for yy, xx in cells).most_common(1)[0][0]] += 1\\n        else:\\n            if start_pos is not None and aa[\\\"bbox\\\"][:2] == start_pos and abs(dx) + abs(dy) > 6:\\n                mx, my = moves[a]; x0, y0, x1, y1 = ab[\\\"bbox\\\"]\\n                cells = [(yy, xx) for yy in range(y0 + my, y1 + my + 1) for xx in range(x0 + mx, x1 + mx + 1)\\n                         if 0 <= yy < 64 and 0 <= xx < 64 and not (x0 <= xx <= x1 and y0 <= yy <= y1)]\\n                if cells:\\n                    hazard[Counter(t.before_frame.grid[yy][xx] for yy, xx in cells).most_common(1)[0][0]] += 1\\n            else:\\n                bad[a] += 1\\n        # collectibles: a small object that vanished where the avatar arrived\\n        bo, _ = extract_objects(t.before_frame.grid); ao, _ = extract_objects(t.after_frame.grid)\\n        after_keys = {(o[\\\"color\\\"], o[\\\"center\\\"]) for o in ao}\\n        for o in bo:\\n            if o[\\\"size\\\"] <= 60 and (o[\\\"color\\\"], o[\\\"center\\\"]) not in after_keys and o[\\\"color\\\"] not in (aa[\\\"color\\\"], ab[\\\"color\\\"]):\\n                if abs(o[\\\"center\\\"][1] - aa[\\\"center\\\"][1]) <= 6 and abs(o[\\\"center\\\"][0] - aa[\\\"center\\\"][0]) <= 6:\\n                    vanish[o[\\\"color\\\"]] += 1\\n    bg = game.frame.background\\n    for a, (dx, dy) in moves.items():\\n        if ok[a] or blocked[a] or bad[a]:\\n            out.append({\\\"kind\\\": \\\"move\\\", \\\"params\\\": {\\\"action\\\": a, \\\"dx\\\": dx, \\\"dy\\\": dy}, \\\"support\\\": ok[a], \\\"counter\\\": bad[a],\\n                        \\\"text\\\": f\\\"{a} moves the avatar by (dx={dx}, dy={dy}) pixels [{ok[a]} moved, {blocked[a]} blocked]\\\"})\\n    for c, n in walls.items():\\n        if c != bg or n >= 2:\\n            k = entered[c] // 5   # entered cells -> roughly moves (an avatar step covers ~5 cells of the colour)\\n            out.append({\\\"kind\\\": \\\"wall\\\", \\\"params\\\": {\\\"color\\\": c}, \\\"support\\\": n, \\\"counter\\\": k,\\n                        \\\"text\\\": f\\\"the avatar cannot enter colour {c} (blocked {n}x\\\" + (f\\\", but entered it {k}x\\\" if k else \\\"\\\") + \\\")\\\"})\\n    for a, n in noop.items():\\n        if a not in moves or ok[a] == 0:\\n            out.append({\\\"kind\\\": \\\"noop\\\", \\\"params\\\": {\\\"action\\\": a}, \\\"support\\\": n, \\\"counter\\\": pressed[a] - n, \\\"text\\\": f\\\"{a} changes nothing ({n}/{pressed[a]} presses)\\\"})\\n    g = nav.gauge() if nav else None\\n    if g:\\n        out.append({\\\"kind\\\": \\\"gauge\\\", \\\"params\\\": {\\\"color\\\": g[\\\"color\\\"], \\\"per_action\\\": g[\\\"per_action\\\"]}, \\\"support\\\": 3, \\\"counter\\\": 0,\\n                    \\\"text\\\": f\\\"the gauge (colour {g['color']}, edge strip) changes by {g['per_action']} per action\\\" + (f\\\"; about {g['actions_left']} actions left\\\" if g.get(\\\"actions_left\\\") is not None else \\\"\\\")})\\n    floor = nav.floor_colors if nav else set()\\n    for c, n in vanish.items():\\n        if c != bg and c not in floor:\\n            out.append({\\\"kind\\\": \\\"collect\\\", \\\"params\\\": {\\\"color\\\": c}, \\\"support\\\": n, \\\"counter\\\": 0, \\\"text\\\": f\\\"touching a colour-{c} object makes it disappear ({n}x)\\\"})\\n    for c, n in hazard.items():\\n        out.append({\\\"kind\\\": \\\"hazard\\\", \\\"params\\\": {\\\"color\\\": c}, \\\"support\\\": n, \\\"counter\\\": 0, \\\"text\\\": f\\\"entering colour {c} sends the avatar back to its start ({n}x)\\\"})\\n    return out\\n\\n\", \"repl.py\": \"\\\"\\\"\\\"Coder mode: the model writes Python against the rulebook library and the harness runs it in-process with a time limit.\\n\\nNamespace the model's code sees (rebuilt every turn):\\n  board, ascii, level, valid_actions, objects, regions, entities, transitions, book, goal, candidates, plans, notes\\n  act(a), click(r, c), press(key), path_to(r, c), predict(a), scene(), check_rule(fn), define_goal(name, fn), note(...),\\n  refute(id, why), set_plan(text), remember(text)\\nEvery action that act() executes is verified against the predictor exactly like a chosen candidate; the level end, a game\\nover and the per-turn action cap unwind the model's code with an exception the harness catches.\\n\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport io\\nimport sys\\nimport time\\nimport traceback\\nfrom contextlib import redirect_stdout\\nfrom typing import Optional\\n\\nfrom arcnav.frame import Frame, masked_ascii\\n\\nfrom .entities import build_scene, scene_text\\nfrom .env import action_label\\n\\nACTION_CAP = 30       # actions per python call\\nTIME_LIMIT = 25.0     # seconds of model code per call (game steps included)\\n\\n\\nclass StopTurn(Exception):\\n    pass\\n\\n\\nclass Repl:\\n    def __init__(self, agent):\\n        self.a = agent\\n        self.notes = \\\"\\\"\\n        self.turn_actions = 0\\n        self.turn_status = \\\"ok\\\"\\n        self.custom_goals: dict = {}\\n\\n    # \\u2500\\u2500 namespace \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\n    def _objects(self, sc) -> list:\\n        return [{\\\"id\\\": o.id, \\\"color\\\": o.color, \\\"size\\\": o.size, \\\"bbox\\\": o.bbox, \\\"center\\\": o.center, \\\"region\\\": o.region, \\\"hud\\\": o.hud,\\n                 \\\"cells\\\": o.cells} for o in sc.objs]\\n\\n    def namespace(self, ev, cands, plans, goal, level0, reviews0) -> dict:\\n        g = self.a.g\\n        sc = ev.scene\\n        self.turn_actions = 0; self.turn_status = \\\"ok\\\"\\n\\n        import re as _re\\n        labels = {c.label: c.actions for c in list(cands) + list(plans) if c.actions}\\n\\n        def _resolve(a):\\n            \\\"\\\"\\\"Strings may be keys, candidate/plan labels ('submit', 'plan:press(W2)', 'click(11@14,31)') or 'MOUSE(r,c)'.\\\"\\\"\\\"\\n            if isinstance(a, (tuple, list)) and len(a) == 2 and all(isinstance(v, (int, float)) for v in a):\\n                return [{\\\"action\\\": \\\"MOUSE\\\", \\\"row\\\": int(a[0]), \\\"col\\\": int(a[1])}]\\n            if isinstance(a, dict):\\n                return [a]\\n            if isinstance(a, str):\\n                t = a.strip()\\n                if t in labels:\\n                    return list(labels[t])\\n                m = _re.match(r\\\"(?:MOUSE|mouse|click|CLICK)\\\\s*\\\\(\\\\s*(?:\\\\d+\\\\s*@\\\\s*)?(\\\\d+)\\\\s*,\\\\s*(\\\\d+)\\\\s*\\\\)\\\\s*$\\\", t)\\n                if m:\\n                    return [{\\\"action\\\": \\\"MOUSE\\\", \\\"row\\\": int(m.group(1)), \\\"col\\\": int(m.group(2))}]\\n                if t.upper() == \\\"MOUSE\\\":\\n                    raise StopTurn(\\\"a click needs coordinates: act((row, col)) or click(row, col)\\\")\\n                return [t.upper()]\\n            return [a]\\n\\n        KEYS = (\\\"UP\\\", \\\"DOWN\\\", \\\"LEFT\\\", \\\"RIGHT\\\", \\\"SPACE\\\", \\\"ACTION7\\\")\\n\\n        def _valid(a):\\n            if isinstance(a, dict):\\n                try:\\n                    r, c = int(a.get(\\\"row\\\")), int(a.get(\\\"col\\\"))\\n                except (TypeError, ValueError):\\n                    raise StopTurn(f\\\"a click needs integer row/col: got {a!r}\\\")\\n                if not (0 <= r < 64 and 0 <= c < 64):\\n                    raise StopTurn(f\\\"click ({r},{c}) is off the 64x64 board\\\")\\n                return {\\\"action\\\": \\\"MOUSE\\\", \\\"row\\\": r, \\\"col\\\": c}\\n            if isinstance(a, str) and a in KEYS:\\n                return a\\n            raise StopTurn(f\\\"act() takes 'UP'/'DOWN'/'LEFT'/'RIGHT'/'SPACE'/'ACTION7', (row, col) for a click, or a candidate/plan label; got {a!r}\\\")\\n\\n        def act(x):\\n            acts = []\\n            for a in (x if isinstance(x, list) and not (len(x) == 2 and all(isinstance(v, (int, float)) for v in x)) else [x]):\\n                acts.extend(_resolve(a))\\n            acts = [_valid(a) for a in acts]\\n            out = []\\n            for a in acts:\\n                if self.turn_actions >= ACTION_CAP:\\n                    raise StopTurn(f\\\"action cap ({ACTION_CAP} per call) reached; look at the board and call again\\\")\\n                self.turn_actions += 1\\n                st = self.a._step(a, \\\"code\\\", level0, reviews0, kind=\\\"code\\\", review_mismatch=False)\\n                pred, v, res = self.a.last_verdict if self.a.last_verdict else (None, None, {})\\n                out.append({\\\"action\\\": action_label(a), \\\"status\\\": st, \\\"predicted\\\": pred.text if pred else \\\"\\\", \\\"actual\\\": v.text if v else \\\"\\\",\\n                            \\\"changed\\\": bool(res.get(\\\"changed\\\"))})\\n                print(f\\\"  {action_label(a)} -> {st}: {v.text[:120] if v else ''}\\\")\\n                if st in (\\\"won\\\", \\\"level\\\"):\\n                    self.turn_status = st; raise StopTurn(\\\"LEVEL COMPLETED \\u2014 the board is new now\\\")\\n                if st == \\\"over\\\":\\n                    self.turn_status = st; raise StopTurn(\\\"GAME OVER \\u2014 the level was reset\\\")\\n                if st == \\\"invalid\\\":\\n                    raise StopTurn(f\\\"{action_label(a)} is not a valid action now ({g.valid_actions})\\\")\\n            return out\\n\\n        def click(r, c):\\n            return act({\\\"action\\\": \\\"MOUSE\\\", \\\"row\\\": int(r), \\\"col\\\": int(c)})\\n\\n        def press(key, n=1):\\n            return act([str(key).upper()] * int(n))\\n\\n        def path_to(r, c):\\n            nav = self.a.evidence().nav\\n            return nav.path_to(int(r), int(c), ignore_wall_target=True) if nav else None\\n\\n        def predict(a):\\n            if isinstance(a, (tuple, list)) and len(a) == 2:\\n                a = {\\\"action\\\": \\\"MOUSE\\\", \\\"row\\\": int(a[0]), \\\"col\\\": int(a[1])}\\n            p = self.a.evidence().predict(a if isinstance(a, dict) else str(a).upper())\\n            return f\\\"{p.kind}: {p.text}\\\"\\n\\n        class _SceneView(dict):\\n            \\\"\\\"\\\"dict with attribute access: scene().board and scene()['board'] both work.\\\"\\\"\\\"\\n            __getattr__ = dict.__getitem__\\n\\n        def scene():\\n            e = self.a.evidence()\\n            return _SceneView(board=e.frame.grid, ascii=e.frame.ascii, objects=self._objects(e.scene), regions=[(r.id, r.color, r.bbox) for r in e.scene.regions], entities=scene_text(e.scene))\\n\\n        def check_rule(fn, name: Optional[str] = None):\\n            \\\"\\\"\\\"fn(before_board, action) -> after_board or None. Scored on every recorded transition of this level; >= 0.8 on >= 4\\n            transitions makes it a verified rule the predictor uses first.\\\"\\\"\\\"\\n            trans = g.level_transitions(); n = ok = 0; examples = []\\n            for t in trans:\\n                try:\\n                    out = fn([row[:] for row in t.before_frame.grid], t.action)\\n                except Exception as e:\\n                    examples.append(f\\\"{action_label(t.action)}: raised {type(e).__name__}: {e}\\\"); continue\\n                if out is None:\\n                    continue\\n                n += 1\\n                good = masked_ascii(Frame(out)) == masked_ascii(t.after_frame)\\n                ok += good\\n                if not good and len(examples) < 3:\\n                    diff = sum(1 for i in range(64) for j in range(64) if out[i][j] != t.after_frame.grid[i][j])\\n                    examples.append(f\\\"{action_label(t.action)}: {diff} cells wrong\\\")\\n            nm = name or getattr(fn, \\\"__name__\\\", \\\"rule\\\")\\n            acc = ok / n if n else 0.0\\n            verdict = \\\"VERIFIED\\\" if n >= 4 and acc >= 0.8 else \\\"not verified\\\"\\n            if verdict == \\\"VERIFIED\\\":\\n                self.a.custom_rules = [r for r in self.a.custom_rules if r[0] != nm] + [(nm, fn, acc, n)]\\n                self.a.book.add(\\\"rules\\\", f\\\"model rule `{nm}` predicts the board after an action; verified by replay ({ok}/{n} transitions)\\\", kind=\\\"other\\\", level=g.level, status=\\\"confirmed\\\", source=\\\"llm\\\")\\n                self.a.save_book()\\n            msg = f\\\"check_rule({nm}): applied to {n} of {len(trans)} recorded transitions, {ok} exact -> {verdict}\\\" + (\\\"; \\\" + \\\"; \\\".join(examples) if examples else \\\"\\\")\\n            print(msg); return {\\\"applicable\\\": n, \\\"correct\\\": ok, \\\"accuracy\\\": acc, \\\"verified\\\": verdict == \\\"VERIFIED\\\"}\\n\\n        def define_goal(name: str, fn, needs_submit: bool = False, text: str = \\\"\\\"):\\n            \\\"\\\"\\\"fn(objects, board) -> bool: a win condition the harness evaluates on every board and refutes automatically.\\\"\\\"\\\"\\n            from . import goals2\\n            cg = goals2.CustomGoal(str(name)[:40], fn, bool(needs_submit))\\n            e = next((x for x in self.a.book.section(\\\"win\\\") if x.text.startswith(f\\\"GOAL {name}:\\\")), None)\\n            if e is None:\\n                e = self.a.book.add(\\\"win\\\", f\\\"GOAL {name}: {text or getattr(fn, '__doc__', '') or 'model-defined predicate'}\\\", kind=\\\"other\\\", level=g.level)\\n            else:\\n                e.status = \\\"hypothesis\\\"; e.note = \\\"\\\"\\n            self.a.preds[e.id] = cg; self.a.save_book()\\n            try:\\n                ok, txt, _ = cg.evaluate(self.a.evidence().scene, self.a.evidence())\\n            except Exception as ex:\\n                ok, txt = False, f\\\"raised {type(ex).__name__}: {ex}\\\"\\n            print(f\\\"define_goal({name}) -> {e.id}; now: {'HOLDS' if ok else 'not yet'} ({txt})\\\"); return e.id\\n\\n        def note(text, section=\\\"rules\\\", kind=\\\"other\\\", params=None):\\n            e = self.a.book.add(section, str(text), kind=kind or \\\"other\\\", params=params or {}, level=g.level); self.a.save_book()\\n            print(f\\\"noted {e.id}\\\"); return e.id\\n\\n        def refute(id_, why=\\\"\\\"):\\n            r = self.a.book.refute(str(id_), str(why) or \\\"refuted by the model\\\"); self.a.save_book(); print(r or f\\\"{id_}: nothing to refute\\\"); return r\\n\\n        def set_plan(text):\\n            self.a.book.plan = str(text)[:600]; self.a.save_book(); print(\\\"plan updated\\\")\\n\\n        def remember(text):\\n            self.notes = (self.notes + \\\"\\\\n\\\" + str(text)).strip()[-1500:]; print(\\\"remembered\\\")\\n\\n        ns = {\\n            \\\"board\\\": ev.frame.grid, \\\"ascii\\\": ev.frame.ascii, \\\"level\\\": g.level, \\\"valid_actions\\\": list(g.valid_actions),\\n            \\\"objects\\\": self._objects(sc), \\\"regions\\\": [(r.id, r.color, r.bbox) for r in sc.regions], \\\"entities\\\": scene_text(sc),\\n            \\\"transitions\\\": [{\\\"action\\\": action_label(t.action), \\\"raw_action\\\": t.action, \\\"before\\\": t.before_frame.grid, \\\"after\\\": t.after_frame.grid, \\\"level\\\": t.level}\\n                            for t in g.transitions[-60:]],\\n            \\\"book\\\": self.a.book.render(), \\\"goal\\\": list(goal.get(\\\"lines\\\", [])) if goal else [],\\n            \\\"candidates\\\": [(c.label, c.prediction) for c in cands if c.kind != \\\"info\\\"], \\\"plans\\\": [(c.label, c.prediction, c.actions) for c in plans if c.kind == \\\"plan\\\"],\\n            \\\"plan_actions\\\": {c.label: c.actions for c in plans if c.kind == \\\"plan\\\"}, \\\"notes\\\": self.notes,\\n            \\\"act\\\": act, \\\"click\\\": click, \\\"press\\\": press, \\\"path_to\\\": path_to, \\\"predict\\\": predict, \\\"scene\\\": scene, \\\"check_rule\\\": check_rule,\\n            \\\"define_goal\\\": define_goal, \\\"note\\\": note, \\\"refute\\\": refute, \\\"set_plan\\\": set_plan, \\\"remember\\\": remember,\\n            \\\"masked\\\": lambda grid: masked_ascii(Frame(grid)), \\\"Counter\\\": __import__(\\\"collections\\\").Counter,\\n        }\\n        return ns\\n\\n    # \\u2500\\u2500 execution \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\n    def run(self, code: str, ns: dict, *, time_limit: float = TIME_LIMIT) -> str:\\n        buf = io.StringIO(); t0 = time.time()\\n\\n        def tracer(frame, event, arg):\\n            if time.time() - t0 > time_limit:\\n                raise StopTurn(f\\\"time limit ({time_limit:.0f}s) reached\\\")\\n            return tracer\\n        old = sys.gettrace()\\n        sys.settrace(tracer)\\n        try:\\n            with redirect_stdout(buf):\\n                try:\\n                    exec(compile(code, \\\"<model>\\\", \\\"exec\\\"), ns)\\n                except StopTurn as e:\\n                    print(f\\\"[stopped: {e}]\\\")\\n                except Exception:\\n                    tb = traceback.format_exc(limit=3)\\n                    print(\\\"[error]\\\\n\\\" + tb[-900:])\\n        finally:\\n            sys.settrace(old)\\n        out = buf.getvalue()\\n        return out[-4000:] if len(out) > 4000 else out\\n\", \"run.py\": \"\\\"\\\"\\\"Runner: play games with the rulebook agent (thread pool), write an experiments-style result JSON + per-game logs/rulebooks.\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport datetime as dt\\nimport json\\nimport logging\\nimport os\\nimport subprocess\\nimport threading\\nimport time\\nfrom concurrent.futures import ThreadPoolExecutor\\nfrom pathlib import Path\\nfrom typing import Optional\\n\\nfrom arcnav.llm import ChatClient\\nfrom arcnav.runner import make_arcade\\n\\nfrom .agent import RulebookAgent\\nfrom .env import Game\\nfrom .llm_io import Model\\n\\nROOT = Path(__file__).resolve().parents[1]\\nDEFAULT_CONFIG = {\\\"base_url\\\": \\\"http://127.0.0.1:1234/v1\\\", \\\"model\\\": \\\"local-qwen\\\", \\\"temperature\\\": 0.6, \\\"top_p\\\": 0.95,\\n                  \\\"think_tokens\\\": 12000, \\\"review_tokens\\\": 6000, \\\"decide_tokens\\\": 1500, \\\"max_minutes\\\": 20, \\\"level_actions\\\": 200, \\\"max_actions\\\": 2000,\\n                  \\\"reviews_per_level\\\": 8, \\\"max_levels\\\": 10, \\\"review_think\\\": \\\"level\\\", \\\"mode\\\": \\\"choose\\\", \\\"compact\\\": False, \\\"jobs\\\": 2}\\n\\n\\ndef _git() -> dict:\\n    try:\\n        rev = subprocess.check_output([\\\"git\\\", \\\"rev-parse\\\", \\\"--short\\\", \\\"HEAD\\\"], cwd=ROOT, text=True, stderr=subprocess.DEVNULL).strip()\\n        dirty = bool(subprocess.check_output([\\\"git\\\", \\\"status\\\", \\\"--porcelain\\\", \\\"rulebook\\\"], cwd=ROOT, text=True, stderr=subprocess.DEVNULL).strip())\\n        return {\\\"commit\\\": rev, \\\"dirty\\\": dirty}\\n    except Exception:\\n        return {}\\n\\n\\ndef play_game(arc, game_id: str, cfg: dict, log_dir: Path, *, lock: threading.Lock, deadline: Optional[float]) -> dict:\\n    with lock:\\n        env = arc.make(game_id)\\n    if env is None:\\n        return {\\\"game_id\\\": game_id, \\\"error\\\": \\\"could not create env\\\"}\\n    game = Game(env, game_id)\\n    agent = RulebookAgent(game, None, log_dir=log_dir, cfg=cfg, deadline=deadline)\\n    if not cfg.get(\\\"no_model\\\"):\\n        client = ChatClient(cfg[\\\"base_url\\\"], cfg[\\\"model\\\"], temperature=cfg[\\\"temperature\\\"], top_p=cfg[\\\"top_p\\\"], max_tokens=cfg[\\\"decide_tokens\\\"])\\n        agent.model = Model(client, think_tokens=cfg[\\\"think_tokens\\\"], decide_tokens=cfg[\\\"decide_tokens\\\"], review_tokens=cfg.get(\\\"review_tokens\\\", 6000), log=agent.log,\\n                            compact=bool(cfg.get(\\\"compact\\\", False)))\\n    try:\\n        return agent.play()\\n    except Exception as e:\\n        logging.exception(\\\"game %s crashed\\\", game_id)\\n        agent.log(f\\\"CRASH {e!r}\\\")\\n        return {\\\"game_id\\\": game_id, \\\"error\\\": f\\\"{type(e).__name__}: {e}\\\", \\\"actions\\\": game.actions_used, \\\"levels_completed\\\": game.level - 1, \\\"levels_total\\\": game.levels_total}\\n\\n\\ndef run(game_ids: list[str], cfg: dict, *, out_dir: Path, tag: str = \\\"\\\", arc=None, deadline: Optional[float] = None) -> dict:\\n    run_id = dt.datetime.now(dt.timezone.utc).strftime(\\\"%Y%m%d-%H%M%S\\\") + f\\\"-{os.getpid()}\\\"\\n    log_dir = out_dir / \\\"logs\\\" / run_id\\n    arc = arc or make_arcade()\\n    lock = threading.Lock(); started = dt.datetime.now(dt.timezone.utc)\\n    with ThreadPoolExecutor(max_workers=int(cfg.get(\\\"jobs\\\", 2))) as ex:\\n        games = list(ex.map(lambda g: play_game(arc, g, cfg, log_dir, lock=lock, deadline=deadline), game_ids))\\n    try:\\n        sc = arc.get_scorecard(); scd = sc.model_dump() if hasattr(sc, \\\"model_dump\\\") else {}\\n    except Exception as e:\\n        scd = {\\\"error\\\": repr(e)}\\n    by_id = {}\\n    for e in scd.get(\\\"environments\\\", []) or []:\\n        if e.get(\\\"runs\\\"):\\n            by_id[e[\\\"id\\\"].split(\\\"-\\\")[0]] = e[\\\"runs\\\"][-1]\\n    for g in games:\\n        r = by_id.get(g[\\\"game_id\\\"].split(\\\"-\\\")[0])\\n        if r:\\n            g[\\\"score\\\"] = r.get(\\\"score\\\"); g[\\\"level_scores\\\"] = r.get(\\\"level_scores\\\"); g[\\\"level_baseline_actions\\\"] = r.get(\\\"level_baseline_actions\\\")\\n    scores = [g.get(\\\"score\\\") or 0.0 for g in games]\\n    result = {\\\"schema\\\": 1, \\\"experiment\\\": \\\"rulebook\\\", \\\"run_id\\\": run_id, \\\"started_at\\\": started.isoformat(), \\\"finished_at\\\": dt.datetime.now(dt.timezone.utc).isoformat(),\\n              \\\"git\\\": _git(), \\\"tag\\\": tag, \\\"config\\\": {\\\"description\\\": \\\"rulebook agent: hypothesis rulebook + deterministic predictor + model choice + review on mismatch\\\", \\\"games\\\": game_ids, \\\"params\\\": cfg},\\n              \\\"aggregate\\\": {\\\"score\\\": sum(scores) / max(1, len(scores)), \\\"levels_completed\\\": sum(g.get(\\\"levels_completed\\\", 0) for g in games),\\n                            \\\"games_level2plus\\\": sum(1 for g in games if g.get(\\\"levels_completed\\\", 0) >= 2), \\\"actions\\\": sum(g.get(\\\"actions\\\", 0) for g in games), \\\"games_played\\\": len(games)},\\n              \\\"games\\\": games, \\\"scorecard\\\": scd}\\n    out_dir.mkdir(parents=True, exist_ok=True)\\n    out = out_dir / f\\\"run-{run_id}.json\\\"; out.write_text(json.dumps(result, indent=2, default=str))\\n    print(f\\\"\\\\n========= rulebook {tag} =========\\\")\\n    for g in games:\\n        print(f\\\"  {g['game_id']:8} levels={g.get('levels_completed', '?'):>2}/{g.get('levels_total', '?')} actions={g.get('actions', '?'):>5} score={g.get('score', 0) or 0:.3f} \\\"\\n              f\\\"mism={g.get('mismatches', '?')} rev={g.get('reviews', '?')} calls={g.get('model_calls', {})} stop={g.get('stop_reason', g.get('error'))}\\\")\\n    print(f\\\"Aggregate score: {result['aggregate']['score']:.4f} | level>=2: {result['aggregate']['games_level2plus']}/{len(games)}\\\\nResult: {out}\\\\nLogs: {log_dir}\\\")\\n    return result\\n\\n\\ndef main(argv=None) -> None:\\n    ap = argparse.ArgumentParser(description=\\\"Play ARC-AGI-3 games with the rulebook agent\\\")\\n    ap.add_argument(\\\"--games\\\", default=\\\"ls20\\\"); ap.add_argument(\\\"--config\\\"); ap.add_argument(\\\"--tag\\\", default=\\\"\\\")\\n    ap.add_argument(\\\"--out\\\", default=str(ROOT / \\\"experiments\\\" / \\\"rulebook\\\" / \\\"results\\\"))\\n    ap.add_argument(\\\"--minutes\\\", type=float); ap.add_argument(\\\"--jobs\\\", type=int); ap.add_argument(\\\"--level-actions\\\", type=int)\\n    ap.add_argument(\\\"--no-model\\\", action=\\\"store_true\\\", help=\\\"deterministic fallback policy only (smoke test)\\\")\\n    ap.add_argument(\\\"--mode\\\", choices=[\\\"choose\\\", \\\"coder\\\"], help=\\\"choose = model picks a candidate label; coder = model writes python against the library\\\")\\n    ap.add_argument(\\\"--gateway\\\", default=\\\"\\\", help=\\\"competition-mode Arcade against this gateway URL (e.g. http://127.0.0.1:8001/ from `make serve`) instead of offline env files\\\")\\n    a = ap.parse_args(argv)\\n    logging.basicConfig(level=logging.WARNING)\\n    cfg = dict(DEFAULT_CONFIG)\\n    if a.config:\\n        cfg.update(json.loads(Path(a.config).read_text()))\\n    if a.minutes: cfg[\\\"max_minutes\\\"] = a.minutes\\n    if a.jobs: cfg[\\\"jobs\\\"] = a.jobs\\n    if a.level_actions: cfg[\\\"level_actions\\\"] = a.level_actions\\n    if a.mode: cfg[\\\"mode\\\"] = a.mode\\n    cfg[\\\"no_model\\\"] = a.no_model\\n    if a.gateway:\\n        os.environ.setdefault(\\\"ARC_API_KEY\\\", \\\"local-dev\\\")\\n        arc = make_arcade(competition=True, base_url=a.gateway)\\n    else:\\n        arc = make_arcade()\\n    games = sorted(e.game_id.split(\\\"-\\\")[0] for e in arc.get_environments()) if a.games == \\\"all\\\" else [g.strip() for g in a.games.split(\\\",\\\") if g.strip()]\\n    run(games, cfg, out_dir=Path(a.out), tag=a.tag, arc=arc)\\n\\n\\nif __name__ == \\\"__main__\\\":\\n    main()\\n\"}}")
for pkg, files in SOURCES.items():
    os.makedirs(f'{WORK}/{pkg}', exist_ok=True)
    for name, body in files.items():
        open(f'{WORK}/{pkg}/{name}', 'w').write(body)
sys.path.insert(0, WORK)
import arc_agi, rulebook.run, rulebook.agent
print('rulebook agent embedded | arc_agi ok |', f'{time.time()-T0:.0f}s')


In [ ]:
# In-notebook vLLM OpenAI server from the attached wheelhouse (installed into a private target dir so the
# notebook kernel's own packages stay untouched) serving the attached HF snapshot.
import glob, os, shutil, subprocess, sys, time, urllib.request, json
WORK = '/kaggle/working'
def _find(ref):
    owner, slug = ref.split('/', 1)
    for p in (f'/kaggle/input/{slug}', f'/kaggle/input/datasets/{owner}/{slug}'):
        if os.path.exists(p):
            return p
    raise FileNotFoundError(ref)
WHEELHOUSE = _find('driessmit1/arc3-vllm-h100-wheelhouse-v3')
_mp = glob.glob('/kaggle/input/models/**/config.json', recursive=True)
MODEL_PATH = os.path.dirname(_mp[0]) if (False and _mp) else _find('driessmit1/vrfai-qwen3-6-27b-fp8-hf-snapshot')
cfgs = glob.glob(MODEL_PATH + '/**/config.json', recursive=True)
MODEL_PATH = os.path.dirname(cfgs[0]) if cfgs else MODEL_PATH
SITE = '/tmp/vllm-site-packages'   # outside /kaggle/working so the kernel output stays small
print('wheelhouse:', WHEELHOUSE, '| model:', MODEL_PATH)
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip())
if not os.path.exists(SITE + '/vllm'):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--find-links', WHEELHOUSE, '--requirement', WHEELHOUSE + '/requirements.lock',
                    '--target', SITE, '--upgrade', '--ignore-installed', '--only-binary', ':all:', '--no-compile', '--disable-pip-version-check',
                    '--no-warn-conflicts', '-q'], check=True)
# FlashInfer JIT-compiles sm120 kernels and links -lcuda: the driver stub lives in /usr/local/nvidia/lib64 on Kaggle
env = dict(os.environ, PYTHONPATH=SITE, USE_TF='0', TRANSFORMERS_NO_TF='1', TRANSFORMERS_NO_TORCHVISION='1', VLLM_NO_USAGE_STATS='1', **{},
           LIBRARY_PATH='/usr/local/nvidia/lib64:' + os.environ.get('LIBRARY_PATH', ''),
           LD_LIBRARY_PATH='/usr/local/nvidia/lib64:' + os.environ.get('LD_LIBRARY_PATH', ''))
cmd = [sys.executable, '-m', 'vllm.entrypoints.openai.api_server', '--model', MODEL_PATH, '--served-model-name', 'local-model',
       '--host', '127.0.0.1', '--port', '1234', '--max-model-len', '65536', '--gpu-memory-utilization', '0.92',
       '--enable-auto-tool-choice', '--enable-prefix-caching', '--generation-config', 'vllm'] + ['--tool-call-parser', 'qwen3_coder', '--reasoning-parser', 'qwen3', '--default-chat-template-kwargs', '{"preserve_thinking": true}']
if env.get('TIKTOKEN_RS_CACHE_DIR') and not os.path.isdir(env['TIKTOKEN_RS_CACHE_DIR']):
    _alt = '/kaggle/input/' + env['TIKTOKEN_RS_CACHE_DIR'].rstrip('/').split('/')[-1]
    env['TIKTOKEN_RS_CACHE_DIR'] = _alt if os.path.isdir(_alt) else env['TIKTOKEN_RS_CACHE_DIR']
    print('tiktoken cache dir:', env['TIKTOKEN_RS_CACHE_DIR'], os.listdir(env['TIKTOKEN_RS_CACHE_DIR']) if os.path.isdir(env['TIKTOKEN_RS_CACHE_DIR']) else 'MISSING')
log = open(f'{WORK}/vllm-server.log', 'w')
VLLM = subprocess.Popen(cmd, env=env, stdout=log, stderr=subprocess.STDOUT)
t0 = time.time()
while True:
    if VLLM.poll() is not None:
        _log = open(f'{WORK}/vllm-server.log').read()
        _err = [l for l in _log.splitlines() if any(k in l for k in ('FAILED', 'error:', 'Error', 'cannot find', 'RuntimeError', 'assert'))]
        raise RuntimeError('vLLM exited. Error lines:\n' + '\n'.join(_err[:40]) + '\n--- tail ---\n' + _log[-3000:])
    try:
        urllib.request.urlopen('http://127.0.0.1:1234/v1/models', timeout=5).read(); break
    except Exception:
        if time.time() - t0 > 1500:
            raise TimeoutError(open(f'{WORK}/vllm-server.log').read()[-4000:])
        time.sleep(5)
print(f'vLLM ready after {time.time()-t0:.0f}s')
req = urllib.request.Request('http://127.0.0.1:1234/v1/chat/completions', data=json.dumps({'model': 'local-model', 'max_tokens': 400,
      'messages': [{'role': 'user', 'content': 'Say hello in five words.'}], **{'chat_template_kwargs': {'enable_thinking': False}}}).encode(),
      headers={'Content-Type': 'application/json'})
try:
    _m = json.loads(urllib.request.urlopen(req, timeout=300).read())['choices'][0]['message']
    print('smoke:', repr((_m.get('content') or '')[:200]), '| reasoning:', repr((_m.get('reasoning_content') or _m.get('reasoning') or '')[:120]))
except Exception as _e:   # the smoke chat is informational; the game run below is the real test
    print('smoke chat failed:', repr(_e)[:300])


In [ ]:
import math, os, sys, time, urllib.request, logging
from pathlib import Path
sys.path.insert(0, '/kaggle/working')
logging.basicConfig(level=logging.WARNING)
from arcnav.runner import make_arcade
from rulebook.run import run, DEFAULT_CONFIG
# no chain-of-thought on Kaggle: the FP8 27B ran past the 12k-token budget on every thinking call in kernel v1 (INIT 362 s, review 181 s, no JSON)
cfg = dict(DEFAULT_CONFIG, model='local-model', base_url='http://127.0.0.1:1234/v1', level_actions=200, max_actions=4000, max_levels=20,
           init_think=False, review_think='never', think_tokens=6000, mode='choose', compact=True, decide_tokens=500, reviews_per_level=3)
out_dir = Path('/kaggle/working/rulebook-results')
if RERUN:
    os.environ['ARC_API_KEY'] = 'test-key-123'
    deadline = time.time() + 60
    while True:  # wait for the gateway sidecar
        try:
            urllib.request.urlopen('http://gateway:8001/api/games', timeout=10).read(); break
        except Exception as e:
            if time.time() > deadline:
                raise RuntimeError(f'gateway not ready: {e!r}')
            time.sleep(5)
    arc = make_arcade(competition=True, base_url='http://gateway:8001/')
    games = [e.game_id for e in arc.get_environments()]
    waves = max(1, math.ceil(len(games) / 12))
    total_left = 470 - (time.time() - T0) / 60
    cfg.update(jobs=12, max_minutes=max(20, min(120, total_left / waves)))
    print(f'rerun: {len(games)} games, {cfg["jobs"]} concurrent, {cfg["max_minutes"]:.0f} min/game, waves={waves}')
    run(games, cfg, out_dir=out_dir, tag='kaggle-rerun', arc=arc, deadline=T0 + 470 * 60)   # the gateway records actions and emits submission.parquet
elif False:
    # rerun replica: the same competition gateway server (arc_agi.server) as a thread, competition mode, full ids, 12 games at once
    import threading, arc_agi
    from arc_agi import OperationMode
    from arc_agi.server import create_app
    _arcade = arc_agi.Arcade(operation_mode=OperationMode.OFFLINE, environments_dir='/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files')
    _app, _api = create_app(_arcade, competition_mode=True)
    threading.Thread(target=lambda: _app.run(host='127.0.0.1', port=8001, threaded=True), daemon=True).start()
    for _ in range(60):
        try:
            urllib.request.urlopen('http://127.0.0.1:8001/api/games', timeout=5).read(); break
        except Exception:
            time.sleep(2)
    os.environ['ARC_API_KEY'] = 'test-key-123'
    arc = make_arcade(competition=True, base_url='http://127.0.0.1:8001')
    games = sorted(e.game_id for e in arc.get_environments())[:12]
    cfg.update(jobs=12, max_minutes=12)
    print(f'replica: {len(games)} games at once, {cfg["max_minutes"]} min/game, ids {games[:3]} ...')
    res = run(games, cfg, out_dir=out_dir, tag='kaggle-replica', arc=arc, deadline=time.time() + (12 + 6) * 60)
    print('replica errors:', [(g['game_id'], g.get('error')) for g in res['games'] if g.get('error')])
    print('replica levels:', [(g['game_id'][:4], g.get('levels_completed'), g.get('actions'), g.get('model_calls', {}).get('decide')) for g in res['games']])
    import pandas as pd
    pd.DataFrame(data=[['1_0', '1', True, 1]], columns=['row_id', 'game_id', 'end_of_game', 'score']).to_parquet('/kaggle/working/submission.parquet', index=False)
else:
    arc = make_arcade(environments_dir='/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files')
    cfg.update(jobs=2, max_minutes=12)
    run(['tn36', 'lp85'], cfg, out_dir=out_dir, tag='kaggle-smoke', arc=arc)
    import pandas as pd   # commit mode: dummy submission so the commit succeeds
    pd.DataFrame(data=[['1_0', '1', True, 1]], columns=['row_id', 'game_id', 'end_of_game', 'score']).to_parquet('/kaggle/working/submission.parquet', index=False)
try:
    VLLM.terminate()
except Exception:
    pass
print(f'done in {(time.time()-T0)/60:.1f} min')
